# 🔬 Continum PersistIQ
### End-to-End Experimentation Intelligence · From First Connection to Proven ROI

---

## What is Continum PersistIQ?

**Continum PersistIQ** is a unified platform that takes a business through the full lifecycle of data-driven decisions — from connecting a fresh data warehouse on day one, through running rigorous experiments, to proving the lift held up months after shipping.

---

## The plug-and-play bootstrap flow

The fastest path from zero to production:

```
1.  Set USE_SYNTHETIC_DATA = False in Cell 3
    Point the snowflake config at your warehouse.

2.  Run Cell 11 (the dispatcher).
    The bootstrap gate fires automatically:
    ┌──────────────────────────────────────────────────────────┐
    │ 🔌 PRODUCTION MODE DETECTED: BOOTSTRAP REQUIRED         |
    │ Run bootstrap now? [Y/n]:                                │
    └──────────────────────────────────────────────────────────┘

3.  Press Enter. bootstrap_from_connection() runs four steps:

    Step 1 — Client name (one question)

    Step 2 — Auto schema discovery
             Profiles every table in the connected warehouse.
             LLM maps catalog → canonical schema.
             Deterministic sanity checks run automatically.

    Step 3 — Human review & approval
             ┌──────────────────────────┬────────────────────┬────────────┐
             │ Canonical table          │ Client table       │ Status     │
             ├──────────────────────────┼────────────────────┼────────────┤
             │ quotes                   │ FACT_QUOTES        │ ✅ OK     │
             │ orders                   │ FACT_ORDERS        │ ✅ OK     │
             │ experiments              │ STATSIG_EXPS       │ ✅ OK     │
             │ accounts                 │ (no match)         │ ⚠️ CHECK  │
             └──────────────────────────┴────────────────────┴────────────┘
             Low-confidence rows are flagged. Mapping errors block commit.
             Accept mapping? [y/N]:

    Step 4 — State committed
             mode → production_ready
             continum_state.json written (survives kernel restart)
             Module [2] Pipeline Health runs to establish a baseline.

4.  Re-run cells 3-Auto, 5-Auto, 6-Auto (takes ~3 seconds).
    CLIENT_SCHEMA is rebuilt from state. Canonical DuckDB views are created.

5.  Run Cell 11 again — all 16 modules now work against your real data.
    No code changes. No manual schema editing.
```

> **POC mode:** leave `USE_SYNTHETIC_DATA = True` and run Cell 11 directly.
> Synthetic data (25k buyers, 180k inquiries, 10 experiments) is ready immediately.

---

## Architecture — 6 layers

```
╔══════════════════════════════════════════════════════════════════════════════╗
║  UI LAYER  (future / optional)                                               ║
║  Streamlit · FastAPI · Custom frontend — wraps the Python API below.         ║
║  See Cell 39 for a working skeleton. The engine has no UI dependency.        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  STATE LAYER — Cell 2b                                                       ║
║  CONTINUM_STATE dict  +  continum_state.json  (persists across restarts)     ║
║  API: get_continum_state() · is_production_ready() · is_synthetic()          ║
║       bootstrap_from_connection(llm)  ·  reset_continum_state()              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  MODULE LAYER — Cells 8–11                                                   ║
║  14 interactive modules. Dispatcher (Cell 11) routes to all of them.         ║
║  Phase 0 (Foundation) · Phase 1 (Planning) · Phase 2 (Live) · Phase 3 (Post) ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  DOCUMENT LAYER — Cell 7b                                                    ║
║  Template-aware PDF generator. Accepts .txt / .md / .pdf / .docx templates.  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  LLM LAYER — Cell 4                                                          ║
║  Local HuggingFace model (Phi-3-mini, ~2.4 GB, offline after first pull).    ║
║  Used for: schema mapping · feature classification · PRD drafting ·          ║
║  causal narrative · pipeline anomaly narration · semantic search.            ║
║  NEVER touches statistics — all numbers come from the engine below.          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  STATISTICAL ENGINE — Cell 7                                                 ║
║  scipy/numpy only. z-test · t-test · power · DiD · ITS · PSM · RDD ·         ║
║  Mediation · mSPRT · χ² · OLS counterfactual · Bonferroni · SLSQP.           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  DATA LAYER — Cells 3/3-Auto, 5/5-Auto, 6/6-Auto                             ║
║  DuckDB in-memory warehouse. Two paths:                                      ║
║  DATA_MODE='synthetic' → auto-generate data (default, zero config).          ║
║  Production:         bootstrap_from_connection() → Cells 3-Auto, 5-Auto,     ║
║                      6-Auto rebuild from CONTINUM_STATE.                     ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

---

## How to run (3 paths)

**Path A — POC / demo (no warehouse needed):**
```
1. Run cells 1 → 11 in order
2. Run Cell 11 → pick any module
   Synthetic data (25k buyers, 10 experiments) is ready immediately.
```

**Path B — Production, first-time setup:**
```
1. Set DATA_MODE = 'production' (or 'csv') in Cell 3
2. Set SNOWFLAKE_ACCOUNT / SNOWFLAKE_USER / SNOWFLAKE_PASSWORD as env vars
3. Run cells 1 → 11
4. Cell 11 fires the bootstrap gate → follow the 4-step flow
5. Re-run cells 3-Auto, 5-Auto, 6-Auto
6. Run Cell 11 → all 16 modules work on real data
```

**Path C — Return session (bootstrap already done):**
```
1. Run cells 1 → 11 in order
   continum_state.json is loaded; mode=production_ready is restored.
2. Run cells 3-Auto, 5-Auto, 6-Auto to activate the production config.
3. Run Cell 11 → pick any module.
```

## 1 · Install Dependencies

In [3]:
#%pip install transformers torch accelerate pandas numpy scipy matplotlib tabulate duckdb sentencepiece reportlab pypdf python-docx statsmodels
# 
#pip install packaging==20.0
#

## 2 · Imports & Device Detection

All Python imports for the session. Auto-detects GPU / MPS / CPU.

In [4]:
import os

os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')
os.environ.setdefault('HF_HUB_DISABLE_IMPLICIT_TOKEN', '1')
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import re, json, time, textwrap, warnings
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Any
from enum import Enum
import logging

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import transformers
try:
    transformers.logging.set_verbosity_error()
except Exception:
    pass

# import snowflake.connector
# import sqlglot
import sqlglot.expressions as exp
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm, t as t_dist
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import duckdb
from tabulate import tabulate

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=UserWarning, module='transformers')
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Silence chatty loggers from HuggingFace hub
for _log_name in ('huggingface_hub', 'huggingface_hub.utils._http',
                  'transformers', 'transformers.modeling_utils',
                  'transformers.configuration_utils',
                  'transformers.tokenization_utils_base'):
    logging.getLogger(_log_name).setLevel(logging.ERROR)

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger('continum_agent')

# ── Detect best available device ─────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():   # Apple Silicon
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}')

# ── Matplotlib dark theme ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f', 'axes.facecolor':  '#1a1a2e',
    'axes.edgecolor':   '#444',    'axes.labelcolor': '#e0e0e0',
    'xtick.color':      '#aaa',    'ytick.color':     '#aaa',
    'text.color':       '#e0e0e0', 'grid.color':      '#333',
    'grid.linestyle':   '--',      'grid.alpha':       0.4,
    'font.family':      'monospace',
    'axes.titlesize':   13,        'axes.labelsize':  11,
})
COLORS = {
    'control':   '#4e9af1', 'treatment': '#f97316',
    'positive':  '#22c55e', 'negative':  '#ef4444',
    'neutral':   '#a1a1aa', 'highlight': '#facc15',
    'bg':        '#1a1a2e', 'accent':    '#7c3aed',
}
# ── Domain constants — configurable per client ───────────────────────────────
SEGMENTS  = ['Core', 'Growth', 'Enterprise', 'Individuals']  # default buyer segments
PLATFORMS = ['web', 'mobile']                                  # default platforms
CATEGORIES = ['Category A', 'Category B', 'Category C', 'Category D', 'Category E']
COUNTRIES  = ['US', 'UK', 'CA', 'AU', 'IN']

# ── statsmodels (optional — used by ARIMA/SARIMA/BSTS/Causal Impact) ────────
try:
    import statsmodels.api as sm                                            # noqa: F401
    from statsmodels.tsa.arima.model import ARIMA as SM_ARIMA              # noqa: F401
    from statsmodels.tsa.statespace.sarimax import SARIMAX as SM_SARIMAX  # noqa: F401
    from statsmodels.tsa.statespace.structural import UnobservedComponents as SM_UC  # noqa: F401
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False
    print('  ℹ️  statsmodels not installed — run: pip install statsmodels')

print('Imports ready')

Device: cpu
  ℹ️  statsmodels not installed — run: pip install statsmodels
Imports ready


## 2b · Bootstrap Orchestrator

**Plug-and-play integration layer.** Manages the transition from synthetic POC mode to production-ready mode with minimal human input.

The orchestrator holds a single state dictionary, `CONTINUM_STATE`, that tracks whether the platform is running on synthetic data or is bootstrapped against a real client warehouse. State persists across kernel restarts via `continum_state.json`.

**Current behavior:** When `USE_SYNTHETIC_DATA = True` (default), this cell is a no-op — all existing cells work unchanged. In subsequent turns, this cell will become the entry point that auto-runs Module [1] Schema Discovery and Module [2] Pipeline Health when connected to a real warehouse.

```
Current turn (Turn 1):  scaffolding only; no behavior change
Turn 2:                 parallel auto-configured cells for 3, 5, 6
Turn 3:                 auto-trigger on production connection + approval gate
Turn 4:                 documentation + UI-readiness proof
```

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# BOOTSTRAP ORCHESTRATOR — State layer for plug-and-play integration
# Modes:
#   'synthetic'                — use built-in synthetic data
#   'production_bootstrapping' — connected to a real warehouse, not yet mapped
#   'production_ready'         — mapping approved, platform ready to run modules
# ─────────────────────────────────────────────────────────────────────────────

import json as _json
import os as _os
from datetime import datetime as _dt

STATE_FILE = 'continum_state.json'

def _default_state():
    return {
        'mode':                 'synthetic',
        'client_name':          None,
        'connection_fingerprint': None,
        'client_schema':        None,
        'bootstrap_timestamp':  None,
        'approved_by':          None,
        'pipeline_baseline':    None,
        'last_health_check':    None,
        'schema_version':       1,
    }


def _load_state():
    """Read state from disk if it exists; otherwise return defaults."""
    if _os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE) as f:
                state = _json.load(f)
                # Fill in any missing keys with defaults (forward compat)
                for k, v in _default_state().items():
                    state.setdefault(k, v)
                return state
        except Exception as e:
            print(f'  ⚠️  Could not load {STATE_FILE}: {e}. Using defaults.')
    return _default_state()


def _save_state(state):
    """Persist state to disk."""
    try:
        with open(STATE_FILE, 'w', encoding='utf-8') as f:
            _json.dump(state, f, indent=2, default=str)
    except Exception as e:
        print(f'  ⚠️  Could not save {STATE_FILE}: {e}')


# Load on cell execution
CONTINUM_STATE = _load_state()


# ─────────────────────────────────────────────────────────────────────────────
# API surface — functions a UI wrapper would call
# ─────────────────────────────────────────────────────────────────────────────

def get_continum_state():
    """Return a copy of current state (read-only from caller's perspective)."""
    return dict(CONTINUM_STATE)


def reset_continum_state():
    """Reset to synthetic mode. Removes the persistence file. For testing / starting over."""
    global CONTINUM_STATE
    CONTINUM_STATE = _default_state()
    if _os.path.exists(STATE_FILE):
        _os.remove(STATE_FILE)
    print('✅ CONTINUM_STATE reset to synthetic mode.')


def is_production_ready():
    """True when the platform is bootstrapped against a real client warehouse."""
    return CONTINUM_STATE.get('mode') == 'production_ready'


def is_synthetic():
    """True when running against built-in synthetic data."""
    return CONTINUM_STATE.get('mode') == 'synthetic'



def bootstrap_from_connection(llm, connection_config=None,
                              skip_confirmation=False,
                              run_health_baseline=True):
    """
    Main entry point for plug-and-play integration.

    Steps
    -----
    1. Print a status banner.
    2. If already production_ready, ask whether to re-run.
    3. Run Module [1] Schema Discovery (programmatic mode — no interactive prompts).
    4. Show a mapping-diff table with confidence scores.
    5. Highlight low-confidence rows; block on human approval unless
       skip_confirmation=True.
    6. On approval: commit to CONTINUM_STATE, persist JSON, set mode=production_ready.
    7. Optionally run Module [2] Pipeline Health to establish a baseline.
    8. Return final state.

    Parameters
    ----------
    llm                 : LLM client (same object passed to all modules)
    connection_config   : dict with 'type' and connection params, e.g.
                          {'type': 'snowflake', 'account': ..., 'user': ...,
                           'password': ..., 'warehouse': ..., 'role': ...}
                          Leave None to use env vars / current DuckDB session.
    skip_confirmation   : bool — if True, commits the mapping without prompting.
                          Use only in automated pipelines where the mapping has
                          already been validated.
    run_health_baseline : bool — run Module [2] after approval to establish
                          a freshness / volume baseline. Default True.
    """
    global CONTINUM_STATE

    print()
    print('╔' + '═'*70 + '╗')
    print('║' + '  🔌  BOOTSTRAP FROM CONNECTION'.ljust(70) + '║')
    print('║' + '  Auto-discover schema → human review → production-ready'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')
    print()

    # ── Already bootstrapped? ────────────────────────────────────────────────
    if CONTINUM_STATE['mode'] == 'production_ready':
        print(f'  ℹ️  Platform already bootstrapped.')
        print(f'     Client    : {CONTINUM_STATE["client_name"]}')
        print(f'     Timestamp : {CONTINUM_STATE["bootstrap_timestamp"]}')
        print()
        raw = input('  Re-run bootstrap? This will replace the existing mapping. [y/N]: ').strip().lower()
        if raw != 'y':
            print('  Keeping existing configuration.')
            return get_continum_state()
        # Reset before re-running
        CONTINUM_STATE['mode'] = 'production_bootstrapping'

    CONTINUM_STATE['mode'] = 'production_bootstrapping'
    _save_state(CONTINUM_STATE)

    # ── Step 1: Get client name ──────────────────────────────────────────────
    print('  Step 1 of 4 — Connection details')
    print('  ─' * 36)
    client_name = input('  Client / project name: ').strip() or 'Client'

    # ── Step 2: Run Schema Discovery (programmatic, non-interactive) ─────────
    print()
    print('  Step 2 of 4 — Schema Discovery')
    print('  ─' * 36)
    print('  Profiling warehouse tables and mapping to canonical schema...')
    print()

    discovery_result = None
    try:
        discovery_result = run_schema_discovery(
            llm,
            _bootstrap_mode=True,
            _client_name=client_name,
        )
    except Exception as e:
        print(f'  ❌ Schema discovery failed: {e}')
        print('     Run Module [1] manually to diagnose the issue.')
        CONTINUM_STATE['mode'] = 'synthetic'
        _save_state(CONTINUM_STATE)
        return get_continum_state()

    if discovery_result is None:
        print('  ❌ Schema discovery returned no result. Aborting bootstrap.')
        CONTINUM_STATE['mode'] = 'synthetic'
        _save_state(CONTINUM_STATE)
        return get_continum_state()

    mapping    = discovery_result['mapping']
    confidence = mapping.get('confidence', 0.0)
    issues     = discovery_result['issues']
    table_map  = mapping.get('table_mapping', {})
    column_map = mapping.get('column_mapping', {})

    # ── Step 3: Human approval gate ──────────────────────────────────────────
    print()
    print('  Step 3 of 4 — Review proposed mapping')
    print('  ─' * 36)
    print()

    LOW_CONFIDENCE = 0.85    # flag mappings below this threshold
    n_low = 0

    # Table mapping table
    print('  ┌────────────────────────┬──────────────────────────────────┬──────────┐')
    print('  │  Canonical table       │  Client table                    │  Status  │')
    print('  ├────────────────────────┼──────────────────────────────────┼──────────┤')
    for canonical, mapped in table_map.items():
        row_conf = confidence if mapped else 0.0
        if not mapped or row_conf < LOW_CONFIDENCE:
            status = '⚠️  REVIEW'
            n_low += 1
        else:
            status = '✅ OK    '
        mapped_str = (mapped or '(no match)')[:32]
        print(f'  │  {canonical:<22}  │  {mapped_str:<32}  │  {status}  │')
    print('  └────────────────────────┴──────────────────────────────────┴──────────┘')

    print()
    print(f'  Overall confidence   : {confidence:.0%}')
    print(f'  Verification issues  : '
          f'{sum(1 for s,_ in issues if s=="error")} errors, '
          f'{sum(1 for s,_ in issues if s=="warn")} warnings')

    # Column mapping (abbreviated — show only the 8 most important)
    PRIORITY_COLS = ['inquiry_id','buyer_id','account_segment','platform',
                     'created_at','converted_to_order','order_value','variant',
                     'category','country','channel','product_id','device_type']
    print()
    print('  Key column mapping:')
    for c in PRIORITY_COLS:
        mapped_c = column_map.get(c)
        ok = '✅' if mapped_c else '⚠️ '
        print(f'    {ok}  {c:<24} → {mapped_c or "(no match)"}')

    if mapping.get('warnings'):
        print()
        print('  LLM warnings:')
        for w in mapping['warnings']:
            print(f'    ⚠️  {w}')

    # Hard block if there are mapping errors (not warnings)
    n_errors = sum(1 for s,_ in issues if s == 'error')
    if n_errors > 0:
        print()
        print(f'  ❌ {n_errors} mapping error(s) must be resolved before proceeding.')
        print('     Run Module [1] manually, review the PDF, correct the mapping,')
        print('     then call bootstrap_from_connection() again.')
        CONTINUM_STATE['mode'] = 'synthetic'
        _save_state(CONTINUM_STATE)
        return get_continum_state()

    # Soft block for low confidence
    if n_low > 0 and not skip_confirmation:
        print()
        print(f'  ⚠️  {n_low} mapping(s) need review (low confidence or no match).')
        print('     Review them above and edit continum_state.json if needed.')

    # Approval prompt
    if not skip_confirmation:
        print()
        raw = input('  Accept mapping and proceed to production mode? [y/N]: ').strip().lower()
        if raw != 'y':
            print()
            print('  Bootstrap cancelled. Edit the mapping and try again.')
            print('  Tip: run Module [1] manually to get the full PDF report.')
            CONTINUM_STATE['mode'] = 'synthetic'
            _save_state(CONTINUM_STATE)
            return get_continum_state()
    else:
        print()
        print('  skip_confirmation=True — proceeding without manual review.')

    # ── Step 4: Commit state ─────────────────────────────────────────────────
    print()
    print('  Step 4 of 4 — Committing configuration')
    print('  ─' * 36)

    from datetime import datetime as _dt
    CONTINUM_STATE.update({
        'mode':                  'production_ready',
        'client_name':           client_name,
        'connection_fingerprint': str(connection_config)[:200] if connection_config else 'duckdb_local',
        'client_schema': {
            'client_name':              client_name,
            'mapping':                  mapping,
            'segment_map':              {},
            'platform_map':             {},
            'cancelled_order_statuses': [],
            'internal_domains':         [],
            'dedup_key':                column_map.get('inquiry_id', 'inquiry_id'),
            'winsorise_pct':            99,
            'min_segment_size':         30,
            'null_ior_default':         0.18,
            'null_aov_default':         5000,
            'null_daily_traffic':       300,
            # Domain constants (empty = use Cell 2 defaults)
            'segments':   [],
            'platforms':  [],
            'categories': [],
            'countries':  [],
        },
        'bootstrap_timestamp': _dt.now().isoformat(timespec='seconds'),
        'approved_by':         'human' if not skip_confirmation else 'automated',
    })
    _save_state(CONTINUM_STATE)

    print(f'  ✅ State committed: mode=production_ready')
    print(f'     Client          : {client_name}')
    print(f'     Timestamp       : {CONTINUM_STATE["bootstrap_timestamp"]}')
    print(f'     Persisted to    : {STATE_FILE}')
    print()
    print('  ── Next steps ──────────────────────────────────────────────────')
    print('  1. Re-run Cell 3-Auto to rebuild CLIENT_SCHEMA from state.')
    print('  2. Re-run Cell 5-Auto to create canonical DuckDB views.')
    print('  3. Re-run Cell 6-Auto to load the production experiment registry.')
    print('  4. All 13 modules now work against your production data.')
    print()

    # ── Optional: pipeline health baseline ──────────────────────────────────
    if run_health_baseline:
        print('  Running Module [2] Pipeline Health to establish baseline...')
        try:
            health = run_pipeline_health(llm)
            CONTINUM_STATE['pipeline_baseline'] = {
                'timestamp':      _dt.now().isoformat(timespec='seconds'),
                'overall':        health.get('overall', 'unknown'),
                'volume_mean':    health.get('findings', {}).get('volume', {}).get('baseline_mean'),
                'null_rates':     {f['column']: f['baseline_pct']
                                   for f in health.get('findings', {}).get('null_spikes', [])},
            }
            CONTINUM_STATE['last_health_check'] = _dt.now().isoformat(timespec='seconds')
            _save_state(CONTINUM_STATE)
            print(f'  ✅ Health baseline saved (overall: {health.get("overall","?")})')
        except Exception as e:
            print(f'  ⚠️  Pipeline health baseline failed: {e}')
            print('     Run Module [2] manually to establish the baseline.')

    print()
    print('  🟢 Bootstrap complete. Platform is production_ready.')
    print('═'*72)
    return get_continum_state()


if CONTINUM_STATE['mode'] == 'synthetic':
    print('   Running against built-in synthetic data (Cells 5 + 6).')
    print('   To connect to a real warehouse: call bootstrap_from_connection(narrative_llm).')
elif CONTINUM_STATE['mode'] == 'production_ready':
    print(f'   Bootstrapped for client: {CONTINUM_STATE["client_name"]}')
    print(f'   Last bootstrap: {CONTINUM_STATE["bootstrap_timestamp"]}')


   Running against built-in synthetic data (Cells 5 + 6).
   To connect to a real warehouse: call bootstrap_from_connection(narrative_llm).


## 3 · Bronze Layer — Source Config & Schema

The entry point of the **medallion architecture**. Defines where raw data comes from and how bronze column names map to canonical silver names.

| Layer | Cell | Responsibility |
|---|---|---|
| 🥉 Bronze | 3 + 5 | Source config, raw ingestion — no transformation |
| 🥈 Silver | 5-Silver | Clean, standardise, unify across sources with DuckDB SQL |
| 🥇 Gold | 6 | Pre-compute analytics tables: pre/post cohorts, dim breakdowns, time-series |

Edit **only `SOURCE_CONFIG`** to onboard a new client — table locations (`bronze_tables`) and the column map (`column_map`) are the only things that differ between clients.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA_MODE — the only variable you need to change between the three run modes
#
#   'synthetic'   POC mode. Auto-generates realistic fake data. Zero config.
#
#   'csv'         Local file mode. Supply CSV exports from your
#                 warehouse. No live DB connection required.
#                 → call register_csv_source() below for each table before
#                   running Cell 16.
#
#   'production'  Live warehouse mode. Connects to Snowflake (or other DB)
#                 and pulls tables through the BronzeIngestionGateway.
# ─────────────────────────────────────────────────────────────────────────────
DATA_MODE = 'synthetic'   # CHANGE THIS: 'synthetic' | 'csv' | 'production'

USE_SYNTHETIC_DATA = (DATA_MODE != 'production')

# ─────────────────────────────────────────────────────────────────────────────
# SOURCE CONFIG — single source of truth for the Bronze layer.
#
# Medallion roles within this dict:
#   bronze_tables  →  raw source locations (ingested as-is, no column rename)
#   column_map     →  bronze col name → canonical name (consumed by Silver SQL)
#   segment_map    →  raw segment values → canonical labels   (applied in Silver)
#   platform_map   →  raw platform values → canonical labels  (applied in Silver)
#   data quality   →  thresholds for Silver cleaning + module pre-checks
#
# ─────────────────────────────────────────────────────────────────────────────
SOURCE_CONFIG = {
    'client_name': 'ABC',

    # ── Snowflake connection (ignored when USE_SYNTHETIC_DATA = True) ─────────
    'snowflake': {
        'account':   os.environ.get('SNOWFLAKE_ACCOUNT',   'your_account'),
        'user':      os.environ.get('SNOWFLAKE_USER',      'your_user'),
        'password':  os.environ.get('SNOWFLAKE_PASSWORD',  'your_password'),
        'warehouse': os.environ.get('SNOWFLAKE_WAREHOUSE', 'COMPUTE_WH'),
        'role':      os.environ.get('SNOWFLAKE_ROLE',      'ANALYST'),
    },

    # ── Bronze table locations ────────────────────────────────────────────────
    'bronze_tables': {
        'quotes':      'QUOTES_SCHEMA.QUOTES',
        'orders':      'ORDERS_SCHEMA.ORDERS',
        'users':       'USERS_SCHEMA.USERS',
        'accounts':    'ACCOUNTS_SCHEMA.ACCOUNT_STATUS',
        'experiments': 'STATSIG_SCHEMA.STATSIG_EXPERIMENTS',
        # Resolved at runtime after Silver materialises
        'inquiries':   'silver_inquiries',
        'all_exp':     'gold_experiment_analysis',
        'traffic':     'silver_traffic',
    },

    # ── Bronze → Silver column map ────────────────────────────────────────────
    'column_map': {
        # Quotes table
        'quote_id':            '_ID',
        'quote_user_id':       'USER',
        'quote_account_id':    'BILLING_ACCOUNT',
        'quote_created_at':    '_CONSTRUCTED',
        'quote_source':        'QUOTE_SOURCE',
        'quote_processes':     'PROCESSES',
        'quote_price':         'TOTAL',
        'quote_status':        'LAST_HISTORY_STATUS',
        # Orders table
        'order_id':            '_ID',
        'order_quote_id':      'QUOTE_ORDER_ID',
        'order_total':         'TOTAL',
        'order_bookings':      'BOOKINGS',
        'order_status':        'LAST_HISTORY_STATUS',
        'order_time':          'ORDER_TIME',
        'order_ship_date':     'SHIP_DATE',
        'order_payment_type':  'PAYMENT_TYPE',
        'order_country':       'SHIPPING_ADDRESS_COUNTRY',
        # Users table
        'user_id':             '_ID',
        'user_account_id':     'ACCOUNT',
        'user_email_flag':     'EMAIL_FLAG',
        'user_customer_flag':  'CURRENT_CUSTOMER_FLAG',
        # Account Status table
        'account_id':          'ACCOUNT_ID',
        'account_segment':     'CONSOLIDATED_BUSINESS_SEGMENT',
        'account_vertical':    'ACCOUNT_VERTICAL_MARKET',
        'account_country':     'BILLING_COUNTRY',
        'account_employees':   'NUMBER_OF_EMPLOYEES',
        # Statsig experiments table
        'exp_user_id':         'USER_ID',
        'exp_group_name':      'GROUP_NAME',
        'exp_experiment_id':   'EXPERIMENT_ID',
        'exp_experiment_name':   'EXPERIMENT_NAME',
        'exp_timestamp':       'TIMESTAMP',
        'exp_account_domain':  'ACCOUNT_DOMAIN',
        # Traffic table
        'traffic_date':        'DATE',
        'total_sessions':      'TOTAL_SESSIONS',
        'new_signups':         'NEW_SIGNUPS',
        'signed_in':           'SIGNED_IN',
        'inquiries_col':       'INQUIRIES',
        # Derived / canonical (what modules use internally)
        'inquiry_id':          'quote_id',
        'buyer_id':            'user_id',
        'converted':           'converted_to_order',
        'order_value':         'order_value',
        'platform':            'quote_source',
        'category':            'quote_processes',
        'variant':             'variant',
        'experiment_name':     'experiment_name',
    },

    # ── Segment value normalisation (applied in Silver) ───────────────────────
    'segment_map': {
        'Individuals':    'Individuals',
        'Small Business': 'SMB',
        'Medium Business':'Growth',
        'Large Business': 'Core',
        'Enterprise':     'Enterprise',
    },

    # ── Platform value normalisation (applied in Silver) ─────────────────────
    'platform_map': {
        'WEBAPP':     'web',
        'FUSION':     'desktop',
        'SOLIDWORKS': 'desktop',
    },

    # ── Silver exclusion filters ──────────────────────────────────────────────
    'cancelled_order_statuses': ['Order Cancelled'],
    'internal_domains':         ['xometry.com', 'staff.xometry.com', 'xometry.eu'],

    # ── Data quality thresholds (Silver cleaning + module pre-checks) ─────────
    'dedup_key':           'quote_id',
    'winsorise_pct':        99,
    'min_segment_size':     30,
    'null_ior_default':     0.18,
    'null_aov_default':     5000,
    'null_daily_traffic':   300,
}

# ── Backward-compat alias ─────────────────────────────────────────────────────
CLIENT_SCHEMA = SOURCE_CONFIG
CLIENT_SCHEMA['tables']  = SOURCE_CONFIG['bronze_tables']
CLIENT_SCHEMA['columns'] = SOURCE_CONFIG['column_map']

# ── CSV / Parquet source registry ─────────────────────────────────────────────
BRONZE_CSV_SOURCES: dict = {}
SCHEMA_REGISTRY:    dict = {}

def register_csv_source(alias: str, path: str, bronze_name: str = None) -> None:
    """Register a CSV or Parquet file as an additional Bronze source table.

    The Silver layer will pick it up automatically if an entry exists in
    SOURCE_CONFIG['column_map'] for the relevant columns.

    Example
    -------
    register_csv_source('events', '/data/user_events_export.csv')
    """
    bname = bronze_name or f'bronze_{alias}'
    BRONZE_CSV_SOURCES[alias] = {'path': path, 'bronze_name': bname}
    print(f'  ✅ CSV source registered: {alias} → {path}  (bronze table: {bname})')

def register_schema(alias: str, schema_path: str, table_name: str = '') -> None:
    """Register an additional DB table as a Bronze source at runtime."""
    full = f'{schema_path}.{table_name}' if table_name and '.' not in schema_path else schema_path
    SCHEMA_REGISTRY[alias] = {'schema': schema_path, 'table': table_name, 'full': full}
    SOURCE_CONFIG['bronze_tables'][alias] = full
    CLIENT_SCHEMA['tables'][alias] = full
    print(f'  ✅ Schema registered: {alias} → {full}')

# ─────────────────────────────────────────────────────────────────────────────
# SILVER SQL BUILDER UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def col(canonical: str, table_hint: str = None) -> str:
    """Resolve canonical name → raw bronze column name for Silver SQL generation.

    Parameters
    ----------
    canonical   : The canonical column name used throughout Silver / Gold / modules
    table_hint  : Optional table alias to check table-specific overrides first
                  e.g. col('created_at', 'orders') checks 'orders.created_at' key
    """
    cols = SOURCE_CONFIG['column_map']
    if table_hint:
        tbl_key = f'{table_hint}.{canonical}'
        if tbl_key in cols:
            return cols[tbl_key]
    return cols.get(canonical, canonical)

def tbl(canonical: str) -> str:
    """Resolve canonical table name → bronze warehouse path or DuckDB table name."""
    tbls = SOURCE_CONFIG['bronze_tables']
    resolved = tbls.get(canonical, canonical)
    return resolved.strip('"').strip("'") if resolved else canonical

def tbl_quoted(canonical: str) -> str:
    return f'"{tbl(canonical)}"'

def resolve_columns(table_canonical: str, *canonical_cols) -> dict:
    """Resolve multiple columns for a table at once.

    Returns a dict mapping canonical_name → raw_column_name.
    Useful for building SELECT lists without repeated col() calls.

    Example
    -------
    c = resolve_columns('orders', 'order_id', 'order_total', 'order_status')
    sql = f"SELECT {c['order_id']}, {c['order_total']} FROM {tbl('orders')}"
    """
    return {c: col(c, table_canonical) for c in canonical_cols}

# ─────────────────────────────────────────────────────────────────────────────
# DATA QUALITY UTILITIES  (shared across all three layers)
# ─────────────────────────────────────────────────────────────────────────────

def safe_query(sql: str, fallback=None):
    """Execute a DuckDB query with full error handling. Returns None on failure.
    Safe to call before Cell 5 — returns None if 'db' is not yet initialised.
    """
    try:
        _db = globals().get('db')
        if _db is None:
            return fallback() if callable(fallback) else None
        result = _db.execute(sql).df()
        return result if not result.empty else (fallback() if callable(fallback) else None)
    except Exception as e:
        logger.warning('Query failed (%s): %s', type(e).__name__, str(e)[:150])
        return fallback() if callable(fallback) else None

def safe_val(v, default=0.0):
    """Safely extract a scalar — guards against NaN, None, and numpy type errors."""
    try:
        x = float(v)
        return x if x == x else float(default)
    except (TypeError, ValueError):
        return float(default)

def winsorise(arr: 'np.ndarray', pct: int = None) -> 'np.ndarray':
    """Clip extreme outliers at the given percentile before statistical testing."""
    pct = pct or SOURCE_CONFIG.get('winsorise_pct', 99)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0: return arr
    upper = np.percentile(arr, pct)
    lower = np.percentile(arr, 100 - pct)
    return np.clip(arr, lower, upper)

def dedup_dataframe(df: 'pd.DataFrame', key_col: str = None) -> 'pd.DataFrame':
    """Remove duplicate rows on dedup_key. Keeps the last occurrence.
    Critical for event-level data where ETL pipelines double-fire.
    """
    key = key_col or SOURCE_CONFIG.get('dedup_key')
    if not key or key not in df.columns: return df
    n_before = len(df)
    df = df.drop_duplicates(subset=[key], keep='last').reset_index(drop=True)
    dropped = n_before - len(df)
    if dropped > 0:
        print(f'  ⚠️  Dedup: removed {dropped:,} duplicate rows on "{key}"')
        logger.warning('Dedup removed %d rows on %s', dropped, key)
    return df

def validate_experiment_data(df: 'pd.DataFrame', exp_name: str) -> dict:
    """Pre-analysis data quality checks. Returns dict with warnings, errors, ok flag."""
    report = {'warnings': [], 'errors': [], 'ok': True}
    if len(df) == 0:
        report['errors'].append('No rows found')
        report['ok'] = False
        return report
    for c_name in ['converted_to_order', 'variant', 'created_at']:
        if c_name not in df.columns:
            report['errors'].append(f'Required column missing: {c_name}')
            report['ok'] = False
            continue
        null_pct = df[c_name].isna().mean() * 100
        if null_pct > 5:
            report['warnings'].append(f'{c_name}: {null_pct:.1f}% null values')
    if 'variant' in df.columns:
        counts = df['variant'].value_counts()
        for v, c in counts.items():
            if c < SOURCE_CONFIG['min_segment_size']:
                report['warnings'].append(f'Variant "{v}" has only {c} rows')
        if len(counts) > 1 and counts.min() / counts.max() < 0.7:
            report['warnings'].append(
                f'Variant imbalance: ratio={counts.min()/counts.max():.2f} — possible SRM')
    if 'order_value' in df.columns:
        vals = df['order_value'].dropna()
        pos  = vals[vals > 0]
        if len(pos) > 10:
            p99, vmax = np.percentile(pos, 99), pos.max()
            if vmax > p99 * 10:
                report['warnings'].append(
                    f'Extreme outlier in order_value: max=${vmax:,.0f} vs p99=${p99:,.0f}. '
                    f'Winsorisation applied automatically.')
    report['ok'] = len(report['errors']) == 0
    return report

# ── Snowflake connector (used by Bronze ingestion in production mode) ─────────
def get_snowflake_conn():
    """Returns a Snowflake connection using credentials from SOURCE_CONFIG."""
    try:
        import snowflake.connector
        cfg = SOURCE_CONFIG['snowflake']
        conn = snowflake.connector.connect(
            account=cfg['account'], user=cfg['user'],
            password=cfg['password'], warehouse=cfg['warehouse'], role=cfg['role'])
        print(f'  ✅ Snowflake connected: {cfg["account"]} / {cfg["warehouse"]}')
        return conn
    except Exception as e:
        print(f'  ❌ Snowflake connection failed: {e}')
        print(f'     Set env vars: SNOWFLAKE_ACCOUNT, SNOWFLAKE_USER, SNOWFLAKE_PASSWORD')
        return None

# ── Synthetic-mode column identity override ───────────────────────────────────
if USE_SYNTHETIC_DATA:
    _identity_cols = [
        'inquiry_id', 'buyer_id', 'converted', 'order_value',
        'created_at', 'platform', 'category', 'variant',
        'experiment_name', 'account_segment', 'country',
    ]
    for _n in _identity_cols:
        SOURCE_CONFIG['column_map'][_n] = _n
    SOURCE_CONFIG['column_map']['account_id']    = 'buyer_id'
    SOURCE_CONFIG['column_map']['converted']     = 'converted_to_order'
    SOURCE_CONFIG['column_map']['order_total']   = 'order_value'
    SOURCE_CONFIG['column_map']['traffic_date']  = 'date'
    SOURCE_CONFIG['column_map']['total_sessions']= 'total_sessions'
    SOURCE_CONFIG['column_map']['new_signups']   = 'new_signups'
    SOURCE_CONFIG['column_map']['signed_in']     = 'signed_in'
    CLIENT_SCHEMA['columns'] = SOURCE_CONFIG['column_map']

def _refresh_runtime_constants() -> None:
    global SEGMENTS, PLATFORMS, CATEGORIES, COUNTRIES
    for attr, key in [('SEGMENTS','segments'),('PLATFORMS','platforms'),
                      ('CATEGORIES','categories'),('COUNTRIES','countries')]:
        vals = SOURCE_CONFIG.get(key, [])
        if vals: globals()[attr] = list(vals)


mode = {'synthetic': '🧪 Synthetic (POC)', 'csv': '📁 CSV Files', 'production': '❄️  Snowflake'}.get(DATA_MODE, DATA_MODE)
print(f'✅ Source Config + Bronze Schema loaded — Mode: {mode}')

# ─────────────────────────────────────────────────────────────────────────────
# CSV MODE SETUP  (only needed when DATA_MODE = 'csv')
#
# Call register_csv_source() once per table below.
# The alias must match a key in SOURCE_CONFIG['bronze_tables'] above.
#
# Example:
#   register_csv_source('quotes',      'exports/quotes_2024.csv')
#   register_csv_source('orders',      'exports/orders_2024.csv')
#   register_csv_source('users',       'exports/users.csv')
# ─────────────────────────────────────────────────────────────────────────────
print(f'   Client   : {SOURCE_CONFIG["client_name"]}')
print(f'   Dedup key: {SOURCE_CONFIG["dedup_key"]}')
print(f'   Winsorise: {SOURCE_CONFIG["winsorise_pct"]}th percentile')
print()
if USE_SYNTHETIC_DATA:
    print('   ┌─ Medallion path (synthetic) ─────────────────────────────────────────')
    print('   │  Bronze  → Cell 5  : generate raw DataFrames, register bronze_* tables')
    print('   │  Silver  → Cell 5-S: DuckDB views — canonicalise, clean, unify sources')
    print('   │  Gold    → Cell 6  : DuckDB views — cohorts, dim breakdowns, time-series')
    print('   └──────────────────────────────────────────────────────────────────────')


✅ Source Config + Bronze Schema loaded — Mode: 🧪 Synthetic (POC)
   Client   : ABC
   Dedup key: quote_id
   Winsorise: 99th percentile

   ┌─ Medallion path (synthetic) ─────────────────────────────────────────
   │  Bronze  → Cell 5  : generate raw DataFrames, register bronze_* tables
   │  Silver  → Cell 5-S: DuckDB views — canonicalise, clean, unify sources
   │  Gold    → Cell 6  : DuckDB views — cohorts, dim breakdowns, time-series
   └──────────────────────────────────────────────────────────────────────


## 3-Auto · Bronze-Auto — Production Config Rebuild

**Active when:** `CONTINUM_STATE['mode'] == 'production_ready'`
*(set by Module [1] Schema Discovery after human approval)*

In production mode this cell:
1. Reads the approved mapping from `CONTINUM_STATE['client_schema']`
2. Rebuilds `SOURCE_CONFIG` (+ `CLIENT_SCHEMA` alias) from the bootstrap-approved column/table mapping
3. Sets `USE_SYNTHETIC_DATA = False` — the Bronze layer will load from the real warehouse
4. Validates the warehouse connection before the Bronze → Silver → Gold pipeline runs

In synthetic mode this is a safe no-op — Cell 3 config stays active.

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3-AUTO — Bronze-Auto: Production Config Rebuild
#
# Reads from CONTINUM_STATE when bootstrapped; otherwise a clean no-op.
# ─────────────────────────────────────────────────────────────────────────────

DATA_PATH = 'synthetic'   # updated below if production_ready

def _build_source_config_from_state(state_schema: dict) -> dict:
    """
    Convert CONTINUM_STATE['client_schema'] (produced by Module [1]) into
    the full SOURCE_CONFIG structure expected by col(), tbl(), Silver SQL,
    and all downstream helpers.
    """
    mapping     = state_schema.get('mapping', {})
    table_map   = mapping.get('table_mapping', {})
    column_map  = mapping.get('column_mapping', {})
    client_name = state_schema.get('client_name', 'Unknown')
    conn_cfg    = state_schema.get('connection', {})

    columns = dict(column_map)
    for c in [
        'inquiry_id', 'buyer_id', 'converted', 'order_value', 'created_at',
        'platform', 'category', 'variant', 'experiment_name', 'account_segment',
        'country', 'order_total', 'traffic_date', 'account_id',
        'quote_id', 'quote_user_id', 'quote_account_id', 'quote_created_at',
        'quote_source', 'quote_processes', 'quote_price', 'quote_status',
        'order_id', 'order_quote_id', 'order_bookings', 'order_status',
        'order_time', 'order_ship_date', 'order_payment_type', 'order_country',
        'user_id', 'user_account_id', 'user_email_flag', 'user_customer_flag',
        'account_vertical', 'account_country', 'account_employees',
        'exp_user_id', 'exp_group_name', 'exp_experiment_id',
        'exp_timestamp', 'exp_account_domain',
    ]:
        columns.setdefault(c, c)

    cfg = {
        'client_name': client_name,
        'snowflake': {
            'account':   conn_cfg.get('account',   os.environ.get('SNOWFLAKE_ACCOUNT', '')),
            'user':      conn_cfg.get('user',       os.environ.get('SNOWFLAKE_USER', '')),
            'password':  conn_cfg.get('password',   os.environ.get('SNOWFLAKE_PASSWORD', '')),
            'warehouse': conn_cfg.get('warehouse',  os.environ.get('SNOWFLAKE_WAREHOUSE', 'COMPUTE_WH')),
            'role':      conn_cfg.get('role',       os.environ.get('SNOWFLAKE_ROLE', 'ANALYST')),
        },
        'bronze_tables': {
            'quotes':      table_map.get('quotes',      ''),
            'orders':      table_map.get('orders',      ''),
            'users':       table_map.get('users',       ''),
            'accounts':    table_map.get('accounts',    ''),
            'experiments': table_map.get('experiments', ''),
            'inquiries':   'silver_inquiries',    # resolved after Silver runs
            'all_exp':     'gold_experiment_analysis',
            'traffic':     'silver_traffic',
        },
        'column_map':   columns,
        'segment_map':  state_schema.get('segment_map',  {}),
        'platform_map': state_schema.get('platform_map', {}),
        'cancelled_order_statuses': state_schema.get('cancelled_order_statuses', []),
        'internal_domains':         state_schema.get('internal_domains',         []),
        'dedup_key':          state_schema.get('dedup_key',        'inquiry_id'),
        'winsorise_pct':      state_schema.get('winsorise_pct',    99),
        'min_segment_size':   state_schema.get('min_segment_size', 30),
        'null_ior_default':   state_schema.get('null_ior_default', 0.18),
        'null_aov_default':   state_schema.get('null_aov_default', 5000),
        'null_daily_traffic': state_schema.get('null_daily_traffic', 300),
        'segments':   state_schema.get('segments',   []),
        'platforms':  state_schema.get('platforms',  []),
        'categories': state_schema.get('categories', []),
        'countries':  state_schema.get('countries',  []),
    }
    cfg['tables']  = cfg['bronze_tables']
    cfg['columns'] = cfg['column_map']
    return cfg


def _validate_bronze_connection(cfg: dict) -> bool:
    """
    Lightweight connectivity check: opens a real Snowflake connection,
    runs SELECT 1, and closes it. Uses only cfg['snowflake'] — does NOT
    reference db (DuckDB), which is created later in Cell 5.
    """
    sf_cfg = cfg.get('snowflake', {})
    if not sf_cfg.get('account'):
        print('     ⚠️  No Snowflake account configured — skipping probe.')
        return True   # optimistic pass; Cell 5 will surface real errors
    try:
        import snowflake.connector as _sf
        _probe_conn = _sf.connect(
            account=sf_cfg['account'],   user=sf_cfg['user'],
            password=sf_cfg['password'], warehouse=sf_cfg['warehouse'],
            role=sf_cfg.get('role', 'ANALYST'),
            login_timeout=10,
        )
        _probe_conn.cursor().execute('SELECT 1')
        _probe_conn.close()
        return True
    except Exception as e:
        print(f'     ⚠️  Snowflake connection probe failed: {e}')
        return False


# ─── Main execution ──────────────────────────────────────────────────────────
_state = globals().get('CONTINUM_STATE', {})

if _state.get('mode') != 'production_ready' or _state.get('client_schema') is None:
    print('  ℹ️  Cell 3-Auto: mode is not production_ready — using existing Cell 3 config.')
    DATA_PATH = 'synthetic'

else:
    print('  🔧 Cell 3-Auto: production_ready detected — rebuilding SOURCE_CONFIG...')

    SOURCE_CONFIG = _build_source_config_from_state(_state['client_schema'])
    CLIENT_SCHEMA = SOURCE_CONFIG   # keep alias live
    USE_SYNTHETIC_DATA = False

    print(f'     Client: {SOURCE_CONFIG["client_name"]}')
    print('     Validating Bronze connection...', end=' ')
    _conn_ok = _validate_bronze_connection(SOURCE_CONFIG)
    print('✅' if _conn_ok else '❌')

    if not _conn_ok:
        print('  ⚠️  Connection failed — falling back to synthetic mode.')
        USE_SYNTHETIC_DATA = True
        DATA_PATH = 'synthetic'
    else:
        DATA_PATH = 'production'
        print(f'  ✅ SOURCE_CONFIG rebuilt. Bronze → Silver → Gold pipeline ready.')
        print(f'     Bronze tables: {", ".join(k+"→"+v for k,v in SOURCE_CONFIG["bronze_tables"].items() if v and "silver" not in v and "gold" not in v)}')
        CLIENT_SCHEMA = SOURCE_CONFIG
        CLIENT_SCHEMA['tables']  = SOURCE_CONFIG['bronze_tables']
        CLIENT_SCHEMA['columns'] = SOURCE_CONFIG['column_map']
        try:
            _refresh_runtime_constants()
        except NameError:
            pass


  ℹ️  Cell 3-Auto: mode is not production_ready — using existing Cell 3 config.


## 3b · Bronze Ingestion Gateway + SILVER_CONFIG

**Bronze ETL/ELT integration layer.** Enforces an allowlist so only declared tables enter DuckDB. ETL/ELT tools (Fivetran, Airbyte, dbt, Stitch, custom pipelines) call `register_etl_push()` to push a raw DataFrame directly into the Bronze layer without any Snowflake connection.

| Concern | Old location | New location |
|---|---|---|
| Raw source table paths | `SOURCE_CONFIG['bronze_tables']` | unchanged |
| Column map (bronze→canonical) | `SOURCE_CONFIG['column_map']` | unchanged |
| Transformation thresholds | `SOURCE_CONFIG` (wrong layer) | `SILVER_CONFIG` |
| Segment/platform normalisation | `SOURCE_CONFIG` (wrong layer) | `SILVER_CONFIG` |
| Exclusion filters | `SOURCE_CONFIG` (wrong layer) | `SILVER_CONFIG` |

**Usage — push mode (ETL/ELT tool sends the data):**
```python
bronze_gateway = BronzeIngestionGateway(db, SOURCE_CONFIG)
bronze_gateway.register_etl_push('quotes',    df_raw_quotes)
bronze_gateway.register_etl_push('orders',    df_raw_orders)
bronze_gateway.register_etl_push('users',     df_raw_users)
bronze_gateway.get_status()   # shows registered vs missing
```

**Usage — pull mode (original Snowflake approach):**
```python
bronze_gateway.pull_from_snowflake(conn, tables=['quotes','orders','experiments'])
```

**Adding a non-standard table:**
```python
bronze_gateway.add_allowed_table('nps_scores', required_cols=['USER_ID','SCORE','DATE'])
bronze_gateway.register_etl_push('nps_scores', df_nps)
```

In [8]:

@dataclass
class SchemaDriftWarning:
    table: str
    added_cols: list
    removed_cols: list
    dtype_changes: dict

    def __str__(self):
        parts = []
        if self.removed_cols:
            parts.append(f"REMOVED cols (will be NULL in Silver): {self.removed_cols}")
        if self.added_cols:
            parts.append(f"NEW cols (ignored by Silver): {self.added_cols}")
        if self.dtype_changes:
            changes = ', '.join(f'{c}: {old}→{new}' for c, (old, new) in self.dtype_changes.items())
            parts.append(f"DTYPE changes: {changes}")
        return f"Schema drift on '{self.table}': " + " | ".join(parts)

class BronzeIngestionGateway:
    """
    Single entry point for all data entering the Bronze layer.

    Responsibilities
    ────────────────
    1. ALLOWLIST  — only tables declared in ALLOWED_TABLES reach DuckDB.
    2. VALIDATION — checks columns, row count, and schema drift on each push.
    3. SEPARATION — enforces Bronze = raw paths only; Silver config is separate.
    4. PUSH MODE  — ETL/ELT tools call register_etl_push(); no code change needed.
    5. PULL MODE  — direct Snowflake/DB pull via pull_from_snowflake().

    Usage (push from ETL/ELT tool)
    ───────────────────────────────
        gateway = BronzeIngestionGateway(db, SOURCE_CONFIG)
        gateway.register_etl_push('quotes',    df_raw_quotes)
        gateway.register_etl_push('orders',    df_raw_orders)
        gateway.register_etl_push('users',     df_raw_users)
        # 'accounts' not sent → stays empty, Silver view still runs (nullable)
        status = gateway.get_status()

    Usage (pull mode — original Snowflake approach)
    ────────────────────────────────────────────────
        gateway.pull_from_snowflake(conn, tables=['quotes','orders','experiments'])
    """

    # ── Canonical set of tables this platform understands ────────────────────
    ALLOWED_TABLES = {
        'quotes', 'orders', 'users', 'accounts', 'experiments', 'traffic',
    }

    # ── Required canonical columns per table (from SOURCE_CONFIG column_map) ─
    REQUIRED_COLUMNS: dict = {
        'quotes':      ['_ID', 'USER', 'BILLING_ACCOUNT', '_CONSTRUCTED'],
        'orders':      ['_ID', 'QUOTE_ORDER_ID', 'TOTAL', 'LAST_HISTORY_STATUS'],
        'users':       ['_ID', 'ACCOUNT'],
        'accounts':    ['ACCOUNT_ID', 'CONSOLIDATED_BUSINESS_SEGMENT'],
        'experiments': ['USER_ID', 'GROUP_NAME', 'EXPERIMENT_ID', 'EXPERIMENT_NAME', 'TIMESTAMP'],
        'traffic':     ['DATE', 'TOTAL_SESSIONS'],
    }

    def __init__(self, db, source_config: dict):
        self._db             = db
        self._cfg            = source_config
        self._registered     : dict[str, dict] = {}   # alias → {rows, cols, bronze_name}
        self._schema_baseline: dict[str, dict] = {}   # alias → {col_name: dtype}
        self._drift_log      : list[SchemaDriftWarning] = []
        self._push_mode      = False   # True once first ETL push arrives
        print('  ✅ BronzeIngestionGateway initialised')
        print(f'     Allowed tables: {sorted(self.ALLOWED_TABLES)}')
        print(f'     Mode: pull (default) — call register_etl_push() to switch to push mode')

    # ── Public API ──────────────────────────────────────────────────────────

    def register_etl_push(
        self,
        table_alias: str,
        df: 'pd.DataFrame',
        *,
        allow_empty: bool = False,
        schema_drift_policy: str = 'warn',   # 'warn' | 'raise' | 'ignore'
    ) -> bool:
        """
        Called by an ETL/ELT tool to push a raw table into the Bronze layer.

        Parameters
        ──────────
        table_alias        : canonical alias ('quotes', 'orders', etc.)
        df                 : raw DataFrame — not yet renamed or transformed
        allow_empty        : if False, rejects tables with 0 rows
        schema_drift_policy: what to do when schema differs from baseline

        Returns True on success, False on validation failure.
        """
        self._push_mode = True
        alias = table_alias.lower().strip()

        # ── Gate 1: allowlist ──────────────────────────────────────────────
        if alias not in self.ALLOWED_TABLES:
            print(f'  ❌ Gateway REJECTED "{alias}": not in ALLOWED_TABLES')
            print(f'     Allowed: {sorted(self.ALLOWED_TABLES)}')
            print(f'     To add it: BronzeIngestionGateway.ALLOWED_TABLES.add("{alias}")')
            return False

        # ── Gate 2: row count ─────────────────────────────────────────────
        if len(df) == 0 and not allow_empty:
            print(f'  ⚠️  Gateway WARNING "{alias}": empty DataFrame — skipped')
            print(f'     To allow empty tables: register_etl_push(..., allow_empty=True)')
            return False

        # ── Gate 3: required columns ──────────────────────────────────────
        required = self.REQUIRED_COLUMNS.get(alias, [])
        # In synthetic mode, required cols are already canonical → skip check
        if not globals().get('USE_SYNTHETIC_DATA', True):
            missing = [c for c in required if c not in df.columns]
            if missing:
                print(f'  ❌ Gateway REJECTED "{alias}": missing required columns: {missing}')
                print(f'     Available columns: {list(df.columns)[:20]}')
                print(f'     Fix: update SOURCE_CONFIG[\'column_map\'] or the ETL mapping.')
                return False

        # ── Gate 4: schema drift detection ───────────────────────────────
        drift = self._detect_schema_drift(alias, df)
        if drift:
            self._drift_log.append(drift)
            msg = str(drift)
            if schema_drift_policy == 'raise':
                raise RuntimeError(f'  ❌ Schema drift — {msg}')
            elif schema_drift_policy == 'warn':
                print(f'  ⚠️  {msg}')
                print(f'     Silver SQL will produce NULLs for missing columns.')
                print(f'     Consider re-running Module [2] Pipeline Health after loading.')
            # 'ignore' → proceed silently

        # ── Register in DuckDB ────────────────────────────────────────────
        bronze_name = f'bronze_{alias}'
        self._db.register(bronze_name, df)
        self._registered[alias] = {
            'rows':        len(df),
            'cols':        list(df.columns),
            'bronze_name': bronze_name,
            'source':      'etl_push',
        }
        # Update schema baseline on first registration
        if alias not in self._schema_baseline:
            self._schema_baseline[alias] = {c: str(df[c].dtype) for c in df.columns}

        print(f'  ✅ Bronze "{alias}" registered → {bronze_name}  ({len(df):,} rows, {len(df.columns)} cols)')
        return True

    def pull_from_snowflake(
        self,
        conn,
        tables: Optional[list] = None,
        row_limit: Optional[int] = None,
    ) -> dict:
        """
        Pull mode: fetch tables directly from Snowflake and register them.

        Parameters
        ──────────
        conn      : active snowflake.connector connection
        tables    : list of aliases to pull (defaults to all ALLOWED_TABLES)
        row_limit : for sampling / testing (None = full table)

        Returns dict of {alias: row_count} for successfully pulled tables.
        """
        to_pull = [t for t in (tables or sorted(self.ALLOWED_TABLES))
                   if t in self.ALLOWED_TABLES]
        results = {}
        limit_sql = f' LIMIT {row_limit}' if row_limit else ''

        for alias in to_pull:
            raw_path = self._cfg.get('bronze_tables', {}).get(alias, '')
            if not raw_path or raw_path.startswith('silver_') or raw_path.startswith('gold_'):
                print(f'  ⚠️  Skipping "{alias}": no raw source path configured')
                continue
            try:
                cursor = conn.cursor()
                sql    = f'SELECT * FROM {raw_path}{limit_sql}'
                cursor.execute(sql)
                df_raw = cursor.fetch_pandas_all()
                cursor.close()
                ok = self.register_etl_push(alias, df_raw)
                if ok:
                    results[alias] = len(df_raw)
            except Exception as e:
                print(f'  ❌ Failed to pull "{alias}" from {raw_path}: {e}')

        return results

    def get_status(self) -> dict:
        """Return a summary of what has been registered vs what is missing."""
        registered  = set(self._registered.keys())
        missing     = self.ALLOWED_TABLES - registered
        status = {
            'mode':           'push' if self._push_mode else 'pull',
            'registered':     {k: v['rows'] for k, v in self._registered.items()},
            'missing':        sorted(missing),
            'drift_warnings': len(self._drift_log),
            'gateway_ready':  len(missing) == 0,
        }

        print()
        print('  ┌─ Bronze Gateway Status ──────────────────────────────────────')
        print(f'  │  Mode             : {status["mode"]}')
        for alias, rows in status['registered'].items():
            print(f'  │  ✅ {alias:<18} : {rows:>9,} rows')
        for alias in status['missing']:
            print(f'  │  ⚠️  {alias:<18} : NOT REGISTERED')
        if self._drift_log:
            print(f'  │  ⚠️  Schema drift warnings: {len(self._drift_log)}')
            for d in self._drift_log:
                print(f'  │      {str(d)[:80]}')
        print(f'  │  Gateway ready    : {"✅ YES" if status["gateway_ready"] else "❌ NO — missing tables above"}')
        print('  └─────────────────────────────────────────────────────────────')
        return status

    def add_allowed_table(self, alias: str, required_cols: list = None):
        """
        Extend the allowlist at runtime for client-specific tables.
        Required for any non-standard source before register_etl_push().

        Example
        ───────
            gateway.add_allowed_table('nps_scores', required_cols=['USER_ID','SCORE','DATE'])
            gateway.register_etl_push('nps_scores', df_nps)
        """
        self.ALLOWED_TABLES.add(alias.lower())
        if required_cols:
            self.REQUIRED_COLUMNS[alias.lower()] = required_cols
        print(f'  ✅ Added "{alias}" to allowlist  (required: {required_cols or "none"})')

    # ── Private helpers ──────────────────────────────────────────────────────

    def _detect_schema_drift(
        self,
        alias: str,
        df: 'pd.DataFrame',
    ) -> Optional[SchemaDriftWarning]:
        """Compare df schema against stored baseline. Returns drift report or None."""
        baseline = self._schema_baseline.get(alias)
        if not baseline:
            return None   # first registration → set baseline, no drift yet

        current_cols  = set(df.columns)
        baseline_cols = set(baseline.keys())

        added         = sorted(current_cols - baseline_cols)
        removed       = sorted(baseline_cols - current_cols)
        dtype_changes = {
            c: (baseline[c], str(df[c].dtype))
            for c in current_cols & baseline_cols
            if str(df[c].dtype) != baseline[c]
        }

        if added or removed or dtype_changes:
            return SchemaDriftWarning(alias, added, removed, dtype_changes)
        return None

# ── SILVER_CONFIG — separate from Bronze SOURCE_CONFIG ───────────────────────

SILVER_CONFIG = {
    # ── Transformation thresholds (Silver cleaning + module pre-checks) ───
    'winsorise_pct':      99,
    'min_segment_size':   30,
    'null_ior_default':   0.18,
    'null_aov_default':   5000.0,
    'null_daily_traffic': 300,

    # ── Segment / platform normalisation (applied in Silver SQL) ──────────
    'segment_map': {
        'Individuals':    'Individuals',
        'Small Business': 'SMB',
        'Medium Business':'Growth',
        'Large Business': 'Core',
        'Enterprise':     'Enterprise',
    },
    'platform_map': {
        'WEBAPP':     'web',
        'FUSION':     'desktop',
        'SOLIDWORKS': 'desktop',
    },

    # ── Exclusion filters ─────────────────────────────────────────────────
    'cancelled_order_statuses': ['Order Cancelled'],
    'internal_domains':         ['xometry.com', 'staff.xometry.com', 'xometry.eu'],

    # ── Dedup key for event-level data ────────────────────────────────────
    'dedup_key': 'quote_id',
}

print('✅ Cell 3b: BronzeIngestionGateway class + SILVER_CONFIG loaded')
print('   Usage (ETL push mode):')
print('     bronze_gateway = BronzeIngestionGateway(db, SOURCE_CONFIG)')
print('     bronze_gateway.register_etl_push("quotes",    df_raw_quotes)')
print('     bronze_gateway.register_etl_push("orders",    df_raw_orders)')
print('     bronze_gateway.register_etl_push("users",     df_raw_users)')
print('     bronze_gateway.get_status()   # ← shows what is missing')
print()
print('   Usage (Snowflake pull mode):')
print('     bronze_gateway.pull_from_snowflake(conn, tables=["quotes","orders","experiments"])')
print()
print('   Add a non-standard table:')
print('     bronze_gateway.add_allowed_table("nps_scores", required_cols=["USER_ID","SCORE"])')
print('     bronze_gateway.register_etl_push("nps_scores", df_nps)')


✅ Cell 3b: BronzeIngestionGateway class + SILVER_CONFIG loaded
   Usage (ETL push mode):
     bronze_gateway = BronzeIngestionGateway(db, SOURCE_CONFIG)
     bronze_gateway.register_etl_push("quotes",    df_raw_quotes)
     bronze_gateway.register_etl_push("orders",    df_raw_orders)
     bronze_gateway.register_etl_push("users",     df_raw_users)
     bronze_gateway.get_status()   # ← shows what is missing

   Usage (Snowflake pull mode):
     bronze_gateway.pull_from_snowflake(conn, tables=["quotes","orders","experiments"])

   Add a non-standard table:
     bronze_gateway.add_allowed_table("nps_scores", required_cols=["USER_ID","SCORE"])
     bronze_gateway.register_etl_push("nps_scores", df_nps)


## 4 · LLM Client

Local HuggingFace model (default). Swap with a cloud provider for production.

In [9]:
# ── Model selection ─────────
AGENT_CONFIG = {
    # 'model_id':    'microsoft/Phi-3-mini-4k-instruct',
    'model_id':  'Qwen/Qwen2.5-1.5B-Instruct', 
    # 'model_id':  'TinyLlama/TinyLlama-1.1B-Chat-v1.0',

    'max_new_tokens':     700,
    'temperature':        0.3,
    'do_sample':          True,
    'repetition_penalty': 1.1,
}


def _sanitise_rope_scaling(config):
    """
    Normalise config.rope_scaling so both "type" and "rope_type" are present.
    This protects against any transformers version that reads one but not
    the other key. Safe no-op if the config has no rope_scaling at all.
    """
    rs = getattr(config, 'rope_scaling', None)
    if not rs or not isinstance(rs, dict):
        return config
    if 'type' in rs and 'rope_type' not in rs:
        rs['rope_type'] = rs['type']
    elif 'rope_type' in rs and 'type' not in rs:
        rs['type'] = rs['rope_type']
    config.rope_scaling = rs
    return config


class TransformersClient:
    """
    LLM client backed by HuggingFace Transformers.

    - No external server, no API key, no CLI tools.
    - Model downloads once (~600 MB to 2.4 GB) on first use; subsequent runs
      are fully offline from the local cache.
    - Used ONLY for narrative text — all stats are pure Python/scipy.
    - The loader for Phi-3 uses transformers\' native implementation via
      AutoConfig + sanitise + AutoModelForCausalLM. We do NOT use
      trust_remote_code because the cached remote Phi-3 modeling file is
      routinely stale and crashes with KeyError: \'type\'.
    """

    def __init__(self, config: dict = None):
        cfg = config or AGENT_CONFIG
        self.model_id       = cfg['model_id']
        self.max_new_tokens = cfg.get('max_new_tokens', 512)
        self.gen_kwargs     = {
            'max_new_tokens':     self.max_new_tokens,
            'temperature':        cfg.get('temperature', 0.3),
            'do_sample':          cfg.get('do_sample', True),
            'repetition_penalty': cfg.get('repetition_penalty', 1.1),
        }
        self._pipe = None   # lazy load — only loads model when first .ask() is called
        print(f'  TransformersClient configured: {self.model_id}')
        print(f'  Model will download on first use if not already cached.')

    # ── Lazy model loader ─────────────────────────────────────────────────────
    def _load(self):
        if self._pipe is not None:
            return
        print(f'  Loading {self.model_id} ...')

        dtype = torch.float16 if DEVICE != 'cpu' else torch.float32


        from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
        try:
            model_config = AutoConfig.from_pretrained(
                self.model_id, trust_remote_code=False)
            model_config = _sanitise_rope_scaling(model_config)
        except Exception as e_cfg:
            print(f'  Could not pre-load config ({type(e_cfg).__name__}). '
                  'Proceeding with defaults.')
            model_config = None

        load_attempts = [
            dict(label='native (eager attention, sanitised config)',
                 trust_remote_code=False,
                 attn_implementation='eager',
                 config=model_config),
            dict(label='native (default attention, sanitised config)',
                 trust_remote_code=False,
                 attn_implementation=None,
                 config=model_config),
            dict(label='native (CPU float32 fallback)',
                 trust_remote_code=False,
                 attn_implementation='eager',
                 config=model_config,
                 force_cpu=True),
        ]

        last_err = None
        for attempt in load_attempts:
            try:
                tok = AutoTokenizer.from_pretrained(
                    self.model_id, trust_remote_code=False)
                model_kwargs = dict(
                    pretrained_model_name_or_path=self.model_id,
                    trust_remote_code=False,
                    dtype=torch.float32 if attempt.get('force_cpu') else dtype,
                )
                if attempt.get('config') is not None:
                    model_kwargs['config'] = attempt['config']
                if attempt.get('attn_implementation'):
                    model_kwargs['attn_implementation'] = attempt['attn_implementation']
                if attempt.get('force_cpu'):
                    model_kwargs['device_map'] = 'cpu'
                else:
                    model_kwargs['device_map'] = 'auto'

                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    mdl = AutoModelForCausalLM.from_pretrained(**model_kwargs)

                # Ensure pad token is set (Phi-3 uses EOS as pad)
                if tok.pad_token_id is None:
                    tok.pad_token_id = tok.eos_token_id

                device_for_pipe = -1 if attempt.get('force_cpu') else (
                    0 if DEVICE == 'cuda' else -1)
                self._pipe = pipeline(
                    'text-generation',
                    model=mdl,
                    tokenizer=tok,
                    device=device_for_pipe,
                )
                print(f'  {self.model_id} loaded: {attempt["label"]}')
                return
            except Exception as e:
                last_err = e
                print(f'  Attempt failed [{attempt["label"]}]: '
                      f'{type(e).__name__}: {str(e)[:120]}')

        raise RuntimeError(
            f'Failed to load {self.model_id}. Last error: {last_err}'
        )

    # ── Core chat method ──────────────────────────────────────────────────────
    def ask(self, prompt: str, system: str = '') -> str:
        """
        Send a prompt to the model and return the response text.
        Uses the chat template (Phi-3 has a well-formed chat template).
        """
        self._load()

        messages = []
        if system:
            messages.append({'role': 'system', 'content': system})
        messages.append({'role': 'user', 'content': prompt})

        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                outputs = self._pipe(
                    messages,
                    **self.gen_kwargs,
                    return_full_text=False,
                    pad_token_id=self._pipe.tokenizer.pad_token_id,
                )
            generated = outputs[0]['generated_text']
            if isinstance(generated, list):
                for msg in reversed(generated):
                    if isinstance(msg, dict) and msg.get('role') == 'assistant':
                        return msg['content'].strip()
                last = generated[-1]
                return (last.get('content', str(last)) if isinstance(last, dict) else str(last)).strip()
            return str(generated).strip()

        except Exception as e:
            logger.error('TransformersClient.ask() error: %s', e)
            return f'[LLM error: {e}]'

    # ── Narrative helper ──────────────────────────────────────────────────────
    def narrate(self, data: dict, context: str) -> str:
        """Generate an executive business narrative from structured result data."""
        system = (
            'You are a senior data scientist presenting findings to VPs '
            'and HODs. Write clear, concise, executive-level insights. Be specific '
            'with numbers. Flag risks and opportunities clearly. Be direct — no '
            'filler phrases. Write in plain business language. Do not use emojis, '
            'icons, or decorative symbols.'
        )
        data_str = json.dumps(data, indent=2, default=str)
        if len(data_str) > 3000:
            data_str = data_str[:3000] + '\n... [truncated]'
        prompt = f'{context}\n\nData:\n{data_str}'
        return self.ask(prompt, system=system)

    def unload(self):
        """Free GPU/CPU memory. Call when done with the agent."""
        if self._pipe is not None:
            del self._pipe
            self._pipe = None
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            print(f'  {self.model_id} unloaded from memory')


# ── Instantiate ───────────────────────────────────────────────────────────────
narrative_llm = TransformersClient(AGENT_CONFIG)
print()
print('TransformersClient ready (model loads on first use)')
print(f'   Model : {AGENT_CONFIG["model_id"]}')
print(f'   Device: {DEVICE}')
print()
print('   To pre-load now:  narrative_llm._load()')
print('   To free memory:   narrative_llm.unload()')
print('   To switch model:  edit AGENT_CONFIG["model_id"] and re-run this cell')

  TransformersClient configured: Qwen/Qwen2.5-1.5B-Instruct
  Model will download on first use if not already cached.

TransformersClient ready (model loads on first use)
   Model : Qwen/Qwen2.5-1.5B-Instruct
   Device: cpu

   To pre-load now:  narrative_llm._load()
   To free memory:   narrative_llm.unload()
   To switch model:  edit AGENT_CONFIG["model_id"] and re-run this cell


## 5 · Bronze Layer — Raw Ingestion

The first layer of the medallion stack. In **synthetic mode** the platform generates realistic raw datasets and registers them as `bronze_*` tables in DuckDB — mimicking what would be loaded from production sources before any cleaning.

In **production mode** this cell loads each raw warehouse table from Snowflake (quotes, orders, users, accounts, experiments, traffic) **as-is** — no column renaming, no joining, no filtering — and registers each as a `bronze_*` DuckDB table.

```
  Source: Snowflake / CSV / Parquet
      ↓  (ingested without transformation)
  DuckDB: bronze_inquiries · bronze_buyers · bronze_exp · bronze_traffic
      ↓
  Silver layer (Cell 5-Silver) — clean, canonicalise, unify
```

The Silver layer handles all renaming and transformation; Bronze is a faithful mirror of the source.

In [10]:
rng = np.random.default_rng(42)

# ── Time bounds — 3 YEARS of history, 8 months of experimentation ─────────────
HIST_START       = pd.Timestamp('2022-08-01')
HIST_END         = pd.Timestamp('2024-12-31')
EXP_START        = pd.Timestamp('2025-01-01')
EXP_END          = pd.Timestamp('2025-08-31')
TODAY            = pd.Timestamp('2025-08-31')

# ── Ground truth IOR uplift (treatment vs control) — kept rich across segments
GT_UPLIFT = {
    ('Core',        'web'):    +0.014,
    ('Core',        'mobile'): +0.010,
    ('Growth',      'web'):    -0.015,
    ('Growth',      'mobile'): -0.012,
    ('Enterprise',  'web'):    +0.006,
    ('Enterprise',  'mobile'): +0.004,
    ('Individuals', 'web'):    -0.004,
    ('Individuals', 'mobile'): -0.002,
}

BASELINE_IOR  = {'Core': 0.195, 'Growth': 0.165, 'Enterprise': 0.240, 'Individuals': 0.130}
GMV_PER_ORDER = {'Core': 4800,  'Growth': 2200,  'Enterprise': 18000, 'Individuals': 950}

# ── BUYERS — 25k buyers with realistic demographics & behavior ────────────────
if DATA_MODE == 'synthetic':
    N_BUYERS   = 25_000
    buyer_ids  = [f'BYR{i:06d}' for i in range(1, N_BUYERS+1)]
    seg_arr    = rng.choice(SEGMENTS, N_BUYERS, p=[0.42, 0.32, 0.12, 0.14])
    plat_arr   = rng.choice(PLATFORMS, N_BUYERS, p=[0.66, 0.34])
    ctry_arr   = rng.choice(COUNTRIES, N_BUYERS, p=[0.70, 0.11, 0.08, 0.07, 0.04])
    join_ts    = pd.to_datetime(rng.integers(
        int((HIST_START - pd.Timedelta(days=730)).timestamp()),
        int(EXP_END.timestamp()), N_BUYERS), unit='s').normalize()

    # Lifetime orders correlated with segment + tenure (more realistic)
    tenure_days = np.clip((EXP_END - join_ts).days, 0, None)   # TimedeltaIndex → int array
    lifetime_order_rate = {'Core': 0.08, 'Growth': 0.04, 'Enterprise': 0.15, 'Individuals': 0.02}
    orders_base = np.array([lifetime_order_rate[s] for s in seg_arr]) * tenure_days / 30
    lifetime_orders = np.clip(rng.poisson(orders_base), 0, 500).astype(int)
    gmv_mean = np.array([GMV_PER_ORDER[s] for s in seg_arr])
    lifetime_gmv = (lifetime_orders * gmv_mean * rng.uniform(0.6, 1.5, N_BUYERS)).round(2)

    df_buyers = pd.DataFrame({
        'buyer_id': buyer_ids,
        'account_segment': seg_arr,
        'primary_platform': plat_arr,
        'country': ctry_arr,
        'industry': rng.choice(
            ['Aerospace','Medical','Auto','Industrial','Consumer','Defense','Energy','Electronics','Research'],
            N_BUYERS, p=[0.15, 0.14, 0.17, 0.19, 0.09, 0.05, 0.07, 0.10, 0.04]),
        'company_size': rng.choice(
            ['1-10','11-50','51-200','201-1000','1000+'], N_BUYERS, p=[0.25,0.30,0.22,0.15,0.08]),
        'joined_at': join_ts,
        'has_billing_profile': rng.choice([True, False], N_BUYERS, p=[0.61, 0.39]),
        'lifetime_orders': lifetime_orders,
        'lifetime_gmv': lifetime_gmv,
        'is_active_30d': rng.choice([True, False], N_BUYERS, p=[0.52, 0.48]),
    })

    # ── HISTORICAL INQUIRIES — 180k rows over 2.4 years with realistic seasonality
    N_HIST = 180_000
    hb_idx  = rng.integers(0, N_BUYERS, N_HIST)
    h_dates = pd.to_datetime(rng.integers(
        int(HIST_START.timestamp()), int(HIST_END.timestamp()), N_HIST), unit='s').normalize()
    h_seg   = seg_arr[hb_idx]
    h_plat  = rng.choice(PLATFORMS, N_HIST, p=[0.66, 0.34])

    # Day-of-week effect (B2B: weekends quieter) + monthly seasonality
    dow_factor      = np.where(pd.DatetimeIndex(h_dates).dayofweek >= 5, 0.92, 1.0)
    month_factor    = 1 + 0.08 * np.sin(2 * np.pi * pd.DatetimeIndex(h_dates).month / 12 - np.pi/6)
    trend_days      = (pd.DatetimeIndex(h_dates) - HIST_START).days.values
    trend_factor    = 1 + 0.0001 * trend_days                 # slow platform growth

    h_ior_base = np.array([BASELINE_IOR[s] for s in h_seg])
    h_ior      = np.clip(h_ior_base * dow_factor * month_factor * trend_factor
                         + rng.normal(0, 0.018, N_HIST), 0.01, 0.80)
    h_converted = rng.random(N_HIST) < h_ior
    h_gmv_per   = np.array([GMV_PER_ORDER[s] for s in h_seg])
    h_price_tier = rng.choice(['budget','mid','premium'], N_HIST, p=[0.30, 0.50, 0.20])
    h_price_mult = np.where(h_price_tier == 'budget', 0.55,
                    np.where(h_price_tier == 'premium', 1.85, 1.0))

    df_hist_inquiries = pd.DataFrame({
        'inquiry_id':      [f'INQ{i:08d}' for i in range(1, N_HIST+1)],
        'buyer_id':        [buyer_ids[i] for i in hb_idx],
        'account_segment': h_seg,
        'platform':        h_plat,
        'category':        rng.choice(CATEGORIES, N_HIST, p=[0.34, 0.26, 0.19, 0.13, 0.08]),
        'country':         ctry_arr[hb_idx],
        'price_tier':      h_price_tier,
        'process_group':   rng.choice(
            ['Prototyping','Low-Volume Production','High-Volume Production','Repair'],
            N_HIST, p=[0.35, 0.34, 0.22, 0.09]),
        'created_at':      h_dates,
        'has_billing_profile': rng.choice([True, False], N_HIST, p=[0.61, 0.39]),
        'converted_to_order': h_converted,
        'order_value':     np.where(h_converted,
                                     h_gmv_per * h_price_mult * rng.uniform(0.65, 1.5, N_HIST),
                                     0).round(2),
        'period': 'historical',
    })

    # ── EXPERIMENT INQUIRIES — 90k rows, 8 months (more per-day volume) ──────────
    N_EXP = 90_000
    eb_idx   = rng.integers(0, N_BUYERS, N_EXP)
    e_dates  = pd.to_datetime(rng.integers(
        int(EXP_START.timestamp()), int(EXP_END.timestamp()), N_EXP), unit='s').normalize()
    e_seg    = seg_arr[eb_idx]
    e_plat   = rng.choice(PLATFORMS, N_EXP, p=[0.66, 0.34])
    e_var    = rng.choice(['control','treatment'], N_EXP, p=[0.50, 0.50])

    e_dow_factor   = np.where(pd.DatetimeIndex(e_dates).dayofweek >= 5, 0.92, 1.0)
    e_month_factor = 1 + 0.08 * np.sin(2 * np.pi * pd.DatetimeIndex(e_dates).month / 12 - np.pi/6)
    e_ior_base = np.array([BASELINE_IOR[s] for s in e_seg])
    treatment_delta = np.array([
        GT_UPLIFT.get((e_seg[i], e_plat[i]), 0.0) if e_var[i] == 'treatment' else 0.0
        for i in range(N_EXP)
    ])
    e_ior = np.clip(e_ior_base * e_dow_factor * e_month_factor + treatment_delta
                    + rng.normal(0, 0.020, N_EXP), 0.01, 0.80)
    e_converted = rng.random(N_EXP) < e_ior
    e_gmv_per   = np.array([GMV_PER_ORDER[s] for s in e_seg])
    e_price_tier = rng.choice(['budget','mid','premium'], N_EXP, p=[0.30, 0.50, 0.20])
    e_price_mult = np.where(e_price_tier == 'budget', 0.55,
                    np.where(e_price_tier == 'premium', 1.85, 1.0))

    df_exp = pd.DataFrame({
        'inquiry_id':          [f'EXP{i:08d}' for i in range(1, N_EXP+1)],
        'buyer_id':            [buyer_ids[i] for i in eb_idx],
        'account_segment':     e_seg,
        'platform':            e_plat,
        'variant':             e_var,
        'category':            rng.choice(CATEGORIES, N_EXP, p=[0.34, 0.26, 0.19, 0.13, 0.08]),
        'country':             ctry_arr[eb_idx],
        'device_type':         rng.choice(['desktop','mobile','tablet'], N_EXP, p=[0.58, 0.32, 0.10]),
        'price_tier':          e_price_tier,
        'process_group':       rng.choice(
            ['Prototyping','Low-Volume Production','High-Volume Production','Repair'],
            N_EXP, p=[0.35, 0.34, 0.22, 0.09]),
        'has_billing_profile': rng.choice([True, False], N_EXP, p=[0.61, 0.39]),
        'created_at':          e_dates,
        'converted_to_order':  e_converted,
        'order_value':         np.where(e_converted,
                                         e_gmv_per * e_price_mult * rng.uniform(0.65, 1.5, N_EXP),
                                         0).round(2),
        'fulfillment_days':    np.where(e_converted, rng.integers(3, 35, N_EXP), np.nan),
        'period':              'experiment',
    })

    # ── PLATFORM TRAFFIC — daily sessions/signups/inquiries with seasonality ─────
    traffic_rows = []
    for d in pd.date_range(HIST_START, EXP_END, freq='D'):
        day_f   = 0.70 if d.dayofweek >= 5 else 1.0
        month_f = 1 + 0.08 * np.sin(2 * np.pi * d.month / 12 - np.pi/6)
        trend   = 1 + 0.00025 * (d - HIST_START).days
        for plat, base in [('web', 4600), ('mobile', 2100)]:
            n = max(int(base * day_f * month_f * trend * rng.normal(1.0, 0.06)), 80)
            traffic_rows.append({
                'date':           d.normalize(),
                'platform':       plat,
                'total_sessions': n,
                'signed_in':      int(n * rng.uniform(0.38, 0.52) * day_f),
                'new_signups':    int(n * rng.uniform(0.015, 0.035)),
                'inquiries':      int(n * rng.uniform(0.058, 0.088)),
            })
    df_traffic = pd.DataFrame(traffic_rows)

    # ── Register in DuckDB ────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────
    # BRONZE REGISTRATION
    # In production mode swap these db.register() calls with Snowflake loads:
    #   cursor.execute(f"SELECT * FROM {SOURCE_CONFIG['bronze_tables']['quotes']}")
    #   df_raw_quotes = cursor.fetch_pandas_all()
    #   db.register('bronze_quotes', df_raw_quotes)
    # ─────────────────────────────────────────────────────────────────────────────

db = duckdb.connect(':memory:')

if DATA_MODE == 'synthetic':
    db.register('bronze_buyers',    df_buyers)
    db.register('bronze_inquiries', df_hist_inquiries)
    db.register('bronze_exp',       df_exp)
    db.register('bronze_traffic',   df_traffic)
    bronze_gateway = None
    print('🥉 Bronze: synthetic data registered')
    print(f'   bronze_buyers: {len(df_buyers):,}  inquiries: {len(df_hist_inquiries):,}  '
          f'exp: {len(df_exp):,}  traffic: {len(df_traffic):,}')

elif DATA_MODE == 'csv':
    bronze_gateway = BronzeIngestionGateway(db, SOURCE_CONFIG)
    _required = list(BronzeIngestionGateway.ALLOWED_TABLES)
    _loaded, _missing = [], []
    import pandas as _pd
    for _alias, _cfg in BRONZE_CSV_SOURCES.items():
        try:
            _df = (_pd.read_csv(_cfg['path'])
                   if str(_cfg['path']).endswith('.csv')
                   else _pd.read_parquet(_cfg['path']))
            ok = bronze_gateway.register_etl_push(_alias, _df)
            if ok:
                _loaded.append(_alias)
        except Exception as _e:
            print(f'  ⚠️  Could not load CSV for "{_alias}": {_e}')
            _missing.append(_alias)
    for _alias in [t for t in _required if t not in _loaded and t not in _missing]:
        print(f'  ⚠️  No CSV registered for "{_alias}" — using empty placeholder.')
        _empty = _pd.DataFrame(columns=['_placeholder'])
        db.register(f'bronze_{_alias}', _empty)
    bronze_gateway.get_status()

elif DATA_MODE == 'production':
    print('  🔌 Production mode: opening Snowflake connection...')
    _sf_conn = get_snowflake_conn()
    if _sf_conn is None:
        print('  ❌ Snowflake connection failed. Check credentials in Cell 3')
        print('     (SOURCE_CONFIG[\'snowflake\'] or env vars).')
        print('     Falling back to synthetic data for this session.')
        db.register('bronze_buyers',    df_buyers)
        db.register('bronze_inquiries', df_hist_inquiries)
        db.register('bronze_exp',       df_exp)
        db.register('bronze_traffic',   df_traffic)
        bronze_gateway = None
    else:
        bronze_gateway = BronzeIngestionGateway(db, SOURCE_CONFIG)
        _pulled = bronze_gateway.pull_from_snowflake(
            _sf_conn,
            tables=sorted(BronzeIngestionGateway.ALLOWED_TABLES),
        )
        _sf_conn.close()
        print('  ✅ Snowflake connection closed.')
        bronze_gateway.get_status()

else:
    raise ValueError(f'Unknown DATA_MODE: {DATA_MODE!r}. Use: synthetic | csv | production')

print(f'\n  Time horizon: {HIST_START.date()} → {EXP_END.date()}')
print(f'  Experiment window: {EXP_START.date()} → {EXP_END.date()}')
if DATA_MODE == 'synthetic':
    print()
    print('  Ground truth treatment effects (IOR delta pp):')
    for (seg, plat), delta in GT_UPLIFT.items():
        print(f'     {seg:<12} {plat:<8}: {delta*100:+.1f}pp')


🥉 Bronze: synthetic data registered
   bronze_buyers: 25,000  inquiries: 180,000  exp: 90,000  traffic: 2,254

  Time horizon: 2022-08-01 → 2025-08-31
  Experiment window: 2025-01-01 → 2025-08-31

  Ground truth treatment effects (IOR delta pp):
     Core         web     : +1.4pp
     Core         mobile  : +1.0pp
     Growth       web     : -1.5pp
     Growth       mobile  : -1.2pp
     Enterprise   web     : +0.6pp
     Enterprise   mobile  : +0.4pp
     Individuals  web     : -0.4pp
     Individuals  mobile  : -0.2pp


## 5-Silver · Silver Layer — Clean + Standardize

**Runs in both synthetic and production mode.**

The Silver layer transforms Bronze tables into clean, canonical, unified DuckDB views. This is the *only* place in the stack that knows about raw column names and source inconsistencies.

| Silver view | Built from | Transforms applied |
|---|---|---|
| `silver_inquiries` | `bronze_inquiries` + orders join | Column rename, type cast, segment/platform map, dedup, filter |
| `silver_buyers` | `bronze_buyers` | Column rename, null coalesce, segment normalise |
| `silver_traffic` | `bronze_traffic` | Column rename, type cast, platform map |
| `silver_exp_inquiries` | `bronze_exp` | Canonicalise, type cast, period label |

Backward-compat aliases keep all 13 modules working unchanged:
`hist_inquiries` → `silver_inquiries`  ·  `buyers` → `silver_buyers`  ·  `platform_traffic` → `silver_traffic`

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5-SILVER — Silver Layer: Clean + Standardize
# ─────────────────────────────────────────────────────────────────────────────

_syn = USE_SYNTHETIC_DATA
_c   = SOURCE_CONFIG['column_map']
_bt  = SOURCE_CONFIG['bronze_tables']
_sm  = SOURCE_CONFIG.get('segment_map', {})
_pm  = SOURCE_CONFIG.get('platform_map', {})
_cancelled = SOURCE_CONFIG.get('cancelled_order_statuses', [])
_internal  = SOURCE_CONFIG.get('internal_domains', [])

print(f'  🥈 Silver Layer — {"Synthetic (canonicalise + type-cast)" if _syn else "Production (full Bronze→Silver transform)"}')

# ── Helpers for building production-mode CASE expressions ─────────────────────
def _seg_case(raw_col: str) -> str:
    if not _sm: return raw_col
    cases = ' '.join(f"WHEN {raw_col} = '{r}' THEN '{c}'" for r, c in _sm.items())
    return f'CASE {cases} ELSE {raw_col} END'

def _plat_case(raw_col: str) -> str:
    if not _pm: return raw_col
    cases = ' '.join(f"WHEN {raw_col} = '{r}' THEN '{c}'" for r, c in _pm.items())
    return f"CASE {cases} ELSE 'web' END"

def _c_(name: str) -> str:
    return _c.get(name, name)

# ─────────────────────────────────────────────────────────────────────────────
# SILVER VIEW: silver_inquiries
# ─────────────────────────────────────────────────────────────────────────────
if _syn:
    db.execute("""
        CREATE OR REPLACE VIEW silver_inquiries AS
            SELECT
                inquiry_id,
                buyer_id,
                account_segment,
                platform,
                category,
                country,
                CAST(created_at AS TIMESTAMP) AS created_at,
                CAST(converted_to_order AS BOOLEAN) AS converted_to_order,
                CAST(order_value AS DOUBLE) AS order_value,
                NULL::DOUBLE AS fulfillment_days,
                has_billing_profile,
                COALESCE(period, 'historical') AS period
            FROM bronze_inquiries
    """)
else:
    _Q = _bt.get('quotes', '')
    _O = _bt.get('orders', '')
    _cancelled_sql = ', '.join(f"'{s}'" for s in _cancelled)
    db.execute(f"""
        CREATE OR REPLACE VIEW silver_inquiries AS
        WITH quote_order_agg AS (
            SELECT
                q.{_c_('quote_id')}                         AS inquiry_id,
                q.{_c_('quote_user_id')}                    AS buyer_id,
                q.{_c_('quote_account_id')}                 AS account_id,
                CAST(q.{_c_('quote_created_at')} AS TIMESTAMP) AS created_at,
                {_plat_case(f"COALESCE(q.{_c_('quote_source')}, 'WEBAPP')")} AS platform,
                COALESCE(q.{_c_('quote_processes')}, 'Unknown')               AS category,
                CASE WHEN COUNT(
                    CASE WHEN o.{_c_('order_status')} NOT IN ({_cancelled_sql}) THEN 1 END
                ) > 0 THEN TRUE ELSE FALSE END              AS converted_to_order,
                COALESCE(SUM(
                    CASE WHEN o.{_c_('order_status')} NOT IN ({_cancelled_sql})
                    THEN CAST(o.{_c_('order_total')} AS DOUBLE) ELSE 0 END
                ), 0.0)                                     AS order_value,
                MIN(CASE WHEN o.{_c_('order_status')} NOT IN ({_cancelled_sql})
                    THEN DATEDIFF('day', q.{_c_('quote_created_at')}, o.{_c_('order_ship_date')})
                END)                                        AS fulfillment_days,
                ROW_NUMBER() OVER (
                    PARTITION BY q.{_c_('quote_id')}
                    ORDER BY q.{_c_('quote_created_at')} DESC
                ) AS rn
            FROM "{_Q}" q
            LEFT JOIN "{_O}" o ON o.{_c_('order_quote_id')} = q.{_c_('quote_id')}
            GROUP BY q.{_c_('quote_id')}, q.{_c_('quote_user_id')}, q.{_c_('quote_account_id')},
                     q.{_c_('quote_created_at')}, q.{_c_('quote_source')}, q.{_c_('quote_processes')}
        ),
        with_accounts AS (
            SELECT
                qo.inquiry_id, qo.buyer_id, qo.created_at, qo.platform,
                qo.category, qo.converted_to_order, qo.order_value, qo.fulfillment_days,
                {_seg_case(f"COALESCE(a.{_c_('account_segment')}, 'Unknown')")} AS account_segment,
                COALESCE(a.{_c_('account_country')}, 'Unknown')                 AS country
            FROM quote_order_agg qo
            LEFT JOIN "{_bt.get('accounts','')}" a ON a.{_c_('account_id')} = qo.account_id
            WHERE qo.rn = 1
        )
        SELECT *, FALSE AS has_billing_profile, 'historical' AS period
        FROM with_accounts
    """)

_n = db.execute('SELECT COUNT(*) FROM silver_inquiries').fetchone()[0]
print(f'     ✅ silver_inquiries      — {_n:>10,} rows')

# ─────────────────────────────────────────────────────────────────────────────
# SILVER VIEW: silver_buyers
# ─────────────────────────────────────────────────────────────────────────────
if _syn:
    db.execute("""
        CREATE OR REPLACE VIEW silver_buyers AS
        SELECT
            buyer_id,
            account_segment,
            primary_platform AS platform,
            country,
            industry,
            company_size,
            CAST(joined_at AS TIMESTAMP) AS joined_at,
            has_billing_profile,
            CAST(lifetime_orders AS INTEGER) AS lifetime_orders,
            CAST(lifetime_gmv    AS DOUBLE)  AS lifetime_gmv,
            is_active_30d
        FROM bronze_buyers
    """)
else:
    _U = _bt.get('users', '')
    _A = _bt.get('accounts', '')
    db.execute(f"""
        CREATE OR REPLACE VIEW silver_buyers AS
        SELECT
            u.{_c_('user_id')}           AS buyer_id,
            {_seg_case(f"COALESCE(a.{_c_('account_segment')}, 'Unknown')")} AS account_segment,
            'web'                         AS platform,
            COALESCE(a.{_c_('account_country')},  'Unknown') AS country,
            COALESCE(a.{_c_('account_vertical')}, 'Unknown') AS industry,
            NULL::VARCHAR                 AS company_size,
            CAST(u.{_c_('quote_created_at')} AS TIMESTAMP) AS joined_at,
            CASE WHEN u.{_c_('user_email_flag')} = 'COMPANY_EMAIL' THEN TRUE ELSE FALSE END AS has_billing_profile,
            0   AS lifetime_orders,
            0.0 AS lifetime_gmv,
            TRUE AS is_active_30d
        FROM "{_U}" u
        LEFT JOIN "{_A}" a ON a.{_c_('account_id')} = u.{_c_('user_account_id')}
    """)

_n = db.execute('SELECT COUNT(*) FROM silver_buyers').fetchone()[0]
print(f'     ✅ silver_buyers         — {_n:>10,} rows')

# ─────────────────────────────────────────────────────────────────────────────
# SILVER VIEW: silver_traffic
# ─────────────────────────────────────────────────────────────────────────────
if _syn:
    db.execute("""
        CREATE OR REPLACE VIEW silver_traffic AS
        SELECT
            CAST(date AS DATE)              AS date,
            platform,
            CAST(total_sessions AS INTEGER) AS total_sessions,
            CAST(signed_in      AS INTEGER) AS signed_in,
            CAST(new_signups    AS INTEGER) AS new_signups,
            CAST(inquiries      AS INTEGER) AS inquiries
        FROM bronze_traffic
    """)
else:
    _T = _bt.get('traffic', '')
    db.execute(f"""
        CREATE OR REPLACE VIEW silver_traffic AS
        SELECT
            CAST({_c_('traffic_date')}  AS DATE)             AS date,
            {_plat_case(_c_('platform') if _c_('platform') != 'platform' else "'web'")} AS platform,
            CAST(COALESCE({_c_('total_sessions')}, 0) AS INTEGER) AS total_sessions,
            CAST(COALESCE({_c_('signed_in')},      0) AS INTEGER) AS signed_in,
            CAST(COALESCE({_c_('new_signups')},     0) AS INTEGER) AS new_signups,
            CAST(COALESCE({_c_('inquiries_col')},   0) AS INTEGER) AS inquiries
        FROM "{_T}"
    """)

_n = db.execute('SELECT COUNT(*) FROM silver_traffic').fetchone()[0]
print(f'     ✅ silver_traffic        — {_n:>10,} rows')

# ─────────────────────────────────────────────────────────────────────────────
# SILVER VIEW: silver_exp_inquiries
# ─────────────────────────────────────────────────────────────────────────────
print(db.execute("DESCRIBE bronze_exp").fetchdf())
if _syn:
    db.execute("""
        CREATE OR REPLACE VIEW silver_exp_inquiries AS
        SELECT
            inquiry_id,
            buyer_id,
            account_segment,
            platform,
            variant,
            category,
            country,
            device_type,
            price_tier,
            process_group,
            has_billing_profile,
            CAST(created_at          AS TIMESTAMP) AS created_at,
            CAST(converted_to_order  AS BOOLEAN)   AS converted_to_order,
            CAST(order_value         AS DOUBLE)     AS order_value,
            CAST(fulfillment_days    AS DOUBLE)     AS fulfillment_days,
            COALESCE(period, 'experiment')          AS period
        FROM bronze_exp
    """)
else:
    _E = _bt.get('experiment', '')
    _internal_sql = ', '.join(f"'{d.lower()}'" for d in _internal)
    db.execute(f"""
        CREATE OR REPLACE VIEW silver_exp_inquiries AS
        SELECT
            i.*,
            e.{_c_('exp_group_name')}       AS variant,
            e.{_c_('exp_experiment_id')}    AS experiment_name
        FROM silver_inquiries i
        INNER JOIN "{_E}" e
            ON e.{_c_('exp_user_id')} = i.buyer_id
        WHERE LOWER(COALESCE(e.{_c_('exp_account_domain')}, ''))
              NOT IN ({_internal_sql})
    """)

_n = db.execute('SELECT COUNT(*) FROM silver_exp_inquiries').fetchone()[0]
print(f'     ✅ silver_exp_inquiries  — {_n:>10,} rows')

# ─────────────────────────────────────────────────────────────────────────────
# BACKWARD-COMPAT ALIASES
# ─────────────────────────────────────────────────────────────────────────────
db.execute('CREATE OR REPLACE VIEW hist_inquiries   AS SELECT * FROM silver_inquiries')
db.execute('CREATE OR REPLACE VIEW buyers           AS SELECT * FROM silver_buyers')
db.execute('CREATE OR REPLACE VIEW platform_traffic AS SELECT * FROM silver_traffic')

SOURCE_CONFIG['bronze_tables']['inquiries'] = 'silver_inquiries'
SOURCE_CONFIG['bronze_tables']['traffic']   = 'silver_traffic'
CLIENT_SCHEMA['tables'] = SOURCE_CONFIG['bronze_tables']

try:
    _refresh_runtime_constants()
except NameError:
    pass

print()
print('  ✅ Silver layer ready.')
print('     Canonical views : silver_inquiries · silver_buyers · silver_traffic · silver_exp_inquiries')
print('     Compat aliases  : hist_inquiries · buyers · platform_traffic')
print('     → Gold layer (Cell 6) builds analytics tables from Silver.')


  🥈 Silver Layer — Synthetic (canonicalise + type-cast)
     ✅ silver_inquiries      —    180,000 rows
     ✅ silver_buyers         —     25,000 rows
     ✅ silver_traffic        —      2,254 rows
            column_name   column_type null   key default extra
0            inquiry_id       VARCHAR  YES  None    None  None
1              buyer_id       VARCHAR  YES  None    None  None
2       account_segment       VARCHAR  YES  None    None  None
3              platform       VARCHAR  YES  None    None  None
4               variant       VARCHAR  YES  None    None  None
5              category       VARCHAR  YES  None    None  None
6               country       VARCHAR  YES  None    None  None
7           device_type       VARCHAR  YES  None    None  None
8            price_tier       VARCHAR  YES  None    None  None
9         process_group       VARCHAR  YES  None    None  None
10  has_billing_profile       BOOLEAN  YES  None    None  None
11           created_at  TIMESTAMP_NS  YES  Non

## 6 · Gold Layer — Experiment Registry & Analytics

The **Gold layer** is where the use case lives. Builds analytics-ready DuckDB tables from Silver for all four analysis types the platform supports.

| Gold table | Use case | Pre-computed for |
|---|---|---|
| `gold_experiment_analysis` | A/B readout, SRM, power checks | Pre/post labels, received_feature flag, DiD prep |
| `gold_pre_post_cohorts` | Difference-in-Differences, ITS | 6-month pre-period + experiment period, aligned by segment |
| `gold_daily_metrics` | Sequential testing, counterfactual | 24-month daily IOR time-series with shipped-feature lifts |
| `gold_dim_breakdowns` | Dimensional slicing, segment readouts | segment × platform × category aggregates |
| `gold_post_ship` | Post-ship ROI monitoring | 90-day post-experiment observations per shipped experiment |

Backward-compat aliases keep all modules working unchanged:
`all_experiments` · `platform_daily_ior` · `post_ship_observations` · `daily_experiment_stats`

In [12]:
# ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# EXPERIMENT REGISTRY — 10 experiments spanning all status types: running, concluded, stopped, shipped, not_started
# ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

EXPERIMENT_REGISTRY = [
    {
        'experiment_name':   'billing_profile_confirmation_v2',
        'description':       'Move billing profile step earlier in checkout funnel.',
        'hypothesis':        'Showing billing profile earlier reduces checkout anxiety and increases IOR.',
        'status':            'concluded',
        'primary_metric':    'converted_to_order',
        'guardrail_metrics': ['order_value', 'fulfillment_days'],
        'variants':          ['control', 'treatment'],
        'start_date':        '2025-02-14',
        'end_date':          '2025-06-30',
        'planned_days':      60,
        'team':              'Checkout',
        'ship_decision':     'partial_ship',
    },
    {
        'experiment_name':   'social_signin_v1',
        'description':       'Google / LinkedIn OAuth on sign-in page — 3-way test.',
        'hypothesis':        'Social login reduces sign-up friction and lifts activated-user rate.',
        'status':            'shipped',
        'primary_metric':    'converted_to_order',
        'guardrail_metrics': ['order_value'],
        'variants':          ['control', 'google_only', 'multi_provider'],
        'start_date':        '2025-01-07',
        'end_date':          '2025-03-31',
        'planned_days':      45,
        'team':              'Auth',
        'ship_decision':     'ship',
    },
    {
        'experiment_name':   'summary_page_cta_test',
        'description':       'Reorder CTA placement on order summary page to boost repeat orders.',
        'hypothesis':        'Prominent reorder CTA increases repeat order rate.',
        'status':            'running',
        'primary_metric':    'converted_to_order',
        'guardrail_metrics': ['order_value', 'fulfillment_days'],
        'variants':          ['control', 'treatment'],
        'start_date':        '2025-05-01',
        'end_date':          None,
        'planned_days':      56,
        'team':              'Growth',
        'ship_decision':     None,
    },
    {
        'experiment_name':   'pricing_display_4way',
        'description':       'Four variants of price display on quote page: bold, anchoring, urgency, control.',
        'hypothesis':        'Anchoring and urgency cues improve perceived value and lift IOR.',
        'status':            'concluded',
        'primary_metric':    'converted_to_order',
        'guardrail_metrics': ['order_value'],
        'variants':          ['control', 'bold_price', 'anchoring', 'urgency'],
        'start_date':        '2024-10-01',
        'end_date':          '2024-12-15',
        'planned_days':      75,
        'team':              'Pricing',
        'ship_decision':     'no_ship',
    },
    {
        'experiment_name':   'mobile_nav_redesign',
        'description':       'Bottom-tab mobile navigation replacing hamburger menu.',
        'hypothesis':        'Faster access to key sections lifts mobile IOR by 5-10%.',
        'status':            'running',
        'primary_metric':    'converted_to_order',
        'guardrail_metrics': ['order_value', 'fulfillment_days'],
        'variants':          ['control', 'treatment'],
        'start_date':        '2025-06-15',
        'end_date':          None,
        'planned_days':      45,
        'team':              'Mobile',
        'ship_decision':     None,
    },
    {
        'experiment_name':   'enterprise_bulk_upload',
        'description':       'Allow enterprise buyers to upload CAD file bundles in one action.',
        'hypothesis':        'Bulk upload reduces quote-time friction for enterprise buyers and lifts IOR.',
        'status':            'stopped',
        'primary_metric':    'converted_to_order',
        'guardrail_metrics': ['order_value', 'fulfillment_days'],
        'variants':          ['control', 'treatment'],
        'start_date':        '2025-03-10',
        'end_date':          '2025-04-02',
        'planned_days':      30,
        'team':              'Enterprise',
        'ship_decision':     'no_ship',
    },
    {
        'experiment_name':   'real_time_quote_preview',
        'description':       'Show live cost estimate as user configures part specs.',
        'hypothesis':        'Instant feedback reduces quote abandonment.',
        'status':            'shipped',
        'primary_metric':    'converted_to_order',
        'guardrail_metrics': ['order_value'],
        'variants':          ['control', 'treatment'],
        'start_date':        '2024-11-01',
        'end_date':          '2024-12-20',
        'planned_days':      45,
        'team':              'Quote',
        'ship_decision':     'ship',
    },
    {
        'experiment_name':   'personalised_recommendations_v1',
        'description':       'ML-driven recommended-parts carousel on home and summary pages.',
        'hypothesis':        'Personalised recommendations increase repeat orders and cross-category discovery.',
        'status':            'not_started',
        'primary_metric':    'converted_to_order',
        'guardrail_metrics': ['order_value'],
        'variants':          ['control', 'treatment'],
        'start_date':        '2025-09-15',
        'end_date':          None,
        'planned_days':      60,
        'team':              'Personalisation',
        'ship_decision':     None,
    },
    {
        'experiment_name':   'chat_support_prompt',
        'description':       'Proactive chat prompt if user hesitates >45s on quote page.',
        'hypothesis':        'Timely help reduces drop-off and improves IOR for hesitating users.',
        'status':            'concluded',
        'primary_metric':    'converted_to_order',
        'guardrail_metrics': ['order_value', 'fulfillment_days'],
        'variants':          ['control', 'treatment'],
        'start_date':        '2025-04-01',
        'end_date':          '2025-05-15',
        'planned_days':      45,
        'team':              'Support',
        'ship_decision':     'ship',
    },
    {
        'experiment_name':   'loyalty_tier_upgrade',
        'description':       'Surface loyalty-tier benefits prominently for near-threshold buyers.',
        'hypothesis':        'Tier visibility near the qualification threshold increases additional orders.',
        'status':            'not_started',
        'primary_metric':    'converted_to_order',
        'guardrail_metrics': ['order_value'],
        'variants':          ['control', 'treatment'],
        'start_date':        '2025-10-01',
        'end_date':          None,
        'planned_days':      45,
        'team':              'Loyalty',
        'ship_decision':     None,
    },
]


CONCURRENT_SHIPS = {
    'billing_profile_confirmation_v2': [
        {'name': 'social_signin_v1',      'ship_date': '2025-04-01',
         'known_ior_lift_pp': +0.012,  'overlaps_population': True},
        {'name': 'homepage_redesign_v1',  'ship_date': '2025-05-15',
         'known_ior_lift_pp': None,    'overlaps_population': True},
        {'name': 'quote_summary_refresh', 'ship_date': '2025-06-10',
         'known_ior_lift_pp': None,    'overlaps_population': False},
    ],
    'social_signin_v1': [
        {'name': 'onboarding_email_sequence', 'ship_date': '2025-02-20',
         'known_ior_lift_pp': +0.005,  'overlaps_population': True},
    ],
    'chat_support_prompt': [
        {'name': 'faq_redesign', 'ship_date': '2025-04-20',
         'known_ior_lift_pp': None, 'overlaps_population': True},
    ],
}


EXTRA_EXP_EFFECTS = {
    'social_signin_v1': {
        'google_only':     {'Core': +0.008, 'Growth': +0.012, 'Enterprise': +0.003, 'Individuals': +0.015},
        'multi_provider':  {'Core': +0.011, 'Growth': +0.016, 'Enterprise': +0.005, 'Individuals': +0.018},
    },
    'summary_page_cta_test': {
        'treatment':       {'Core': +0.004, 'Growth': -0.002, 'Enterprise': +0.006, 'Individuals': +0.001},
    },
    'pricing_display_4way': {
        'bold_price':      {'Core': +0.005, 'Growth': +0.008, 'Enterprise': -0.002, 'Individuals': +0.010},
        'anchoring':       {'Core': +0.012, 'Growth': +0.003, 'Enterprise': +0.015, 'Individuals': -0.003},
        'urgency':         {'Core': -0.006, 'Growth': +0.002, 'Enterprise': -0.008, 'Individuals': +0.004},
    },
    'mobile_nav_redesign': {
        'treatment':       {'Core': +0.007, 'Growth': +0.009, 'Enterprise': +0.003, 'Individuals': +0.006},
    },
    'enterprise_bulk_upload': {
        'treatment':       {'Core': -0.002, 'Growth': -0.004, 'Enterprise': +0.008, 'Individuals': -0.001},
    },
    'real_time_quote_preview': {
        'treatment':       {'Core': +0.018, 'Growth': +0.014, 'Enterprise': +0.009, 'Individuals': +0.012},
    },
    'chat_support_prompt': {
        'treatment':       {'Core': +0.006, 'Growth': +0.011, 'Enterprise': +0.002, 'Individuals': +0.013},
    },
    'personalised_recommendations_v1': {},   # not yet started
    'loyalty_tier_upgrade':            {},   # not yet started
}


extra_frames = []
for exp_info in EXPERIMENT_REGISTRY:
    exp_name = exp_info['experiment_name']
    if exp_name == 'billing_profile_confirmation_v2':
        continue     # headline experiment uses df_exp generated in cell 5
    if exp_info['status'] == 'not_started':
        continue     # not yet running — no data to generate

    variants = exp_info['variants']
    effects  = EXTRA_EXP_EFFECTS.get(exp_name, {})
    start    = pd.Timestamp(exp_info['start_date'])
    end      = pd.Timestamp(exp_info['end_date']) if exp_info['end_date'] else EXP_END

    n_rows = int(rng.integers(6_000, 12_000))

    eb       = rng.integers(0, N_BUYERS, n_rows)
    e_seg    = seg_arr[eb]
    e_plat   = rng.choice(PLATFORMS, n_rows, p=[0.66, 0.34])
    e_var    = rng.choice(variants, n_rows)
    e_dates  = pd.to_datetime(rng.integers(
        int(start.timestamp()), int(end.timestamp()), n_rows), unit='s').normalize()

    dow  = np.where(pd.DatetimeIndex(e_dates).dayofweek >= 5, 0.92, 1.0)
    mon  = 1 + 0.08 * np.sin(2 * np.pi * pd.DatetimeIndex(e_dates).month / 12 - np.pi/6)

    ior_base = np.array([BASELINE_IOR[s] for s in e_seg])
    delta    = np.array([effects.get(e_var[i], {}).get(e_seg[i], 0.0)
                          for i in range(n_rows)])
    ior      = np.clip(ior_base * dow * mon + delta + rng.normal(0, 0.020, n_rows),
                       0.01, 0.80)
    converted = rng.random(n_rows) < ior
    gmv_mean  = np.array([GMV_PER_ORDER[s] for s in e_seg])
    price_tier = rng.choice(['budget','mid','premium'], n_rows, p=[0.30, 0.50, 0.20])
    price_mul  = np.where(price_tier == 'budget', 0.55,
                  np.where(price_tier == 'premium', 1.85, 1.0))

    df_extra = pd.DataFrame({
        'inquiry_id':          [f'{exp_name[:4].upper()}{i:07d}' for i in range(n_rows)],
        'buyer_id':            [buyer_ids[i] for i in eb],
        'account_segment':     e_seg,
        'platform':            e_plat,
        'variant':             e_var,
        'experiment_name':     exp_name,
        'category':            rng.choice(CATEGORIES, n_rows, p=[0.34, 0.26, 0.19, 0.13, 0.08]),
        'country':             ctry_arr[eb],
        'device_type':         rng.choice(['desktop','mobile','tablet'], n_rows, p=[0.58, 0.32, 0.10]),
        'price_tier':          price_tier,
        'process_group':       rng.choice(
            ['Prototyping','Low-Volume Production','High-Volume Production','Repair'],
            n_rows, p=[0.35, 0.34, 0.22, 0.09]),
        'has_billing_profile': rng.choice([True, False], n_rows, p=[0.61, 0.39]),
        'created_at':          e_dates,
        'converted_to_order':  converted,
        'order_value':         np.where(converted,
                                         gmv_mean * price_mul * rng.uniform(0.65, 1.5, n_rows),
                                         0).round(2),
        'fulfillment_days':    np.where(converted, rng.integers(3, 35, n_rows), np.nan),
        'period':              'experiment',
    })
    extra_frames.append(df_extra)
    print(f'  Generated {len(df_extra):>6,} rows for {exp_name:<36} ({len(variants)} variants)')

df_exp['experiment_name'] = 'billing_profile_confirmation_v2'
df_all_experiments = pd.concat([df_exp] + extra_frames, ignore_index=True)

# ─────────────────────────────────────────────────────────────────────────────
# BRONZE REGISTRATION (experiment data)
# ─────────────────────────────────────────────────────────────────────────────
db.register('bronze_exp_all', df_all_experiments)
db.register('experiment_registry', pd.DataFrame(EXPERIMENT_REGISTRY))

# ─────────────────────────────────────────────────────────────────────────────
# GOLD VIEW: gold_experiment_analysis
# ─────────────────────────────────────────────────────────────────────────────
SHIPPED_TO_SEGMENTS = {'billing_profile_confirmation_v2': ['Core', 'Enterprise']}
_shipped_sql = {k: ', '.join(f"'{s}'" for s in v) for k, v in SHIPPED_TO_SEGMENTS.items()}
_exp_start_map = {e['experiment_name']: e['start_date'] for e in EXPERIMENT_REGISTRY}

db.execute("""
    CREATE OR REPLACE VIEW gold_experiment_analysis AS
    SELECT
        *,
        CASE WHEN created_at < TIMESTAMP '2025-02-14' THEN 'pre' ELSE 'post' END AS period_label,
        CASE
            WHEN experiment_name = 'billing_profile_confirmation_v2'
             AND account_segment IN ('Core', 'Enterprise') THEN TRUE
            ELSE FALSE
        END AS received_feature
    FROM bronze_exp_all
""")

_n = db.execute('SELECT COUNT(*) FROM gold_experiment_analysis').fetchone()[0]
print(f'     ✅ gold_experiment_analysis — {_n:>10,} rows  (pre/post + received_feature)')

# ─────────────────────────────────────────────────────────────────────────────
# GOLD VIEW: gold_pre_post_cohorts
# ─────────────────────────────────────────────────────────────────────────────
db.execute("""
    CREATE OR REPLACE VIEW gold_pre_post_cohorts AS
    -- Pre-period: 6 months before the headline experiment started
    SELECT
        inquiry_id, buyer_id, account_segment, platform, category, country,
        created_at, converted_to_order, order_value,
        'pre'           AS period_label,
        'pre_period'    AS variant,
        'billing_profile_confirmation_v2' AS experiment_name,
        account_segment IN ('Core', 'Enterprise') AS received_feature
    FROM silver_inquiries
    WHERE created_at >= TIMESTAMP '2024-08-14'
      AND created_at <  TIMESTAMP '2025-02-14'

    UNION ALL

    -- Post-period: treatment arm of the headline experiment
    SELECT
        inquiry_id, buyer_id, account_segment, platform, category, country,
        created_at, converted_to_order, order_value,
        'post'          AS period_label,
        variant,
        experiment_name,
        received_feature
    FROM gold_experiment_analysis
    WHERE experiment_name = 'billing_profile_confirmation_v2'
""")

_n = db.execute('SELECT COUNT(*) FROM gold_pre_post_cohorts').fetchone()[0]
print(f'     ✅ gold_pre_post_cohorts   — {_n:>10,} rows  (DiD pre+post combined)')

# ─────────────────────────────────────────────────────────────────────────────
# GOLD VIEW: gold_dim_breakdowns
# ─────────────────────────────────────────────────────────────────────────────
db.execute("""
    CREATE OR REPLACE VIEW gold_dim_breakdowns AS
    SELECT
        account_segment,
        platform,
        category,
        DATE_TRUNC('month', created_at)             AS period_month,
        COUNT(DISTINCT inquiry_id)                  AS n_inquiries,
        COUNT(DISTINCT buyer_id)                    AS n_buyers,
        AVG(CAST(converted_to_order AS DOUBLE))     AS ior,
        SUM(CASE WHEN converted_to_order THEN order_value ELSE 0 END) AS total_gmv,
        AVG(CASE WHEN converted_to_order THEN order_value END)        AS avg_order_value,
        STDDEV(CAST(converted_to_order AS DOUBLE))  AS ior_stddev
    FROM silver_inquiries
    GROUP BY account_segment, platform, category, DATE_TRUNC('month', created_at)
""")

_n = db.execute('SELECT COUNT(*) FROM gold_dim_breakdowns').fetchone()[0]
print(f'     ✅ gold_dim_breakdowns     — {_n:>10,} rows  (segment × platform × category × month)')

# ─────────────────────────────────────────────────────────────────────────────
# DAILY TIME-SERIES (generated as bronze data, exposed via Gold views)
# ─────────────────────────────────────────────────────────────────────────────
daily_rows = []
for exp_info in EXPERIMENT_REGISTRY:
    exp_name = exp_info['experiment_name']
    start  = pd.Timestamp(exp_info['start_date'])
    end    = pd.Timestamp(exp_info['end_date']) if exp_info['end_date'] else EXP_END
    exp_df = df_all_experiments[df_all_experiments['experiment_name'] == exp_name].copy()
    exp_df['day'] = (exp_df['created_at'] - start).dt.days
    for day_num in sorted(exp_df['day'].unique()):
        day_df = exp_df[exp_df['day'] <= day_num]
        for var in exp_info['variants']:
            var_df = day_df[day_df['variant'] == var]
            if len(var_df) < 5: continue
            daily_rows.append({
                'experiment_name':  exp_name,
                'day_number':       int(day_num),
                'date':             start + pd.Timedelta(days=int(day_num)),
                'variant':          var,
                'n_users':          len(var_df),
                'n_converted':      int(var_df['converted_to_order'].sum()),
                'ior':              float(var_df['converted_to_order'].mean()),
                'avg_order_value':  float(var_df['order_value'].mean()),
                'total_gmv':        float(var_df['order_value'].sum()),
            })
df_daily = pd.DataFrame(daily_rows)
db.register('daily_experiment_stats', df_daily)

# ── EXTENDED HISTORICAL TIME-SERIES (needed for counterfactual forecasting) ───

hist_ts_rows = []
hist_ts_start = pd.Timestamp('2023-01-01')   # 2 full years before experiment end
hist_ts_end   = pd.Timestamp('2025-09-28')   # 90 days post-ship (billing_profile end=2025-06-30)

for d in pd.date_range(hist_ts_start, hist_ts_end, freq='D'):
    days_since_start = (d - hist_ts_start).days
    trend = 0.00005 * days_since_start        # slow upward drift

    # Weekly pattern (Mon=1.0, Fri=0.95, Sat=0.72, Sun=0.68)
    weekly = [1.00, 0.99, 0.98, 0.97, 0.95, 0.72, 0.68][d.dayofweek]

    monthly_effect = {
        1:+0.003, 2:+0.001, 3:+0.004, 4:+0.002, 5:+0.001, 6: 0.000,
        7:-0.004, 8:-0.008, 9:-0.002, 10:+0.006, 11:+0.009, 12:+0.007
    }[d.month]

    base_ior = 0.183 + trend + monthly_effect

    concurrent_lift = 0.0
    for exp_info in EXPERIMENT_REGISTRY:
        exp_start = pd.Timestamp(exp_info['start_date'])
        exp_end   = pd.Timestamp(exp_info['end_date']) if exp_info['end_date'] else hist_ts_end
        if exp_start <= d <= exp_end and exp_info['ship_decision'] not in ('no_ship', None):
            # This experiment was running — add a partial effect (50% of treatment reach)
            if exp_info['experiment_name'] == 'billing_profile_confirmation_v2':
                concurrent_lift += 0.006   # partial ship lift
            elif exp_info['experiment_name'] == 'social_signin_v1':
                concurrent_lift += 0.010   # full ship lift

    for exp_info in EXPERIMENT_REGISTRY:
        if exp_info['ship_decision'] in ('no_ship', None, 'running'): continue
        ship_date = pd.Timestamp(exp_info['end_date']) if exp_info['end_date'] else None
        if ship_date and d > ship_date:
            if exp_info['experiment_name'] == 'billing_profile_confirmation_v2':
                concurrent_lift += 0.005 * max(0.6, 1 - (d-ship_date).days * 0.002)
            elif exp_info['experiment_name'] == 'social_signin_v1':
                concurrent_lift += 0.008 * max(0.7, 1 - (d-ship_date).days * 0.0015)

    noise = float(rng.normal(0, 0.006))
    daily_ior = float(np.clip(base_ior * weekly + concurrent_lift + noise, 0.05, 0.60))

    is_weekend = d.dayofweek >= 5
    daily_inq = int(max(50, rng.normal(480 if is_weekend else 720, 80)))
    daily_conv = int(daily_inq * daily_ior)
    avg_aov = float(rng.normal(4900, 600))

    hist_ts_rows.append({
        'date':           d,
        'ior':            round(daily_ior, 5),
        'n_inquiries':    daily_inq,
        'n_orders':       daily_conv,
        'gmv':            round(daily_conv * avg_aov, 0),
        'avg_order_value':round(avg_aov, 0),
        'day_of_week':    d.dayofweek,
        'month':          d.month,
        'quarter':        d.quarter,
        'is_weekend':     is_weekend,
        'days_since_start': days_since_start,
    })

df_hist_ts = pd.DataFrame(hist_ts_rows)
post_ship_rows = []
for exp_info in EXPERIMENT_REGISTRY:
    if exp_info['status'] != 'concluded' or exp_info['ship_decision'] == 'no_ship':
        continue
    exp_name = exp_info['experiment_name']
    end_date = pd.Timestamp(exp_info['end_date'])
    exp_df_  = df_all_experiments[df_all_experiments['experiment_name'] == exp_name]
    ctrl_ior = exp_df_[exp_df_['variant']=='control']['converted_to_order'].mean()
    best_var = exp_info['variants'][1]
    treat_ior= exp_df_[exp_df_['variant']==best_var]['converted_to_order'].mean()
    ship_lift= treat_ior - ctrl_ior
    for day in range(1, 91):
        obs_date = end_date + pd.Timedelta(days=day)
        obs_row  = df_hist_ts[df_hist_ts['date'] == obs_date]
        if len(obs_row) > 0:
            observed_ior = float(obs_row.iloc[0]['ior'])
        else:
            decay = max(0.5, 1 - day * 0.003)
            observed_ior = float(np.clip(ctrl_ior + ship_lift*decay + rng.normal(0,0.004), 0.01,0.99))
        post_ship_rows.append({
            'experiment_name':    exp_name,
            'day_post_ship':      day,
            'observation_date':   obs_date,
            'observed_ior':       round(observed_ior, 5),
            'baseline_ior':       round(ctrl_ior, 5),
            'expected_ior':       round(ctrl_ior + ship_lift, 5),
            'ior_delta_naive':    round(observed_ior - ctrl_ior, 5),
            'daily_gmv':          round(observed_ior * 500 * float(rng.normal(4900,600)), 0),
            'concurrent_ships':   [c['name'] for c in CONCURRENT_SHIPS.get(exp_name,[])
                                   if pd.Timestamp(c['ship_date']) <= obs_date],
        })

df_post_ship = pd.DataFrame(post_ship_rows)
db.register('post_ship_observations', df_post_ship)

# ── LEARNINGS REPOSITORY ──────────────────────────────────────────────────────
LEARNINGS_STORE = [
    {'id':'L001','experiment_name':'billing_profile_confirmation_v2',
     'ship_decision':'partial_ship',
     'outcome':'Core +1.4pp IOR (sig), Growth -1.5pp (sig). Net positive only for Core & Enterprise.',
     'key_learning':'Billing profile friction hurts high-intent Core buyers most.',
     'what_worked':'Earlier billing prompt for buyers with saved profiles.',
     'what_didnt':'New buyers abandoned at higher rates.',
     'recommendation':'Ship to Core and Enterprise. Hold for Growth.',
     'follow_up_experiments':['billing_profile_v3_new_buyers','checkout_guest_mode_test'],
     'recorded_by':'Analytics Team','recorded_at':'2025-07-05',
     'tags':['checkout','billing','ior','segmented_rollout']},
    {'id':'L002','experiment_name':'social_signin_v1',
     'ship_decision':'ship',
     'outcome':'Multi-provider +1.6pp IOR for Growth, +1.1pp for Core.',
     'key_learning':'Social login disproportionately benefits Growth segment.',
     'what_worked':'LinkedIn OAuth — reduces sign-up time by ~65%.',
     'what_didnt':'Mobile-specific benefit was smaller than expected.',
     'recommendation':'Ship multi-provider. Add Apple Sign-In in v2 for mobile.',
     'follow_up_experiments':['social_signin_v2_apple','onboarding_flow_redesign'],
     'recorded_by':'Analytics Team','recorded_at':'2025-04-10',
     'tags':['auth','signup','acquisition','mobile']},
    {'id':'L003','experiment_name':'pricing_display_4way',
     'ship_decision':'no_ship',
     'outcome':'Anchoring positive for Enterprise (+1.5pp) but negative for Growth (-0.8pp). Urgency negative.',
     'key_learning':'Urgency tactics hurt trust with B2B buyers.',
     'what_worked':'Price anchoring for high-AOV Enterprise.',
     'what_didnt':'Urgency backfired across all segments.',
     'recommendation':'Never use urgency for B2B. Test anchoring for Enterprise only.',
     'follow_up_experiments':['enterprise_price_anchoring_v2','pricing_transparency_test'],
     'recorded_by':'Analytics Team','recorded_at':'2025-01-10',
     'tags':['pricing','ior','enterprise','b2b_psychology']},
]
df_learnings = pd.DataFrame(LEARNINGS_STORE)
db.register('experiment_learnings', df_learnings)

# ─────────────────────────────────────────────────────────────────────────────
# GOLD VIEW: gold_daily_metrics
# ─────────────────────────────────────────────────────────────────────────────
db.register('_bronze_daily_ior', df_hist_ts)
db.execute("""
    CREATE OR REPLACE VIEW gold_daily_metrics AS
    SELECT
        date,
        ior,
        n_inquiries,
        n_orders,
        gmv,
        avg_order_value,
        day_of_week,
        month,
        quarter,
        is_weekend,
        days_since_start,
        -- Statistical annotation columns for sequential testing
        LOG(ior / NULLIF(1 - ior, 0)) AS log_odds,
        ior - LAG(ior, 7) OVER (ORDER BY date) AS ior_wow_delta
    FROM _bronze_daily_ior
""")
_n = db.execute('SELECT COUNT(*) FROM gold_daily_metrics').fetchone()[0]
print(f'     ✅ gold_daily_metrics      — {_n:>10,} rows  (24-month daily IOR time-series)')

# ─────────────────────────────────────────────────────────────────────────────
# GOLD VIEW: gold_post_ship
# ─────────────────────────────────────────────────────────────────────────────
db.register('_bronze_post_ship', df_post_ship)
db.execute("""
    CREATE OR REPLACE VIEW gold_post_ship AS
    SELECT
        experiment_name,
        day_post_ship,
        observation_date,
        observed_ior,
        baseline_ior,
        expected_ior,
        ior_delta_naive,
        daily_gmv,
        concurrent_ships,
        -- Derived: % of expected lift retained on this day
        CASE WHEN (expected_ior - baseline_ior) > 0
             THEN ROUND((observed_ior - baseline_ior) / (expected_ior - baseline_ior) * 100, 1)
             ELSE NULL END AS pct_lift_retained
    FROM _bronze_post_ship
""")
_n = db.execute('SELECT COUNT(*) FROM gold_post_ship').fetchone()[0]
print(f'     ✅ gold_post_ship          — {_n:>10,} rows  (90-day post-ship observations)')

# ─────────────────────────────────────────────────────────────────────────────
# DAILY EXPERIMENT STATS (cumulative by day — used by sequential testing module)
# ─────────────────────────────────────────────────────────────────────────────
db.register('daily_experiment_stats', df_daily)
_n = db.execute('SELECT COUNT(*) FROM daily_experiment_stats').fetchone()[0]
print(f'     ✅ daily_experiment_stats  — {_n:>10,} rows  (cumulative per-variant per-day)')

# ─────────────────────────────────────────────────────────────────────────────
# BACKWARD-COMPAT ALIASES
# ─────────────────────────────────────────────────────────────────────────────
db.execute('CREATE OR REPLACE VIEW all_experiments       AS SELECT * FROM gold_experiment_analysis')
db.execute('CREATE OR REPLACE VIEW platform_daily_ior   AS SELECT * FROM gold_daily_metrics')
db.execute('CREATE OR REPLACE VIEW post_ship_observations AS SELECT * FROM gold_post_ship')

# PSM + DiD ancillary tables (kept as registered DataFrames — used by causal modules)
df_did_ready = pd.concat([
    df_hist_inquiries.assign(
        period_label='pre',
        experiment_name='billing_profile_confirmation_v2',
        received_feature=df_hist_inquiries['account_segment'].isin(['Core', 'Enterprise']),
        variant='pre_period',
    )[df_hist_inquiries['created_at'] >= pd.Timestamp('2024-08-14')],
    df_all_experiments[
        (df_all_experiments['experiment_name'] == 'billing_profile_confirmation_v2') &
        (df_all_experiments['variant'] == 'treatment')
    ].assign(period_label='post', received_feature=True),
], ignore_index=True)

df_psm_features = df_buyers.copy()
df_psm_features['has_orders']   = (df_psm_features['lifetime_orders'] > 0).astype(int)
df_psm_features['is_us']        = (df_psm_features['country'] == 'US').astype(int)
df_psm_features['is_web']       = (df_psm_features['primary_platform'] == 'web').astype(int)
df_psm_features['high_gmv']     = (df_psm_features['lifetime_gmv'] > df_psm_features['lifetime_gmv'].median()).astype(int)
df_psm_features['segment_num']  = pd.Categorical(df_psm_features['account_segment']).codes
df_hist_inquiries_labeled = df_hist_inquiries.assign(
    period_label='pre',
    experiment_name='billing_profile_confirmation_v2',
    received_feature=df_hist_inquiries['account_segment'].isin(['Core', 'Enterprise']),
    variant='pre_period',
)

db.register('did_ready',    df_did_ready)
db.register('psm_features', df_psm_features)
db.register('hist_labeled', df_hist_inquiries_labeled)

print()
print('✅ Gold layer ready.')
print('   Gold tables  : gold_experiment_analysis · gold_pre_post_cohorts')
print('                  gold_daily_metrics · gold_dim_breakdowns · gold_post_ship')
print('   Compat views : all_experiments · platform_daily_ior · post_ship_observations')
print(f'   Experiments  : {len(EXPERIMENT_REGISTRY)} in EXPERIMENT_REGISTRY')
print(f'   Learnings    : {len(LEARNINGS_STORE)} entries in LEARNINGS_STORE')


  Generated  7,591 rows for social_signin_v1                     (3 variants)
  Generated 10,565 rows for summary_page_cta_test                (2 variants)
  Generated 10,164 rows for pricing_display_4way                 (4 variants)
  Generated 10,839 rows for mobile_nav_redesign                  (2 variants)
  Generated  6,156 rows for enterprise_bulk_upload               (2 variants)
  Generated  7,686 rows for real_time_quote_preview              (2 variants)
  Generated 11,885 rows for chat_support_prompt                  (2 variants)
     ✅ gold_experiment_analysis —    154,886 rows  (pre/post + received_feature)
     ✅ gold_pre_post_cohorts   —    118,161 rows  (DiD pre+post combined)
     ✅ gold_dim_breakdowns     —      1,160 rows  (segment × platform × category × month)
     ✅ gold_daily_metrics      —      1,002 rows  (24-month daily IOR time-series)
     ✅ gold_post_ship          —        180 rows  (90-day post-ship observations)
     ✅ daily_experiment_stats  —      1,663 

## 6-Auto · Gold-Auto — Production Analytics

**Active when:** `DATA_PATH == 'production'` (set by Cell 3-Auto)

In production mode this cell:
1. Loads `EXPERIMENT_REGISTRY` from the live experiments table (Statsig / LaunchDarkly / custom)
2. Builds `gold_experiment_analysis` from `silver_exp_inquiries` (which joins Silver inquiries × experiment assignments)
3. Materialises all Gold views (`gold_pre_post_cohorts`, `gold_dim_breakdowns`, `gold_daily_metrics`, `gold_post_ship`) from Silver data
4. Registers backward-compat aliases so all modules continue to work unchanged

In synthetic mode this is a clean no-op — Cell 6 handles Gold materialisation.

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6-AUTO — Gold-Auto: Production Analytics
# ─────────────────────────────────────────────────────────────────────────────

_data_path = globals().get('DATA_PATH', 'synthetic')

if _data_path != 'production':
    print('  ℹ️  Cell 6-Auto: DATA_PATH is not production — Cell 6 Gold layer is active.')

else:
    print('  🔧 Cell 6-Auto: building Gold layer from Silver (production mode)...')

    _t = SOURCE_CONFIG['bronze_tables']
    _col_ = lambda n: SOURCE_CONFIG['column_map'].get(n, n)

    _exp_src = _t.get('experiments', '')
    _loaded_registry = False

    if _exp_src:
        try:
            _exp_meta = db.execute(f"""
                SELECT
                    {_col_('exp_experiment_id')}     AS experiment_name,
                    COUNT(DISTINCT {_col_('exp_user_id')})   AS n_users,
                    STRING_AGG(DISTINCT {_col_('exp_group_name')}, ' | ') AS variants,
                    MIN({_col_('exp_timestamp')})::DATE       AS start_date,
                    MAX({_col_('exp_timestamp')})::DATE       AS end_date
                FROM "{_exp_src}"
                GROUP BY {_col_('exp_experiment_id')}
                ORDER BY start_date DESC
            """).df()

            _existing = {e['experiment_name']: e for e in globals().get('EXPERIMENT_REGISTRY', [])}
            EXPERIMENT_REGISTRY = []
            for _, row in _exp_meta.iterrows():
                name = str(row['experiment_name'])
                base = _existing.get(name, {})
                EXPERIMENT_REGISTRY.append({
                    'experiment_name':   name,
                    'description':       base.get('description', f'Live experiment: {name}'),
                    'hypothesis':        base.get('hypothesis', ''),
                    'status':            base.get('status', 'running'),
                    'primary_metric':    base.get('primary_metric', 'converted_to_order'),
                    'guardrail_metrics': base.get('guardrail_metrics', ['order_value']),
                    'variants':          str(row['variants']).split(' | '),
                    'start_date':        str(row['start_date']),
                    'end_date':          str(row['end_date']),
                    'planned_days':      base.get('planned_days', None),
                    'team':              base.get('team', 'Unknown'),
                    'ship_decision':     base.get('ship_decision', None),
                })
            db.register('experiment_registry', pd.DataFrame(EXPERIMENT_REGISTRY))
            _loaded_registry = True
            print(f'     ✅ EXPERIMENT_REGISTRY — {len(EXPERIMENT_REGISTRY)} experiments from live table')
        except Exception as e:
            print(f'     ⚠️  Could not load experiment registry: {e}')

    if not _loaded_registry:
        print('     ℹ️  Using hardcoded EXPERIMENT_REGISTRY from Cell 6')

    # ── GOLD: gold_experiment_analysis ───────────────────────────────────────
    try:
        _shipped_to = {'billing_profile_confirmation_v2': "('Core', 'Enterprise')"}
        _shipped_sql = ' '.join(
            f"WHEN experiment_name = '{k}' AND account_segment IN {v} THEN TRUE"
            for k, v in _shipped_to.items()
        )
        db.execute(f"""
            CREATE OR REPLACE VIEW gold_experiment_analysis AS
            SELECT
                *,
                CASE
                    WHEN created_at < (
                        SELECT MIN(CAST(start_date AS TIMESTAMP))
                        FROM experiment_registry
                        WHERE experiment_name = silver_exp_inquiries.experiment_name
                    ) THEN 'pre' ELSE 'post'
                END AS period_label,
                CASE {_shipped_sql} ELSE FALSE END AS received_feature
            FROM silver_exp_inquiries
        """)
        _n = db.execute('SELECT COUNT(*) FROM gold_experiment_analysis').fetchone()[0]
        print(f'     ✅ gold_experiment_analysis  — {_n:>10,} rows')
    except Exception as e:
        print(f'     ⚠️  gold_experiment_analysis failed: {e}')

    # ── GOLD: gold_pre_post_cohorts ───────────────────────────────────────────
    try:
        db.execute("""
            CREATE OR REPLACE VIEW gold_pre_post_cohorts AS
            SELECT
                inquiry_id, buyer_id, account_segment, platform, category, country,
                created_at, converted_to_order, order_value,
                'pre' AS period_label, 'pre_period' AS variant,
                (SELECT experiment_name FROM experiment_registry
                 WHERE status = 'concluded' LIMIT 1) AS experiment_name,
                account_segment IN ('Core', 'Enterprise') AS received_feature
            FROM silver_inquiries
            WHERE created_at >= (CURRENT_TIMESTAMP - INTERVAL '9 months')
              AND created_at <  (
                  SELECT MIN(CAST(start_date AS TIMESTAMP))
                  FROM experiment_registry
                  WHERE status = 'concluded'
                  LIMIT 1)

            UNION ALL

            SELECT
                inquiry_id, buyer_id, account_segment, platform, category, country,
                created_at, converted_to_order, order_value,
                period_label, variant, experiment_name, received_feature
            FROM gold_experiment_analysis
        """)
        _n = db.execute('SELECT COUNT(*) FROM gold_pre_post_cohorts').fetchone()[0]
        print(f'     ✅ gold_pre_post_cohorts     — {_n:>10,} rows')
    except Exception as e:
        print(f'     ⚠️  gold_pre_post_cohorts failed: {e}')

    # ── GOLD: gold_dim_breakdowns ─────────────────────────────────────────────
    try:
        db.execute("""
            CREATE OR REPLACE VIEW gold_dim_breakdowns AS
            SELECT
                account_segment, platform, category,
                DATE_TRUNC('month', created_at) AS period_month,
                COUNT(DISTINCT inquiry_id)      AS n_inquiries,
                COUNT(DISTINCT buyer_id)        AS n_buyers,
                AVG(CAST(converted_to_order AS DOUBLE)) AS ior,
                SUM(CASE WHEN converted_to_order THEN order_value ELSE 0 END) AS total_gmv,
                AVG(CASE WHEN converted_to_order THEN order_value END) AS avg_order_value,
                STDDEV(CAST(converted_to_order AS DOUBLE)) AS ior_stddev
            FROM silver_inquiries
            GROUP BY account_segment, platform, category, DATE_TRUNC('month', created_at)
        """)
        _n = db.execute('SELECT COUNT(*) FROM gold_dim_breakdowns').fetchone()[0]
        print(f'     ✅ gold_dim_breakdowns        — {_n:>10,} rows')
    except Exception as e:
        print(f'     ⚠️  gold_dim_breakdowns failed: {e}')

    # ── BACKWARD-COMPAT ALIASES ───────────────────────────────────────────────
    try:
        db.execute('CREATE OR REPLACE VIEW all_experiments       AS SELECT * FROM gold_experiment_analysis')
        db.execute('CREATE OR REPLACE VIEW platform_daily_ior   AS SELECT * FROM gold_daily_metrics')   # if Cell 6 ran
        db.execute('CREATE OR REPLACE VIEW post_ship_observations AS SELECT * FROM gold_post_ship')     # if Cell 6 ran
    except Exception as _e:
        pass  

    print()
    print('  ✅ Gold-Auto complete. Modules ready for production data.')


  ℹ️  Cell 6-Auto: DATA_PATH is not production — Cell 6 Gold layer is active.


## 7 · Statistical Engine

All statistical computation — z-tests, t-tests, power, DiD, ITS, PSM, RDD, mediation, synthetic control, mSPRT. Pure scipy/numpy. Identical across clients.

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# PROPORTION TEST
# ─────────────────────────────────────────────────────────────────────────────
def proportion_test(
    n_ctrl: int, conv_ctrl: int,
    n_treat: int, conv_treat: int,
    alpha: float = 0.05,
) -> dict:
    """Two-proportion z-test. Returns full result dict."""
    p_c = conv_ctrl  / n_ctrl  if n_ctrl  > 0 else 0
    p_t = conv_treat / n_treat if n_treat > 0 else 0
    delta = p_t - p_c
    rel   = delta / p_c if p_c > 0 else 0

    # Pooled proportion
    p_pool = (conv_ctrl + conv_treat) / (n_ctrl + n_treat)
    se     = np.sqrt(p_pool * (1 - p_pool) * (1/n_ctrl + 1/n_treat)) if (n_ctrl+n_treat) > 0 else 1
    z_stat = delta / se if se > 0 else 0
    p_val  = 2 * norm.sf(abs(z_stat))  # two-tailed

    # 95% CI on the delta
    se_delta = np.sqrt(p_c*(1-p_c)/n_ctrl + p_t*(1-p_t)/n_treat) if (n_ctrl*n_treat) > 0 else 0
    z_crit   = norm.ppf(1 - alpha/2)
    ci_lo    = delta - z_crit * se_delta
    ci_hi    = delta + z_crit * se_delta

    # Cohen's h effect size
    h = 2 * np.arcsin(np.sqrt(p_t)) - 2 * np.arcsin(np.sqrt(p_c))

    return {
        'n_control': n_ctrl, 'n_treatment': n_treat,
        'conv_control': conv_ctrl, 'conv_treatment': conv_treat,
        'rate_control': round(p_c, 6), 'rate_treatment': round(p_t, 6),
        'delta_abs': round(delta, 6), 'delta_rel': round(rel, 6),
        'delta_pp': round(delta * 100, 4),
        'z_stat': round(z_stat, 4), 'p_value': round(p_val, 6),
        'ci_lo_pp': round(ci_lo * 100, 4), 'ci_hi_pp': round(ci_hi * 100, 4),
        'effect_size_h': round(h, 4),
        'is_significant': p_val < alpha,
        'direction': 'positive' if delta > 0 else 'negative' if delta < 0 else 'neutral',
        'alpha': alpha,
    }


# ─────────────────────────────────────────────────────────────────────────────
# MEANS TEST (for GMV, revenue, etc.)
# ─────────────────────────────────────────────────────────────────────────────
def means_test(
    ctrl_vals: np.ndarray,
    treat_vals: np.ndarray,
    alpha: float = 0.05,
    apply_winsorise: bool = True,
) -> dict:
    """
    Welch's t-test for two independent samples.
    apply_winsorise=True clips extreme outliers at CLIENT_SCHEMA winsorise_pct
    before testing — prevents a single $500K Enterprise order from driving
    false significance on GMV metrics.
    """
    ctrl_vals  = ctrl_vals[~np.isnan(ctrl_vals)]
    treat_vals = treat_vals[~np.isnan(treat_vals)]
    if len(ctrl_vals) < 2 or len(treat_vals) < 2:
        return {'error': 'insufficient data'}
    # Winsorise to remove outlier influence before testing
    if apply_winsorise:
        try:
            ctrl_vals  = winsorise(ctrl_vals)
            treat_vals = winsorise(treat_vals)
        except Exception:
            pass  

    t_stat, p_val = stats.ttest_ind(ctrl_vals, treat_vals, equal_var=False)
    delta_mean = treat_vals.mean() - ctrl_vals.mean()
    rel        = delta_mean / ctrl_vals.mean() if ctrl_vals.mean() != 0 else 0

    # Cohen's d
    pooled_sd = np.sqrt((ctrl_vals.std()**2 + treat_vals.std()**2) / 2)
    cohens_d  = delta_mean / pooled_sd if pooled_sd > 0 else 0

    # 95% CI
    se = np.sqrt(ctrl_vals.var()/len(ctrl_vals) + treat_vals.var()/len(treat_vals))
    z_crit = norm.ppf(1 - alpha/2)
    return {
        'n_control': len(ctrl_vals), 'n_treatment': len(treat_vals),
        'mean_control': round(ctrl_vals.mean(), 2), 'mean_treatment': round(treat_vals.mean(), 2),
        'delta_mean': round(delta_mean, 2), 'delta_rel': round(rel, 4),
        't_stat': round(t_stat, 4), 'p_value': round(p_val, 6),
        'ci_lo': round(delta_mean - z_crit*se, 2),
        'ci_hi': round(delta_mean + z_crit*se, 2),
        'effect_size_d': round(cohens_d, 4),
        'is_significant': p_val < alpha,
        'direction': 'positive' if delta_mean > 0 else 'negative' if delta_mean < 0 else 'neutral',
        'alpha': alpha,
    }


# ─────────────────────────────────────────────────────────────────────────────
# POWER CALCULATOR
# ─────────────────────────────────────────────────────────────────────────────
def compute_sample_size(
    baseline_rate: float,
    mde_abs: float,       # absolute MDE (e.g. 0.01 = 1pp)
    alpha: float = 0.05,
    power: float = 0.80,
    n_variants: int = 2,  # control + treatment
) -> dict:
    """Sample size per variant for a two-proportion test."""
    p1 = baseline_rate
    p2 = baseline_rate + mde_abs
    p2 = np.clip(p2, 0.001, 0.999)

    z_alpha = norm.ppf(1 - alpha/2)   # two-tailed
    z_beta  = norm.ppf(power)

    # Standard formula
    n = (
        (z_alpha * np.sqrt(2 * p1 * (1-p1)) + z_beta * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2
        / (mde_abs**2)
    )
    n_per_variant = int(np.ceil(n))
    n_total       = n_per_variant * n_variants

    # Effect size (Cohen's h)
    h = abs(2 * np.arcsin(np.sqrt(p2)) - 2 * np.arcsin(np.sqrt(p1)))

    return {
        'baseline_rate': baseline_rate,
        'target_rate': round(p2, 6),
        'mde_abs': mde_abs,
        'mde_rel': round(mde_abs / baseline_rate * 100, 2),
        'alpha': alpha, 'power': power, 'n_variants': n_variants,
        'n_per_variant': n_per_variant,
        'n_total': n_total,
        'effect_size_h': round(h, 4),
    }


def compute_duration(
    n_total: int,
    daily_traffic: float,       
    traffic_share: float = 1.0,   
) -> dict:
    """How many days to collect n_total observations."""
    effective_daily = daily_traffic * traffic_share
    days  = int(np.ceil(n_total / effective_daily)) if effective_daily > 0 else 9999
    weeks = round(days / 7, 1)
    return {
        'daily_eligible': round(effective_daily, 1),
        'days_required': days,
        'weeks_required': weeks,
        'end_date': (pd.Timestamp.today() + pd.Timedelta(days=days)).strftime('%Y-%m-%d'),
    }


# ─────────────────────────────────────────────────────────────────────────────
# OPPORTUNITY SIZING
# ─────────────────────────────────────────────────────────────────────────────
def compute_opportunity(
    monthly_inquiries:   float,
    current_ior:         float,   # current order rate (0–1)
    target_ior:          float,   # target order rate (0–1)
    avg_order_value:     float,   # $ per order
    avg_gross_margin:    float,   # gross margin fraction (0–1)
    time_horizon_months: float = 12,
) -> dict:
    """Revenue opportunity from closing the IOR gap."""
    current_orders  = monthly_inquiries * current_ior
    target_orders   = monthly_inquiries * target_ior
    incremental_orders_mo = target_orders - current_orders
    incremental_rev_mo    = incremental_orders_mo * avg_order_value
    incremental_gm_mo     = incremental_rev_mo * avg_gross_margin

    horizon_orders = incremental_orders_mo * time_horizon_months
    horizon_rev    = incremental_rev_mo    * time_horizon_months
    horizon_gm     = incremental_gm_mo     * time_horizon_months

    return {
        'monthly_inquiries': monthly_inquiries,
        'current_ior': current_ior, 'target_ior': target_ior,
        'ior_gap_pp': round((target_ior - current_ior)*100, 2),
        'current_orders_monthly': round(current_orders, 1),
        'target_orders_monthly':  round(target_orders, 1),
        'incremental_orders_monthly': round(incremental_orders_mo, 1),
        'incremental_revenue_monthly': round(incremental_rev_mo, 0),
        'incremental_gm_monthly': round(incremental_gm_mo, 0),
        f'incremental_orders_{int(time_horizon_months)}mo': round(horizon_orders, 0),
        f'incremental_revenue_{int(time_horizon_months)}mo': round(horizon_rev, 0),
        f'incremental_gm_{int(time_horizon_months)}mo': round(horizon_gm, 0),
        'avg_order_value': avg_order_value,
        'gross_margin': avg_gross_margin,
        'time_horizon_months': time_horizon_months,
    }


print('✅ Statistical utilities loaded')
print('   proportion_test()   — two-proportion z-test with CI + effect size')
print('   means_test()        — Welch t-test with CI + Cohen\'s d')
print('   compute_sample_size() — power-based sample size for proportions')
print('   compute_duration()  — days required given traffic')
print('   compute_opportunity() — revenue opportunity from metric gap')


✅ Statistical utilities loaded
   proportion_test()   — two-proportion z-test with CI + effect size
   means_test()        — Welch t-test with CI + Cohen's d
   compute_sample_size() — power-based sample size for proportions
   compute_duration()  — days required given traffic
   compute_opportunity() — revenue opportunity from metric gap


## 7b · Template-Aware Document Generator

Four helpers used by modules [1], [4], [8]: `ask_for_template` · `build_llm_prompt_from_template` · `parse_sections_from_llm_output` · `render_document_pdf`. Produces designed PDFs; falls back to `.txt` if ReportLab is missing.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# STRING UTILITIES — available to ALL modules
# ─────────────────────────────────────────────────────────────────────────────

def _strip_decorative_chars(text: str) -> str:
    """
    Remove common emojis and decorative Unicode symbols from generated text
    so saved business documents stay clean. Keeps basic punctuation, letters,
    digits, and standard separators.
    """
    if not text:
        return text
    emoji_pattern = re.compile(
        '['
        '\U0001F300-\U0001FAFF'
        '\U0001F600-\U0001F64F'
        '\U0001F680-\U0001F6FF'
        '\U0001F1E0-\U0001F1FF'
        '\U00002600-\U000027BF'
        '\U0001F900-\U0001F9FF'
        '\U00002B00-\U00002BFF'
        ']+',
        flags=re.UNICODE,
    )
    text = emoji_pattern.sub('', text)
    replacements = {
        '═': '=', '━': '-', '─': '-', '│': '|',
        '╔': '=', '╗': '=', '╚': '=', '╝': '=',
        '╠': '=', '╣': '=', '╦': '=', '╩': '=', '╬': '=',
        '┌': '+', '┐': '+', '└': '+', '┘': '+',
        '├': '+', '┤': '+', '┬': '+', '┴': '+', '┼': '+',
        '•': '-', '·': '-', '◦': '-',
        '►': '>', '▶': '>', '◄': '<', '◀': '<',
        '✅': '[OK]', '✔': '[OK]', '❌': '[X]', '✘': '[X]',
        '⚠': '[!]', '⚠️': '[!]', '🚨': '[ALERT]',
    }
    for src_ch, dst in replacements.items():
        text = text.replace(src_ch, dst)
    return text


# ─────────────────────────────────────────────────────────────────────────────
# TEMPLATE-AWARE DOCUMENT GENERATOR
# ─────────────────────────────────────────────────────────────────────────────

from collections import OrderedDict
from datetime import datetime

# ── PDF engine (ReportLab) ───────────────────────────────────────────────────
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.colors import HexColor
    from reportlab.lib.units import mm
    from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                    Table, TableStyle, HRFlowable, KeepTogether)
    _PDF_OK = True
except ImportError:
    _PDF_OK = False

# ── Optional template readers (pypdf / python-docx) ──────────────────────────
try:
    from pypdf import PdfReader as _PdfReader
    _PDF_READ_OK = True
except ImportError:
    _PDF_READ_OK = False

try:
    import docx as _docx_mod
    _DOCX_READ_OK = True
except ImportError:
    _DOCX_READ_OK = False

DOC_GENERATOR_READY = _PDF_OK

PDF_PALETTE = {
    'primary':    '#1a3a8c',
    'accent':     '#4e9af1',
    'secondary':  '#f97316',
    'success':    '#22c55e',
    'text':       '#1a1a1a',
    'subtle':     '#6b7280',
    'card_bg':    '#f7f9fc',
    'rule':       '#d6dce5',
}


# ─── 1. Template ingestion ───────────────────────────────────────────────────

def _read_template_file(path):
    """Return raw text of a .txt / .md / .pdf / .docx template."""
    path = os.path.expanduser(path.strip().strip('"').strip("'"))
    if not os.path.isfile(path):
        raise FileNotFoundError('Template file not found: ' + path)
    ext = os.path.splitext(path)[1].lower()

    if ext in ('.txt', '.md', '.markdown', ''):
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()

    if ext == '.pdf':
        if not _PDF_READ_OK:
            raise RuntimeError('pypdf not available — install with: pip install pypdf')
        reader = _PdfReader(path)
        return '\n\n'.join((p.extract_text() or '') for p in reader.pages)

    if ext == '.docx':
        if not _DOCX_READ_OK:
            raise RuntimeError('python-docx not available — install with: pip install python-docx')
        d = _docx_mod.Document(path)
        chunks = []
        for para in d.paragraphs:
            txt = para.text
            style = (para.style.name or '').lower() if para.style else ''
            if 'heading' in style:
                level = 1
                m = re.search(r'(\d+)', style)
                if m:
                    try: level = max(1, min(6, int(m.group(1))))
                    except ValueError: pass
                chunks.append('#' * level + ' ' + txt)
            else:
                chunks.append(txt)
        return '\n'.join(chunks)

    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()


def _extract_section_headers_from_template(raw):
    """Pull section headings out of a user-supplied template."""
    out, seen = [], set()
    for line in raw.splitlines():
        stripped = line.strip()
        if not stripped or len(stripped) > 80:
            continue
        header = None

        m = re.match(r'^(#{1,3})\s+(.+?)\s*$', stripped)
        if m:
            header = m.group(2).strip().rstrip(':')
        elif (sum(1 for c in stripped if c.isalpha()) >= 3
              and stripped == stripped.upper()
              and any(c.isalpha() for c in stripped)
              and not stripped.startswith(('-', '*', '•'))):
            alpha = [c for c in stripped if c.isalpha()]
            if alpha and len(alpha) / max(len(stripped), 1) > 0.4:
                header = stripped.rstrip(':')
        elif re.match(r'^\d+[.)]\s+[A-Z]', stripped):
            header = re.sub(r'^\d+[.)]\s+', '', stripped).rstrip(':')
        elif (stripped.endswith(':') and stripped.count(':') == 1
              and len(stripped) <= 50 and stripped[0].isalpha()):
            core = stripped.rstrip(':')
            if re.match(r'^[A-Za-z][A-Za-z0-9 \-/&()]+$', core):
                header = core

        if not header:
            continue
        h_clean = re.sub(r'\s+', ' ', header).strip()
        if len(h_clean) < 3:
            continue
        if h_clean.lower() in ('yes', 'no', 'todo', 'tbd', 'example', 'note', 'notes'):
            continue
        key = h_clean.upper()
        if key not in seen:
            seen.add(key)
            out.append(h_clean)
    return out


def ask_for_template(doc_kind, default_sections):
    """
    Interactive prompt. Asks the user if they have a template to follow.
    Returns (sections_or_None, raw_template_text_or_None).
    """
    kind = (doc_kind or 'document').strip()
    print()
    print('  ' + '┄' * 70)
    print('  📐  Template check — ' + kind.upper())
    print('  ' + '┄' * 70)
    print('  Do you have a defined format / template for your ' + kind + '?')
    print('  You can upload a .txt, .md, .pdf, or .docx file and the platform')
    print('  will match its section structure. Press Enter or type N to use')
    print('  the built-in default layout.')
    print()

    preview = ', '.join(default_sections[:4])
    if len(default_sections) > 4:
        preview += f', … (+{len(default_sections) - 4} more)'
    print('  Default sections: ' + preview)
    print()

    ans = input('  ❓ Use a custom template? [y/N]: ').strip().lower()
    if ans not in ('y', 'yes'):
        print('  → Using default layout.')
        return None, None

    while True:
        raw_path = input('  📎 Enter path to template file (or blank to cancel): ').strip()
        if not raw_path:
            print('  → Using default layout.')
            return None, None
        try:
            raw = _read_template_file(raw_path)
        except Exception as e:
            print(f'     ⚠️  Could not read template: {e}')
            retry = input('     Try a different path? [Y/n]: ').strip().lower()
            if retry in ('n', 'no'):
                print('  → Falling back to default layout.')
                return None, None
            continue

        sections = _extract_section_headers_from_template(raw)
        if len(sections) < 2:
            print(f'     ⚠️  Only {len(sections)} section header(s) detected in that file.')
            print('     The template needs 2+ clear headings (markdown #, ALL CAPS, or "Title:")')
            retry = input('     Try a different file? [Y/n]: ').strip().lower()
            if retry in ('n', 'no'):
                print('  → Falling back to default layout.')
                return None, None
            continue

        print(f'\n  ✅ Detected {len(sections)} section(s) from your template:')
        for i, s in enumerate(sections, 1):
            print(f'     {i:>2}. {s}')
        confirm = input('\n  Use these sections? [Y/n]: ').strip().lower()
        if confirm in ('', 'y', 'yes'):
            return sections, raw
        print('  → Falling back to default layout.')
        return None, None


# ─── 2. Prompt builder ───────────────────────────────────────────────────────

def build_llm_prompt_from_template(role, context_block, sections_to_fill,
                                   content_guidance=None, style_notes=None):
    """Build a prompt that asks the LLM to produce exactly the listed sections."""
    header_bullets = '\n'.join('  - ' + s for s in sections_to_fill)

    blank_lines = []
    for s in sections_to_fill:
        blank_lines.append(s.upper())
        blank_lines.append('[Content for this section.]')
        blank_lines.append('')
    blank_template = '\n'.join(blank_lines).rstrip()

    default_style = (
        'Write in plain business language. Do not use emojis, icons, or '
        'decorative symbols. Do not wrap the output in code fences or '
        'markdown. For items with a "Field: value" structure (metrics, '
        'tracking events, etc.) keep each field on its own line — the '
        'renderer will format these as styled cards.'
    )
    style = default_style + ('\n' + style_notes if style_notes else '')

    guidance = ''
    if content_guidance:
        guidance = '\nREFERENCE MATERIAL (use only what fits):\n' + content_guidance.strip() + '\n'

    prompt = (
        role.strip() + '\n\n'
        + 'CONTEXT:\n' + context_block.strip() + '\n'
        + guidance + '\n'
        + 'You must produce a document with EXACTLY these sections, in this order:\n'
        + header_bullets + '\n\n'
        + 'STYLE:\n' + style + '\n\n'
        + 'Each section must start on its own line with its heading in UPPERCASE and\n'
        + 'nothing else on that line. Do not add extra sections. Do not renumber or\n'
        + 'reword the headings. Do not add a preamble before the first heading.\n\n'
        + 'Output the sections using this exact skeleton (replace the bracketed text\n'
        + 'with real content; keep the headings verbatim):\n\n'
        + blank_template + '\n'
    ).strip()
    return prompt


# ─── 3. Output parser ────────────────────────────────────────────────────────

def parse_sections_from_llm_output(raw, expected_sections):
    """Split LLM free-text output into a dict keyed by section name."""
    positions = []
    for header in expected_sections:
        pattern = re.compile(
            r'(?im)^\s*(?:#{1,6}\s+|\d+[.)]\s+|\*+\s*)?'
            + re.escape(header) + r'\s*:?\s*$'
        )
        m = pattern.search(raw)
        if m:
            positions.append((m.start(), m.end(), header))

    found_headers = {h for (_, _, h) in positions}
    for header in expected_sections:
        if header in found_headers:
            continue
        pattern = re.compile(re.escape(header), re.IGNORECASE)
        m = pattern.search(raw)
        if m:
            positions.append((m.start(), m.end(), header))

    positions.sort(key=lambda t: t[0])

    out = OrderedDict((h, '') for h in expected_sections)
    for i, (_, end, header) in enumerate(positions):
        next_start = positions[i+1][0] if i+1 < len(positions) else len(raw)
        content = raw[end:next_start].strip()
        content = '\n'.join(line.rstrip() for line in content.splitlines())
        content = re.sub(r'^\n+', '', content)
        out[header] = content

    if not any(out.values()) and expected_sections:
        out[expected_sections[0]] = raw.strip()

    return out


# ─── 4. PDF renderer ─────────────────────────────────────────────────────────

def _escape_pdf(s):
    if s is None: return ''
    return (str(s).replace('&', '&amp;')
                  .replace('<', '&lt;')
                  .replace('>', '&gt;'))


def _render_section_body(content, styles, accent_hex):
    """Turn section content into a list of ReportLab flowables."""
    style_body   = styles['body']
    style_bullet = styles['bullet']
    flowables    = []
    lines        = (content or '').splitlines()
    card_buffer, bullet_buffer, para_buffer = [], [], []

    def _flush_card():
        if not card_buffer: return
        rows = [
            [Paragraph('<b>' + _escape_pdf(lbl) + '</b>', style_body),
             Paragraph(_escape_pdf(val), style_body)]
            for (lbl, val) in card_buffer
        ]
        tbl = Table(rows, colWidths=[42*mm, 123*mm], hAlign='LEFT')
        tbl.setStyle(TableStyle([
            ('BACKGROUND',    (0, 0), (-1, -1), HexColor(PDF_PALETTE['card_bg'])),
            ('LINEBEFORE',    (0, 0), (0, -1),  2.2, HexColor(accent_hex)),
            ('BOX',           (0, 0), (-1, -1), 0.3, HexColor(PDF_PALETTE['rule'])),
            ('VALIGN',        (0, 0), (-1, -1), 'TOP'),
            ('LEFTPADDING',   (0, 0), (-1, -1), 8),
            ('RIGHTPADDING',  (0, 0), (-1, -1), 8),
            ('TOPPADDING',    (0, 0), (-1, -1), 4),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 4),
        ]))
        flowables.append(tbl)
        flowables.append(Spacer(1, 6))
        card_buffer.clear()

    def _flush_bullets():
        if not bullet_buffer: return
        for b in bullet_buffer:
            flowables.append(Paragraph('• ' + _escape_pdf(b), style_bullet))
        flowables.append(Spacer(1, 4))
        bullet_buffer.clear()

    def _flush_paras():
        if not para_buffer: return
        text = ' '.join(p.strip() for p in para_buffer if p.strip())
        if text:
            flowables.append(Paragraph(_escape_pdf(text), style_body))
        para_buffer.clear()

    for line in lines:
        stripped = line.strip()
        if not stripped:
            _flush_paras(); _flush_bullets(); _flush_card()
            continue
        if stripped.startswith(('- ', '* ', '• ')):
            _flush_paras(); _flush_card()
            bullet_buffer.append(stripped[2:].strip())
            continue
        m = re.match(r'^([A-Z][A-Za-z0-9 /()\-]{1,32}):\s+(.+)$', stripped)
        if m:
            _flush_paras(); _flush_bullets()
            card_buffer.append((m.group(1).strip(), m.group(2).strip()))
            continue
        if re.match(r'^[A-Z][A-Za-z0-9 /()\-]{1,32}:\s*$', stripped):
            _flush_paras(); _flush_bullets(); _flush_card()
            continue
        _flush_card(); _flush_bullets()
        para_buffer.append(stripped)

    _flush_card(); _flush_bullets(); _flush_paras()
    return flowables


def _write_plain_text_fallback(title, subtitle, sections, metadata, path):
    lines = ['=' * 72, title]
    if subtitle: lines.append(subtitle)
    lines += ['=' * 72, '']
    if metadata:
        for k, v in metadata.items(): lines.append(str(k) + ': ' + str(v))
        lines.append('')
    for sec_name, content in sections.items():
        lines.append('-' * 60)
        lines.append(sec_name.upper())
        lines.append('-' * 60)
        lines.append(content or '(no content)')
        lines.append('')
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))


def render_document_pdf(title, subtitle, sections, output_path,
                        metadata=None, accent_color=None):
    """Render a designed PDF document. Falls back to .txt if ReportLab missing."""
    if not _PDF_OK:
        fallback = os.path.splitext(output_path)[0] + '.txt'
        _write_plain_text_fallback(title, subtitle, sections, metadata, fallback)
        print('     (ReportLab not installed; wrote plain text fallback)')
        return fallback

    accent  = accent_color or PDF_PALETTE['accent']
    primary = PDF_PALETTE['primary']

    if not isinstance(sections, OrderedDict):
        sections = OrderedDict(sections.items() if hasattr(sections, 'items') else sections)

    def _draw_chrome(canvas_obj, doc):
        canvas_obj.saveState()
        canvas_obj.setFillColor(HexColor(accent))
        canvas_obj.rect(0, A4[1] - 6*mm, A4[0], 6*mm, stroke=0, fill=1)
        if doc.page > 1:
            canvas_obj.setFont('Helvetica', 9)
            canvas_obj.setFillColor(HexColor(PDF_PALETTE['subtle']))
            canvas_obj.drawString(18*mm, A4[1] - 13*mm, title[:80])
            canvas_obj.setStrokeColor(HexColor(PDF_PALETTE['rule']))
            canvas_obj.setLineWidth(0.4)
            canvas_obj.line(18*mm, A4[1] - 16*mm, A4[0] - 18*mm, A4[1] - 16*mm)
        canvas_obj.setFont('Helvetica', 8)
        canvas_obj.setFillColor(HexColor(PDF_PALETTE['subtle']))
        canvas_obj.drawString(18*mm, 12*mm,
                              'Generated by Continum PersistIQ')
        canvas_obj.drawRightString(A4[0] - 18*mm, 12*mm, 'Page ' + str(doc.page))
        canvas_obj.setStrokeColor(HexColor(PDF_PALETTE['rule']))
        canvas_obj.setLineWidth(0.4)
        canvas_obj.line(18*mm, 15*mm, A4[0] - 18*mm, 15*mm)
        canvas_obj.restoreState()

    doc = SimpleDocTemplate(output_path, pagesize=A4,
                            leftMargin=18*mm, rightMargin=18*mm,
                            topMargin=22*mm, bottomMargin=22*mm,
                            title=title, author='Continum PersistIQ')

    ss = getSampleStyleSheet()
    styles = {
        'title': ParagraphStyle('DocTitle', parent=ss['Heading1'],
                                fontName='Helvetica-Bold', fontSize=22,
                                textColor=HexColor(primary),
                                spaceAfter=4, leading=26),
        'subtitle': ParagraphStyle('DocSubtitle', parent=ss['Normal'],
                                   fontName='Helvetica', fontSize=11,
                                   textColor=HexColor(PDF_PALETTE['subtle']),
                                   spaceAfter=12, leading=14),
        'section': ParagraphStyle('SectionHead', parent=ss['Heading2'],
                                  fontName='Helvetica-Bold', fontSize=13,
                                  textColor=HexColor(accent),
                                  spaceBefore=16, spaceAfter=6, leading=16),
        'body': ParagraphStyle('Body', parent=ss['BodyText'],
                               fontName='Helvetica', fontSize=10,
                               textColor=HexColor(PDF_PALETTE['text']),
                               leading=14, spaceAfter=6),
        'bullet': ParagraphStyle('Bullet', parent=ss['BodyText'],
                                 fontName='Helvetica', fontSize=10,
                                 textColor=HexColor(PDF_PALETTE['text']),
                                 leading=14, leftIndent=14,
                                 bulletIndent=4, spaceAfter=3),
        'meta_key': ParagraphStyle('MetaKey', parent=ss['Normal'],
                                   fontName='Helvetica-Bold', fontSize=9,
                                   textColor=HexColor(PDF_PALETTE['subtle']),
                                   leading=12),
        'meta_val': ParagraphStyle('MetaVal', parent=ss['Normal'],
                                   fontName='Helvetica', fontSize=10,
                                   textColor=HexColor(PDF_PALETTE['text']),
                                   leading=13),
    }

    story = [Paragraph(_escape_pdf(title), styles['title'])]
    if subtitle:
        story.append(Paragraph(_escape_pdf(subtitle), styles['subtitle']))
    story.append(HRFlowable(width='100%', thickness=1.4,
                            color=HexColor(accent),
                            spaceBefore=2, spaceAfter=10))

    meta = dict(metadata or {})
    meta.setdefault('Generated', datetime.now().strftime('%d %b %Y · %H:%M'))
    meta_rows = [
        [Paragraph(_escape_pdf(str(k).upper()), styles['meta_key']),
         Paragraph(_escape_pdf(str(v)),         styles['meta_val'])]
        for (k, v) in meta.items()
    ]
    if meta_rows:
        meta_table = Table(meta_rows, colWidths=[35*mm, 130*mm], hAlign='LEFT')
        meta_table.setStyle(TableStyle([
            ('BACKGROUND',    (0, 0), (-1, -1), HexColor(PDF_PALETTE['card_bg'])),
            ('BOX',           (0, 0), (-1, -1), 0.4, HexColor(PDF_PALETTE['rule'])),
            ('INNERGRID',     (0, 0), (-1, -1), 0.25, HexColor(PDF_PALETTE['rule'])),
            ('LEFTPADDING',   (0, 0), (-1, -1), 8),
            ('RIGHTPADDING',  (0, 0), (-1, -1), 8),
            ('TOPPADDING',    (0, 0), (-1, -1), 5),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
            ('VALIGN',        (0, 0), (-1, -1), 'TOP'),
        ]))
        story.append(meta_table)
        story.append(Spacer(1, 14))

    for sec_name, content in sections.items():
        content = (content or '').strip() or '(No content generated for this section.)'
        header_block = [
            Paragraph(_escape_pdf(sec_name.upper()), styles['section']),
            HRFlowable(width=60*mm, thickness=1.2,
                       color=HexColor(accent),
                       spaceBefore=-2, spaceAfter=6),
        ]
        body_flowables = _render_section_body(content, styles, accent)
        if body_flowables:
            story.append(KeepTogether(header_block + body_flowables[:1]))
            story.extend(body_flowables[1:])
        else:
            story.extend(header_block)
        story.append(Spacer(1, 6))

    doc.build(story, onFirstPage=_draw_chrome, onLaterPages=_draw_chrome)
    return output_path


if DOC_GENERATOR_READY:
    print('✅ Template-aware document generator ready')
    print('   helpers : ask_for_template · build_llm_prompt_from_template')
    print('             parse_sections_from_llm_output · render_document_pdf')
    print('   readers : pypdf={}  python-docx={}'.format(_PDF_READ_OK, _DOCX_READ_OK))
else:
    print('⚠️  ReportLab not installed — PDFs will fall back to .txt')
    print('   pip install reportlab pypdf python-docx   to enable full functionality')


✅ Template-aware document generator ready
   helpers : ask_for_template · build_llm_prompt_from_template
             parse_sections_from_llm_output · render_document_pdf
   readers : pypdf=True  python-docx=True


## 8 · Phase 0 — Foundation Modules

Before any experimentation: connect a warehouse, validate the data layer.

**Module [1]** — Schema Discovery & Mapping. Auto-generates a `CLIENT_SCHEMA` block from any connected source.
**Module [2]** — Pipeline Health Monitor. Daily anomaly scan over the canonical tables (volume, distribution, freshness, schema, nulls).

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# MODULE 1 — SCHEMA DISCOVERY & MAPPING (Phase 0 — Foundation)
# ─────────────────────────────────────────────────────────────────────────────

CANONICAL_TABLES = {
    # Core transaction tables (required for basic analysis)
    'inquiries':    'Aggregated inquiry/quote/lead rows — primary unit of analysis for conversion rate',
    'quotes':       'Quote / inquiry / lead records — one row per quote / request submitted',
    'orders':       'Order / transaction / purchase records — one row per completed transaction',
    'users':        'User / buyer / customer records — one row per registered user',
    'accounts':     'Account / company / organisation records — one row per business account',
    'experiments':  'Experiment assignment records — one row per user-variant exposure',
    # Optional extended tables
    'traffic':      'Daily platform traffic — sessions, sign-ups, page views by day',
    'products':     'Product / SKU / listing catalogue — one row per product',
    'sessions':     'Session / clickstream records — one row per session or page view',
    'events':       'Raw event log — one row per user action / event',
    'returns':      'Return / refund / cancellation records — one row per return',
    'inventory':    'Inventory / stock records — one row per SKU / location combination',
}

CANONICAL_COLUMNS = {
    # Identity & time (always required)
    'inquiry_id':         'Unique inquiry / quote / lead / request identifier (string or int)',
    'buyer_id':           'Unique user / buyer / customer / member identifier (string or int)',
    'created_at':         'Timestamp the inquiry / event was created (datetime)',
    # Segmentation (highly recommended)
    'account_segment':    'Account or user segment classification (e.g. Core / Growth / Enterprise / SMB)',
    'platform':           'Platform of origin (e.g. web / mobile / desktop / app / api)',
    'category':           'Product, process, or service category (e.g. Electronics / Clothing / CNC)',
    'country':            'Buyer, shipping, or registration country (ISO code or full name)',
    # Conversion / outcome
    'converted_to_order': 'Boolean — did this inquiry/lead/cart become a completed order/purchase?',
    'order_value':        'Transaction value in the client currency (numeric, 0 when not converted)',
    # Experiment
    'variant':            'Experiment variant assignment (e.g. control / treatment / variant_a)',
    'experiment_name':    'Experiment identifier the user was exposed to',
    # Optional but commonly used
    'product_id':         'Product or SKU identifier (for product-level experiments)',
    'channel':            'Marketing or acquisition channel (e.g. organic / paid / email / referral)',
    'device_type':        'Device type (e.g. desktop / mobile / tablet)',
    'industry':           'Industry or vertical classification of the account',
}


def _profile_dataframe(df, table_name, sample_size=2000):
    """Compute per-column profile: nulls, cardinality, dtype, sample values."""
    n_rows = len(df)
    sample = df.sample(min(sample_size, n_rows), random_state=42) if n_rows else df
    profile = []
    for col in df.columns:
        s = sample[col]
        non_null = s.dropna()
        # Try to detect the value family
        family = 'unknown'
        if non_null.empty:
            family = 'all_null'
        elif s.dtype.kind in ('i', 'u'):
            family = 'integer'
        elif s.dtype.kind == 'f':
            family = 'float'
        elif s.dtype.kind == 'b':
            family = 'boolean'
        elif s.dtype.kind == 'M':
            family = 'datetime'
        elif s.dtype.kind == 'O' or str(s.dtype) == 'str':
            sample_strs = non_null.astype(str)
            if sample_strs.str.match(r'^\d{4}-\d{2}-\d{2}').any():
                family = 'date_string'
            elif sample_strs.str.match(r'^[a-zA-Z0-9_]{8,}$').any() and non_null.nunique() > 0.5 * len(non_null):
                family = 'identifier'
            elif non_null.nunique() <= 20:
                family = 'categorical'
            else:
                family = 'string'

        sample_vals = non_null.head(3).tolist()
        profile.append({
            'table':        table_name,
            'column':       col,
            'dtype':        str(s.dtype),
            'family':       family,
            'null_pct':     round(s.isna().mean() * 100, 2),
            'cardinality':  int(non_null.nunique()),
            'sample':       sample_vals,
        })
    return profile


def _build_mapping_prompt(profiles_by_table):
    """Compact representation of the catalog for the LLM."""
    catalog_lines = []
    for tbl, cols in profiles_by_table.items():
        catalog_lines.append(f'\nTABLE: {tbl}  ({len(cols)} columns)')
        for c in cols[:25]:    # cap to avoid prompt bloat
            sample_str = str(c['sample'])[:60]
            catalog_lines.append(
                f"  - {c['column']:<32} {c['family']:<14} null={c['null_pct']:>5.1f}%  "
                f"card={c['cardinality']:>6}  sample={sample_str}"
            )
        if len(cols) > 25:
            catalog_lines.append(f'  ... ({len(cols) - 25} more columns)')

    canonical_tables_text = '\n'.join(f'  - {k}: {v}' for k, v in CANONICAL_TABLES.items())
    canonical_cols_text   = '\n'.join(f'  - {k}: {v}' for k, v in CANONICAL_COLUMNS.items())

    prompt = f"""You are a data engineer mapping a client warehouse to a canonical experimentation schema.

CANONICAL TABLES needed by the platform:
{canonical_tables_text}

CANONICAL COLUMNS the platform expects (after mapping):
{canonical_cols_text}

CLIENT WAREHOUSE CATALOG (profiled):
{chr(10).join(catalog_lines)}

For each canonical table, pick the BEST matching client table from the catalog above.
For each canonical column, pick the BEST matching column from the chosen table.
If no good match exists, return null.

Return ONLY a JSON object in this exact shape, no other text:

{{
  "table_mapping": {{
    "quotes":      "<client_table_name_or_null>",
    "orders":      "<client_table_name_or_null>",
    "users":       "<client_table_name_or_null>",
    "accounts":    "<client_table_name_or_null>",
    "experiments": "<client_table_name_or_null>",
    "inquiries":   "<client_table_name_or_null>",
    "traffic":     "<client_table_name_or_null>"
  }},
  "column_mapping": {{
    "inquiry_id":         "<client_column_or_null>",
    "buyer_id":           "<client_column_or_null>",
    "account_segment":    "<client_column_or_null>",
    "platform":           "<client_column_or_null>",
    "category":           "<client_column_or_null>",
    "country":            "<client_column_or_null>",
    "created_at":         "<client_column_or_null>",
    "converted_to_order": "<client_column_or_null>",
    "order_value":        "<client_column_or_null>",
    "variant":            "<client_column_or_null>",
    "experiment_name":    "<client_column_or_null>"
  }},
  "confidence": <a number 0.0-1.0 reflecting your overall confidence>,
  "warnings":   ["<any caveat the user should review>", ...]
}}
"""
    return prompt


def _verify_mapping(mapping, profiles_by_table):
    """
    Deterministic sanity checks on the LLM's proposed mapping.
    Returns a list of (severity, message) issues.
    """
    issues = []
    table_mapping = mapping.get('table_mapping', {})
    column_mapping = mapping.get('column_mapping', {})

    for canonical, mapped in table_mapping.items():
        if mapped is None or mapped == 'null':
            issues.append(('warn', f'No table mapped for canonical "{canonical}"'))
        elif mapped not in profiles_by_table:
            issues.append(('error', f'Mapped table "{mapped}" not found in catalog'))

    all_columns = {c['column'] for cols in profiles_by_table.values() for c in cols}
    for canonical, mapped in column_mapping.items():
        if mapped is None or mapped == 'null':
            issues.append(('warn', f'No column mapped for canonical "{canonical}"'))
        elif mapped not in all_columns:
            issues.append(('error', f'Mapped column "{mapped}" not found in any table'))

    created_at_col = column_mapping.get('created_at')
    if created_at_col:
        for cols in profiles_by_table.values():
            for c in cols:
                if c['column'] == created_at_col:
                    if c['family'] not in ('datetime', 'date_string'):
                        issues.append(('warn',
                            f'"{created_at_col}" mapped to created_at but family is {c["family"]}'))

    conv_col = column_mapping.get('converted_to_order')
    if conv_col:
        for cols in profiles_by_table.values():
            for c in cols:
                if c['column'] == conv_col:
                    if c['family'] not in ('boolean', 'integer'):
                        issues.append(('warn',
                            f'"{conv_col}" mapped to converted_to_order but family is {c["family"]}'))

    val_col = column_mapping.get('order_value')
    if val_col:
        for cols in profiles_by_table.values():
            for c in cols:
                if c['column'] == val_col:
                    if c['family'] not in ('integer', 'float'):
                        issues.append(('warn',
                            f'"{val_col}" mapped to order_value but family is {c["family"]}'))

    return issues


def _format_client_schema_block(client_name, mapping):
    """Generate a paste-ready CLIENT_SCHEMA dict from the mapping."""
    table_mapping  = mapping.get('table_mapping', {})
    column_mapping = mapping.get('column_mapping', {})

    def _fmt_table(canonical):
        v = table_mapping.get(canonical)
        return repr(v) if v else "''   # ← MAP ME"

    def _fmt_col(canonical):
        v = column_mapping.get(canonical)
        return repr(v) if v else "''   # ← MAP ME"

    col_lines = ['\n'.join(
        f"        '{k}': {_fmt_col(k)},"
        for k in column_mapping.keys()
    )]
    all_tables = list(CANONICAL_TABLES.keys())
    tbl_lines = '\n'.join(
        f"        '{k}': {_fmt_table(k)},"
        for k in all_tables
    )

    return f"""CLIENT_SCHEMA = {{
    'client_name': {client_name!r},
    'tables': {{
{tbl_lines}
    }},
    'columns': {{
        'inquiry_id':         {_fmt_col('inquiry_id')},
        'buyer_id':           {_fmt_col('buyer_id')},
        'account_segment':    {_fmt_col('account_segment')},
        'platform':           {_fmt_col('platform')},
        'category':           {_fmt_col('category')},
        'country':            {_fmt_col('country')},
        'created_at':         {_fmt_col('created_at')},
        'converted_to_order': {_fmt_col('converted_to_order')},
        'order_value':        {_fmt_col('order_value')},
        'variant':            {_fmt_col('variant')},
        'experiment_name':    {_fmt_col('experiment_name')},
        # Extended columns discovered:
{col_lines[0] if col_lines else ''}
    }},
    # Review the mapping above; edit any '' placeholders before using.
}}"""


def run_schema_discovery(llm, _bootstrap_mode=False, _client_name=None):
    """
    Module 1 — Schema Discovery & Mapping.

    Parameters
    ----------
    llm               : LLM client
    _bootstrap_mode   : bool — if True, skip interactive prompts and return
                        the result dict for use by bootstrap_from_connection().
                        The PDF and schema file are still written.
    _client_name      : str — client name to use when _bootstrap_mode=True.
    """
    print()
    print('╔' + '═'*70 + '╗')
    print('║' + '  🔍  SCHEMA DISCOVERY & MAPPING (Phase 0 — Foundation)'.ljust(70) + '║')
    print('║' + '  Auto-generate a CLIENT_SCHEMA from a connected warehouse'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')

    if _bootstrap_mode:
        # Non-interactive path — use DuckDB session tables, client name from caller
        client_name = _client_name or 'Client'
        print(f'\n  Running in bootstrap mode for client: {client_name}')
    else:
        # ── Source selection ──────────────────────────────────────────────────
        print()
        print('  Available sources:')
        print('    [1] Synthetic data (DuckDB tables registered in this session)')
        print('    [2] Snowflake / Postgres / external warehouse (advanced — needs connection)')
        while True:
            choice = input('  ❓ Choose source [1/2] (default 1): ').strip() or '1'
            if choice in ('1', '2'): break
            print('     ⚠️  Choose 1 or 2')

        if choice == '2':
            print('\n  ℹ️  External warehouse profiling is left as an integration step.')
            print('     For this session, falling back to the synthetic catalog so you')
            print('     can see the full discovery flow end-to-end.')

        client_name = input('\n  Client name (e.g. "Xometry"): ').strip() or 'Demo Client'

    # ── Catalog scan from DuckDB ─────────────────────────────────────────────
    print('\n  🔎 Scanning DuckDB catalog...')
    try:
        catalog_df = db.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
            ORDER BY table_name
        """).df()
    except Exception:
        catalog_df = db.execute("SHOW TABLES").df()
        catalog_df.columns = ['table_name']

    if len(catalog_df) == 0:
        print('  ⚠️  No tables found. Run cells 5 and 6 first to register synthetic data.')
        return None

    print(f'     Found {len(catalog_df)} tables: {", ".join(catalog_df["table_name"].tolist())}')

    # ── Profile each table ───────────────────────────────────────────────────
    print('\n  🧪 Profiling tables (sampling rows, computing nulls / cardinality / families)...')
    profiles_by_table = {}
    for tbl in catalog_df['table_name']:
        try:
            df_tbl = db.execute(f'SELECT * FROM "{tbl}" LIMIT 5000').df()
            profiles_by_table[tbl] = _profile_dataframe(df_tbl, tbl)
            print(f'     ✅ {tbl:<32} {len(df_tbl):>6,} rows sampled, {len(df_tbl.columns)} columns')
        except Exception as e:
            print(f'     ⚠️  {tbl}: {e}')

    if not profiles_by_table:
        print('  ❌ No tables could be profiled.')
        return None

    # ── LLM mapping ──────────────────────────────────────────────────────────
    print('\n  🤖 Asking the LLM to map catalog → canonical schema...')
    prompt = _build_mapping_prompt(profiles_by_table)
    raw = llm.ask(prompt)
    try:
        json_match = re.search(r'\{[\s\S]*\}', raw)
        mapping = json.loads(json_match.group()) if json_match else {}
    except Exception as e:
        print(f'  ⚠️  Could not parse LLM response as JSON ({e}); proceeding with empty mapping.')
        mapping = {'table_mapping': {}, 'column_mapping': {}, 'confidence': 0.0,
                   'warnings': ['LLM output was not valid JSON; review carefully.']}

    confidence = mapping.get('confidence', 0.0)
    print(f'\n  Mapping confidence: {confidence:.0%}')

    # ── Verify ───────────────────────────────────────────────────────────────
    print('\n  ✅ Verifying mapping with deterministic checks...')
    issues = _verify_mapping(mapping, profiles_by_table)
    n_errors = sum(1 for sev, _ in issues if sev == 'error')
    n_warns  = sum(1 for sev, _ in issues if sev == 'warn')
    if n_errors:
        print(f'     ❌ {n_errors} error(s):')
        for sev, msg in issues:
            if sev == 'error': print(f'        - {msg}')
    if n_warns:
        print(f'     ⚠️  {n_warns} warning(s):')
        for sev, msg in issues:
            if sev == 'warn': print(f'        - {msg}')
    if not issues:
        print('     ✅ All checks passed.')

    # ── Display the proposed mapping ─────────────────────────────────────────
    print('\n  ── Proposed table mapping ──')
    for canonical, mapped in mapping.get('table_mapping', {}).items():
        marker = '✅' if mapped else '⚠️ '
        print(f'    {marker} {canonical:<14} → {mapped or "(no match)"}')

    print('\n  ── Proposed column mapping ──')
    for canonical, mapped in mapping.get('column_mapping', {}).items():
        marker = '✅' if mapped else '⚠️ '
        print(f'    {marker} {canonical:<22} → {mapped or "(no match)"}')

    # ── Generate paste-ready CLIENT_SCHEMA + PDF ─────────────────────────────
    schema_block = _format_client_schema_block(client_name, mapping)
    schema_path = f'client_schema_{client_name.lower().replace(" ", "_")}.py'
    with open(schema_path, 'w', encoding='utf-8') as f:
        f.write(f'# Auto-generated by Continum PersistIQ — Schema Discovery\n')
        f.write(f'# Client: {client_name}\n')
        f.write(f'# Mapping confidence: {confidence:.0%}\n')
        f.write(f'# Review every mapping before pasting into Cell 3.\n\n')
        f.write(schema_block + '\n')
    print(f'\n  📁 Schema block saved → {schema_path}')

    # PDF report
    from collections import OrderedDict
    pdf_sections = OrderedDict([
        ('OVERVIEW',
            f'Schema discovery against {len(profiles_by_table)} tables in the connected source. '
            f'LLM-proposed mapping confidence: {confidence:.0%}. '
            f'{n_errors} error(s) and {n_warns} warning(s) flagged by deterministic checks.'),
        ('TABLE MAPPING', '\n'.join(
            f'- {canonical} → {mapped or "(no match)"}'
            for canonical, mapped in mapping.get('table_mapping', {}).items())),
        ('COLUMN MAPPING', '\n'.join(
            f'- {canonical} → {mapped or "(no match)"}'
            for canonical, mapped in mapping.get('column_mapping', {}).items())),
        ('VERIFICATION ISSUES',
            '\n'.join(f'- [{sev.upper()}] {msg}' for sev, msg in issues) or 'None — all checks passed.'),
        ('NEXT STEPS',
            '- Review every mapped column in the generated schema file.\n'
            '- Replace any (no match) placeholders before deploying.\n'
            '- Paste the CLIENT_SCHEMA block into Cell 3 of the notebook.\n'
            '- Set USE_SYNTHETIC_DATA = False and re-run cells 3, 5, 6.'),
    ])

    pdf_out = render_document_pdf(
        title='Schema Discovery Report',
        subtitle=f'Client: {client_name}',
        sections=pdf_sections,
        output_path='schema_discovery_report.pdf',
        metadata={
            'Client':        client_name,
            'Tables found':  str(len(profiles_by_table)),
            'Confidence':    f'{confidence:.0%}',
            'Errors':        str(n_errors),
            'Warnings':      str(n_warns),
        },
        accent_color=PDF_PALETTE['accent'],
    )
    print(f'  📁 PDF report saved → {pdf_out}')

    return {
        'client_name':   client_name,
        'tables_found':  list(profiles_by_table.keys()),
        'mapping':       mapping,
        'issues':        issues,
        'schema_file':   schema_path,
        'pdf_report':    pdf_out,
    }


# ─────────────────────────────────────────────────────────────────────────────
# MODULE 2 — PIPELINE HEALTH MONITOR (Phase 0 — Foundation)
# ─────────────────────────────────────────────────────────────────────────────

def _detect_volume_anomaly(daily_counts, baseline_days=28, alert_z=2.5):
    """
    Compare today's volume to the seasonal-adjusted forecast from the prior baseline_days.
    Returns dict with z-score and severity.
    """
    if len(daily_counts) < baseline_days + 1:
        return {'status': 'insufficient_data', 'z_score': None, 'severity': 'info'}

    history = daily_counts.iloc[-(baseline_days + 1):-1]
    today   = float(daily_counts.iloc[-1])

    today_dow = daily_counts.index[-1].dayofweek
    same_dow  = history[history.index.dayofweek == today_dow]
    if len(same_dow) >= 3:
        baseline_mean = float(same_dow.mean())
        baseline_std  = float(same_dow.std()) or 1.0
    else:
        baseline_mean = float(history.mean())
        baseline_std  = float(history.std()) or 1.0

    z = (today - baseline_mean) / baseline_std
    pct_change = (today - baseline_mean) / baseline_mean * 100 if baseline_mean else 0
    severity = 'critical' if abs(z) > alert_z else 'warning' if abs(z) > 1.5 else 'ok'

    return {
        'status':         'analysed',
        'today_value':    today,
        'baseline_mean':  baseline_mean,
        'baseline_std':   baseline_std,
        'z_score':        round(z, 3),
        'pct_change':     round(pct_change, 2),
        'severity':       severity,
    }


def _detect_distribution_shift(today_counts, baseline_counts, alert_p=0.001):
    """χ² test comparing today's category split against baseline."""
    from scipy.stats import chisquare
    today_norm = today_counts.reindex(baseline_counts.index, fill_value=0).astype(float)
    if today_norm.sum() == 0 or baseline_counts.sum() == 0:
        return {'status': 'insufficient_data', 'severity': 'info'}

    expected = baseline_counts / baseline_counts.sum() * today_norm.sum()
    expected = expected.replace(0, 1e-6)
    chi2, p = chisquare(today_norm.values, f_exp=expected.values)
    severity = 'critical' if p < alert_p else 'warning' if p < 0.01 else 'ok'
    return {
        'status':       'analysed',
        'chi2':         round(float(chi2), 3),
        'p_value':      round(float(p), 6),
        'severity':     severity,
        'today_split':  {k: int(v) for k, v in today_norm.items()},
        'baseline_split': {k: int(v) for k, v in baseline_counts.items()},
    }


def _detect_freshness(latest_ts, sla_hours=24):
    """Compare most-recent record to SLA. Always returns all keys."""
    now = pd.Timestamp.now()
    if latest_ts is None or pd.isna(latest_ts):
        return {
            'status':    'no_data',
            'latest':    'n/a',
            'age_hours': 0.0,
            'sla_hours': sla_hours,
            'severity':  'critical',
        }
    age_hours = (now - pd.Timestamp(latest_ts)).total_seconds() / 3600
    severity = 'critical' if age_hours > sla_hours * 2 \
        else 'warning' if age_hours > sla_hours else 'ok'
    return {
        'status':     'analysed',
        'latest':     str(latest_ts),
        'age_hours':  round(age_hours, 2),
        'sla_hours':  sla_hours,
        'severity':   severity,
    }


def _detect_null_spike(df, columns, baseline_null_rates, alert_pp=10):
    """Find columns where null rate jumped vs baseline by alert_pp percentage points."""
    findings = []
    for col in columns:
        if col not in df.columns: continue
        current = df[col].isna().mean() * 100
        baseline = baseline_null_rates.get(col, 0.0)
        jump = current - baseline
        if jump > alert_pp:
            findings.append({
                'column':       col,
                'baseline_pct': round(baseline, 2),
                'current_pct':  round(current, 2),
                'jump_pp':      round(jump, 2),
                'severity':     'critical' if jump > alert_pp * 2 else 'warning',
            })
    return findings


def _detect_schema_drift(current_columns, expected_columns):
    """Compare current schema vs expected; report new / missing columns."""
    current_set  = set(current_columns)
    expected_set = set(expected_columns)
    return {
        'new_columns':     sorted(current_set - expected_set),
        'missing_columns': sorted(expected_set - current_set),
        'severity':        'critical' if (expected_set - current_set) \
                            else 'warning' if (current_set - expected_set) else 'ok',
    }


def run_pipeline_health(llm):
    """
    Module 2 — Pipeline Health Monitor.
    Scans the canonical tables for anomalies and produces a designed PDF.
    """
    print()
    print('╔' + '═'*70 + '╗')
    print('║' + '  🩺  PIPELINE HEALTH MONITOR (Phase 0 — Foundation)'.ljust(70) + '║')
    print('║' + '  Volume · Distribution · Freshness · Schema · Null spikes'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')

    print()
    print('  Scanning canonical tables for anomalies...')

    # ── Pull primary table for analysis ──────────────────────────────────────
    try:
        df_inq = db.execute("""
            SELECT created_at, account_segment, platform, category, has_billing_profile
            FROM hist_inquiries
            ORDER BY created_at
        """).df()
        df_inq['created_at'] = pd.to_datetime(df_inq['created_at'])
    except Exception as e:
        print(f'  ❌ Could not read hist_inquiries: {e}')
        return None

    print(f'  ✅ Loaded {len(df_inq):,} historical inquiries '
          f'({df_inq["created_at"].min().date()} → {df_inq["created_at"].max().date()})')

    findings = {}

    # ── Volume drift ──────────────────────────────────────────────────────
    daily = df_inq.groupby(df_inq['created_at'].dt.normalize()).size()
    findings['volume'] = _detect_volume_anomaly(daily)
    print(f'\n  [1/5] Volume drift')
    v = findings['volume']
    if v['status'] == 'analysed':
        icon = {'ok':'✅','warning':'⚠️ ','critical':'🚨'}[v['severity']]
        print(f'     {icon} Today: {v["today_value"]:.0f}   Baseline: {v["baseline_mean"]:.0f}   '
              f'Δ={v["pct_change"]:+.1f}%   z={v["z_score"]:+.2f}')
    else:
        print(f'     ℹ️  {v["status"]}')

    # ── Distribution shift on category ────────────────────────────────────
    today_date = daily.index[-1]
    today_rows = df_inq[df_inq['created_at'].dt.normalize() == today_date]
    baseline_rows = df_inq[df_inq['created_at'].dt.normalize() < today_date].tail(28 * 500)
    today_cat   = today_rows['category'].value_counts()
    base_cat    = baseline_rows['category'].value_counts()
    findings['category_shift'] = _detect_distribution_shift(today_cat, base_cat)
    print(f'\n  [2/5] Distribution shift — category')
    d = findings['category_shift']
    if d['status'] == 'analysed':
        icon = {'ok':'✅','warning':'⚠️ ','critical':'🚨'}[d['severity']]
        print(f'     {icon} χ²={d["chi2"]:.2f}   p={d["p_value"]:.4f}')
    else:
        print(f'     ℹ️  {d["status"]}')

    # ── Distribution shift on platform ────────────────────────────────────
    today_plat = today_rows['platform'].value_counts()
    base_plat  = baseline_rows['platform'].value_counts()
    findings['platform_shift'] = _detect_distribution_shift(today_plat, base_plat)
    p = findings['platform_shift']
    if p['status'] == 'analysed':
        icon = {'ok':'✅','warning':'⚠️ ','critical':'🚨'}[p['severity']]
        print(f'         platform: {icon} χ²={p["chi2"]:.2f}   p={p["p_value"]:.4f}')

    # ── Freshness ─────────────────────────────────────────────────────────
    latest = df_inq['created_at'].max()
    findings['freshness'] = _detect_freshness(latest, sla_hours=24*30)  # synthetic data is months old
    f = findings['freshness']
    icon = {'ok':'✅','warning':'⚠️ ','critical':'🚨'}.get(f['severity'], 'ℹ️ ')
    print(f'\n  [3/5] Freshness')
    _age_str = f'{f["age_hours"]:.1f}h' if isinstance(f.get('age_hours'), (int, float)) else 'n/a'
    _sla_str = f'{f["sla_hours"]}h' if isinstance(f.get('sla_hours'), (int, float)) else 'n/a'
    print(f'     {icon} Latest record: {f.get("latest","n/a")}   '
          f'Age: {_age_str}   SLA: {_sla_str}')

    # ── Null spike ────────────────────────────────────────────────────────
    full_null_rates = (df_inq.isna().mean() * 100).to_dict()
    today_nulls = _detect_null_spike(today_rows,
        ['account_segment', 'platform', 'category', 'has_billing_profile'],
        full_null_rates, alert_pp=10)
    findings['null_spikes'] = today_nulls
    print(f'\n  [4/5] Null-rate spikes')
    if today_nulls:
        for n in today_nulls:
            icon = {'warning':'⚠️ ','critical':'🚨'}[n['severity']]
            print(f'     {icon} {n["column"]:<24} baseline={n["baseline_pct"]:.1f}%   '
                  f'today={n["current_pct"]:.1f}%   Δ={n["jump_pp"]:+.1f}pp')
    else:
        print('     ✅ No spikes detected.')

    # ── Schema drift ──────────────────────────────────────────────────────
    expected_cols = {'created_at','account_segment','platform','category','has_billing_profile'}
    findings['schema'] = _detect_schema_drift(df_inq.columns, expected_cols)
    s = findings['schema']
    icon = {'ok':'✅','warning':'⚠️ ','critical':'🚨'}[s['severity']]
    print(f'\n  [5/5] Schema drift')
    print(f'     {icon} New: {s["new_columns"] or "—"}   Missing: {s["missing_columns"] or "—"}')

    # ── Aggregate severity ───────────────────────────────────────────────────
    severities = []
    for _k, _v in findings.items():
        if isinstance(_v, list):
            severities.extend(item['severity'] for item in _v)
        elif isinstance(_v, dict):
            severities.append(_v.get('severity', 'info'))
    if 'critical' in severities:
        overall = '🚨 CRITICAL'
    elif 'warning' in severities:
        overall = '⚠️  WARNING'
    else:
        overall = '✅ HEALTHY'

    print('\n' + '─' * 72)
    print(f'  Overall pipeline status: {overall}')
    print('─' * 72)

    # ── LLM narration ────────────────────────────────────────────────────────
    print('\n  🤖 Generating plain-English summary via LLM...')
    findings_summary = json.dumps(
        {fk: (fv if not isinstance(fv, list) else fv[:5]) for fk, fv in findings.items()},
        default=str, indent=2)[:3000]

    narration_prompt = textwrap.dedent(f"""
        You are a senior data engineer reviewing today's pipeline health.
        Below are the structured findings from automated checks.
        Write a 4-6 sentence executive-friendly summary explaining:
        (1) the overall health,
        (2) the most concerning finding (if any) and likely cause,
        (3) what action the team should take next.
        Be specific. Avoid generic advice. Do not use emojis.

        Findings:
        {findings_summary}
    """).strip()

    try:
        narration = llm.ask(narration_prompt)
        try:
            narration = _strip_decorative_chars(narration)
        except NameError:
            pass
    except Exception as e:
        narration = f'(LLM narration failed: {e})'

    print('\n  ── Narrative ──')
    for line in narration.split('\n'):
        if line.strip(): print(f'    {line}')

    # ── PDF report ───────────────────────────────────────────────────────────
    from collections import OrderedDict
    overall_plain = overall.replace('🚨 ', '').replace('⚠️  ', '').replace('✅ ', '')
    pdf_sections = OrderedDict([
        ('STATUS', f'{overall_plain}  —  scan completed at {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}'),
        ('NARRATIVE', narration),
        ('VOLUME DRIFT',
            (f'- Today: {v["today_value"]:.0f} inquiries\n'
             f'- 28-day baseline: {v["baseline_mean"]:.0f}\n'
             f'- Change: {v["pct_change"]:+.1f}%\n'
             f'- z-score: {v["z_score"]:+.2f}\n'
             f'- Severity: {v["severity"].upper()}') if v['status'] == 'analysed'
            else 'Insufficient data for this check.'),
        ('DISTRIBUTION SHIFTS',
            (f'- Category split: χ²={d["chi2"]:.2f}, p={d["p_value"]:.4f}, severity={d["severity"].upper()}\n'
             f'- Platform split: χ²={p["chi2"]:.2f}, p={p["p_value"]:.4f}, severity={p["severity"].upper()}'
             ) if d['status'] == 'analysed' else 'Insufficient data.'),
        ('FRESHNESS',
            f'- Latest record: {f.get("latest", "n/a")}\n'
            f'- Age: {f.get("age_hours", 0):.1f} hours\n'
            f'- SLA: {f.get("sla_hours", 0)} hours\n'
            f'- Severity: {f["severity"].upper()}'),
        ('NULL-RATE SPIKES',
            ('\n'.join(
                f'- {n["column"]}: baseline {n["baseline_pct"]:.1f}% → '
                f'today {n["current_pct"]:.1f}% (Δ {n["jump_pp"]:+.1f}pp), {n["severity"].upper()}'
                for n in today_nulls)
             ) or 'No null-rate spikes detected.'),
        ('SCHEMA DRIFT',
            f'- New columns: {", ".join(s["new_columns"]) or "none"}\n'
            f'- Missing columns: {", ".join(s["missing_columns"]) or "none"}\n'
            f'- Severity: {s["severity"].upper()}'),
    ])

    accent = (PDF_PALETTE.get('warning', '#f59e0b') if 'WARNING' in overall
              else PDF_PALETTE.get('success', '#22c55e') if 'HEALTHY' in overall
              else PDF_PALETTE.get('secondary', '#f97316'))
    pdf_out = render_document_pdf(
        title='Pipeline Health Report',
        subtitle=f'Daily scan — {pd.Timestamp.now().strftime("%Y-%m-%d")}',
        sections=pdf_sections,
        output_path='pipeline_health_report.pdf',
        metadata={
            'Overall':      overall_plain,
            'Tables scanned': '1 (hist_inquiries)',
            'Checks run':   '5',
            'Critical':     str(sum(1 for s in severities if s == 'critical')),
            'Warnings':     str(sum(1 for s in severities if s == 'warning')),
        },
        accent_color=accent,
    )
    print(f'\n  📁 Pipeline health report saved → {pdf_out}')

    return {
        'overall':    overall,
        'findings':   findings,
        'narrative':  narration,
        'pdf_report': pdf_out,
    }


# ─────────────────────────────────────────────────────────────────────────────
# MODULE 14 — WATCHTOWER (Phase 0 · Foundation)
# ─────────────────────────────────────────────────────────────────────────────

WATCHTOWER_METRICS = [
    {'name': 'IOR',         'sql_numerator': 'SUM(CAST(converted_to_order AS INTEGER))',
                             'sql_denominator': 'COUNT(*)',      'table': 'hist_inquiries'},
    {'name': 'Volume',      'sql_numerator': 'COUNT(*)',
                             'sql_denominator': None,            'table': 'hist_inquiries'},
    {'name': 'AOV',         'sql_numerator': 'AVG(CASE WHEN converted_to_order THEN order_value END)',
                             'sql_denominator': None,            'table': 'hist_inquiries'},
]

WATCHTOWER_DIMENSIONS = ['account_segment', 'platform', 'price_tier', 'process_group']
WATCHTOWER_ALERT_Z    = 2.5   
WATCHTOWER_BASELINE   = 28    


def _compute_metric_series(metric: dict, dim: str, level: str,
                             start_date: 'pd.Timestamp',
                             end_date:   'pd.Timestamp') -> 'pd.Series':
    """
    Compute daily time series for a metric at a specific dimensional slice.
    Returns a pandas Series indexed by date.
    """
    tbl = metric['table']
    num = metric['sql_numerator']
    den = metric['sql_denominator']

    dim_filter = f"AND {dim} = '{level}'" if dim != 'all' else ''

    if den:
        metric_expr = f'({num}) / NULLIF(({den}), 0)'
    else:
        metric_expr = num

    sql = f"""
        SELECT
            created_at::DATE AS day,
            {metric_expr}    AS metric_value
        FROM "{tbl}"
        WHERE created_at::DATE BETWEEN '{start_date.date()}' AND '{end_date.date()}'
          {dim_filter}
        GROUP BY created_at::DATE
        ORDER BY day
    """
    try:
        df = db.execute(sql).df()
        if df.empty:
            return pd.Series(dtype=float)
        df['day'] = pd.to_datetime(df['day'])
        return df.set_index('day')['metric_value'].astype(float)
    except Exception as e:
        return pd.Series(dtype=float)


def _detect_slice_anomaly(series: 'pd.Series',
                           baseline_days: int = WATCHTOWER_BASELINE,
                           alert_z: float = WATCHTOWER_ALERT_Z) -> dict:
    """
    Run the same day-of-week adjusted anomaly detection as Module 2's
    _detect_volume_anomaly(), applied to a single dimensional slice.
    """
    if len(series) < baseline_days + 1:
        return {'status': 'insufficient_data', 'severity': 'info'}

    history = series.iloc[-(baseline_days + 1):-1]
    today   = float(series.iloc[-1])
    if pd.isna(today):
        return {'status': 'no_data', 'severity': 'warning'}

    today_dow = series.index[-1].dayofweek
    same_dow  = history[history.index.dayofweek == today_dow]
    if len(same_dow) >= 3:
        baseline_mean = float(same_dow.mean())
        baseline_std  = float(same_dow.std()) or 1.0
    else:
        baseline_mean = float(history.mean())
        baseline_std  = float(history.std()) or 1.0

    z          = (today - baseline_mean) / baseline_std
    pct_change = (today - baseline_mean) / baseline_mean * 100 if baseline_mean else 0
    severity   = 'critical' if abs(z) > alert_z else                  'warning'  if abs(z) > 1.5     else 'ok'

    return {
        'status':        'analysed',
        'today_value':   round(today, 6),
        'baseline_mean': round(baseline_mean, 6),
        'z_score':       round(z, 3),
        'pct_change':    round(pct_change, 2),
        'severity':      severity,
    }


def _cross_reference_experiments(anomaly_dim: str, anomaly_level: str,
                                   anomaly_date: 'pd.Timestamp') -> list:
    """
    Check if any running experiment overlaps with the anomaly's dimension / level.
    Returns a list of experiment names that could explain the anomaly.
    """
    matches = []
    for exp in globals().get('EXPERIMENT_REGISTRY', []):
        if exp.get('status') not in ('running', 'concluded'):
            continue
        start = pd.Timestamp(exp['start_date'])
        end   = pd.Timestamp(exp['end_date']) if exp.get('end_date') else pd.Timestamp.now()
        if start <= anomaly_date <= end:
            matches.append({
                'experiment_name': exp['experiment_name'],
                'status':          exp['status'],
                'team':            exp.get('team', '—'),
                'start':           str(exp['start_date']),
            })
    return matches


def _cross_reference_pipeline_baseline(metric_name: str, dim: str, level: str,
                                         z_score: float) -> str:
    """
    Check if the alert direction is consistent with the pipeline health baseline.
    If the pipeline baseline shows a historical volume drop in the same slice,
    the anomaly may be a data pipeline issue rather than a business metric change.
    Returns a diagnosis hint string.
    """
    baseline = globals().get('CONTINUM_STATE', {}).get('pipeline_baseline')
    if not baseline:
        return ''

    # Simple heuristic: if this is a volume metric and the baseline was already
    # CRITICAL/WARNING when established, flag as potentially pipeline-related
    overall = baseline.get('overall', '')
    if metric_name == 'Volume' and 'CRITICAL' in str(overall):
        return '⚠️ Pipeline baseline was CRITICAL — may be a data issue'
    if metric_name == 'Volume' and 'WARNING' in str(overall):
        return '⚠️ Pipeline baseline was WARNING — check Module 2 first'

    return ''


def run_watchtower(llm):
    """
    Module 14 — Watchtower Dimensional Anomaly Detection.

    Scans every metric × dimension × level combination for anomalies.
    Cross-references running experiments to disambiguate real signals
    from experiment effects or technical failures.
    Produces a structured alert table + LLM narration + PDF report.
    """
    print()
    print('╔' + '═'*70 + '╗')
    print('║' + '  🔭  WATCHTOWER — Dimensional Anomaly Detection'.ljust(70) + '║')
    print('║' + '  Phase 0 · Foundation · Module 14'.ljust(70) + '║')
    print('║' + '  Monitoring: IOR · Volume · AOV  ×  Segment · Platform · Tier · Process'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')

    try:
        date_range = db.execute(
            'SELECT MIN(created_at)::DATE, MAX(created_at)::DATE FROM hist_inquiries'
        ).fetchone()
        hist_start = pd.Timestamp(date_range[0])
        hist_end   = pd.Timestamp(date_range[1])
    except Exception as e:
        print(f'  ❌ Could not determine date range: {e}')
        return None

    scan_start = hist_end - pd.Timedelta(days=WATCHTOWER_BASELINE + 7)
    print(f'\n  Scan window : {scan_start.date()} → {hist_end.date()}')
    print(f'  Baseline    : {WATCHTOWER_BASELINE} days (day-of-week adjusted)')
    print(f'  Alert threshold: |z| > {WATCHTOWER_ALERT_Z}')
    print()

    alerts     = []  
    scan_count = 0

    print('  Scanning metric × dimension × level...')
    print('  (This may take 20–40 seconds for large datasets)')
    print()

    for metric in WATCHTOWER_METRICS:
        for dim in WATCHTOWER_DIMENSIONS:
            try:
                levels = db.execute(
                    f'SELECT DISTINCT {dim} FROM hist_inquiries '
                    f'WHERE {dim} IS NOT NULL ORDER BY {dim}'
                ).df()[dim].tolist()
            except Exception:
                continue

            for level in levels:
                series = _compute_metric_series(
                    metric, dim, level, scan_start, hist_end)
                if series.empty or len(series) < 5:
                    continue

                result = _detect_slice_anomaly(series)
                scan_count += 1

                if result['status'] == 'analysed' and result['severity'] in ('warning', 'critical'):
                    # Cross-reference with running experiments
                    xref = _cross_reference_experiments(dim, level, hist_end)

                    _pipeline_hint = _cross_reference_pipeline_baseline(
                        metric['name'], dim, level, result['z_score'])
                    alerts.append({
                        'metric':         metric['name'],
                        'dimension':      dim,
                        'level':          str(level),
                        'z_score':        result['z_score'],
                        'pct_change':     result['pct_change'],
                        'today_value':    result['today_value'],
                        'baseline':       result['baseline_mean'],
                        'severity':       result['severity'],
                        'experiments':    xref,
                        'pipeline_hint':  _pipeline_hint,
                    })

    sev_order = {'critical': 0, 'warning': 1}
    alerts.sort(key=lambda a: (sev_order.get(a['severity'], 2), -abs(a['z_score'])))

    print(f'  Scanned {scan_count:,} metric × slice combinations.')
    n_critical = sum(1 for a in alerts if a['severity'] == 'critical')
    n_warning  = sum(1 for a in alerts if a['severity'] == 'warning')
    print(f'  Found: {n_critical} critical alert(s), {n_warning} warning(s).')

    # ── Display alert table ───────────────────────────────────────────────────
    if not alerts:
        print()
        print('  ✅ No anomalies detected. All metric × slice combinations are within bounds.')
    else:
        print()
        print('  ┌──────────────┬────────────────────┬────────────┬─────────┬──────────┬──────────────────────────────┐')
        print('  │ Severity     │ Metric × Slice     │ Today val  │ z-score │ Δ%       │ Experiments overlapping      │')
        print('  ├──────────────┼────────────────────┼────────────┼─────────┼──────────┼──────────────────────────────┤')
        for a in alerts[:20]:   # cap display at 20
            sev_str  = '🚨 CRITICAL' if a['severity'] == 'critical' else '⚠️  WARNING '
            slice_str = f'{a["dimension"]}={a["level"]}'[:18]
            metric_str= f'{a["metric"]} · {slice_str}'[:18]
            today_str = f'{a["today_value"]:.4f}'
            z_str     = f'{a["z_score"]:+.2f}'
            pct_str   = f'{a["pct_change"]:+.1f}%'
            exp_names = ', '.join(x['experiment_name'][:15] for x in a['experiments'][:2]) or '—'
            hint_str = a.get('pipeline_hint','')
            note = '🔧' if hint_str else ' '
            print(f'  │ {sev_str:<12} │ {metric_str:<18} │ {today_str:<10} │ {z_str:<7} │ {pct_str:<8} │ {note} {exp_names:<26} │')
        print('  └──────────────┴────────────────────┴────────────┴─────────┴──────────┴──────────────────────────────┘')

        if len(alerts) > 20:
            print(f'  ... and {len(alerts) - 20} more. See PDF report for full list.')

    # ── LLM narration ─────────────────────────────────────────────────────────
    print()
    print('  🤖 Generating Watchtower narrative...')

    alert_summary = '\n'.join(
        f'{a["severity"].upper()}: {a["metric"]} × {a["dimension"]}={a["level"]} '
        f'z={a["z_score"]:+.2f} ({a["pct_change"]:+.1f}%) '
        f'overlaps_with=[{", ".join(x["experiment_name"] for x in a["experiments"][:2])}]'
        for a in alerts[:10]
    ) or 'No anomalies detected.'

    narration_prompt = textwrap.dedent(f"""
        You are a senior data engineer reviewing today's Watchtower scan.
        Write a 4-6 sentence executive summary covering:
        (1) Overall status — how many critical alerts vs normal.
        (2) The most urgent finding and the most likely cause (technical failure,
            experiment effect, or genuine business change).
        (3) Whether any alerts correlate with running experiments and what that implies.
        (4) Recommended immediate action.
        Be specific. Avoid generic advice. Do not use emojis.

        Alert summary:
        {alert_summary}
    """).strip()

    try:
        narration = llm.ask(narration_prompt)
        try:
            narration = _strip_decorative_chars(narration)
        except NameError:
            pass
    except Exception as e:
        narration = f'(LLM narration unavailable: {e})'

    print()
    print('  ── Watchtower Narrative ──')
    for line in narration.split('\n'):
        if line.strip():
            print(f'    {line}')

    # ── PDF report ────────────────────────────────────────────────────────────
    from collections import OrderedDict
    alert_table_str = '\n'.join(
        f'- [{a["severity"].upper()}] {a["metric"]} × {a["dimension"]}={a["level"]}: '
        f'z={a["z_score"]:+.2f}, Δ={a["pct_change"]:+.1f}%'
        + (f', experiments: {", ".join(x["experiment_name"] for x in a["experiments"][:2])}'
           if a["experiments"] else '')
        for a in alerts
    ) or 'No anomalies detected.'

    pdf_sections = OrderedDict([
        ('STATUS',
            f'{n_critical} CRITICAL · {n_warning} WARNING · scanned {scan_count:,} combinations'),
        ('NARRATIVE', narration),
        ('ALERT DETAILS', alert_table_str),
        ('EXPERIMENT CROSS-REFERENCE',
            '\n'.join(
                f'- [{a["severity"].upper()}] {a["metric"]} × {a["dimension"]}={a["level"]} '
                f'overlaps: {", ".join(x["experiment_name"] for x in a["experiments"])}'
                for a in alerts if a["experiments"]
            ) or 'No anomalies overlap with running or recently concluded experiments.'),
        ('RECOMMENDED ACTIONS',
            '- For CRITICAL alerts not overlapping experiments: investigate as technical failure.\n'
            '- For alerts overlapping a running experiment: check Module 8 (Health Monitor).\n'
            '- For gradual declines over multiple days: run Module 2 (Pipeline Health) to rule out data issues.\n'
            '- For alerts in a recently shipped segment: consider pausing rollout and running Module 12 (ROI Tracker).'),
    ])

    overall_plain = 'CRITICAL' if n_critical > 0 else ('WARNING' if n_warning > 0 else 'HEALTHY')
    accent = (PDF_PALETTE.get('secondary', '#f97316') if n_critical > 0
              else PDF_PALETTE.get('warning', '#f59e0b') if n_warning > 0
              else PDF_PALETTE.get('success', '#22c55e'))
    pdf_out = render_document_pdf(
        title='Watchtower Report',
        subtitle=f'Dimensional anomaly scan — {hist_end.date()}',
        sections=pdf_sections,
        output_path='watchtower_report.pdf',
        metadata={
            'Scan date':       str(hist_end.date()),
            'Slices scanned':  str(scan_count),
            'Critical alerts': str(n_critical),
            'Warnings':        str(n_warning),
            'Overall':         overall_plain,
        },
        accent_color=accent,
    )
    print(f'\n  📁 Watchtower report saved → {pdf_out}')

    return {
        'overall':       overall_plain,
        'alerts':        alerts,
        'n_critical':    n_critical,
        'n_warning':     n_warning,
        'scan_count':    scan_count,
        'narrative':     narration,
        'pdf_report':    pdf_out,
    }


## 9 · Phase 1 — Planning Modules

Module [1] Brief + Method Recommendation · [2] Opportunity Sizing · [3] Power Calculator · [4] KPI & Tracking Plan · [5] Audience Selection.

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# BALANCE TEST ENGINE — Covariate balance validation for experiment assignment
# ─────────────────────────────────────────────────────────────────────────────


from scipy.stats import ttest_ind, chi2_contingency
SMD_THRESHOLD   = 0.10  
BALANCE_MAX_ITER = 10   

BALANCE_COVARIATES = [
    ('account_segment',    'categorical'),
    ('platform',           'categorical'),
    ('lifetime_orders',    'continuous'),
    ('personal_ior',       'continuous'),
    ('avg_order_value',    'continuous'),
    ('days_since_last',    'continuous'),
    ('n_inquiries',        'continuous'),
]


def _compute_smd(vals_a: 'pd.Series', vals_b: 'pd.Series') -> float:
    """
    Standardised Mean Difference between two groups on a continuous covariate.
    SMD = (mean_a - mean_b) / pooled_SD
    """
    n_a, n_b = len(vals_a.dropna()), len(vals_b.dropna())
    if n_a < 5 or n_b < 5:
        return float('nan')
    mu_a, mu_b = vals_a.mean(), vals_b.mean()
    sd_a, sd_b = vals_a.std(), vals_b.std()
    pooled_sd  = np.sqrt(((n_a - 1) * sd_a**2 + (n_b - 1) * sd_b**2) / (n_a + n_b - 2))
    if pooled_sd == 0:
        return 0.0
    return float(abs(mu_a - mu_b) / pooled_sd)


def _run_balance_battery(assignments: 'pd.DataFrame') -> dict:
    """
    Run the full covariate balance battery on the assignment DataFrame.

    assignments must have columns: buyer_id, group, + covariate columns
    (merged from df_buyers + computed features during propensity scoring).

    Returns a dict: {covariate: {smd, p_value, mean_ctrl, mean_trt, flag}}
    """
    groups   = sorted(assignments['group'].unique())
    control  = 'control' if 'control' in groups else groups[0]
    treatment_groups = [g for g in groups if g != control]
    ctrl_df  = assignments[assignments['group'] == control]

    results = {}
    for cov_name, ctype in BALANCE_COVARIATES:
        if cov_name not in assignments.columns:
            continue
        for trt in treatment_groups:
            trt_df = assignments[assignments['group'] == trt]
            key    = f'{cov_name}' if len(treatment_groups) == 1 else f'{cov_name}__{trt}'
            row    = {'covariate': cov_name, 'treatment': trt, 'type': ctype}

            if ctype == 'continuous':
                a = pd.to_numeric(ctrl_df[cov_name], errors='coerce').dropna()
                b = pd.to_numeric(trt_df[cov_name],  errors='coerce').dropna()
                if len(a) < 5 or len(b) < 5:
                    continue
                smd    = _compute_smd(a, b)
                _, pv  = ttest_ind(a, b, equal_var=False)
                row.update({
                    'smd':      round(smd, 4),
                    'p_value':  round(float(pv), 4),
                    'mean_ctrl': round(float(a.mean()), 4),
                    'mean_trt':  round(float(b.mean()), 4),
                    'flag':      smd > SMD_THRESHOLD,
                })

            else:  # categorical
                ctrl_counts = ctrl_df[cov_name].value_counts()
                trt_counts  = trt_df[cov_name].value_counts()
                all_cats    = sorted(set(ctrl_counts.index) | set(trt_counts.index))
                contingency = np.array([
                    [ctrl_counts.get(c, 0) for c in all_cats],
                    [trt_counts.get(c, 0)  for c in all_cats],
                ])
                try:
                    _, pv, _, _ = chi2_contingency(contingency)
                except Exception:
                    pv = 1.0
                # For categorical, use the max proportion difference as SMD proxy
                ctrl_prop = ctrl_counts / ctrl_counts.sum()
                trt_prop  = trt_counts  / trt_counts.sum()
                max_diff  = float((ctrl_prop - trt_prop).abs().max()) if not trt_prop.empty else 0.0
                row.update({
                    'smd':      round(max_diff, 4),
                    'p_value':  round(float(pv), 4),
                    'mean_ctrl': str(ctrl_counts.idxmax()) if not ctrl_counts.empty else '—',
                    'mean_trt':  str(trt_counts.idxmax())  if not trt_counts.empty else '—',
                    'flag':      pv < 0.05,
                })

            results[key] = row

    return results


def _print_balance_report(results: dict, n_ctrl: int, n_trt: int) -> bool:
    """
    Print the balance table. Returns True if all checks pass, False if any flag.
    """
    n_flags = sum(1 for r in results.values() if r.get('flag', False))
    print()
    print('  ── Covariate Balance Report ──────────────────────────────────────────')
    print(f'  {"Covariate":<26} {"Ctrl mean":<16} {"Trt mean":<16} {"SMD":>6}  {"p-val":>6}  {"Status"}')
    print('  ' + '─'*80)

    for key, r in results.items():
        icon   = '⚠️ ' if r.get('flag') else '✅ '
        smd    = f'{r["smd"]:.4f}' if isinstance(r["smd"], float) else '—'
        pv     = f'{r["p_value"]:.4f}'
        mc     = f'{r["mean_ctrl"]:.4f}' if isinstance(r["mean_ctrl"], float) else str(r["mean_ctrl"])
        mt     = f'{r["mean_trt"]:.4f}'  if isinstance(r["mean_trt"],  float) else str(r["mean_trt"])
        print(f'  {r["covariate"]:<26} {mc:<16} {mt:<16} {smd:>6}  {pv:>6}  {icon}')

    print('  ' + '─'*80)
    print(f'  n(control)={n_ctrl:,}  n(treatment)={n_trt:,}')
    threshold_note = f'  SMD threshold: {SMD_THRESHOLD}  (flag if SMD > {SMD_THRESHOLD} or p < 0.05 for categoricals)'
    print(threshold_note)

    if n_flags == 0:
        print()
        print('  ✅ Balance: PASS — all covariates within acceptable bounds.')
        print('     Groups are statistically equivalent. Safe to launch.')
    else:
        print()
        print(f'  ⚠️  Balance: FAIL — {n_flags} covariate(s) flagged.')
        print('     Imbalanced groups risk confounding the experiment results.')

    return n_flags == 0


def _plot_love_plot(balance_results: dict, exp_name: str = '') -> str:
    """
    Generate a Love plot (Austin 2009) showing SMD per covariate.
    A vertical dashed line at SMD=0.10 marks the balance threshold.
    Returns the saved filename.
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    cov_list = [r['covariate'] for r in balance_results.values()]
    smds     = [r['smd'] if isinstance(r['smd'], float) else 0.0
                for r in balance_results.values()]
    flags    = [r.get('flag', False) for r in balance_results.values()]

    if not cov_list:
        return None

    fig, ax = plt.subplots(figsize=(8, max(3, len(cov_list) * 0.55)))
    colors = ['#E74C3C' if f else '#2ECC71' for f in flags]
    y_pos  = range(len(cov_list))

    ax.barh(list(y_pos), smds, color=colors, alpha=0.80, edgecolor='white', height=0.65)
    ax.axvline(x=SMD_THRESHOLD, color='#E74C3C', linestyle='--', linewidth=1.4,
               label=f'Threshold (SMD={SMD_THRESHOLD})', alpha=0.7)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(cov_list, fontsize=10)
    ax.set_xlabel('Standardised Mean Difference (SMD)', fontsize=10)
    title = f'Love Plot — Covariate Balance{" · " + exp_name if exp_name else ""}'
    ax.set_title(title, fontsize=11, fontweight='bold', color='#1B4F72')
    ax.axvline(x=0, color='grey', linewidth=0.5, alpha=0.4)
    ax.set_xlim(left=0)

    pass_patch  = mpatches.Patch(color='#2ECC71', alpha=0.8, label='Balanced (SMD ≤ 0.10)')
    fail_patch  = mpatches.Patch(color='#E74C3C', alpha=0.8, label='Imbalanced (SMD > 0.10)')
    ax.legend(handles=[pass_patch, fail_patch,
                        plt.Line2D([0],[0], color='#E74C3C', linestyle='--', label=f'Threshold ({SMD_THRESHOLD})')],
              fontsize=9, loc='lower right')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()

    fname = f'balance_love_plot{"_" + exp_name if exp_name else ""}.png'
    plt.savefig(fname, dpi=130, bbox_inches='tight')
    plt.close()
    return fname


def _rerandomise_until_balanced(features_df, n_per_group, n_groups,
                                 exp_name, max_iter=BALANCE_MAX_ITER):
    """
    Re-draw group assignment until covariate balance passes, or max_iter is reached.

    Uses simple random re-assignment (no complex optimisation) — this is the
    rerandomisation approach from Morgan & Rudin (2012): draw randomly, test
    balance, reject and redraw if balance fails.

    Returns (assignments_df, balance_results, passed: bool)
    """
    group_names = ['control'] + [f'treatment_{i}' if n_groups > 2 else 'treatment'
                                  for i in range(1, n_groups)]
    n_total = n_per_group * n_groups
    eligible = features_df.sample(min(n_total, len(features_df)),
                                   random_state=None).reset_index(drop=True)

    for attempt in range(1, max_iter + 1):
        # Random shuffle and assign groups
        shuffled = eligible.sample(frac=1, random_state=attempt * 17).reset_index(drop=True)
        shuffled['group'] = np.repeat(group_names,
                                       [len(shuffled) // n_groups + (1 if i < len(shuffled) % n_groups else 0)
                                        for i in range(n_groups)])[:len(shuffled)]
        shuffled['experiment_name']  = exp_name
        shuffled['selection_mode']   = 'propensity_balanced'
        shuffled['propensity_score'] = features_df.get('propensity_score',
                                        pd.Series(np.nan, index=shuffled.index))

        balance = _run_balance_battery(shuffled)
        passed  = all(not r.get('flag', False) for r in balance.values())

        if passed:
            print(f'     ✅ Balance achieved on attempt {attempt}/{max_iter}')
            return shuffled, balance, True

        if attempt < max_iter:
            n_flags = sum(1 for r in balance.values() if r.get('flag'))
            print(f'     ↩️  Attempt {attempt}: {n_flags} flag(s) — re-drawing...')

    print(f'     ⚠️  Balance not achieved after {max_iter} attempts.')
    print('         Proceeding with best available assignment.')
    print('         Consider reducing MDE or increasing sample size.')
    return shuffled, balance, False


# ─────────────────────────────────────────────────────────────────────────────
# FUNNEL TAXONOMY
# ─────────────────────────────────────────────────────────────────────────────

FUNNEL_TAXONOMY = {
    'acquisition': {
        'label':       'Traffic & Sign-Up (Acquisition)',
        'description': '''
            This stage focuses on all top-of-funnel activities responsible for bringing new users 
            into the ecosystem and converting them into registered or identifiable users.
            It includes both paid and organic acquisition channels such as SEO, ads, referrals, 
            partnerships, and direct traffic. Additionally, it covers the full authentication 
            experience including login, signup, and onboarding entry points.

            The primary goal at this stage is to maximize high-quality traffic and efficiently 
            convert visitors into signed-up users while maintaining cost efficiency and traffic relevance.
        ''',
        'keywords':    ['login','sign','auth','oauth','social','register','onboard',
                        'traffic','landing','seo','ad','referral','invite'],
        'primary_metric': 'sign_up_rate',

        'questions': [
            ('monthly_visitors',    'Monthly unique visitors to the page/funnel',  'visitors/month', False),
            ('signup_rate',         'Current sign-up or sign-in rate (%)',          'rate %',         True),
            ('activation_rate',     'Of new sign-ups, % who submit a first inquiry within 30 days', 'rate %', True),
            ('aov',                 'Average order value ($)',                      'dollars',        False),
            ('gross_margin',        'Gross margin (%)',                             'rate %',         True),
            ('horizon',             'Time horizon for sizing (months)',             'months',         False),

            ('bounce_rate',         'Percentage of visitors who leave without interaction', 'rate %', False),
            ('traffic_quality_score','Weighted score of traffic based on downstream conversion', 'score', False),
            ('cost_per_visitor',    'Average acquisition cost per visitor ($)', 'dollars', False),
            ('cost_per_signup',     'Customer acquisition cost per signup ($)', 'dollars', False),
            ('channel_mix',         'Traffic distribution by channel (%)', 'distribution %', False),
            ('form_completion_rate','% users who start vs complete signup form', 'rate %', False),
        ],

        'downstream': [('activation_rate', None), ('ior', None)],

        'tracking_plan': '''
            - Track source/medium, campaign, and keyword attribution using UTM parameters.
            - Implement event tracking for: page_view → signup_start → signup_complete.
            - Measure drop-offs at each field level in signup forms (field analytics).
            - Cohort users by acquisition channel to evaluate downstream activation and revenue quality.
            - Use multi-touch attribution models to understand contribution of channels.
            - Track device, geography, and page load speed as influencing factors.
            - Set up funnel visualization dashboards (e.g., visitor → signup → activation).
        ''',

        'mde_benchmarks': {'typical_min_rel': 0.10, 'typical_max_rel': 0.30,
                           'label': '10–30% relative lift on sign-up rate'},
    },

    'activation': {
        'label':       'Activation & Onboarding',
        'description': '''
            This stage ensures that newly acquired users reach their first meaningful action 
            (activation milestone), which strongly correlates with long-term retention and monetization.
            It includes onboarding flows, tutorials, guided setups, nudges, lifecycle emails, 
            and any mechanism that helps users realize product value quickly.

            The focus is on reducing time-to-value (TTV) and eliminating friction in early user experience.
        ''',
        'keywords':    ['onboard','first','welcome','setup','profile','complete','wizard',
                        'activation','getting started','tutorial','nudge','email'],
        'primary_metric': 'activation_rate',

        'questions': [
            ('monthly_signups',     'Monthly new sign-ups (users entering onboarding)', 'users/month', False),
            ('activation_rate',     'Current % who complete the activation step (%)',  'rate %',      True),
            ('ior',                 'IOR for activated users (%)',                      'rate %',      True),
            ('aov',                 'Average order value ($)',                          'dollars',     False),
            ('gross_margin',        'Gross margin (%)',                                 'rate %',      True),
            ('horizon',             'Time horizon (months)',                             'months',      False),

            ('time_to_activation',  'Median time taken to reach activation (hours/days)', 'time', False),
            ('onboarding_completion_rate','% users completing onboarding flow', 'rate %', False),
            ('drop_off_step',       'Step with highest drop-off in onboarding funnel', 'step index', False),
            ('nudge_effectiveness', '% lift in activation due to nudges/emails', 'rate %', False),
            ('feature_adoption_rate','% users using key features within first session', 'rate %', False),
        ],

        'downstream': [('ior', None)],

        'tracking_plan': '''
            - Track step-by-step onboarding funnel with timestamps.
            - Instrument key activation events (e.g., profile completion, first action).
            - Use cohort analysis to compare activation across signup dates and channels.
            - Measure impact of lifecycle messaging (email, push) on activation.
            - Track time-to-first-key-action as a critical KPI.
            - Run A/B tests on onboarding variants (guided vs unguided).
            - Capture qualitative feedback (surveys, session recordings).
        ''',

        'mde_benchmarks': {'typical_min_rel': 0.05, 'typical_max_rel': 0.20,
                           'label': '5–20% relative lift on activation rate'},
    },

    'conversion': {
        'label':       'Conversion Rate (Checkout / Funnel / Lead-to-Sale)',
        'description': '''
            This stage captures the efficiency of converting high-intent users into paying customers.
            It includes the entire purchase journey: browsing/browsing to cart, cart to checkout,
            checkout to payment, payment confirmation. Applies equally to e-commerce (cart → purchase),
            marketplace (inquiry/quote → order), SaaS (trial → paid), and retail (browse → buy).

            Optimization here directly impacts revenue and is often sensitive to friction, trust,
            pricing clarity, and UX performance.
        ''',
        'keywords':    ['checkout','funnel','cart','billing','payment','order','quote',
                        'ior','conversion','buy','purchase','accept','confirm','submit',
                        'lead','sale','transaction','basket'],
        'primary_metric': 'ior', 

        'questions': [
            ('monthly_inquiries',   'Monthly inquiries / leads / carts / quotes (primary volume metric)',  'units/month', False),
            ('ior',                 'Current conversion rate (cart-to-purchase, quote-to-order, lead-to-sale %) ', 'rate %', True),
            ('aov',                 'Average order value ($)',                  'dollars',         False),
            ('gross_margin',        'Gross margin (%)',                         'rate %',          True),
            ('horizon',             'Time horizon (months)',                    'months',          False),

            ('checkout_dropoff_rate','% users dropping off during checkout', 'rate %', False),
            ('payment_success_rate','% successful payments vs attempts', 'rate %', False),
            ('error_rate',          'Technical or validation error rate during checkout', 'rate %', False),
            ('avg_checkout_time',   'Average time to complete checkout', 'time', False),
            ('cart_abandonment_rate','% carts abandoned before purchase', 'rate %', False),
        ],

        'downstream': [],

        'tracking_plan': '''
            - Track each step in checkout funnel (cart → address → payment → confirmation).
            - Capture payment failures with detailed error codes.
            - Monitor latency and page performance across checkout steps.
            - Segment conversion by device, payment method, and geography.
            - Track coupon usage and pricing exposure.
            - Run funnel A/B tests (e.g., fewer steps, guest checkout).
            - Implement session replay to identify UX friction.
        ''',

        'mde_benchmarks': {'typical_min_rel': 0.05, 'typical_max_rel': 0.15,
                           'label': '5–15% relative IOR lift (0.5–2pp absolute)'},
    },

    'retention': {
        'label':       'Repeat Orders & Retention',
        'description': '''
            This stage focuses on maximizing customer lifetime value by encouraging repeat usage 
            and reducing churn. It includes post-purchase experiences, re-engagement campaigns, 
            loyalty programs, and personalized recommendations.

            Strong retention indicates product-market fit and sustainable growth.
        ''',
        'keywords':    ['repeat','retention','return','reorder','summary','post.order',
                        'ltv','churn','re-engage','loyalty','upsell','cross-sell'],
        'primary_metric': 'repeat_order_rate',

        'questions': [
            ('monthly_orders',      'Monthly completed orders',                  'orders/month', False),
            ('repeat_rate',         'Current repeat order rate (%)',              'rate %',       True),
            ('aov',                 'Average order value for repeat orders ($)',  'dollars',      False),
            ('gross_margin',        'Gross margin (%)',                           'rate %',       True),
            ('horizon',             'Time horizon (months)',                      'months',       False),

            ('customer_ltv',        'Customer lifetime value ($)', 'dollars', False),
            ('churn_rate',          'Percentage of users not returning', 'rate %', False),
            ('repeat_frequency',    'Average number of repeat purchases per user', 'count', False),
            ('cohort_retention',    'Retention rate by cohort over time', 'rate %', False),
            ('email_reengagement_rate','% users reactivated via campaigns', 'rate %', False),
        ],

        'downstream': [],

        'tracking_plan': '''
            - Build cohort retention tables (weekly/monthly cohorts).
            - Track repeat purchase intervals and frequency distribution.
            - Attribute repeat orders to re-engagement campaigns.
            - Monitor churn signals (inactivity duration, drop in usage).
            - Track LTV by acquisition source.
            - Measure effectiveness of loyalty programs and incentives.
        ''',

        'mde_benchmarks': {'typical_min_rel': 0.05, 'typical_max_rel': 0.15,
                           'label': '5–15% relative lift on repeat order rate'},
    },

    'engagement': {
        'label':       'UI / UX & Engagement',
        'description': '''
            This stage includes all improvements related to user interaction, interface design, 
            and engagement mechanisms. While these changes may not directly drive revenue, 
            they significantly influence user behavior and downstream conversion.

            It includes search, recommendations, notifications, UI layouts, and interaction design.
        ''',
        'keywords':    ['ui','ux','design','layout','search','recommend','notification',
                        'email','push','banner','modal','tooltip','button','cta','page'],
        'primary_metric': 'ior',

        'questions': [
            ('monthly_users',       'Monthly active users who see this feature',  'users/month', False),
            ('current_ctr',         'Current click-through or engagement rate (%)', 'rate %',    True),
            ('ctr_to_ior_rate',     'Of engaged users, % who eventually convert (%)', 'rate %',  True),
            ('aov',                 'Average order value ($)',                    'dollars',     False),
            ('gross_margin',        'Gross margin (%)',                           'rate %',      True),
            ('horizon',             'Time horizon (months)',                      'months',      False),

            ('session_duration',    'Average session time (minutes)', 'time', False),
            ('pages_per_session',   'Average pages viewed per session', 'count', False),
            ('interaction_depth',   'Number of interactions per session', 'count', False),
            ('feature_usage_rate',  '% users engaging with feature', 'rate %', False),
            ('scroll_depth',        'Average scroll percentage on pages', 'rate %', False),
        ],

        'downstream': [('ctr_to_ior_rate', None)],

        'tracking_plan': '''
            - Track user interaction events (clicks, scrolls, hovers).
            - Use heatmaps and session recordings for UX insights.
            - Segment engagement by user cohorts and device types.
            - Measure feature adoption and repeat usage.
            - Run A/B tests on UI components (buttons, layouts, messaging).
            - Track notification performance (open rate, CTR).
        ''',

        'mde_benchmarks': {'typical_min_rel': 0.03, 'typical_max_rel': 0.10,
                           'label': '3–10% relative lift (UX changes tend to be smaller)'},
    },

    'pricing': {
        'label':       'Pricing & Monetisation',
        'description': '''
            This stage focuses on optimizing pricing strategies to maximize revenue and profitability.
            It includes pricing display, discount strategies, bundling, tiering, and psychological pricing.

            Pricing changes often have trade-offs between conversion rate and average order value, 
            making careful experimentation critical.
        ''',
        'keywords':    ['price','pricing','discount','fee','rate','tier','plan','package',
                        'revenue','monetis','anchor','display','promo','coupon'],
        'primary_metric': 'ior',

        'questions': [
            ('monthly_inquiries',   'Monthly quotes that see the pricing change', 'inquiries/month', False),
            ('ior',                 'Current IOR (%)',                           'rate %',          True),
            ('aov',                 'Current average order value ($)',           'dollars',         False),
            ('aov_delta_pct',       'Expected % change in AOV from pricing change (%)', 'rate %',  True),
            ('gross_margin',        'Gross margin (%)',                          'rate %',          True),
            ('horizon',             'Time horizon (months)',                     'months',          False),

            ('price_elasticity',    'Sensitivity of demand to price changes', 'elasticity', False),
            ('discount_uplift',     'Conversion lift due to discounts', 'rate %', False),
            ('margin_after_discount','Effective margin after discounts (%)', 'rate %', False),
            ('plan_selection_distribution','% users selecting each pricing tier', 'distribution %', False),
            ('revenue_per_user',    'Average revenue per user ($)', 'dollars', False),
        ],

        'downstream': [],

        'tracking_plan': '''
            - Track exposure to pricing variants (A/B testing).
            - Measure both conversion rate and AOV simultaneously.
            - Segment pricing performance by customer cohorts.
            - Track discount usage and incremental revenue impact.
            - Monitor margin impact post pricing changes.
            - Conduct elasticity analysis using historical experiments.
        ''',

        'mde_benchmarks': {'typical_min_rel': 0.03, 'typical_max_rel': 0.12,
                           'label': '3–12% relative lift (pricing changes have mixed effects)'},
    },
}


# ─────────────────────────────────────────────────────────────────────────────
# FUNNEL CLASSIFIER — uses LLM to map free-text description to taxonomy
# ─────────────────────────────────────────────────────────────────────────────

def classify_feature(description: str, llm) -> str:
    """
    Returns a key from FUNNEL_TAXONOMY, or 'other' for novel/uncategorised features.

    Three-stage classification:
    Stage 1: keyword scoring (instant, no LLM)
    Stage 2: LLM disambiguation when scores are tied or zero
    Stage 3: 'other' escape hatch — when LLM confidence is low,
             returns 'other' so the caller can trigger grounding questions
             instead of silently falling back to a wrong category.
    """
    desc_lower = description.lower()

    # Stage 1: keyword scoring
    scores = {}
    for cat, meta in FUNNEL_TAXONOMY.items():
        hits = sum(1 for kw in meta['keywords'] if kw in desc_lower)
        if hits > 0:
            scores[cat] = hits

    if scores:
        top_score = max(scores.values())
        top_cats  = [c for c, s in scores.items() if s == top_score]
        if len(top_cats) == 1 and top_score >= 2:
            return top_cats[0]   # strong unambiguous match — skip LLM

    # Stage 2: LLM disambiguation
    all_cats = list(FUNNEL_TAXONOMY.keys()) + ['other']
    categories_text = '\n'.join(
        f'  {k}: {v["label"]} — {v["description"]}'
        for k, v in FUNNEL_TAXONOMY.items()
    )
    prompt = (
        f'Classify this product feature into exactly one category.\n'
        f'Feature: "{description}"\n\n'
        f'Categories:\n{categories_text}\n'
        f'  other: Does not fit any category above\n\n'
        f'Return ONLY the category key (one of: {", ".join(all_cats)}).\n'
        f'If unsure or it spans multiple categories equally, return: other\n'
        f'No explanation. Just the key.'
    )
    resp = llm.ask(prompt).strip().lower().split()[0]

    for k in all_cats:
        if k in resp:
            return k

    # Stage 3: fallback with warning
    if scores:
        best = max(scores, key=scores.get)
        logger.warning(
            'classify_feature: LLM returned ambiguous response "%s" for "%s". '
            'Falling back to "%s" (keyword match). Consider using "other" path.',
            resp[:30], description[:50], best)
        return best

    logger.warning('classify_feature: no keyword match and LLM ambiguous for "%s". '
                   'Returning "other".', description[:50])
    return 'other'


def handle_other_category(desc: str, llm) -> tuple:
    """
    Escape hatch for features that don't fit the 6 standard categories.
    Asks 3 grounding questions to determine the funnel position and
    the right metrics, then builds a custom opportunity plan. Output is
    a designed PDF document (template-aware).
    """
    print()
    print("  ℹ️  This feature does not clearly fit one of the standard funnel categories.")
    print("  I will ask 3 quick grounding questions to understand it better.")
    print()

    q1 = input(
        '  ❓ [1/3] What specific USER ACTION changes because of this feature?\n'
        '         (e.g. "user can now 3D-preview part before ordering",\n'
        '          "user sees fewer required fields in checkout")\n'
        '  → ').strip()

    q2 = input(
        '\n  ❓ [2/3] What metric would PROVE this feature worked?\n'
        '         (e.g. "more users place an order after viewing the part",\n'
        '          "checkout completion rate increases")\n'
        '  → ').strip()

    q3 = input(
        '\n  ❓ [3/3] What could BREAK or get WORSE if this feature ships?\n'
        '         (e.g. "page load time increases", "AOV drops because users\n'
        '          order simpler parts after seeing 3D preview")\n'
        '  → ').strip()

    # ── Template-aware section list ──────────────────────────────────────────
    default_sections = [
        'FUNNEL POSITION',
        'PRIMARY METRICS',
        'SECONDARY METRICS',
        'GUARDRAIL METRICS',
        'DATA TRACKING REQUIREMENTS',
        'OPPORTUNITY SIZING APPROACH',
    ]
    user_sections, _ = ask_for_template('Custom Measurement Plan', default_sections)
    sections_to_use = user_sections if user_sections else default_sections

    print()
    print('  🤖 Generating custom measurement plan for this feature...')

    context_block = (
        'Feature: "{}"\n'
        'User action that changes: {}\n'
        'Success metric: {}\n'
        'Risk / what could break: {}'
    ).format(desc, q1, q2, q3)

    guidance = (
        'Keep "Field: value" lines (Metric, Definition, Why primary, Direction, '
        'Track, When, Properties, Risk, Threshold) each on its own line so the '
        'renderer formats them as styled cards. For PRIMARY, SECONDARY, and '
        'GUARDRAIL metrics use the Metric / Definition / Why / Direction pattern. '
        'For DATA TRACKING REQUIREMENTS use the Track / When / Properties / Why '
        'needed pattern. OPPORTUNITY SIZING APPROACH should be 2-3 sentences in '
        'plain prose about proxy metrics and estimation method.'
    )

    prompt = build_llm_prompt_from_template(
        role='You are a senior product analytics expert. A PM described a feature that '
             'does not fit standard funnel categories. Generate a custom measurement plan.',
        context_block=context_block,
        sections_to_fill=sections_to_use,
        content_guidance=guidance,
    )

    plan = llm.ask(prompt)
    try:
        plan = _strip_decorative_chars(plan)
    except NameError:
        pass

    print('\n' + '═'*72)
    print('  📋  CUSTOM MEASUREMENT PLAN (Uncategorised Feature)')
    print('  Feature: ' + desc[:65])
    print('═'*72)
    print(plan)
    print('═'*72)
    print()
    print('  ⚠️  This plan was generated for a non-standard feature.')
    print('  Recommend human review before adding to your PRD.')

    # ── Parse + render PDF ───────────────────────────────────────────────────
    parsed = parse_sections_from_llm_output(plan, sections_to_use)
    fname = 'custom_measurement_plan.pdf'
    out_path = render_document_pdf(
        title='Custom Measurement Plan',
        subtitle='Feature: ' + desc[:90],
        sections=parsed,
        output_path=fname,
        metadata={
            'Feature':        desc[:120],
            'User action':    q1[:120],
            'Success signal': q2[:120],
            'Risk':           q3[:120],
            'Category':       'Uncategorised (custom plan)',
        },
        accent_color=PDF_PALETTE['secondary'],
    )
    print('  📁 Saved → ' + out_path)

    custom_answers = {
        'user_action': q1, 'success_metric': q2,
        'risk': q3, 'plan': plan,
        'sections_used': sections_to_use,
        'output_file': out_path,
    }
    return custom_answers, plan


# ─────────────────────────────────────────────────────────────────────────────
# BASELINE PULLER — pulls relevant metrics from historical data
# ─────────────────────────────────────────────────────────────────────────────

def pull_baselines(category: str) -> dict:
    """
    Query DuckDB for baseline metrics, null-safe and schema-config-aware.

    Real-world hardening:
    - Every value has a fallback default (no raw None/NaN propagation)
    - Queries use col() / tbl() so column/table names are client-configurable
    - Traffic query handles both pre-aggregated and row-level granularity
    - Segment-specific IOR baselines for more accurate per-segment MDE
    """
    cfg = CLIENT_SCHEMA
    inq_tbl  = tbl('inquiries')
    traf_tbl = tbl('traffic')
    conv_col = col('converted')
    val_col  = col('order_value')
    seg_col  = col('account_segment')
    date_col = col('created_at')

    # ── Overall inquiry baselines ──────────────────────────────────────────────
    hist_raw = safe_query(f"""
        SELECT
            COUNT(DISTINCT {col('inquiry_id')})
                / NULLIF(DATEDIFF('day', MIN({date_col}), MAX({date_col})) + 1, 0)
                * 30.4                                               AS monthly_inquiries,
            AVG(CAST({conv_col} AS DOUBLE))                         AS ior,
            AVG(CASE WHEN {conv_col} THEN {val_col} END)            AS aov,
            STDDEV(CAST({conv_col} AS DOUBLE))                      AS ior_stddev,
            SUM(CAST({conv_col} AS INTEGER))
                / NULLIF(DATEDIFF('month', MIN({date_col}), MAX({date_col})) + 1, 0)
                                                                     AS monthly_orders
        FROM {inq_tbl}
        WHERE {conv_col} IS NOT NULL
    """)

    if hist_raw is None:
        hist = {'monthly_inquiries': cfg['null_daily_traffic']*30.4,
                'ior': cfg['null_ior_default'], 'aov': cfg['null_aov_default'],
                'ior_stddev': 0.02, 'monthly_orders': cfg['null_daily_traffic']*30.4*cfg['null_ior_default']}
    else:
        r = hist_raw.iloc[0]
        hist = {
            'monthly_inquiries': safe_val(r['monthly_inquiries'], cfg['null_daily_traffic']*30.4),
            'ior':               safe_val(r['ior'],               cfg['null_ior_default']),
            'aov':               safe_val(r['aov'],               cfg['null_aov_default']),
            'ior_stddev':        safe_val(r['ior_stddev'],        0.02),
            'monthly_orders':    safe_val(r['monthly_orders'],    10),
        }

    # ── Segment-specific IOR baselines (for per-segment MDE) ─────────────────
    seg_raw = safe_query(f"""
        SELECT
            {seg_col}                              AS segment,
            AVG(CAST({conv_col} AS DOUBLE))        AS ior,
            COUNT(*)                               AS n_inquiries,
            AVG(CASE WHEN {conv_col} THEN {val_col} END) AS aov
        FROM {inq_tbl}
        WHERE {conv_col} IS NOT NULL
          AND {seg_col}  IS NOT NULL
        GROUP BY {seg_col}
        HAVING COUNT(*) >= {cfg['min_segment_size']}
        ORDER BY n_inquiries DESC
    """)
    segment_baselines = {}
    if seg_raw is not None:
        for _, row in seg_raw.iterrows():
            segment_baselines[str(row['segment'])] = {
                'ior': safe_val(row['ior'], hist['ior']),
                'n':   int(row['n_inquiries']),
                'aov': safe_val(row['aov'], hist['aov']),
            }

    # ── Repeat-order rate ──────────────────────────────────────────────────────
    repeat_raw = safe_query(f"""
        SELECT
            SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END) * 1.0
                / NULLIF(COUNT(*), 0)  AS repeat_rate
        FROM (
            SELECT {col('account_id')}, COUNT(*) AS order_count
            FROM {inq_tbl}
            WHERE {conv_col} = true
            GROUP BY {col('account_id')}
        ) buyer_orders
    """)
    repeat_rate = safe_val(
        repeat_raw.iloc[0]['repeat_rate'] if repeat_raw is not None else None, 0.40)

    # ── Traffic baselines ──────────────────────────────────────────────────────
    granularity = cfg.get('traffic_granularity', 'pre_aggregated')

    if granularity == 'pre_aggregated':
        traffic_raw = safe_query(f"""
            SELECT
                SUM({col('total_sessions')})
                    / NULLIF(COUNT(DISTINCT {col('traffic_date')}), 0) * 30.4 AS monthly_visitors,
                SUM({col('new_signups')})
                    / NULLIF(COUNT(DISTINCT {col('traffic_date')}), 0) * 30.4 AS monthly_signups,
                SUM({col('signed_in')})
                    / NULLIF(COUNT(DISTINCT {col('traffic_date')}), 0) * 30.4 AS monthly_signins
            FROM {traf_tbl}
            WHERE {col('traffic_date')} >= 
                  ((SELECT MAX({col('traffic_date')}) FROM {traf_tbl})
                   - INTERVAL 3 MONTH)
        """)
    else:
        traffic_raw = safe_query(f"""
            SELECT
                COUNT(*) / NULLIF(COUNT(DISTINCT {col('traffic_date')}), 0) * 30.4 AS monthly_visitors,
                SUM(CASE WHEN is_new_user THEN 1 ELSE 0 END)
                    / NULLIF(COUNT(DISTINCT {col('traffic_date')}), 0) * 30.4  AS monthly_signups,
                SUM(CASE WHEN is_signed_in THEN 1 ELSE 0 END)
                    / NULLIF(COUNT(DISTINCT {col('traffic_date')}), 0) * 30.4  AS monthly_signins
            FROM {traf_tbl}
            WHERE {col('traffic_date')} >= 
                  ((SELECT MAX({col('traffic_date')}) FROM {traf_tbl})
                   - INTERVAL 3 MONTH)
        """)

    default_vis = hist['monthly_inquiries'] * 15
    if traffic_raw is None:
        traffic = {'monthly_visitors': default_vis,
                   'monthly_signups': default_vis * 0.03,
                   'monthly_signins': default_vis * 0.45}
    else:
        r = traffic_raw.iloc[0]
        traffic = {
            'monthly_visitors': safe_val(r['monthly_visitors'], default_vis),
            'monthly_signups':  safe_val(r['monthly_signups'],  default_vis * 0.03),
            'monthly_signins':  safe_val(r['monthly_signins'],  default_vis * 0.45),
        }

    monthly_visitors = max(traffic['monthly_visitors'], 1)
    signup_rate = traffic['monthly_signups'] / monthly_visitors

    return {
        'monthly_visitors':    round(monthly_visitors, 0),
        'monthly_signups':     round(traffic['monthly_signups'], 0),
        'monthly_signins':     round(traffic['monthly_signins'], 0),
        'signup_rate':         round(signup_rate, 4),
        'monthly_inquiries':   round(hist['monthly_inquiries'], 0),
        'ior':                 round(hist['ior'], 4),
        'ior_stddev':          round(hist['ior_stddev'], 4),
        'aov':                 round(hist['aov'], 0),
        'monthly_orders':      round(hist['monthly_orders'], 0),
        'repeat_rate':         round(repeat_rate, 4),
        'activation_rate':     0.20,
        'segment_baselines':   segment_baselines,  
    }




# ─────────────────────────────────────────────────────────────────────────────
# MDE RECOMMENDER — never asks user; computes from data + benchmarks
# ─────────────────────────────────────────────────────────────────────────────

def recommend_mde(baseline_rate: float, category: str, baselines: dict) -> dict:
    """
    Returns a recommended MDE (absolute and relative) based on:
    1. Statistical minimum: smallest effect detectable with ~4 weeks of data
    2. Business minimum: smallest effect worth caring about (1% of baseline)
    3. Industry benchmark range for this category
    The recommended MDE is the MAXIMUM of (1) and (2) — conservative but practical.
    """
    from scipy.stats import norm as _norm

    if category == 'acquisition':
        daily_n = baselines.get('monthly_visitors', 10000) / 30.4 / 2
    elif category in ('conversion', 'pricing'):
        daily_n = baselines.get('monthly_inquiries', 1000) / 30.4 / 2
    else:
        daily_n = baselines.get('monthly_inquiries', 1000) / 30.4 / 2

    n_4weeks = daily_n * 28
    p = baseline_rate
    se = np.sqrt(2 * p * (1-p) / n_4weeks) if n_4weeks > 0 else 0.01
    stat_min_abs = round(1.96 * se * 2, 4)   # 80% power, approx
    stat_min_rel = round(stat_min_abs / p * 100, 1) if p > 0 else 10.0

    biz_min_abs = round(p * 0.01, 4)   # 1% relative
    biz_min_rel = 1.0

    bench = FUNNEL_TAXONOMY[category]['mde_benchmarks']
    bench_mid_abs = round(p * (bench['typical_min_rel'] + bench['typical_max_rel']) / 2, 4)
    bench_mid_rel = round((bench['typical_min_rel'] + bench['typical_max_rel']) / 2 * 100, 1)

    recommended_abs = round(max(stat_min_abs, biz_min_abs), 4)
    recommended_rel = round(recommended_abs / p * 100, 1) if p > 0 else 10.0

    return {
        'stat_min_abs':     stat_min_abs,
        'stat_min_rel_pct': stat_min_rel,
        'biz_min_abs':      biz_min_abs,
        'biz_min_rel_pct':  biz_min_rel,
        'bench_range':      bench['label'],
        'bench_mid_abs':    bench_mid_abs,
        'bench_mid_rel_pct': bench_mid_rel,
        'recommended_abs':  recommended_abs,
        'recommended_rel_pct': recommended_rel,
        'n_4weeks':         int(n_4weeks),
    }


# ─────────────────────────────────────────────────────────────────────────────
# DYNAMIC OPPORTUNITY COMPUTATION
# ─────────────────────────────────────────────────────────────────────────────

def compute_dynamic_opportunity(category: str, answers: dict) -> dict:
    """
    Computes incremental GMV by walking the funnel chain for the given category.
    `answers` contains whatever questions were asked for this category.
    """
    mde_info = answers['_mde']
    rec_abs  = mde_info['recommended_abs']

    horizon  = int(answers.get('horizon', 12))
    gm       = answers.get('gross_margin', 0.30)
    aov      = answers.get('aov', answers.get('aov', 5000))

    funnel_chain = []
    incremental  = 0

    if category == 'acquisition':
        visitors     = answers['monthly_visitors']
        curr_su      = answers['signup_rate']
        target_su    = curr_su + rec_abs
        act_rate     = answers['activation_rate']
        ior          = answers['ior']
        extra_su     = visitors * (target_su - curr_su)
        extra_inq    = extra_su * act_rate
        incremental  = extra_inq * ior
        funnel_chain = [
            ('Visitors', f'{visitors:,.0f}/mo', '', ''),
            ('Sign-up rate', f'{curr_su*100:.2f}%', f'{target_su*100:.2f}%', f'+{(target_su-curr_su)*100:.3f}pp'),
            ('Extra sign-ups', f'{extra_su:,.1f}/mo', '', ''),
            ('Activation rate', f'{act_rate*100:.1f}%', '', f'→ {extra_inq:,.1f} extra inquiries'),
            ('IOR', f'{ior*100:.2f}%', '', f'→ {incremental:,.1f} extra orders/mo'),
        ]

    elif category == 'activation':
        monthly_su   = answers['monthly_signups']
        curr_act     = answers['activation_rate']
        target_act   = curr_act + rec_abs
        ior          = answers['ior']
        extra_act    = monthly_su * (target_act - curr_act)
        incremental  = extra_act * ior
        funnel_chain = [
            ('Monthly sign-ups', f'{monthly_su:,.0f}/mo', '', ''),
            ('Activation rate', f'{curr_act*100:.2f}%', f'{target_act*100:.2f}%', f'+{(target_act-curr_act)*100:.3f}pp'),
            ('Extra activated', f'{extra_act:,.1f}/mo', '', ''),
            ('IOR', f'{ior*100:.2f}%', '', f'→ {incremental:,.1f} extra orders/mo'),
        ]

    elif category in ('conversion', 'pricing'):
        monthly_inq   = answers['monthly_inquiries']
        curr_ior      = answers['ior']
        target_ior    = curr_ior + rec_abs
        aov_mult      = 1 + answers.get('aov_delta_pct', 0)
        effective_aov = aov * aov_mult

        ior_incremental = monthly_inq * (target_ior - curr_ior)
        incremental     = ior_incremental

        funnel_chain = [
            ('Monthly inquiries', f'{monthly_inq:,.0f}/mo', '', ''),
            ('IOR', f'{curr_ior*100:.2f}%', f'{target_ior*100:.2f}%',
             f'+{(target_ior-curr_ior)*100:.3f}pp'),
            ('Extra orders/mo', f'{ior_incremental:,.1f}', '', ''),
        ]

        aov_gmv_uplift = 0.0
        if category == 'pricing' and abs(aov_mult - 1) > 0.001:
            aov_gmv_uplift = monthly_inq * curr_ior * (effective_aov - aov)
            funnel_chain.append((
                'AOV uplift on existing orders',
                f'${aov:,.0f}', f'${effective_aov:,.0f}',
                f'+${aov_gmv_uplift:,.0f} GMV/mo'
            ))
            aov = effective_aov 

    elif category == 'retention':
        monthly_ord  = answers['monthly_orders']
        curr_rep     = answers['repeat_rate']
        target_rep   = curr_rep + rec_abs
        incremental  = monthly_ord * (target_rep - curr_rep)
        funnel_chain = [
            ('Monthly orders', f'{monthly_ord:,.0f}/mo', '', ''),
            ('Repeat order rate', f'{curr_rep*100:.2f}%', f'{target_rep*100:.2f}%', f'+{(target_rep-curr_rep)*100:.3f}pp'),
            ('Extra repeat orders', f'{incremental:,.1f}/mo', '', ''),
        ]

    elif category == 'engagement':
        monthly_users = answers['monthly_users']
        curr_ctr      = answers['current_ctr']
        target_ctr    = curr_ctr + rec_abs
        c2o           = answers['ctr_to_ior_rate']
        extra_engaged = monthly_users * (target_ctr - curr_ctr)
        incremental   = extra_engaged * c2o
        funnel_chain  = [
            ('Monthly users', f'{monthly_users:,.0f}/mo', '', ''),
            ('Engagement rate', f'{curr_ctr*100:.2f}%', f'{target_ctr*100:.2f}%', f'+{(target_ctr-curr_ctr)*100:.3f}pp'),
            ('Extra engaged', f'{extra_engaged:,.1f}/mo', '', ''),
            ('Engage→Order rate', f'{c2o*100:.1f}%', '', f'→ {incremental:,.1f} extra orders/mo'),
        ]

    aov_gmv_uplift = locals().get('aov_gmv_uplift', 0.0)  # only set for pricing
    monthly_gmv = incremental * aov + aov_gmv_uplift
    monthly_gm  = monthly_gmv * gm

    return {
        'category':                 category,
        'funnel_chain':             funnel_chain,
        'mde_info':                 mde_info,
        'incremental_orders_mo':    round(incremental, 1),
        'incremental_gmv_mo':       round(monthly_gmv, 0),
        'incremental_gm_mo':        round(monthly_gm, 0),
        'incremental_gmv_total':    round(monthly_gmv * horizon, 0),
        'incremental_gm_total':     round(monthly_gm  * horizon, 0),
        'time_horizon_months':      horizon,
        'aov':                      aov,
        'gross_margin':             gm,
    }


# ─────────────────────────────────────────────────────────────────────────────
# MAIN DYNAMIC OPPORTUNITY SIZER
# ─────────────────────────────────────────────────────────────────────────────

def run_opportunity_sizing(llm):
    print('\n' + '═'*72)
    print('  📐  DYNAMIC OPPORTUNITY SIZING')
    print('═'*72)

    # ── Step 1: Feature description ───────────────────────────────────────────
    print('\n  Describe the feature or change you are sizing.')
    print('  (Be specific — e.g. "social login button on sign-up page", '
                            '"reorder billing step in checkout", "new summary page layout")')
    print()
    while True:
        desc = input('  ❓ Feature description: ').strip()
        if len(desc) >= 5: break
        print('     ⚠️  Please describe the feature (at least 5 characters)')

    # ── Step 2: Classify ────────────────────────────────────────────────────────
    print('\n  🔍 Classifying feature...')
    category = classify_feature(desc, llm)

    # Handle 'other' — novel features outside the 6 standard categories
    if category == 'other':
        custom_answers, plan = handle_other_category(desc, llm)
        return {'category': 'other', 'custom_plan': plan}

    taxonomy  = FUNNEL_TAXONOMY[category]
    baselines = pull_baselines(category)

    # ── Data quality check on baselines table ────────────────────────────────
    inq_table = tbl('inquiries')
    try:
        raw_df = safe_query(f'SELECT * FROM {inq_table} LIMIT 50000')
        if raw_df is not None:
            raw_df = dedup_dataframe(raw_df)
            dq_report = validate_experiment_data(raw_df, 'baselines')
            if dq_report['warnings']:
                print(f'\n  ⚠️  Data quality warnings on baseline table:')
                for w in dq_report['warnings']:
                    print(f'     • {w}')
            if not dq_report['ok']:
                print(f'  🚨 Data quality errors — baselines may be unreliable:')
                for e in dq_report['errors']:
                    print(f'     • {e}')
    except Exception as _dq_err:
        logger.warning('DQ check failed: %s', _dq_err)

    print(f'  → Classified as: {taxonomy["label"]}')
    print(f'     {taxonomy["description"]}')

    # ── Step 3: MDE recommendation ────────────────────────────────────────────
    # Determine the primary baseline rate for this category
    primary_metric = taxonomy['primary_metric']
    baseline_rate  = baselines.get(primary_metric, baselines['ior'])
    mde_info       = recommend_mde(baseline_rate, category, baselines)

    print(f'\n  📊 Auto-detected baselines & recommended MDE:')
    print(f'     Primary metric ({primary_metric}): {baseline_rate*100:.3f}%')
    print(f'     Stat. minimum detectable (4 wks): {mde_info["stat_min_rel_pct"]:+.1f}% rel  = {mde_info["stat_min_abs"]*100:+.3f}pp abs')
    print(f'     Business minimum meaningful:      {mde_info["biz_min_rel_pct"]:+.1f}% rel  = {mde_info["biz_min_abs"]*100:+.3f}pp abs')
    print(f'     Industry benchmark ({category}):  {mde_info["bench_range"]}')
    print(f'     ✅ Recommended MDE:               {mde_info["recommended_rel_pct"]:+.1f}% rel  = {mde_info["recommended_abs"]*100:+.3f}pp abs')
    print(f'       (= smallest effect worth detecting, based on your traffic & benchmarks)')

    # Show per-segment baselines for context
    seg_bases = baselines.get('segment_baselines', {})
    if seg_bases:
        print(f'\n  📊 Segment-specific IOR baselines (for segment-level MDE):')
        for seg, sb in seg_bases.items():
            seg_mde = recommend_mde(sb['ior'], category, baselines)
            print(f'     {seg:<18}: IOR={sb["ior"]*100:.2f}%  '
                  f'MDE={seg_mde["recommended_rel_pct"]:+.1f}% rel '
                  f'(n={sb["n"]:,})')

    # ── Step 4: Ask only relevant questions ───────────────────────────────────
    print(f'\n  ── {len(taxonomy["questions"])} questions for {taxonomy["label"]} ──')
    print('     (press Enter to accept the auto-detected default)\n')

    answers = {'_mde': mde_info}

    answers.setdefault('ior',             baselines['ior'])
    answers.setdefault('activation_rate', baselines['activation_rate'])
    answers.setdefault('monthly_signups', baselines['monthly_signups'])
    answers.setdefault('monthly_orders',  baselines['monthly_orders'])
    answers.setdefault('repeat_rate',     baselines['repeat_rate'])
    answers.setdefault('monthly_visitors',baselines['monthly_visitors'])
    answers.setdefault('monthly_inquiries',baselines['monthly_inquiries'])
    answers.setdefault('monthly_users',   baselines['monthly_signins'])

    DEFAULTS = {
        'monthly_visitors':  baselines['monthly_visitors'],
        'signup_rate':       baselines['signup_rate'] * 100,
        'monthly_signups':   baselines['monthly_signups'],
        'activation_rate':   baselines['activation_rate'] * 100,
        'monthly_inquiries': baselines['monthly_inquiries'],
        'ior':               baselines['ior'] * 100,
        'monthly_orders':    baselines['monthly_orders'],
        'repeat_rate':       baselines['repeat_rate'] * 100,
        'aov':               baselines['aov'],
        'gross_margin':      30.0,
        'horizon':           12,
        'monthly_users':     baselines['monthly_signins'],
        'current_ctr':       2.5,
        'ctr_to_ior_rate':   baselines['ior'] * 100,
        'aov_delta_pct':     0.0,
    }

    for key, question_text, unit, is_pct in taxonomy['questions']:
        default = DEFAULTS.get(key, 0)
        while True:
            if is_pct:
                hint = f' [{default:.3f}%]'
            else:
                hint = f' [{default:,.0f} {unit}]' if unit else f' [{default}]'
            raw = input(f'     ❓ {question_text}{hint}: ').strip()
            if raw == '':
                val = float(default)
            else:
                try:
                    val = float(raw)
                except ValueError:
                    print('        ⚠️  Please enter a number'); continue
            if is_pct: val = val / 100
            answers[key] = val
            break

    # ── Step 5: Compute opportunity ───────────────────────────────────────────
    result = compute_dynamic_opportunity(category, answers)

    # ── Step 6: Print results ─────────────────────────────────────────────────
    print('\n' + '─'*72)
    print('  📊  RESULTS')
    print('─'*72)
    print(f'  Feature category   : {taxonomy["label"]}')
    print(f'  Feature description: {desc}')
    print(f'  Recommended MDE    : {mde_info["recommended_rel_pct"]:+.1f}% relative '
          f'= {mde_info["recommended_abs"]*100:+.3f}pp absolute')
    print(f'\n  Funnel chain:')
    for row in result['funnel_chain']:
        if row[2]:  # has current → target
            print(f'    {row[0]:<26} {row[1]:<15} → {row[2]:<15} {row[3]}')
        else:
            print(f'    {row[0]:<26} {row[1]:<15} {row[3]}')
    print(f'\n  ── Using MDE as the uplift target ──')
    print(f'  Incremental orders/month : {result["incremental_orders_mo"]:,.1f}')
    print(f'  Incremental GMV/month    : ${result["incremental_gmv_mo"]:,.0f}')
    print(f'  Incremental GM/month     : ${result["incremental_gm_mo"]:,.0f}')
    print(f'  ── {result["time_horizon_months"]}-month horizon ──')
    print(f'  Incremental GMV total    : ${result["incremental_gmv_total"]:,.0f}')
    print(f'  Incremental GM total     : ${result["incremental_gm_total"]:,.0f}')
    print('─'*72)
    print('  💡 This is the MINIMUM opportunity (at the MDE uplift).')
    print('     If the feature delivers more lift, the upside scales linearly.')

    # ── Step 7: Visualisation ─────────────────────────────────────────────────
    _plot_opportunity(result, desc, taxonomy, mde_info)

    # ── Step 8: LLM Narrative ─────────────────────────────────────────────────
    print('\n  🤖 Generating executive summary...')
    narrative = llm.narrate(
        {'feature': desc, 'category': taxonomy['label'], 'result': result,
         'mde': mde_info, 'funnel': result['funnel_chain']},
        context=(
            f'Opportunity sizing for feature: "{desc}". '
            f'Category: {taxonomy["label"]}. '
            f'MDE used: {mde_info["recommended_rel_pct"]:.1f}% relative. '
            f'Provide: (1) headline opportunity in plain language, '
            f'(2) key assumptions and what could make the number higher/lower, '
            f'(3) recommendation on whether to proceed with an A/B test.'
        )
    )
    print('\n' + '─'*72)
    print('  🤖  EXECUTIVE SUMMARY')
    print('─'*72)
    print(narrative)
    print('─'*72)
    return result


def _plot_opportunity(result, desc, taxonomy, mde_info):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.patch.set_facecolor('#0f0f0f')
    fig.suptitle(f'📐 Opportunity Sizing: {desc[:55]}', fontsize=13,
                 color=COLORS['highlight'], fontweight='bold')

    # 1. MDE breakdown
    ax = axes[0]
    labels  = ['Statistical\nMinimum', 'Business\nMinimum', 'Industry\nBenchmark', 'Recommended\nMDE']
    values  = [mde_info['stat_min_rel_pct'], mde_info['biz_min_rel_pct'],
               mde_info['bench_mid_rel_pct'], mde_info['recommended_rel_pct']]
    colours = [COLORS['neutral'], COLORS['neutral'], COLORS['control'], COLORS['highlight']]
    bars    = ax.bar(labels, values, color=colours, width=0.55)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold', color='white')
    ax.set_title('MDE Breakdown\n(relative %)', color=COLORS['highlight'])
    ax.set_ylabel('Relative MDE (%)')

    # 2. Cumulative GMV over horizon
    ax = axes[1]
    months  = np.arange(1, result['time_horizon_months']+1)
    cum_gmv = months * result['incremental_gmv_mo'] / 1e6
    cum_gm  = months * result['incremental_gm_mo']  / 1e6
    ax.fill_between(months, cum_gmv, alpha=0.25, color=COLORS['treatment'])
    ax.plot(months, cum_gmv, color=COLORS['treatment'], lw=2.5, label='GMV')
    ax.fill_between(months, cum_gm,  alpha=0.25, color=COLORS['positive'])
    ax.plot(months, cum_gm,  color=COLORS['positive'],  lw=2.5, label='Gross Margin')
    ax.set_xlabel('Month'); ax.set_ylabel('Cumulative ($M)')
    ax.set_title('Cumulative Opportunity', color=COLORS['highlight'])
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:.2f}M'))

    # 3. Funnel breakdown
    ax = axes[2]
    ax.axis('off')
    ax.set_title('Funnel Chain', color=COLORS['highlight'])
    for ri, row in enumerate(result['funnel_chain']):
        ax.text(0.03, 0.95-ri*0.17, row[0], transform=ax.transAxes,
                fontsize=9, color='#aaa', va='top')
        display_val = f'{row[1]}' + (f' → {row[2]}' if row[2] else '')
        ax.text(0.03, 0.88-ri*0.17, display_val, transform=ax.transAxes,
                fontsize=9.5, color='white', va='top', fontweight='bold')
        if row[3]:
            ax.text(0.03, 0.81-ri*0.17, row[3], transform=ax.transAxes,
                    fontsize=9, color=COLORS['positive'], va='top', fontstyle='italic')

    plt.tight_layout()
    plt.savefig('opportunity_sizing.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print('  📁 Chart saved → opportunity_sizing.png')


print('✅ Dynamic opportunity sizer loaded')
print('   Supports categories:', list(FUNNEL_TAXONOMY.keys()))


# ═════════════════════════════════════════════════════════════════════════════
# MODULE 5 — AUDIENCE SELECTION
# ═════════════════════════════════════════════════════════════════════════════

# ── Propensity scoring rules per funnel category ───────────────────────────
PROPENSITY_RULES = {
    'conversion': {
        'description': 'Users likely to respond to checkout/funnel changes',
        'high_propensity': {
            'ior_range':      (0.10, 0.30),   # mid-range IOR — not already maxed out
            'n_inquiries_min': 3,              # have submitted quotes before
            'has_billing':     None,           # both relevant
            'segments':        ['Core','Growth'],
        },
        'exclude': {
            'ior_range': (0.40, 1.0),          # already converting very well
            'n_inquiries_max': 1,              # one-time users unlikely to respond
        },
    },
    'acquisition': {
        'description': 'Users likely to respond to sign-up / onboarding changes',
        'high_propensity': {
            'ior_range':      (0.0, 0.20),
            'n_inquiries_min': 0,
            'recency_days_max': 90,
            'segments':        ['Growth','Individuals'],
        },
        'exclude': {
            'n_inquiries_min': 20,             # already very established
        },
    },
    'retention': {
        'description': 'Users likely to respond to re-engagement / repeat order nudges',
        'high_propensity': {
            'orders_range':   (1, 5),          # made 1-5 orders — could repeat
            'recency_days_range': (30, 180),   # not too recent, not churned
            'segments':        ['Core','Enterprise'],
        },
        'exclude': {
            'orders_range': (0, 0),            # never ordered
        },
    },
    'engagement': {
        'description': 'Users likely to respond to UI / UX changes',
        'high_propensity': {
            'n_inquiries_min': 5,
            'ior_range':      (0.05, 0.35),
            'segments':        ['Core','Growth','Enterprise'],
        },
        'exclude': {},
    },
    'pricing': {
        'description': 'Users likely to respond to price display changes',
        'high_propensity': {
            'ior_range':      (0.08, 0.35),
            'n_inquiries_min': 2,
            'segments':        ['Core','Enterprise'],
        },
        'exclude': {},
    },
}


def _build_user_features() -> 'pd.DataFrame':
    """
    Build a per-user feature matrix from buyers + hist_inquiries.
    Used for propensity scoring and matching.
    """
    user_history = db.execute("""
        SELECT
            buyer_id,
            COUNT(*)                                          AS n_inquiries,
            AVG(CAST(converted_to_order AS DOUBLE))          AS personal_ior,
            AVG(CASE WHEN converted_to_order THEN order_value END) AS avg_order_value,
            MAX(created_at)                                   AS last_inquiry_date,
            MIN(created_at)                                   AS first_inquiry_date,
            SUM(CAST(converted_to_order AS INTEGER))          AS total_orders
        FROM hist_inquiries
        GROUP BY buyer_id
    """).df()

    today_dt = pd.Timestamp(TODAY)
    user_history['days_since_last']  = (today_dt - pd.to_datetime(user_history['last_inquiry_date'])).dt.days
    user_history['days_since_first'] = (today_dt - pd.to_datetime(user_history['first_inquiry_date'])).dt.days
    user_history['personal_ior']     = user_history['personal_ior'].fillna(0)
    user_history['avg_order_value']  = user_history['avg_order_value'].fillna(0)

    # Merge with buyer profiles
    features = df_buyers.merge(user_history, on='buyer_id', how='left')
    features['n_inquiries']     = features['n_inquiries'].fillna(0)
    features['personal_ior']    = features['personal_ior'].fillna(0)
    features['total_orders']    = features['total_orders'].fillna(0)
    features['days_since_last'] = features['days_since_last'].fillna(999)
    features['tenure_days']     = (today_dt - pd.to_datetime(features['joined_at'])).dt.days.fillna(0)

    # Normalised features for propensity model
    for col in ['n_inquiries','personal_ior','lifetime_gmv','tenure_days']:
        mn, mx = features[col].min(), features[col].max()
        features[f'{col}_norm'] = (features[col] - mn) / (mx - mn + 1e-8)

    features['is_web']   = (features['primary_platform'] == 'web').astype(int)
    features['is_us']    = (features['country'] == 'US').astype(int)
    features['has_bill'] = features['has_billing_profile'].astype(int)
    features['seg_num']  = pd.Categorical(features['account_segment'],
                                          categories=SEGMENTS).codes

    return features


def _score_propensity(features: 'pd.DataFrame', category: str,
                      hypothesis: str, target_audience: str, llm) -> 'pd.Series':
    """
    Compute a propensity score [0, 1] for each user reflecting how likely they
    are to show a measurable response to this feature.

    Score = weighted combination of:
      - Behavioral fit: does the user's IOR/activity match the feature's target range?
      - Segment fit: are they in the target audience segments?
      - Variance contribution: do they have enough variance to detect the MDE?
      - Recency: are they currently active?
    """
    rules    = PROPENSITY_RULES.get(category, PROPENSITY_RULES['engagement'])
    high     = rules.get('high_propensity', {})
    score    = pd.Series(0.5, index=features.index)   # start neutral

    # Behavioral fit
    if 'ior_range' in high:
        lo, hi = high['ior_range']
        in_range = (features['personal_ior'] >= lo) & (features['personal_ior'] <= hi)
        score += np.where(in_range, 0.25, -0.10)

    if 'n_inquiries_min' in high:
        active = features['n_inquiries'] >= high['n_inquiries_min']
        score += np.where(active, 0.15, -0.05)

    if 'orders_range' in high:
        lo, hi = high['orders_range']
        in_range = (features['total_orders'] >= lo) & (features['total_orders'] <= hi)
        score += np.where(in_range, 0.20, -0.05)

    if 'recency_days_max' in high:
        recent = features['days_since_last'] <= high['recency_days_max']
        score += np.where(recent, 0.15, -0.10)

    if 'recency_days_range' in high:
        lo, hi = high['recency_days_range']
        in_range = (features['days_since_last'] >= lo) & (features['days_since_last'] <= hi)
        score += np.where(in_range, 0.20, -0.05)

    # Segment fit
    if 'segments' in high:
        in_seg = features['account_segment'].isin(high['segments'])
        score += np.where(in_seg, 0.20, 0.0)

    # Exclusion rules
    excl = rules.get('exclude', {})
    if 'ior_range' in excl:
        lo, hi = excl['ior_range']
        exclude_mask = (features['personal_ior'] >= lo) & (features['personal_ior'] <= hi)
        score[exclude_mask] = -1.0   # force exclusion

    if 'n_inquiries_max' in excl:
        exclude_mask = features['n_inquiries'] <= excl['n_inquiries_max']
        score[exclude_mask] = score[exclude_mask] - 0.30

    # Variance contribution: users near the MDE boundary contribute most
    # IOR near 0.0 or 1.0 has low variance; near 0.5 has highest variance
    variance_contribution = 4 * features['personal_ior'] * (1 - features['personal_ior'])
    score += variance_contribution * 0.10

    return np.clip(score, 0.0, 1.0)


def run_audience_selection(llm):
    print('\n' + '╔' + '═'*70 + '╗')
    print('║' + '  🎯  AUDIENCE SELECTION — Module 5'.ljust(70) + '║')
    print('║' + '  Who goes into control and treatment?'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')

    print("""
  Two modes:
  [A] Manual   — paste or enter a list of user IDs for each group
  [B] Auto     — model selects users based on feature hypothesis and
                 propensity scoring; forms matched, balanced groups
""")
    while True:
        mode = input('  ❓ Choose mode [A/B]: ').strip().upper()
        if mode in ('A','B'): break
        print('     ⚠️  Enter A or B')

    # ── Get experiment context ─────────────────────────────────────────────────
    print('\n  ── Experiment Context ──')
    exp_name   = input('  ❓ Experiment name (used to tag output): ').strip() or 'new_experiment'
    desc       = input('  ❓ Feature description: ').strip() or 'checkout funnel change'
    hypothesis = input('  ❓ Hypothesis: ').strip() or 'will improve IOR for Core buyers'
    target_aud = input('  ❓ Target audience: ').strip() or 'active buyers'
    n_variants = int(input('  ❓ Number of groups (2 = A/B, 3 = A/B/C): ').strip() or '2')

    # ── Classify feature category ──────────────────────────────────────────────
    print('\n  🔍 Classifying feature...')
    category = classify_feature(desc, llm)
    print(f'  → Category: {FUNNEL_TAXONOMY[category]["label"]}')

    # ── Mode A: Manual ─────────────────────────────────────────────────────────
    if mode == 'A':
        print('\n  Manual mode: enter user IDs for each group.')
        print('  Format: comma-separated list of buyer_ids (e.g. BYR00001,BYR00042,...)')
        print('  Or paste a file path to a .txt file with one ID per line.')
        print()

        groups = {}
        group_names = ['control'] + [f'treatment_{i}' if n_variants > 2 else 'treatment'
                                      for i in range(1, n_variants)]
        for g in group_names:
            raw = input(f'  ❓ User IDs for [{g}]: ').strip()
            if raw.endswith('.txt'):
                try:
                    ids = [l.strip() for l in open(raw).readlines() if l.strip()]
                except: ids = []
            else:
                ids = [x.strip() for x in raw.split(',') if x.strip()]
            groups[g] = ids
            print(f'     {len(ids):,} users assigned to {g}')

        # Build assignment DataFrame
        rows = []
        for g, ids in groups.items():
            for uid in ids:
                rows.append({'buyer_id': uid, 'group': g, 'experiment_name': exp_name,
                             'selection_mode': 'manual', 'propensity_score': None})
        assignments = pd.DataFrame(rows)

    # ── Mode B: Auto (propensity-based) ───────────────────────────────────────
    else:
        print('\n  Auto mode: computing propensity scores...')

        # Target sample size from power calc
        baselines = pull_baselines(category)
        ior       = baselines.get('ior', 0.18)
        mde_abs   = ior * 0.10
        ss        = compute_sample_size(ior, mde_abs, 0.05, 0.80, n_variants)
        n_per_group = ss['n_per_variant']
        n_needed    = n_per_group * n_variants
        print(f'  Target: {n_per_group:,} per group ({n_needed:,} total) to detect {mde_abs*100:.3f}pp MDE')

        # Build user features
        print('  Building user feature matrix...')
        features = _build_user_features()
        print(f'  {len(features):,} users in feature matrix')

        # Score propensity
        print('  Scoring response propensity...')
        features['propensity_score'] = _score_propensity(features, category, hypothesis, target_aud, llm)

        # Show propensity distribution
        eligible = features[features['propensity_score'] > 0].copy()
        eligible = eligible.sort_values('propensity_score', ascending=False)
        print(f'\n  Propensity score distribution ({len(eligible):,} eligible users):')
        for pct_label, lo, hi in [('High (>0.7)', 0.70, 1.01),
                                   ('Medium (0.4-0.7)', 0.40, 0.70),
                                   ('Low (<0.4)', 0.0, 0.40)]:
            n = ((eligible['propensity_score'] >= lo) & (eligible['propensity_score'] < hi)).sum()
            pct = n / len(eligible) * 100
            bar = '█' * int(pct / 3)
            print(f'    {pct_label:<18} {n:>5,}  ({pct:.0f}%)  {bar}')

        # Check if we have enough high-propensity users
        high_prop = eligible[eligible['propensity_score'] >= 0.60]
        if len(high_prop) < n_needed:
            print(f'\n  ⚠️  Only {len(high_prop):,} high-propensity users available, need {n_needed:,}.')
            lower = input(f'  ❓ Use propensity ≥ 0.40 instead? [Y/n]: ').strip().lower()
            if lower != 'n':
                high_prop = eligible[eligible['propensity_score'] >= 0.40]
                print(f'  Using {len(high_prop):,} users with score ≥ 0.40')

        if len(high_prop) < n_needed:
            print(f'  ⚠️  Still short — using top {min(n_needed, len(eligible)):,} users by propensity score.')
            high_prop = eligible.head(min(n_needed, len(eligible)))

        # Select pool of top-propensity users
        selected = high_prop.head(n_needed).copy()

        # ── Stratified assignment: ensure balanced segment distribution ───────
        selected = selected.sort_values(['account_segment', 'propensity_score'], ascending=[True, False])
        group_names  = ['control'] + [f'treatment_{i}' if n_variants > 2 else 'treatment'
                                       for i in range(1, n_variants)]
        selected['group'] = [group_names[i % n_variants] for i in range(len(selected))]
        selected['experiment_name'] = exp_name
        selected['selection_mode']  = 'auto_propensity'

        assignments = selected[['buyer_id','group','experiment_name',
                                 'selection_mode','propensity_score',
                                 'account_segment','primary_platform',
                                 'country','personal_ior','n_inquiries']].copy()

    # ── Validate group composition ─────────────────────────────────────────────
    print('\n  ── Group Composition Report ──')
    print(f'\n  {"Group":<20} {"N":>7}  {"% of total":>12}')
    print('  ' + '─'*44)
    for g, grp in assignments.groupby('group'):
        pct = len(grp) / len(assignments) * 100
        print(f'  {g:<20} {len(grp):>7,}  ({pct:.1f}%)')

    # Segment breakdown per group
    if 'account_segment' in assignments.columns:
        print(f'\n  Segment breakdown by group:')
        seg_cross = assignments.groupby(['group','account_segment']).size().unstack(fill_value=0)
        print(seg_cross.to_string())

    # Check balance
    group_sizes = assignments['group'].value_counts()
    if len(group_sizes) > 1:
        balance_ratio = group_sizes.min() / group_sizes.max()
        if balance_ratio < 0.80:
            print(f'\n  ⚠️  Group imbalance detected (ratio={balance_ratio:.2f}). Consider re-balancing.')
        else:
            print(f'\n  ✅ Groups are balanced (ratio={balance_ratio:.2f})')

    # Expected power with this cohort
    if mode == 'B' and 'personal_ior' in assignments.columns:
        cohort_ior = assignments[assignments['group']=='control']['personal_ior'].mean()
        cohort_mde = cohort_ior * 0.10
        n_ctrl     = (assignments['group']=='control').sum()
        n_trt      = (assignments['group'] != 'control').sum()
        expected_ss = compute_sample_size(cohort_ior, cohort_mde, 0.05, 0.80, n_variants)
        print(f'\n  Expected power analysis for this cohort:')
        print(f'    Cohort baseline IOR    : {cohort_ior*100:.3f}%')
        print(f'    Target MDE (10% rel)   : {cohort_mde*100:.3f}pp')
        print(f'    Required per group     : {expected_ss["n_per_variant"]:,}')
        print(f'    Assigned to control    : {n_ctrl:,}')
        if n_ctrl >= expected_ss['n_per_variant']:
            print(f'    ✅ Cohort is sufficient to detect the MDE at 80% power')
        else:
            shortfall = expected_ss['n_per_variant'] - n_ctrl
            print(f'    ⚠️  {shortfall:,} more users needed in each group to reach 80% power')

    # ── Covariate balance check ───────────────────────────────────────────────
    print()
    print('  ── [Auto] Running covariate balance tests ──')
    print('     Verifying treatment and control groups are statistically equivalent')
    print('     on all observable dimensions before the experiment launches...')
    print()

    _features_for_balance = None
    try:
        _features_for_balance = _build_user_features()
        assignments = assignments.merge(
            _features_for_balance[['buyer_id','account_segment','lifetime_orders',
                                    'n_inquiries','personal_ior','avg_order_value',
                                    'days_since_last']],
            on='buyer_id', how='left', suffixes=('','_feat')
        )
    except Exception as _e:
        print(f'     ⚠️  Could not merge feature data for balance tests: {_e}')

    _balance_results = _run_balance_battery(assignments)
    _ctrl_n   = (assignments['group'] == 'control').sum()
    _trt_n    = (assignments['group'] != 'control').sum()
    _balanced = _print_balance_report(_balance_results, _ctrl_n, _trt_n)

    if not _balanced and mode == 'B':
        print()
        raw_rerand = input('  ❓ Re-randomise to improve balance? [Y/n]: ').strip().lower()
        if raw_rerand != 'n':
            print()
            print(f'  ↩️  Re-randomising (up to {BALANCE_MAX_ITER} attempts)...')
            _n_per_group = min(_ctrl_n, _trt_n)
            _n_groups    = assignments['group'].nunique()
            try:
                _features_for_balance = _features_for_balance if _features_for_balance is not None                     else _build_user_features()
                assignments, _balance_results, _balanced = _rerandomise_until_balanced(
                    _features_for_balance, _n_per_group, _n_groups, exp_name
                )
                _ctrl_n  = (assignments['group'] == 'control').sum()
                _trt_n   = (assignments['group'] != 'control').sum()
                print()
                _balanced = _print_balance_report(_balance_results, _ctrl_n, _trt_n)
            except Exception as _e:
                print(f'     ⚠️  Re-randomisation failed: {_e}')
    elif not _balanced and mode == 'A':
        print()
        print('  ℹ️  Manual assignment cannot be re-randomised automatically.')
        print('     Review the flagged covariates and adjust the user lists manually,')
        print('     or switch to Auto mode (B) for balanced group selection.')

    # Generate Love plot
    try:
        _love_path = _plot_love_plot(_balance_results, exp_name)
        if _love_path:
            print(f'  📊 Love plot saved → {_love_path}')
    except Exception as _le:
        print(f'  ⚠️  Love plot skipped: {_le}')

    assignments['balance_pass'] = _balanced
    _n_flags = sum(1 for r in _balance_results.values() if r.get('flag', False))
    print()
    if _balanced:
        print('  ✅ Groups are scientifically valid. Safe to launch experiment.')
    else:
        print(f'  ⚠️  {_n_flags} balance flag(s) remain. Proceed with caution.')
        print('     Results from this experiment may be confounded by the imbalanced covariates.')

    # ── Visualise ──────────────────────────────────────────────────────────────
    _plot_audience_selection(assignments, mode, category)

    # ── Save assignment file for engineering ───────────────────────────────────
    fname = f'audience_assignments_{exp_name}.csv'
    assignments.to_csv(fname, index=False)
    print(f'\n  📁 Assignments saved → {fname}')
    print('  Engineering team: use this file to configure your feature flag targeting.')
    print('  Column "group" = the experiment bucket each user_id should be assigned to.')

    # Register in DuckDB for use by later modules
    db.register('audience_assignments', assignments)
    print(f'\n  ✅ Registered in DuckDB as "audience_assignments" — available to all modules.')

    # ── LLM narrative ─────────────────────────────────────────────────────────
    summary = {
        'experiment': exp_name,
        'feature': desc,
        'category': FUNNEL_TAXONOMY.get(category, {}).get('label', category),
        'mode': 'Manual' if mode == 'A' else 'Propensity-based auto-selection',
        'n_assigned': len(assignments),
        'groups': assignments['group'].value_counts().to_dict(),
        'segment_distribution': assignments['account_segment'].value_counts().to_dict()
            if 'account_segment' in assignments.columns else {},
    }
    narrative = llm.narrate(summary,
        f'Audience selection for experiment "{exp_name}". '
        f'Provide: (1) quality assessment of the selected groups, '
        f'(2) whether the composition is right for the hypothesis, '
        f'(3) any risks with this audience selection, '
        f'(4) recommendation on whether to proceed.')
    print('\n  🤖 ' + '─'*68)
    print(narrative)
    print('  ' + '─'*68)
    return assignments


def _plot_audience_selection(assignments: 'pd.DataFrame', mode: str, category: str):
    has_propensity = 'propensity_score' in assignments.columns and \
                     assignments['propensity_score'].notna().any()
    has_segment    = 'account_segment' in assignments.columns

    ncols = 3 if has_propensity else 2
    fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 5))
    if ncols == 1: axes = [axes]
    fig.patch.set_facecolor('#0f0f0f')
    fig.suptitle(f'🎯 Audience Selection — {mode} Mode', fontsize=13,
                 color=COLORS['highlight'], fontweight='bold')

    group_colors = {'control': COLORS['control'], 'treatment': COLORS['treatment'],
                    'treatment_1': COLORS['treatment'], 'treatment_2': COLORS['accent']}

    # Chart 1: group size
    ax = axes[0]
    counts = assignments['group'].value_counts()
    colors_bar = [group_colors.get(g, COLORS['neutral']) for g in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors_bar, width=0.5)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{v:,}', ha='center', fontsize=11, fontweight='bold', color='white')
    ax.set_title('Group Sizes', color=COLORS['highlight'])
    ax.set_ylabel('Number of users')

    # Chart 2: segment breakdown
    if has_segment:
        ax2 = axes[1]
        groups   = assignments['group'].unique()
        segments = SEGMENTS
        x = np.arange(len(segments))
        w = 0.8 / len(groups)
        for gi, g in enumerate(sorted(groups)):
            grp_counts = assignments[assignments['group']==g]['account_segment'].value_counts()
            vals = [grp_counts.get(s, 0) for s in segments]
            offset = (gi - len(groups)/2 + 0.5) * w
            ax2.bar(x + offset, vals, w, label=g,
                    color=group_colors.get(g, COLORS['neutral']), alpha=0.85)
        ax2.set_xticks(x); ax2.set_xticklabels(segments)
        ax2.set_title('Segment Distribution by Group', color=COLORS['highlight'])
        ax2.set_ylabel('Users'); ax2.legend(fontsize=8)

    # Chart 3: propensity score distribution (auto mode only)
    if has_propensity:
        ax3 = axes[-1]
        for g in sorted(assignments['group'].unique()):
            scores = assignments[assignments['group']==g]['propensity_score'].dropna()
            ax3.hist(scores, bins=20, alpha=0.65, label=g,
                     color=group_colors.get(g, COLORS['neutral']), density=True)
        ax3.set_xlabel('Propensity Score')
        ax3.set_ylabel('Density')
        ax3.set_title('Propensity Score Distribution by Group\n(should overlap — similar users)',
                      color=COLORS['highlight'])
        ax3.legend(fontsize=8)
        ax3.axvline(0.60, color=COLORS['highlight'], lw=1.5, linestyle='--',
                    label='High propensity threshold')

    plt.tight_layout()
    plt.savefig('audience_selection.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print('  📁 Chart saved → audience_selection.png')


print('✅ Module 5: Audience Selection loaded')
print('   Modes: Manual (upload user IDs) | Auto (propensity-based selection)')


✅ Dynamic opportunity sizer loaded
   Supports categories: ['acquisition', 'activation', 'conversion', 'retention', 'engagement', 'pricing']
✅ Module 5: Audience Selection loaded
   Modes: Manual (upload user IDs) | Auto (propensity-based selection)


In [18]:
def run_power_calculator(llm, use_synthetic_baseline: bool = True):
    print('\n' + '═'*72)
    print('  ⚡  POWER CALCULATOR')
    print('═'*72)

    # ── Auto-pull baselines ────────────────────────────────────────────────────
    if use_synthetic_baseline:
        base = db.execute("""
            SELECT
                COUNT(*) / DATEDIFF('day', MIN(created_at), MAX(created_at)) AS daily_inq,
                AVG(CAST(converted_to_order AS DOUBLE))                      AS overall_ior
            FROM hist_inquiries
        """).df().iloc[0]
        auto_daily_traffic = round(base['daily_inq'], 0)
        auto_baseline_rate = round(base['overall_ior'], 4)

        print(f'\n  📊 Auto-detected from historical data:')
        print(f'     Daily inquiries  : {auto_daily_traffic:,.0f}')
        print(f'     Overall IOR      : {auto_baseline_rate*100:.2f}%')

    print()

    def prompt_float(question, default=None, min_val=None, max_val=None):
        while True:
            hint = f' [{default}]' if default is not None else ''
            raw  = input(f'  ❓ {question}{hint}: ').strip()
            if raw == '' and default is not None: return float(default)
            try:
                v = float(raw)
                if min_val is not None and v < min_val:
                    print(f'     ⚠️  Must be ≥ {min_val}'); continue
                if max_val is not None and v > max_val:
                    print(f'     ⚠️  Must be ≤ {max_val}'); continue
                return v
            except ValueError:
                print('     ⚠️  Please enter a number')

    def prompt_int(question, default=None, min_val=1):
        while True:
            hint = f' [{default}]' if default is not None else ''
            raw  = input(f'  ❓ {question}{hint}: ').strip()
            if raw == '' and default is not None: return int(default)
            try:
                v = int(raw)
                if v < min_val: print(f'     ⚠️  Must be ≥ {min_val}'); continue
                return v
            except ValueError:
                print('     ⚠️  Please enter a whole number')

    print('  Enter experiment parameters (press Enter for defaults):\n')

    baseline    = prompt_float('Baseline conversion/IOR rate (0–1)',
                               default=auto_baseline_rate if use_synthetic_baseline else None,
                               min_val=0.001, max_val=0.999)
    mde_pct     = prompt_float('MDE — minimum detectable effect (%, e.g. 10 = detect 10% relative lift)',
                               default=10.0, min_val=0.1)
    mde_abs     = baseline * mde_pct / 100
    print(f'                               → absolute MDE = {mde_abs*100:.3f}pp')

    alpha       = prompt_float('Significance level α (e.g. 0.05)',
                               default=0.05, min_val=0.001, max_val=0.30)
    power       = prompt_float('Statistical power 1−β (e.g. 0.80)',
                               default=0.80, min_val=0.50, max_val=0.99)
    n_variants  = prompt_int('Number of variants including control (e.g. 2 = A/B)',
                             default=2, min_val=2)
    daily_traffic = prompt_float('Daily eligible traffic entering the experiment',
                                 default=auto_daily_traffic if use_synthetic_baseline else None,
                                 min_val=1)
    traffic_share = prompt_float('Fraction of traffic in experiment (0–1)',
                                 default=1.0, min_val=0.01, max_val=1.0)

    # ── Compute ───────────────────────────────────────────────────────────────
    ss   = compute_sample_size(baseline, mde_abs, alpha, power, n_variants)
    dur  = compute_duration(ss['n_total'], daily_traffic, traffic_share)

    print('\n' + '─'*72)
    print('  📊  RESULTS')
    print('─'*72)
    print(f'  Baseline rate         : {baseline*100:.2f}%')
    print(f'  MDE                   : {mde_pct:.1f}% relative  = {mde_abs*100:.3f}pp absolute')
    print(f'  Effect size (Cohen h) : {ss["effect_size_h"]:.4f}')
    print(f'  α  (significance)     : {alpha:.3f}   Power: {power:.2f}')
    print(f'  Variants              : {n_variants}')
    print(f'  ─── Sample requirements ───')
    print(f'  Per variant           : {ss["n_per_variant"]:,}')
    print(f'  Total (all variants)  : {ss["n_total"]:,}')
    print(f'  ─── Duration ───')
    print(f'  Daily eligible traffic: {dur["daily_eligible"]:,.1f}')
    print(f'  Required duration     : {dur["days_required"]:,} days  ({dur["weeks_required"]} weeks)')
    print(f'  Estimated end date    : {dur["end_date"]}')
    print('─'*72)

    # ── Sensitivity analysis ──────────────────────────────────────────────────
    mde_range   = np.linspace(0.005, 0.20, 50)   # 0.5% – 20% relative MDE
    power_range = [0.70, 0.80, 0.90]
    alpha_range = [0.01, 0.05, 0.10]

    fig = plt.figure(figsize=(18, 11))
    fig.patch.set_facecolor('#0f0f0f')
    gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    sens_colors = ['#4e9af1','#f97316','#22c55e']

    # Plot 1: Sample size vs MDE for different powers
    ax1 = fig.add_subplot(gs[0, 0])
    for pw, col in zip(power_range, sens_colors):
        ns = [compute_sample_size(baseline, baseline*m, alpha, pw, n_variants)['n_per_variant']
              for m in mde_range]
        ax1.plot(mde_range*100, ns, color=col, linewidth=2, label=f'Power={pw:.0%}')
    ax1.axvline(mde_pct, color=COLORS['highlight'], linestyle='--', alpha=0.8, label=f'Your MDE={mde_pct:.0f}%')
    ax1.set_xlabel('MDE (% relative)'); ax1.set_ylabel('Sample size per variant')
    ax1.set_title('Sample Size vs MDE', color=COLORS['highlight'])
    ax1.legend(fontsize=8); ax1.set_yscale('log')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))

    # Plot 2: Duration vs MDE for different powers
    ax2 = fig.add_subplot(gs[0, 1])
    for pw, col in zip(power_range, sens_colors):
        ds = [compute_duration(
                compute_sample_size(baseline, baseline*m, alpha, pw, n_variants)['n_total'],
                daily_traffic, traffic_share)['days_required']
              for m in mde_range]
        ax2.plot(mde_range*100, ds, color=col, linewidth=2, label=f'Power={pw:.0%}')
    ax2.axvline(mde_pct, color=COLORS['highlight'], linestyle='--', alpha=0.8)
    ax2.axhline(dur['days_required'], color=COLORS['highlight'], linestyle=':', alpha=0.6,
                label=f'Your result: {dur["days_required"]}d')
    ax2.set_xlabel('MDE (% relative)'); ax2.set_ylabel('Duration (days)')
    ax2.set_title('Duration vs MDE', color=COLORS['highlight'])
    ax2.legend(fontsize=8)

    # Plot 3: Sample size vs α for different MDEs
    ax3 = fig.add_subplot(gs[0, 2])
    alphas = np.linspace(0.01, 0.20, 40)
    mde_lines = [mde_pct*0.5, mde_pct, mde_pct*1.5]
    for m, col in zip(mde_lines, sens_colors):
        ns = [compute_sample_size(baseline, baseline*m/100, a, power, n_variants)['n_per_variant']
              for a in alphas]
        ax3.plot(alphas, ns, color=col, linewidth=2, label=f'MDE={m:.0f}%')
    ax3.axvline(alpha, color=COLORS['highlight'], linestyle='--', alpha=0.8, label=f'Your α={alpha}')
    ax3.set_xlabel('Significance level α'); ax3.set_ylabel('Sample size per variant')
    ax3.set_title('Sample Size vs α', color=COLORS['highlight'])
    ax3.legend(fontsize=8)

    # Plot 4: Heatmap — duration (days) by MDE × traffic share
    ax4 = fig.add_subplot(gs[1, :])
    mde_grid    = np.linspace(0.01, 0.25, 20)
    share_grid  = np.linspace(0.10, 1.0, 20)
    heat = np.zeros((len(share_grid), len(mde_grid)))
    for i, sh in enumerate(share_grid):
        for j, md in enumerate(mde_grid):
            ss_h = compute_sample_size(baseline, baseline*md, alpha, power, n_variants)
            dur_h = compute_duration(ss_h['n_total'], daily_traffic, sh)
            heat[i, j] = dur_h['days_required']

    im = ax4.imshow(heat, aspect='auto', origin='lower', cmap='RdYlGn_r',
                    extent=[mde_grid[0]*100, mde_grid[-1]*100, share_grid[0]*100, share_grid[-1]*100])
    plt.colorbar(im, ax=ax4, label='Days required')
    ax4.set_xlabel('MDE (% relative)'); ax4.set_ylabel('Traffic share (%)')
    ax4.set_title('Duration Heatmap: MDE × Traffic Share  (red=long, green=fast)',
                  color=COLORS['highlight'])
    ax4.scatter([mde_pct], [traffic_share*100], color=COLORS['highlight'],
                marker='*', s=300, zorder=5, label='Your config')
    ax4.legend()

    plt.suptitle('⚡  Power Calculator — Sensitivity Analysis', fontsize=15,
                 color=COLORS['highlight'], fontweight='bold', y=1.01)
    plt.savefig('power_calculator.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print('  📁 Chart saved → power_calculator.png')

    # ── LLM Narrative ─────────────────────────────────────────────────────────
    print('\n  🤖 Generating experiment design summary...')
    narrative = llm.narrate(
        {'sample_size': ss, 'duration': dur},
        context=(
            f'A/B experiment power calculation. '
            f'Baseline IOR: {baseline*100:.2f}%, MDE: {mde_pct:.1f}% relative ({mde_abs*100:.3f}pp absolute). '
            f'Provide: (1) clear summary of what the numbers mean, '
            f'(2) business risk of running shorter/longer, '
            f'(3) specific recommendations for the experiment team.'
        )
    )
    print('\n' + '─'*72)
    print('  🤖  EXPERIMENT DESIGN SUMMARY')
    print('─'*72)
    print(narrative)
    print('─'*72)
    return ss, dur

print('✅ Power calculator module ready')


✅ Power calculator module ready


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# EXPERIMENT SELECTOR
# ─────────────────────────────────────────────────────────────────────────────

def select_experiment() -> tuple:
    """
    Lists all experiments from the data.
    User picks one by number.
    Returns (experiment_name, df_experiment, variant_list, control_name)
    """
    print('\n  Fetching experiment list from data...')
    exp_summary = db.execute("""
        SELECT
            e.experiment_name,
            COUNT(*)                                              AS n_rows,
            COUNT(DISTINCT e.variant)                            AS n_variants,
            STRING_AGG(DISTINCT e.variant, ' | ')                AS variants,
            MIN(e.created_at)::DATE                              AS start_date,
            MAX(e.created_at)::DATE                              AS end_date,
            AVG(CAST(e.converted_to_order AS DOUBLE))*100        AS overall_ior_pct,
            r.description,
            r.status,
            r.team
        FROM all_experiments e
        LEFT JOIN experiment_registry r USING (experiment_name)
        GROUP BY e.experiment_name, r.description, r.status, r.team
        ORDER BY start_date DESC
    """).df()

    print('\n  ┌' + '─'*70 + '┐')
    print('  │  AVAILABLE EXPERIMENTS' + ' '*47 + '│')
    print('  ├' + '─'*70 + '┤')
    for i, (_, row) in enumerate(exp_summary.iterrows()):
        status_icon = '🟢' if row.get('status') == 'running' else '✅' if row.get('status') == 'concluded' else '⬜'
        print(f"  │  [{i+1}] {status_icon} {row['experiment_name']:<40}"[:73].ljust(73) + '│')
        print(f"  │       {row['n_rows']:>6,} rows  |  {row['n_variants']} variants: {row['variants'][:40]}".ljust(73) + '│')
        desc = str(row.get('description',''))[:65] or 'No description'
        print(f"  │       {desc}".ljust(73) + '│')
        print(f"  │       {row['start_date']} → {row['end_date']}  |  Team: {str(row.get('team','?'))[:15]}  |  IOR: {row['overall_ior_pct']:.2f}%".ljust(73) + '│')
        if i < len(exp_summary)-1:
            print('  ├' + '─'*70 + '┤')
    print('  └' + '─'*70 + '┘')

    while True:
        raw = input(f'\n  ❓ Select experiment [1–{len(exp_summary)}]: ').strip()
        try:
            idx = int(raw) - 1
            if 0 <= idx < len(exp_summary): break
        except ValueError: pass
        print(f'     ⚠️  Enter a number between 1 and {len(exp_summary)}')

    selected_name = exp_summary.iloc[idx]['experiment_name']
    df_selected   = df_all_experiments[df_all_experiments['experiment_name'] == selected_name].copy()
    variants      = sorted(df_selected['variant'].unique().tolist())
    control       = 'control' if 'control' in variants else variants[0]

    print(f'\n  ✅ Selected: {selected_name}')
    print(f'     Variants detected: {variants}')
    print(f'     Control group    : "{control}"')
    print(f'     Rows             : {len(df_selected):,}')

    return selected_name, df_selected, variants, control


# ─────────────────────────────────────────────────────────────────────────────
# PAIRWISE COMPARISON ENGINE — handles N variants
# ─────────────────────────────────────────────────────────────────────────────

def run_pairwise_comparisons(
    df: pd.DataFrame,
    variants: list,
    control: str,
    alpha: float = 0.05,
    bonferroni: bool = True,
) -> pd.DataFrame:
    """
    For N variants, computes:
    - Each treatment vs control (primary comparisons)
    - Each treatment vs every other treatment (secondary comparisons)
    Applies Bonferroni correction if requested.
    """
    treatments = [v for v in variants if v != control]

    # All unique pairs
    pairs = [(control, t) for t in treatments]                    # vs control
    pairs += [(treatments[i], treatments[j])                      # vs each other
              for i in range(len(treatments))
              for j in range(i+1, len(treatments))]

    # Bonferroni correction: alpha / number of comparisons
    n_comparisons = len(pairs)
    alpha_adj     = alpha / n_comparisons if bonferroni and n_comparisons > 1 else alpha

    rows = []
    for var_a, var_b in pairs:
        grp_a = df[df['variant'] == var_a]
        grp_b = df[df['variant'] == var_b]
        if len(grp_a) < 30 or len(grp_b) < 30:
            continue

        n_a = len(grp_a); conv_a = int(grp_a['converted_to_order'].sum())
        n_b = len(grp_b); conv_b = int(grp_b['converted_to_order'].sum())
        pr  = proportion_test(n_a, conv_a, n_b, conv_b, alpha_adj)
        mr  = means_test(grp_a['order_value'].values, grp_b['order_value'].values, alpha_adj, apply_winsorise=True)

        is_primary = (var_a == control)
        rows.append({
            'comparison':    f'{var_a} vs {var_b}',
            'baseline':      var_a,
            'variant':       var_b,
            'is_primary':    is_primary,  # True = vs control
            'n_baseline':    n_a,
            'n_variant':     n_b,
            'ior_baseline':  pr['rate_control'],
            'ior_variant':   pr['rate_treatment'],
            'delta_pp':      pr['delta_pp'],
            'ci_lo_pp':      pr['ci_lo_pp'],
            'ci_hi_pp':      pr['ci_hi_pp'],
            'p_value':       pr['p_value'],
            'sig':           pr['is_significant'],
            'direction':     pr['direction'],
            'effect_h':      pr['effect_size_h'],
            'gmv_baseline':  mr.get('mean_control', 0),
            'gmv_variant':   mr.get('mean_treatment', 0),
            'gmv_delta':     mr.get('delta_mean', 0),
            'p_value_gmv':   mr.get('p_value', 1.0),
            'sig_gmv':       mr.get('is_significant', False),
            'alpha_used':    alpha_adj,
            'bonferroni':    bonferroni,
            'n_comparisons': n_comparisons,
        })

    return pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────────────────────
# DIMENSION ANALYSIS FOR N VARIANTS
# ─────────────────────────────────────────────────────────────────────────────

def analyze_dimension_multivariant(
    df: pd.DataFrame,
    dim: str,
    variants: list,
    control: str,
    alpha: float,
) -> pd.DataFrame:
    """For each level of `dim`, runs each treatment vs control."""
    rows = []
    for level, grp in df.groupby(dim):
        ctrl = grp[grp['variant'] == control]
        if len(ctrl) < 20: continue
        for trt in [v for v in variants if v != control]:
            tr = grp[grp['variant'] == trt]
            if len(tr) < 20: continue
            pr = proportion_test(len(ctrl), int(ctrl['converted_to_order'].sum()),
                                 len(tr),   int(tr['converted_to_order'].sum()), alpha)
            rows.append({
                'dimension': dim, 'level': str(level), 'variant': trt,
                'n_ctrl': len(ctrl), 'n_treat': len(tr),
                'ior_ctrl': pr['rate_control'], 'ior_treat': pr['rate_treatment'],
                'delta_pp': pr['delta_pp'], 'ci_lo_pp': pr['ci_lo_pp'], 'ci_hi_pp': pr['ci_hi_pp'],
                'p_value': pr['p_value'], 'sig': pr['is_significant'], 'direction': pr['direction'],
            })
    return pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN POST-EXPERIMENT ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

def run_post_experiment_analysis(llm):
    print('\n' + '═'*72)
    print('  🔬  POST-EXPERIMENT ANALYSIS  (Multi-Variant)')
    print('═'*72)

    # ── Step 1: Pick experiment ───────────────────────────────────────────────
    exp_name, df, variants, control = select_experiment()
    df = dedup_dataframe(df)
    dq = validate_experiment_data(df, exp_name)
    if dq.get('warnings') or dq.get('errors'):
        for msg in dq.get('warnings',[]): print(f'  ⚠️  {msg}')
        for msg in dq.get('errors',  []): print(f'  🚨 ERROR: {msg}')
    else:
        print('  ✅ Data quality: clean')

    # ── Step 2: Configure analysis ────────────────────────────────────────────
    print('\n  Configuration (press Enter for defaults):')
    alpha_raw = input('  ❓ Significance level α [0.05]: ').strip()
    alpha     = float(alpha_raw) if alpha_raw else 0.05

    bonferroni = True
    if len(variants) > 2:
        bon_raw = input(f'  ❓ Apply Bonferroni correction for {len(variants)-1} treatments? [Y/n]: ').strip().lower()
        bonferroni = bon_raw != 'n'

    seg_filter  = input('  ❓ Filter to segment(s)? (comma-sep or Enter for all): ').strip()
    plat_filter = input('  ❓ Filter to platform? (web/mobile or Enter for all): ').strip()

    if seg_filter:
        segs = [s.strip() for s in seg_filter.split(',')]
        df   = df[df['account_segment'].isin(segs)]
        print(f'  → Filtered to segments: {segs}  ({len(df):,} rows)')
    if plat_filter:
        df   = df[df['platform'] == plat_filter.lower()]
        print(f'  → Filtered to platform: {plat_filter}  ({len(df):,} rows)')

    n_comparisons = len(variants) - 1 + max(0, (len(variants)-1)*(len(variants)-2)//2)
    alpha_adj     = alpha / n_comparisons if bonferroni and n_comparisons > 1 else alpha
    print(f'\n  α = {alpha}' + (f'  → Bonferroni-adjusted: {alpha_adj:.5f} ({n_comparisons} comparisons)' if bonferroni and n_comparisons > 1 else ''))

    # ── Step 3: SRM check ─────────────────────────────────────────────────────
    print('\n  ── [1/5] Sample Ratio Mismatch ──')
    from scipy.stats import chi2 as _chi2
    counts  = df['variant'].value_counts()
    n_total = counts.sum()
    expected = n_total / len(variants)
    chi2_stat = sum((counts.get(v,0) - expected)**2 / expected for v in variants)
    p_srm = 1 - _chi2.cdf(chi2_stat, df=len(variants)-1)
    srm_flag = '🚨 SRM DETECTED' if p_srm < 0.01 else '✅ No SRM'
    for v in variants:
        print(f'     {v:<22}: {counts.get(v,0):>7,}  ({counts.get(v,0)/n_total*100:.1f}%)')
    print(f'     χ²={chi2_stat:.3f}  p={p_srm:.4f}  {srm_flag}')

    # ── Step 4: Overall pairwise comparisons ──────────────────────────────────
    print('\n  ── [2/5] Pairwise Comparisons ──')
    pairwise_df = run_pairwise_comparisons(df, variants, control, alpha, bonferroni)

    primary = pairwise_df[pairwise_df['is_primary'] == True]
    secondary = pairwise_df[pairwise_df['is_primary'] == False]

    print(f'\n  Primary comparisons (vs "{control}"):')
    for _, r in primary.iterrows():
        sig_str = f'✅ p={r["p_value"]:.4f}' if r['sig'] else f'⚠️  p={r["p_value"]:.4f} n.s.'
        print(f'    {r["variant"]:<22} IOR: {r["ior_baseline"]*100:.3f}% → {r["ior_variant"]*100:.3f}%  '
              f'Δ={r["delta_pp"]:+.4f}pp [{r["ci_lo_pp"]:+.3f}, {r["ci_hi_pp"]:+.3f}]  {sig_str}')

    if len(secondary) > 0:
        print(f'\n  Head-to-head (treatment vs treatment):')
        for _, r in secondary.iterrows():
            sig_str = f'✅ p={r["p_value"]:.4f}' if r['sig'] else f'⚠️  p={r["p_value"]:.4f} n.s.'
            print(f'    {r["baseline"]:<14} vs {r["variant"]:<14} Δ={r["delta_pp"]:+.4f}pp  {sig_str}')

    # ── Step 5: Dimensional breakdown ─────────────────────────────────────────
    print('\n  ── [3/5] Dimensional Analysis ──')
    dims = ['account_segment','platform','country','device_type','category','has_billing_profile']
    dim_results = []
    for dim in dims:
        if dim not in df.columns: continue
        res = analyze_dimension_multivariant(df, dim, variants, control, alpha_adj)
        if len(res) == 0: continue
        dim_results.append(res)
        print(f'\n  📊 {dim.upper().replace("_"," ")}')

        for trt_name, trt_grp in res.groupby('variant'):
            print(f'    [{trt_name}]')
            for _, row in trt_grp.iterrows():
                sig_icon = '✅' if row['sig'] else '——'
                dir_icon = '↑' if row['direction']=='positive' else '↓' if row['direction']=='negative' else '→'
                print(f'      {row["level"]:<20} {dir_icon} {row["delta_pp"]:+.4f}pp '
                      f'[{row["ci_lo_pp"]:+.3f},{row["ci_hi_pp"]:+.3f}] '
                      f'p={row["p_value"]:.4f} {sig_icon}')

    combined = pd.concat(dim_results, ignore_index=True) if dim_results else pd.DataFrame()

    # ── Step 6: Winners / Losers per variant ──────────────────────────────────
    print('\n  ── [4/5] Winners / Losers per Variant ──')
    if not combined.empty:
        for trt in [v for v in variants if v != control]:
            trt_data = combined[combined['variant'] == trt]
            wins  = trt_data[trt_data['sig'] & (trt_data['direction']=='positive')]
            loses = trt_data[trt_data['sig'] & (trt_data['direction']=='negative')]
            neuts = trt_data[~trt_data['sig']]
            print(f'\n  Variant: [{trt}]')
            print(f'    🟢 Winning dimensions ({len(wins)}):')
            for _, r in wins.iterrows():
                print(f'       {r["dimension"]:<22} {r["level"]:<18} Δ={r["delta_pp"]:+.3f}pp  p={r["p_value"]:.4f}')
            print(f'    🔴 Losing dimensions ({len(loses)}):')
            for _, r in loses.iterrows():
                print(f'       {r["dimension"]:<22} {r["level"]:<18} Δ={r["delta_pp"]:+.3f}pp  p={r["p_value"]:.4f}')
            print(f'    ⚪ Neutral ({len(neuts)} not significant)')

    # ── Step 7: Visualisations ────────────────────────────────────────────────
    print('\n  ── [5/5] Generating Visualisations ──')
    _plot_multivariant(df, pairwise_df, combined, variants, control, exp_name, alpha)

    # ── Step 8: LLM Narrative ─────────────────────────────────────────────────
    print('\n  🤖 Generating executive analysis...')
    ship_summary = [{
        'variant': r['variant'],
        'delta_pp': r['delta_pp'], 'p_value': r['p_value'],
        'significant': r['sig'], 'direction': r['direction'],
        'ci': [r['ci_lo_pp'], r['ci_hi_pp']],
    } for _, r in primary.iterrows()]
    dim_summary  = {}
    if not combined.empty:
        for trt in [v for v in variants if v != control]:
            trt_d = combined[combined['variant']==trt]
            dim_summary[trt] = {
                'winners': trt_d[trt_d['sig'] & (trt_d['direction']=='positive')][['dimension','level','delta_pp']].to_dict('records'),
                'losers':  trt_d[trt_d['sig'] & (trt_d['direction']=='negative')][['dimension','level','delta_pp']].to_dict('records'),
            }
    narrative = llm.narrate(
        {'experiment': exp_name, 'pairwise': ship_summary, 'dimensions': dim_summary,
         'srm': {'detected': p_srm<0.01, 'p': round(p_srm,4)}, 'bonferroni': bonferroni},
        context=(
            f'Post-experiment analysis for "{exp_name}". {len(variants)} variants including control. '
            f'Provide: (1) Ship/No-Ship/Partial-Ship recommendation per variant, '
            f'(2) which variant wins and why, (3) segment-level nuance, '
            f'(4) caveats (SRM, multiple comparisons, novelty effect), '
            f'(5) recommended rollout strategy.'
        )
    )
    print('\n' + '─'*72)
    print('  🤖  EXECUTIVE ANALYSIS')
    print('─'*72)
    print(narrative)
    print('─'*72)
    return pairwise_df, combined


def _plot_multivariant(df, pairwise_df, combined, variants, control, exp_name, alpha):
    treatments = [v for v in variants if v != control]
    n_treats   = len(treatments)
    bar_colors = [COLORS['treatment'], COLORS['accent'], COLORS['positive'], COLORS['highlight']][:n_treats]

    n_rows_plot = 2 + (1 if not combined.empty else 0)
    fig = plt.figure(figsize=(20, 7 * n_rows_plot))
    fig.patch.set_facecolor('#0f0f0f')
    gs  = GridSpec(n_rows_plot, 3, figure=fig, hspace=0.55, wspace=0.38)

    # ── Row 1: Overall IOR per variant ────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    all_vars  = [control] + treatments
    all_cols  = [COLORS['control']] + bar_colors
    ior_vals  = [df[df['variant']==v]['converted_to_order'].mean()*100 for v in all_vars]
    bars      = ax1.bar(all_vars, ior_vals, color=all_cols, width=0.55)
    for bar, v in zip(bars, ior_vals):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
                 f'{v:.3f}%', ha='center', fontsize=10, fontweight='bold', color='white')
    ax1.set_title('IOR per Variant', color=COLORS['highlight'])
    ax1.set_ylabel('IOR (%)')
    plt.setp(ax1.get_xticklabels(), rotation=20, ha='right', fontsize=8)

    # ── Row 1: CI forest plot for primary comparisons ─────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    primary = pairwise_df[pairwise_df['is_primary'] == True].reset_index(drop=True)
    for i, row in primary.iterrows():
        col = COLORS['positive'] if (row['sig'] and row['direction']=='positive') \
              else COLORS['negative'] if (row['sig'] and row['direction']=='negative') \
              else COLORS['neutral']
        ax2.plot([row['ci_lo_pp'], row['ci_hi_pp']], [i, i], color=col, lw=3, solid_capstyle='round')
        ax2.scatter([row['delta_pp']], [i], color=col, s=80, zorder=5)
    ax2.axvline(0, color='white', lw=1, linestyle='--', alpha=0.7)
    ax2.set_yticks(range(len(primary)))
    ax2.set_yticklabels([r['variant'] for _, r in primary.iterrows()], fontsize=9)
    ax2.set_xlabel('IOR delta (pp)')
    ax2.set_title(f'IOR Δ vs {control}\n(95% CI, Bonferroni-adj if >1 treatment)', color=COLORS['highlight'])

    # ── Row 1: P-value comparison ─────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    pvals  = [r['p_value'] for _, r in primary.iterrows()]
    labels = [r['variant'][:15] for _, r in primary.iterrows()]
    colors_p = [COLORS['positive'] if p < alpha else COLORS['negative'] for p in pvals]
    ax3.barh(labels, pvals, color=colors_p)
    ax3.axvline(alpha, color=COLORS['highlight'], lw=2, linestyle='--', label=f'α={alpha}')
    for i, p in enumerate(pvals):
        ax3.text(p + 0.005, i, f'{p:.4f}', va='center', fontsize=9, color='white')
    ax3.set_xlabel('p-value'); ax3.set_title('P-values per Variant', color=COLORS['highlight'])
    ax3.legend(fontsize=9)

    # ── Row 2: Segment × Variant heatmap ─────────────────────────────────────
    if not combined.empty and 'account_segment' in combined['dimension'].values:
        ax4 = fig.add_subplot(gs[1, :])
        seg_data = combined[combined['dimension']=='account_segment']
        seg_levels = sorted(seg_data['level'].unique())
        heat = np.full((len(treatments), len(seg_levels)), np.nan)
        for ti, trt in enumerate(treatments):
            for si, seg in enumerate(seg_levels):
                row = seg_data[(seg_data['variant']==trt) & (seg_data['level']==seg)]
                if len(row) > 0:
                    heat[ti, si] = row.iloc[0]['delta_pp']
        vmax = np.nanmax(np.abs(heat)) if not np.all(np.isnan(heat)) else 1
        im = ax4.imshow(heat, cmap='RdYlGn', aspect='auto', vmin=-vmax, vmax=vmax)
        plt.colorbar(im, ax=ax4, label='IOR delta (pp)')
        ax4.set_xticks(range(len(seg_levels))); ax4.set_xticklabels(seg_levels)
        ax4.set_yticks(range(len(treatments))); ax4.set_yticklabels(treatments)
        for ti in range(len(treatments)):
            for si in range(len(seg_levels)):
                v = heat[ti, si]
                if not np.isnan(v):
                    row = seg_data[(seg_data['variant']==treatments[ti]) & (seg_data['level']==seg_levels[si])]
                    sig_mark = '✅' if (len(row)>0 and row.iloc[0]['sig']) else ''
                    ax4.text(si, ti, f'{v:+.2f}pp{sig_mark}', ha='center', va='center',
                             fontsize=10, color='white', fontweight='bold')
        ax4.set_title('IOR Delta Heatmap: Variant × Segment  (✅ = significant)', color=COLORS['highlight'])

    # ── Row 3: Dimension winners summary per variant ───────────────────────────
    if not combined.empty and n_rows_plot > 2:
        for ti, trt in enumerate(treatments[:3]):  # max 3 treatments in this row
            ax = fig.add_subplot(gs[2, ti])
            trt_dim = combined[combined['variant']==trt].copy()
            winners  = trt_dim[trt_dim['sig'] & (trt_dim['direction']=='positive')]
            losers   = trt_dim[trt_dim['sig'] & (trt_dim['direction']=='negative')]
            ax.axis('off')
            ax.set_title(f'[{trt}]\nWinners / Losers', color=bar_colors[ti] if ti < len(bar_colors) else COLORS['neutral'])
            lines = []
            lines.append(('🟢 WINNING', COLORS['positive']))
            for _, r in winners.iterrows():
                lines.append((f"  {r['dimension'][:10]} / {r['level'][:10]}  {r['delta_pp']:+.3f}pp", 'white'))
            lines.append(('🔴 LOSING', COLORS['negative']))
            for _, r in losers.iterrows():
                lines.append((f"  {r['dimension'][:10]} / {r['level'][:10]}  {r['delta_pp']:+.3f}pp", 'white'))
            if not winners.empty and not losers.empty:
                lines.append(('─'*30, '#444'))
                lines.append((f'Net: {int(len(winners))} win, {int(len(losers))} lose', COLORS['highlight']))
            for row_i, (txt, col) in enumerate(lines[:14]):
                ax.text(0.03, 0.97-row_i*0.07, txt, transform=ax.transAxes,
                        fontsize=8.5, color=col, va='top', fontfamily='monospace')

    plt.suptitle(f'🔬 Post-Experiment: {exp_name}  |  {len(variants)} variants', fontsize=13,
                 color=COLORS['highlight'], fontweight='bold', y=1.01)
    plt.savefig('post_experiment_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print('  📁 Chart saved → post_experiment_analysis.png')


print('✅ Multi-variant post-experiment module loaded')
print('   Supports: N variants, experiment selection, Bonferroni correction, head-to-head comparisons')


✅ Multi-variant post-experiment module loaded
   Supports: N variants, experiment selection, Bonferroni correction, head-to-head comparisons


In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# METRICS KNOWLEDGE BASE
# ─────────────────────────────────────────────────────────────────────────────

METRICS_KB = {
    'acquisition': {
        'primary_pool': [
            'Sign-up completion rate (% of users who start and finish sign-up)',
            'New registered users per day / week',
            'Sign-up-to-first-login rate',
        ],
        'secondary_pool': [
            'Sign-up method distribution (email vs social provider)',
            'Median time to complete sign-up (seconds)',
            'Sign-up page bounce rate',
            'Email verification completion rate (for email sign-ups)',
            'Day-7 retention of new sign-ups (did they return?)',
            'First inquiry within 30 days of sign-up (activation proxy)',
        ],
        'guardrail_pool': [
            'Existing user login success rate (must not break current auth)',
            'Account takeover / fraud incident rate',
            'Auth-related support ticket volume',
            'Password reset request rate',
        ],
        'tracking_event_types': [
            'Page load / view on sign-up or login page',
            'Sign-up method selected (which provider or email)',
            'Sign-up form submit attempt',
            'Sign-up success or failure + error code',
            'Email verification link clicked',
            'First login after sign-up',
        ],
    },
    'activation': {
        'primary_pool': [
            'Activation rate (% of sign-ups who reach first key action within N days)',
            'Time to first meaningful action (median days)',
            'Onboarding completion rate',
        ],
        'secondary_pool': [
            'Step-by-step onboarding funnel drop-off by step',
            'Feature discovery rate (% who interact with key features)',
            'Help / tooltip click rate during onboarding',
            'Onboarding skip rate',
            'Profile completeness score at end of onboarding',
        ],
        'guardrail_pool': [
            'Onboarding abandonment rate (must not increase vs baseline)',
            'Email unsubscribe rate from onboarding sequences',
            'Support tickets from new users',
        ],
        'tracking_event_types': [
            'Onboarding step started and completed (per step)',
            'Onboarding skipped or exited early',
            'First key action taken (first inquiry, first upload, etc.)',
            'Help content viewed during onboarding',
            'Profile field completed',
        ],
    },
    'conversion': {
        'primary_pool': [
            'Inquiry-to-order rate / IOR (% of quotes that become orders)',
            'Checkout completion rate (% who reach order confirmation)',
            'Cart / quote abandonment rate (inverse — should decrease)',
        ],
        'secondary_pool': [
            'Checkout funnel step-by-step drop-off rates',
            'Median time from quote acceptance to order placement',
            'Billing profile adoption rate',
            'Payment method distribution',
            'Order error / rejection rate',
            'Re-attempt rate after a failed checkout',
        ],
        'guardrail_pool': [
            'Average order value (must not decrease)',
            'Payment failure rate',
            'Order cancellation rate within 24 hours',
            'Checkout-related support tickets',
            'Revenue per day (no unintended revenue drop)',
        ],
        'tracking_event_types': [
            'Each checkout step entered and exited (with step name)',
            'Checkout abandoned (with last step reached)',
            'Payment method selected',
            'Billing profile created or reused',
            'Order placed successfully',
            'Order placement failed with error type',
            'Order confirmation page viewed',
        ],
    },
    'retention': {
        'primary_pool': [
            'Repeat order rate (% of customers who order again within 90 days)',
            'Time between first and second order (median days)',
            'Monthly active buyer rate',
        ],
        'secondary_pool': [
            'Reorder CTA click rate on the summary or confirmation page',
            'Orders per buyer per quarter',
            'Return visit rate within 30 days of last order',
            'Re-engagement email open and click rates',
            'NPS or CSAT score from post-order survey',
        ],
        'guardrail_pool': [
            'Order cancellation rate (must not increase)',
            'Churn rate (buyers with no activity for 90+ days)',
            'Support ticket rate post-order',
        ],
        'tracking_event_types': [
            'Order summary / confirmation page viewed',
            'Reorder CTA clicked',
            'Return visit to platform (session start) after order',
            'New quote / inquiry started after prior order',
            'Post-order survey submitted',
        ],
    },
    'engagement': {
        'primary_pool': [
            'Feature adoption rate (% of eligible users who interact with the feature)',
            'Click-through rate on the changed element',
            'Task completion rate (% who achieve the intended action)',
        ],
        'secondary_pool': [
            'Time spent on the affected page or flow',
            'Scroll depth on redesigned pages',
            'Secondary action rate (actions taken after the primary one)',
            'Return visits to the feature within 7 days',
            'User preference or feedback signal (thumbs up/down, rating)',
        ],
        'guardrail_pool': [
            'Overall page bounce rate (must not increase)',
            'Page load / response time (performance must not degrade)',
            'Accessibility complaints or error reports',
            'Downstream conversion rate (engagement must lead to orders)',
        ],
        'tracking_event_types': [
            'Feature / component viewed (impression)',
            'Feature / component interacted with (click, hover, expand)',
            'Task started and completed within the feature',
            'User dismissed or closed the feature',
            'Error or empty-state encountered',
        ],
    },
    'pricing': {
        'primary_pool': [
            'Inquiry-to-order rate / IOR (pricing should not hurt conversion)',
            'Average order value (AOV) — should increase if pricing change is designed to',
            'Revenue per inquiry (IOR x AOV combined signal)',
        ],
        'secondary_pool': [
            'Price-sensitivity signals (users requesting requotes after seeing price)',
            'Discount redemption rate',
            'Tier or plan upgrade rate',
            'Time from price display to order placement',
            'Quote comparison rate (did users compare multiple quotes more?)',
        ],
        'guardrail_pool': [
            'Customer satisfaction score (pricing changes can hurt perception)',
            'Complaint and dispute rate',
            'Churn rate in the 30 days post-exposure',
            'Refund or cancellation rate',
        ],
        'tracking_event_types': [
            'Price displayed to user (with price value and context)',
            'Price details expanded or inspected',
            'Quote accepted or rejected after price view',
            'Requote requested after price shown',
            'Pricing-related support contact initiated',
        ],
    },
}


# ─────────────────────────────────────────────────────────────────────────────
# INTERNAL HELPERS — normalise LLM output and build fallback content
# ─────────────────────────────────────────────────────────────────────────────

def _strip_markdown(text: str) -> str:
    """Remove markdown symbols so plain-text card renderer gets clean input."""
    import re
    if not text:
        return ''
    # Remove code fences
    text = re.sub(r'```[^\n]*\n?', '', text)
    # Remove heading markers (## Header → Header)
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    # Remove bold/italic markers
    text = re.sub(r'\*{1,3}([^*]+)\*{1,3}', r'\1', text)
    text = re.sub(r'_{1,2}([^_]+)_{1,2}', r'\1', text)
    # Collapse 3+ blank lines to 2
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def _normalise_section_output(raw: str) -> str:
    """Ensure each metric/tracking item is on its own line in Field: value format.

    Small LLMs tend to run Field: value items together on one line or use
    inline markdown. This normaliser moves each recognised field label to
    its own line so the PDF card renderer can format them correctly.
    """
    import re
    text = _strip_markdown(raw)

    FIELD_LABELS = [
        'Metric:', 'Definition:', 'Why primary:', 'Why secondary:',
        'Why guardrail:', 'Expected direction:', 'Expected magnitude:',
        'Risk:', 'Threshold:', 'Track:', 'When:', 'Properties:', 'Why needed:',
        'Event name:', 'Trigger:', 'Note:',
    ]

    for label in FIELD_LABELS:
        pattern = re.compile(r'(?<!\n)(' + re.escape(label) + r')', re.IGNORECASE)
        text = pattern.sub(r'\n\1', text)

    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def _build_primary_metrics_fallback(kb: dict, desc: str, maturity: str,
                                     preference: str) -> str:
    """Build a clean PRIMARY METRICS section directly from METRICS_KB.

    Used when the LLM call fails or returns unusable output.
    Picks 2-3 metrics from primary_pool and formats them as Field: value cards.
    """
    threshold_map = {
        'mvp':       '5-10% degradation tolerance',
        'iteration': '2-3% maximum degradation',
        'critical':  '0.5-1% hard stop threshold',
    }
    direction_map = {
        'primary_pool': 'Increase',
    }
    items = kb.get('primary_pool', [])[:3]
    lines = []
    for metric in items:
        name = metric.split('(')[0].strip().rstrip('/')
        lines.append(f'Metric: {name}')
        lines.append(f'Definition: {metric}')
        lines.append(f'Why primary: Directly measures the impact of this feature on the core funnel outcome.')
        lines.append(f'Expected direction: Increase')
        lines.append(f'Expected magnitude: +2-5% relative lift')
        lines.append('')
    return '\n'.join(lines).strip()


def _build_secondary_metrics_fallback(kb: dict, desc: str) -> str:
    """Build a clean SECONDARY METRICS section from METRICS_KB."""
    items = kb.get('secondary_pool', [])[:4]
    lines = []
    for metric in items:
        name = metric.split('(')[0].strip()
        lines.append(f'Metric: {name}')
        lines.append(f'Definition: {metric}')
        lines.append(f'Why secondary: Provides diagnostic signal to understand why the primary metric moved or did not move.')
        lines.append(f'Expected direction: Varies')
        lines.append(f'Expected magnitude: Monitor for meaningful change')
        lines.append('')
    return '\n'.join(lines).strip()


def _build_guardrail_fallback(kb: dict, maturity: str) -> str:
    """Build a clean GUARDRAIL METRICS section from METRICS_KB."""
    threshold_map = {
        'mvp':       '5-10%',
        'iteration': '2-3%',
        'critical':  '0.5-1%',
    }
    threshold = threshold_map.get(maturity, '3-5%')
    items = kb.get('guardrail_pool', [])
    lines = []
    for metric in items:
        name = metric.split('(')[0].strip().rstrip('—')
        lines.append(f'Metric: {name}')
        lines.append(f'Definition: {metric}')
        lines.append(f'Risk: If this metric degrades the feature is harming the user experience or business KPIs.')
        lines.append(f'Threshold: Must not degrade by more than {threshold} — halt experiment if breached.')
        lines.append(f'Expected direction: Neutral or better')
        lines.append('')
    return '\n'.join(lines).strip()


def _build_tracking_fallback(kb: dict, desc: str, instrumentation: str) -> str:
    """Build a clean DATA TRACKING REQUIREMENTS section from METRICS_KB."""
    depth_map = {
        'none':    kb.get('tracking_event_types', []),
        'partial': kb.get('tracking_event_types', [])[:4],
        'full':    kb.get('tracking_event_types', [])[:2],
    }
    events = depth_map.get(instrumentation, kb.get('tracking_event_types', []))
    lines = []
    for event in events:
        # Build a snake_case event name from the description
        import re
        raw_name = event.split('(')[0].split('/')[0].strip().lower()
        snake = re.sub(r'[^a-z0-9]+', '_', raw_name).strip('_')
        lines.append(f'Track: {snake}')
        lines.append(f'When: {event}')
        lines.append(f'Properties: user_id (string), timestamp (datetime), session_id (string), feature_variant (string)')
        lines.append(f'Why needed: Enables calculation of the primary and secondary metrics above.')
        lines.append('')
    return '\n'.join(lines).strip()


def _build_open_questions_fallback(desc: str, category: str) -> str:
    """Build an OPEN QUESTIONS & ASSUMPTIONS section."""
    return '\n'.join([
        f'1. Confirm the baseline conversion rate is stable across weekdays before launching — check for day-of-week variation in the past 30 days.',
        f'2. Verify that the proposed tracking event names do not collide with existing analytics events in the data warehouse.',
        f'3. Agree on the minimum detectable effect before the experiment launches — run the power calculator (Module 5) to confirm sample size and duration.',
        f'4. Confirm which user segments are in scope for this feature — does it apply to all buyers or a specific tier?',
        f'5. Clarify who owns the guardrail monitoring during the experiment — analytics, engineering, or product?',
    ])


# ─────────────────────────────────────────────────────────────────────────────
# PER-SECTION LLM CALL HELPERS
# ─────────────────────────────────────────────────────────────────────────────

_SECTION_SYSTEM = (
    'You are a senior product analytics expert. '
    'Write in plain business English. '
    'No markdown symbols (no #, **, *, >, -), no code fences, no emojis. '
    'Use ONLY plain "Field: value" lines separated by blank lines. '
    'Do not repeat the section heading. '
    'Do not add a preamble or explanation outside the Field: value blocks.'
)

_CARD_FORMAT_REMINDER = (
    'FORMAT RULES (must follow exactly):\n'
    '- Every item starts on a new line with "Metric: <name>"\n'
    '- Each field (Definition, Why, Expected direction, Expected magnitude) '
    'is on its own line\n'
    '- Separate each item with ONE blank line\n'
    '- No bullet points, no markdown, no headers\n'
    '- Example of correct format:\n'
    'Metric: Checkout completion rate\n'
    'Definition: Percentage of users who complete all checkout steps.\n'
    'Why primary: Directly measures whether the simplified flow reduces drop-off.\n'
    'Expected direction: Increase\n'
    'Expected magnitude: +3-8% relative lift\n'
)


def _call_llm_for_section(llm, section_name: str, prompt: str,
                            fallback_fn, max_tokens: int = 450) -> str:
    """Call the LLM for one section. Returns normalised text or the fallback.

    Uses a higher max_new_tokens than the global AGENT_CONFIG to ensure
    each section gets adequate space. Falls back gracefully on any failure.
    """
    try:
        # Temporarily override max_new_tokens for this call
        original_max = llm.max_new_tokens
        original_kwargs = dict(llm.gen_kwargs)
        llm.max_new_tokens = max_tokens
        llm.gen_kwargs['max_new_tokens'] = max_tokens

        raw = llm.ask(prompt, system=_SECTION_SYSTEM)

        # Restore
        llm.max_new_tokens = original_max
        llm.gen_kwargs = original_kwargs

        if not raw or len(raw.strip()) < 30:
            print(f'     ⚠️  LLM returned too little for {section_name} — using structured fallback')
            return fallback_fn()

        normalised = _normalise_section_output(raw)

        # Sanity check: if the output has no "Field: value" lines at all,
        # the LLM ignored the format — use the fallback instead
        import re
        field_lines = re.findall(
            r'^(?:Metric|Definition|Why|Expected|Risk|Threshold|Track|When|Properties|Why needed):',
            normalised, re.MULTILINE | re.IGNORECASE
        )
        if len(field_lines) < 2:
            print(f'     ⚠️  LLM output for {section_name} lacks structured fields — using structured fallback')
            return fallback_fn()

        return normalised

    except Exception as e:
        print(f'     ⚠️  LLM call failed for {section_name} ({e}) — using structured fallback')
        # Restore on exception
        try:
            llm.max_new_tokens = original_max
            llm.gen_kwargs = original_kwargs
        except Exception:
            pass
        return fallback_fn()


# ─────────────────────────────────────────────────────────────────────────────
# SECTION GENERATORS — one function per section
# ─────────────────────────────────────────────────────────────────────────────

def _gen_primary_metrics(llm, desc, category, kb, maturity, preference,
                          context_block) -> str:
    """Generate PRIMARY METRICS — 2-3 metrics with Definition/Why/Direction/Magnitude."""
    candidates = '\n'.join(f'- {m}' for m in kb['primary_pool'])
    preference_note = {
        'leading':  'Prefer early-signal leading indicators.',
        'balanced': 'Include one leading indicator and one lagging revenue outcome.',
        'lagging':  'Focus on revenue-tied lagging outcomes (conversion rate, AOV, GMV).',
    }.get(preference, '')

    prompt = (
        f'Feature: {desc}\n'
        f'Funnel stage: {category}\n'
        f'Maturity: {maturity}\n\n'
        f'Candidate primary metrics:\n{candidates}\n\n'
        f'{preference_note}\n\n'
        f'Choose 2-3 of the most relevant primary metrics for this feature. '
        f'For each, write:\n'
        f'Metric: <name>\n'
        f'Definition: <exact numerator / denominator in one sentence>\n'
        f'Why primary: <one sentence — what signal does it give for this feature>\n'
        f'Expected direction: Increase or Decrease\n'
        f'Expected magnitude: <realistic range, e.g. +2-5pp or +5-10% relative>\n\n'
        f'{_CARD_FORMAT_REMINDER}'
    )
    return _call_llm_for_section(
        llm, 'PRIMARY METRICS', prompt,
        fallback_fn=lambda: _build_primary_metrics_fallback(kb, desc, maturity, preference),
        max_tokens=480,
    )


def _gen_secondary_metrics(llm, desc, category, kb, context_block) -> str:
    """Generate SECONDARY METRICS — 3-4 supporting/diagnostic metrics."""
    candidates = '\n'.join(f'- {m}' for m in kb['secondary_pool'])
    prompt = (
        f'Feature: {desc}\n'
        f'Funnel stage: {category}\n\n'
        f'Candidate secondary metrics:\n{candidates}\n\n'
        f'Choose 3-4 of the most relevant secondary (diagnostic) metrics. '
        f'These help understand WHY the primary metric moved or did not move.\n'
        f'For each, write:\n'
        f'Metric: <name>\n'
        f'Definition: <exact measurement in one sentence>\n'
        f'Why secondary: <one sentence — what diagnostic signal does it provide>\n'
        f'Expected direction: Increase or Decrease or Neutral\n'
        f'Expected magnitude: <realistic expectation>\n\n'
        f'{_CARD_FORMAT_REMINDER}'
    )
    return _call_llm_for_section(
        llm, 'SECONDARY METRICS', prompt,
        fallback_fn=lambda: _build_secondary_metrics_fallback(kb, desc),
        max_tokens=480,
    )


def _gen_guardrail_metrics(llm, desc, category, kb, maturity,
                            context_block) -> str:
    """Generate GUARDRAIL METRICS — metrics that must not degrade."""
    threshold_map = {
        'mvp':       '5-10%  (MVP: tolerate some variance)',
        'iteration': '2-3%   (v2: tight guardrails)',
        'critical':  '0.5-1% (critical path: hard stop if breached)',
    }
    threshold_note = threshold_map.get(maturity, '3-5%')
    candidates = '\n'.join(f'- {m}' for m in kb['guardrail_pool'])

    prompt = (
        f'Feature: {desc}\n'
        f'Funnel stage: {category}\n'
        f'Maturity: {maturity} (threshold guideline: {threshold_note})\n\n'
        f'Candidate guardrail metrics:\n{candidates}\n\n'
        f'List ALL relevant guardrail metrics. '
        f'These are red lines — if any degrade beyond the threshold, the experiment must stop.\n'
        f'For each, write:\n'
        f'Metric: <name>\n'
        f'Definition: <exact measurement>\n'
        f'Risk: <one sentence — why degradation here is dangerous for the business>\n'
        f'Threshold: Must not degrade by more than {threshold_note.split(" ")[0]} — halt if breached.\n'
        f'Expected direction: Neutral (must not worsen)\n\n'
        f'{_CARD_FORMAT_REMINDER.replace("Why primary", "Risk").replace("Expected magnitude", "Threshold")}'
    )
    return _call_llm_for_section(
        llm, 'GUARDRAIL METRICS', prompt,
        fallback_fn=lambda: _build_guardrail_fallback(kb, maturity),
        max_tokens=480,
    )


def _gen_tracking_requirements(llm, desc, category, kb, instrumentation,
                                 context_block) -> str:
    """Generate DATA TRACKING REQUIREMENTS — concrete events with properties."""
    depth_map = {
        'none':    '6-10 concrete tracking events covering the full user path, not just the new feature.',
        'partial': '4-6 NEW events specific to this feature. For each, note if it extends an existing event.',
        'full':    '2-3 events maximum. Focus on metric definitions that tie to existing events.',
    }
    depth_note = depth_map.get(instrumentation, '4-6 events')
    event_types = '\n'.join(f'- {e}' for e in kb['tracking_event_types'])

    prompt = (
        f'Feature: {desc}\n'
        f'Funnel stage: {category}\n'
        f'Instrumentation: {instrumentation} ({depth_note})\n\n'
        f'Relevant tracking event types:\n{event_types}\n\n'
        f'Design the tracking events needed to measure the metrics above.\n'
        f'For each event, write:\n'
        f'Track: <snake_case_event_name>\n'
        f'When: <exact user action or system event that fires this>\n'
        f'Properties: <comma-separated list with data types — e.g. user_id (string), step_name (string), timestamp (datetime)>\n'
        f'Why needed: <which metric above this event enables>\n\n'
        f'FORMAT RULES:\n'
        f'- Event names must be in snake_case (e.g. checkout_step_completed)\n'
        f'- Properties must include user_id, timestamp, and feature_variant as standard\n'
        f'- No markdown, no bullet points, no headers\n'
        f'- Separate each event with ONE blank line\n'
        f'- Example:\n'
        f'Track: checkout_step_completed\n'
        f'When: User successfully completes a checkout step and advances to the next\n'
        f'Properties: user_id (string), step_name (string), step_index (integer), time_on_step_seconds (float), feature_variant (string), timestamp (datetime)\n'
        f'Why needed: Enables calculation of checkout completion rate and per-step drop-off rates.\n'
    )
    return _call_llm_for_section(
        llm, 'DATA TRACKING REQUIREMENTS', prompt,
        fallback_fn=lambda: _build_tracking_fallback(kb, desc, instrumentation),
        max_tokens=520,
    )


def _gen_open_questions(llm, desc, category, context_block) -> str:
    """Generate OPEN QUESTIONS & ASSUMPTIONS — 4-5 items to resolve before launch."""
    prompt = (
        f'Feature: {desc}\n'
        f'Funnel stage: {category}\n\n'
        f'List 4-5 specific open questions or assumptions the analytics and product team '
        f'must resolve BEFORE this experiment launches.\n'
        f'Focus on: baseline stability, event name collisions, segment scope, '
        f'holdout group design, power calculation inputs, data quality, and attribution.\n\n'
        f'Write each question on its own line starting with a number and a period.\n'
        f'Make each question concrete and specific to this feature — not generic.\n'
        f'Example format:\n'
        f'1. Confirm that the baseline checkout completion rate is stable across weekdays — check for day-of-week variation in the past 30 days before setting the MDE.\n'
        f'2. Verify the proposed event names (checkout_step_completed, checkout_abandoned) do not collide with existing analytics events.\n'
    )
    return _call_llm_for_section(
        llm, 'OPEN QUESTIONS & ASSUMPTIONS', prompt,
        fallback_fn=lambda: _build_open_questions_fallback(desc, category),
        max_tokens=380,
    )


# ─────────────────────────────────────────────────────────────────────────────
# MAIN MODULE FUNCTION
# ─────────────────────────────────────────────────────────────────────────────


def _gen_metrics_bundle(llm, desc, category, kb, maturity, preference, instrumentation, context_block) -> dict:
    """
    One LLM call generates primary + secondary + guardrail metrics together.
    Falls back section-by-section from METRICS_KB if output is malformed.
    """
    threshold_map = {'mvp': '5-10%', 'iteration': '2-3%', 'critical': '0.5-1%'}
    threshold = threshold_map.get(maturity, '3-5%')
    preference_note = {
        'leading':  'Prefer early-signal leading indicators for primary.',
        'balanced': 'Include one leading and one lagging metric for primary.',
        'lagging':  'Focus on revenue-tied lagging outcomes for primary.',
    }.get(preference, '')

    primary_cands   = '\n'.join(f'- {m}' for m in kb['primary_pool'])
    secondary_cands = '\n'.join(f'- {m}' for m in kb['secondary_pool'])
    guardrail_cands = '\n'.join(f'- {m}' for m in kb['guardrail_pool'])

    prompt = (
        f'Feature: {desc}\nFunnel stage: {category}\n'
        f'Maturity: {maturity} — guardrail threshold: {threshold}\n'
        f'{preference_note}\n\n'
        f'Generate THREE metric sections. Each section header in UPPERCASE on its own line.\n'
        f'Use only "Field: value" lines — no markdown, no bullets, no preamble.\n\n'
        f'PRIMARY METRICS\n'
        f'Choose 2-3 from: {primary_cands}\n'
        f'For each: Metric: / Definition: / Why primary: / Expected direction: / Expected magnitude:\n\n'
        f'SECONDARY METRICS\n'
        f'Choose 3-4 from: {secondary_cands}\n'
        f'For each: Metric: / Definition: / Why secondary: / Expected direction:\n\n'
        f'GUARDRAIL METRICS\n'
        f'Choose all relevant from: {guardrail_cands}\n'
        f'For each: Metric: / Definition: / Risk: / Threshold: Must not degrade by more than {threshold}.\n'
        f'\nStart immediately with PRIMARY METRICS — no other text before it.\n'
        + _CARD_FORMAT_REMINDER
    )

    try:
        original_max = llm.max_new_tokens
        original_kwargs = dict(llm.gen_kwargs)
        llm.max_new_tokens = 900
        llm.gen_kwargs['max_new_tokens'] = 900
        raw = llm.ask(prompt, system=_SECTION_SYSTEM)
        llm.max_new_tokens = original_max
        llm.gen_kwargs = original_kwargs
    except Exception as e:
        logger.warning('_gen_metrics_bundle LLM call failed: %s', e)
        raw = ''

    raw = _normalise_section_output(raw)
    sections = parse_sections_from_llm_output(
        raw, ['PRIMARY METRICS', 'SECONDARY METRICS', 'GUARDRAIL METRICS'])

    result = {}
    fallback_map = {
        'PRIMARY METRICS':   lambda: _build_primary_metrics_fallback(kb, desc, maturity, preference),
        'SECONDARY METRICS': lambda: _build_secondary_metrics_fallback(kb, desc),
        'GUARDRAIL METRICS': lambda: _build_guardrail_fallback(kb, maturity),
    }
    for sec_name, fb_fn in fallback_map.items():
        content = sections.get(sec_name, '').strip()
        field_hits = len(re.findall(
            r'^(?:Metric|Definition|Why|Expected|Risk|Threshold):',
            content, re.MULTILINE | re.IGNORECASE
        ))
        if not content or field_hits < 2:
            print(f'     ⚠️  Structured fallback applied for {sec_name}')
            result[sec_name] = fb_fn()
        else:
            result[sec_name] = content
    return result


def _gen_tracking_and_questions_bundle(llm, desc, category, kb, instrumentation, context_block) -> dict:
    """
    One LLM call generates tracking events + open questions together.
    Falls back section-by-section from METRICS_KB if output is malformed.
    """
    depth_map = {
        'none':    '6-8 concrete tracking events covering the full user path.',
        'partial': '4-5 NEW events specific to this feature.',
        'full':    '2-3 events that tie to existing event tracking.',
    }
    depth_note  = depth_map.get(instrumentation, '4-5 events')
    event_types = '\n'.join(f'- {e}' for e in kb['tracking_event_types'])

    prompt = (
        f'Feature: {desc}\nFunnel stage: {category}\nInstrumentation: {instrumentation} ({depth_note})\n\n'
        f'Relevant event types:\n{event_types}\n\n'
        f'Generate TWO sections. Each header in UPPERCASE on its own line.\n'
        f'No markdown, no bullets in the tracking section, no preamble.\n\n'
        f'DATA TRACKING REQUIREMENTS\n'
        f'For each event: Track: <snake_case_name> / When: <exact trigger> / '
        f'Properties: user_id (string), timestamp (datetime), feature_variant (string), <specific props> / '
        f'Why needed: <which metric this enables>\n'
        f'Separate events with ONE blank line.\n\n'
        f'OPEN QUESTIONS & ASSUMPTIONS\n'
        f'List 4-5 numbered questions to resolve before launch. '
        f'Be specific to this feature — not generic.\n'
        f'Cover: baseline stability, event name collisions, segment scope, power calc inputs.\n'
        f'\nStart immediately with DATA TRACKING REQUIREMENTS — no other text before it.\n'
    )

    try:
        original_max = llm.max_new_tokens
        original_kwargs = dict(llm.gen_kwargs)
        llm.max_new_tokens = 700
        llm.gen_kwargs['max_new_tokens'] = 700
        raw = llm.ask(prompt, system=_SECTION_SYSTEM)
        llm.max_new_tokens = original_max
        llm.gen_kwargs = original_kwargs
    except Exception as e:
        logger.warning('_gen_tracking_and_questions_bundle LLM call failed: %s', e)
        raw = ''

    raw = _normalise_section_output(raw)
    sections = parse_sections_from_llm_output(
        raw, ['DATA TRACKING REQUIREMENTS', 'OPEN QUESTIONS & ASSUMPTIONS'])

    result = {}
    fallback_map = {
        'DATA TRACKING REQUIREMENTS': lambda: _build_tracking_fallback(kb, desc, instrumentation),
        'OPEN QUESTIONS & ASSUMPTIONS': lambda: _build_open_questions_fallback(desc, category),
    }
    for sec_name, fb_fn in fallback_map.items():
        content = sections.get(sec_name, '').strip()
        field_hits = len(re.findall(
            r'^(?:Track|When|Properties|Why needed|\d+\.)',
            content, re.MULTILINE | re.IGNORECASE
        ))
        if not content or field_hits < 2:
            print(f'     ⚠️  Structured fallback applied for {sec_name}')
            result[sec_name] = fb_fn()
        else:
            result[sec_name] = content
    return result


# ─────────────────────────────────────────────────────────────────────────────
# EFFICIENCY IMPROVEMENT 6 — Infer maturity / instrumentation / preference
# ─────────────────────────────────────────────────────────────────────────────

def _infer_kpi_config(desc: str, llm) -> tuple:
    """
    Infer maturity, instrumentation, and preference from the feature description.
    Returns (maturity, instrumentation, preference, uncertain_keys).
    """
    prompt = (
        f'Feature: "{desc}"\n\n'
        'Infer the KPI planning configuration. Return ONLY valid JSON — no other text:\n'
        '{\n'
        '  "maturity": "mvp" | "iteration" | "critical",\n'
        '  "instrumentation": "none" | "partial" | "full",\n'
        '  "preference": "leading" | "balanced" | "lagging",\n'
        '  "uncertain": ["maturity"]\n'
        '}\n'
        'maturity=mvp if first launch. instrumentation=full if v2/existing. '
        'preference=lagging if revenue/conversion explicitly mentioned. '
        'uncertain lists only keys that are genuinely ambiguous (1-2 max).'
    )
    try:
        raw = llm.ask(prompt)
        m = re.search(r'\{[\s\S]*\}', raw)
        if m:
            d = json.loads(m.group())
            maturity        = d.get('maturity',        'iteration')
            instrumentation = d.get('instrumentation', 'partial')
            preference      = d.get('preference',      'balanced')
            uncertain       = d.get('uncertain',       [])
            valid_m  = {'mvp', 'iteration', 'critical'}
            valid_i  = {'none', 'partial', 'full'}
            valid_p  = {'leading', 'balanced', 'lagging'}
            if maturity not in valid_m: maturity = 'iteration'
            if instrumentation not in valid_i: instrumentation = 'partial'
            if preference not in valid_p: preference = 'balanced'
            return maturity, instrumentation, preference, uncertain
    except Exception as e:
        logger.warning('_infer_kpi_config failed: %s', e)
    return 'iteration', 'partial', 'balanced', ['maturity', 'instrumentation', 'preference']


def run_metrics_and_tracking(llm):
    """
    KPI + Data Tracking Planner. Produces a designed PDF with:
      - Primary, secondary, guardrail metrics grounded in the real baselines
      - Concrete tracking events (names, triggers, properties)
      - Guardrail thresholds calibrated to feature maturity

    Architecture change vs. original:
      Each section is generated by a SEPARATE focused LLM call (not one
      monolithic call for all 5 sections). This guarantees every section
      has adequate token budget. If an LLM call fails or returns
      unstructured output, a structured fallback is built automatically
      from METRICS_KB — so no section is ever left empty.
    """
    print('\n' + 'x'*72)
    print('  KPI METRICS & DATA TRACKING PLANNER')
    print('  For Product Managers, Analytics, and Engineering')
    print('x'*72)

    # ── Step 1: Feature description ───────────────────────────────────────────
    print('\n  Describe the feature or initiative you are planning.')
    print('  The more specific you are, the more tailored the output will be.')
    print()
    while True:
        desc = input('  ? Feature description: ').strip()
        if len(desc) >= 10:
            break
        print('     Please describe the feature in a bit more detail')



    # ── Steps 2-4: Infer maturity / instrumentation / preference (Improvement 6)
    # One LLM call replaces 3 serial input() prompts.
    print('\n  Inferring configuration from feature description...')
    maturity, instrumentation, preference, uncertain_keys = _infer_kpi_config(desc, llm)

    CONFIG_OPTIONS = {
        'maturity':        (['mvp', 'iteration', 'critical'],
                            {'mvp': 'MVP / first launch',
                             'iteration': 'v2 iteration',
                             'critical': 'Critical revenue path'}),
        'instrumentation': (['none', 'partial', 'full'],
                            {'none': 'Greenfield',
                             'partial': 'Some events exist',
                             'full': 'Fully instrumented'}),
        'preference':      (['leading', 'balanced', 'lagging'],
                            {'leading': 'Leading indicators',
                             'balanced': 'Balanced',
                             'lagging': 'Lagging / revenue outcomes'}),
    }
    inferred = {
        'maturity': maturity,
        'instrumentation': instrumentation,
        'preference': preference,
    }
    print(f'  Inferred: maturity={maturity}, instrumentation={instrumentation}, preference={preference}')

    if uncertain_keys:
        print(f'\n  Confirming {len(uncertain_keys)} ambiguous value(s):\n')
        for key in uncertain_keys:
            if key not in CONFIG_OPTIONS:
                continue
            options, labels = CONFIG_OPTIONS[key]
            opts_str = '  /  '.join(
                f'[{i+1}] {labels[o]}' for i, o in enumerate(options))
            default_idx = options.index(inferred[key]) + 1
            raw = input(
                f'  ❓ {key.title()}: {opts_str}  (default {default_idx}): '
            ).strip() or str(default_idx)
            if raw.isdigit() and 1 <= int(raw) <= len(options):
                inferred[key] = options[int(raw)-1]

    maturity        = inferred['maturity']
    instrumentation = inferred['instrumentation']
    preference      = inferred['preference']

    maturity_label = {
        'mvp':       'MVP / first launch — higher variance tolerance',
        'iteration': 'v2 iteration — tighter guardrails',
        'critical':  'Critical revenue path — tightest guardrails',
    }[maturity]
    instr_label = {
        'none':    'Greenfield — design all tracking events from scratch',
        'partial': 'Partial — extend existing events with feature-specific properties',
        'full':    'Full — metric definitions only, minimal new tracking',
    }[instrumentation]

    # ── Step 5: Optional PRD context ──────────────────────────────────────────
    print('\n  Optional: paste additional context (problem statement, goals,')
    print('  target users). Press Enter twice when done, or just Enter to skip.')
    print()
    context_lines = []
    while True:
        line = input('  ').strip()
        if line == '' and (not context_lines or context_lines[-1] == ''):
            break
        context_lines.append(line)
    prd_context = ' '.join(l for l in context_lines if l).strip()

    # ── Step 6: Classify feature ──────────────────────────────────────────────
    print('\n  Classifying feature...')
    category = classify_feature(desc, llm)

    if category == 'other':
        print('  Feature classified as uncategorised — switching to guided mode.')
        custom_answers, plan = handle_other_category(desc, llm)
        return {'category': 'other', 'custom_plan': plan,
                'output_file': custom_answers.get('output_file')}

    taxonomy = FUNNEL_TAXONOMY[category]
    kb = METRICS_KB[category]

    print(f'  -> Funnel position : {taxonomy["label"]}')
    print(f'  -> Maturity        : {maturity_label}')
    print(f'  -> Instrumentation : {instr_label}')

    # ── Step 7: Template check ────────────────────────────────────────────────
    default_sections = [
        'PRIMARY METRICS',
        'SECONDARY METRICS',
        'GUARDRAIL METRICS',
        'DATA TRACKING REQUIREMENTS',
        'OPEN QUESTIONS & ASSUMPTIONS',
    ]
    user_sections, _ = ask_for_template('KPI & Tracking Plan', default_sections)
    sections_to_use = user_sections if user_sections else default_sections

    context_block = (
        f'Feature description: "{desc}"\n'
        f'Funnel position: {taxonomy["label"]}\n'
        f'Feature maturity: {maturity_label}\n'
        f'Instrumentation state: {instr_label}\n'
        f'Metric preference: {preference}'
        + (f'\nAdditional context: {prd_context}' if prd_context else '')
    )

    # ── Step 8: Generate all sections in 2 bundled LLM calls
    sections = {}
    section_set = set(sections_to_use)
    METRIC_SECTIONS  = {'PRIMARY METRICS', 'SECONDARY METRICS', 'GUARDRAIL METRICS'}
    TRACKING_SECTIONS = {'DATA TRACKING REQUIREMENTS', 'OPEN QUESTIONS & ASSUMPTIONS'}

    if METRIC_SECTIONS & section_set:
        print('\n  Generating measurement plan (2 bundled calls)...')
        print('  [1/2] Metrics (primary + secondary + guardrail)...', end=' ', flush=True)
        metric_sections = _gen_metrics_bundle(
            llm, desc, category, kb, maturity, preference, instrumentation, context_block)
        total_cards = sum(v.count('Metric:') for v in metric_sections.values())
        print(f'done  ({total_cards} metrics)')
        sections.update({k: v for k, v in metric_sections.items() if k in section_set})
    else:
        print('\n  Generating measurement plan (1 bundled call)...')

    if TRACKING_SECTIONS & section_set:
        print('  [2/2] Tracking events + open questions...', end=' ', flush=True)
        tracking_sections = _gen_tracking_and_questions_bundle(
            llm, desc, category, kb, instrumentation, context_block)
        n_events = tracking_sections.get('DATA TRACKING REQUIREMENTS', '').count('Track:')
        print(f'done  ({n_events} events)')
        sections.update({k: v for k, v in tracking_sections.items() if k in section_set})

    # ── Step 9: Display ───────────────────────────────────────────────────────
    _display_metrics_plan(desc, category, taxonomy, sections)

    # ── Step 10: Render PDF ───────────────────────────────────────────────────
    fname = 'metrics_tracking_plan.pdf'
    # Build ordered dict matching sections_to_use order
    from collections import OrderedDict
    sections_ordered = OrderedDict(
        (s, sections.get(s, '(No content generated for this section.)'))
        for s in sections_to_use
    )

    out_path = render_document_pdf(
        title='KPI & Data Tracking Plan',
        subtitle='Feature: ' + desc[:90],
        sections=sections_ordered,
        output_path=fname,
        metadata={
            'Feature':         desc[:120],
            'Funnel stage':    taxonomy['label'],
            'Maturity':        maturity_label,
            'Instrumentation': instr_label,
            'Preference':      preference,
        },
        accent_color=PDF_PALETTE['accent'],
    )
    print(f'\n  Measurement plan saved -> {out_path}')

    return {
        **sections,
        'category':      category,
        'maturity':      maturity,
        'instrumentation': instrumentation,
        'preference':    preference,
        'output_file':   out_path,
        'sections_used': sections_to_use,
    }


def _display_metrics_plan(desc, category, taxonomy, sections):
    SEP = '=' * 72
    print('\n' + SEP)
    print(f'  MEASUREMENT PLAN')
    print(f'  Feature  : {desc[:65]}')
    print(f'  Category : {taxonomy["label"]}')
    print(SEP)

    # Section key → (display title, subtitle)
    # Keys match what parse/generation functions produce (UPPERCASE SECTION NAMES)
    SECTION_META = [
        ('PRIMARY METRICS',
         'PRIMARY METRICS',
         'The north-star numbers. If these move as expected, the feature worked.'),
        ('SECONDARY METRICS',
         'SECONDARY METRICS',
         'Supporting signals. Help diagnose WHY primary moved or did not.'),
        ('GUARDRAIL METRICS',
         'GUARDRAIL / NO-HARM METRICS',
         'Red lines. The experiment halts if any of these degrade.'),
        ('DATA TRACKING REQUIREMENTS',
         'DATA TRACKING REQUIREMENTS',
         'Events engineering must instrument before the experiment launches.'),
        ('OPEN QUESTIONS & ASSUMPTIONS',
         'OPEN QUESTIONS & ASSUMPTIONS',
         'Items to resolve before launch.'),
    ]

    for sec_key, title, subtitle in SECTION_META:
        content = sections.get(sec_key, '').strip()
        if not content or content.startswith('(No content'):
            continue
        print(f'\n  {title}')
        print(f'  {subtitle}')
        print('  ' + '-' * 68)
        for line in content.split('\n'):
            stripped = line.strip()
            if not stripped:
                print()
                continue
            print(f'  {stripped}')
        print()
    print(SEP)


print('Module 6: KPI Metrics & Data Tracking Planner loaded')
print('  run_metrics_and_tracking(narrative_llm)')
print('  Each section generated by a focused LLM call + structured fallback')


Module 6: KPI Metrics & Data Tracking Planner loaded
  run_metrics_and_tracking(narrative_llm)
  Each section generated by a focused LLM call + structured fallback


## 10 · Phase 2 — Live Monitoring Modules

Module [6] Experiment Health Monitor (SRM, guardrails, ETA). Module [7] Sequential Testing (always-valid mSPRT p-values).

In [21]:
# ═════════════════════════════════════════════════════════════════════════════
# MODULE E — EXPERIMENT HEALTH MONITOR
# ═════════════════════════════════════════════════════════════════════════════

def run_health_monitor(llm):
    print('\n' + '╔' + '═'*70 + '╗')
    print('║' + '  🩺  EXPERIMENT HEALTH MONITOR'.ljust(70) + '║')
    print('║' + '  Real-time status · SRM · Guardrails · ETA to significance'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')

    # Only show running experiments
    running = [e for e in EXPERIMENT_REGISTRY if e['status'] == 'running']
    if not running:
        print('\n  No running experiments found.')
        return

    print('\n  Running experiments:')
    for i, e in enumerate(running):
        days_in = (pd.Timestamp.today() - pd.Timestamp(e['start_date'])).days
        print(f'  [{i+1}] {e["experiment_name"]}  (day {days_in} of planned {e["planned_days"]})')
        print(f'       {e["description"]}')

    while True:
        raw = input('\n  ❓ Select experiment [1]: ').strip() or '1'
        try:
            idx = int(raw)-1
            if 0 <= idx < len(running): break
        except ValueError: pass
        print('     ⚠️  Invalid choice')

    exp_info  = running[idx]
    exp_name  = exp_info['experiment_name']
    variants  = exp_info['variants']
    control   = 'control'
    start_dt  = pd.Timestamp(exp_info['start_date'])
    today_sim = EXP_END if USE_SYNTHETIC_DATA else pd.Timestamp.now().normalize()  # synthetic: deterministic date; production: real today
    days_elapsed = (today_sim - start_dt).days

    exp_df = df_all_experiments[df_all_experiments['experiment_name'] == exp_name].copy()
    exp_df = dedup_dataframe(exp_df)   # remove duplicates before health checks
    dq = validate_experiment_data(exp_df, exp_name)
    if dq['warnings']:
        print(f'\n  ⚠️  DQ warnings: ' + ' | '.join(dq['warnings'][:3]))

    print(f'\n  ── Checking health of: {exp_name} ──')
    print(f'  Days elapsed: {days_elapsed} / planned {exp_info["planned_days"]}')
    print(f'  Progress    : {min(days_elapsed/exp_info["planned_days"]*100, 100):.0f}%')

    # ── 1. Sample ratio mismatch ──────────────────────────────────────────────
    from scipy.stats import chi2 as _chi2
    counts   = exp_df['variant'].value_counts()
    n_total  = counts.sum()
    expected = n_total / len(variants)
    chi2_val = sum((counts.get(v,0) - expected)**2 / expected for v in variants)
    p_srm    = 1 - _chi2.cdf(chi2_val, df=len(variants)-1)
    srm_flag = '🚨 SRM DETECTED — investigate before trusting results' if p_srm < 0.01 else '✅ Clean'

    print(f'\n  [1/5] Sample Ratio Mismatch')
    for v in variants:
        n = counts.get(v, 0)
        print(f'     {v:<18}: {n:>6,}  ({n/n_total*100:.1f}%)')
    print(f'  χ²={chi2_val:.3f}  p={p_srm:.4f}  {srm_flag}')

    # ── 2. Primary metric trajectory ─────────────────────────────────────────
    print(f'\n  [2/5] Primary Metric Trajectory (IOR)')
    alpha  = 0.05 / max(1, len(variants)-1)  # Bonferroni
    latest = {}
    for v in variants:
        vdf = exp_df[exp_df['variant'] == v]
        n, c = len(vdf), int(vdf['converted_to_order'].sum())
        ior  = c/n if n > 0 else 0
        latest[v] = {'n': n, 'c': c, 'ior': ior}
        print(f'  {v:<18}: IOR={ior*100:.3f}%  n={n:,}  conversions={c:,}')

    ctrl_n, ctrl_c   = latest[control]['n'], latest[control]['c']
    treat_results    = {}
    for v in [x for x in variants if x != control]:
        tr_n, tr_c = latest[v]['n'], latest[v]['c']
        pr = proportion_test(ctrl_n, ctrl_c, tr_n, tr_c, alpha)
        treat_results[v] = pr
        sig_label = f'✅ SIGNIFICANT (p={pr["p_value"]:.4f})' if pr['is_significant'] \
                    else f'⏳ not yet significant (p={pr["p_value"]:.4f})'
        print(f'\n  {v} vs {control}: Δ={pr["delta_pp"]:+.4f}pp  CI=[{pr["ci_lo_pp"]:+.3f},{pr["ci_hi_pp"]:+.3f}]  {sig_label}')

    # ── 3. ETA to significance ────────────────────────────────────────────────
    print(f'\n  [3/5] ETA to Significance')
    baseline_ior = latest[control]['ior']
    current_obs  = ctrl_n
    if baseline_ior > 0 and current_obs > 0:
        # Current effect size — use it to estimate required n
        for v in [x for x in variants if x != control]:
            obs_delta = abs(latest[v]['ior'] - baseline_ior)
            if obs_delta < 0.001:
                print(f'  {v}: Effect too small to estimate ETA (<0.1pp observed)')
                continue
            ss     = compute_sample_size(baseline_ior, obs_delta, alpha, 0.80, len(variants))
            needed = ss['n_per_variant']
            daily_rate = current_obs / max(days_elapsed, 1)
            if current_obs >= needed:
                print(f'  {v}: ✅ Already have sufficient sample ({current_obs:,} ≥ {needed:,})')
            else:
                days_needed = int(np.ceil((needed - current_obs) / max(daily_rate, 1)))
                eta_date    = (today_sim + pd.Timedelta(days=days_needed)).strftime('%Y-%m-%d')
                print(f'  {v}: Need {needed:,} per variant. At {daily_rate:.0f}/day → {days_needed} more days (ETA: {eta_date})')

    # ── 4. Guardrail checks ───────────────────────────────────────────────────
    print(f'\n  [4/5] Guardrail Metrics')
    guardrail_cols = [c for c in ['order_value', 'fulfillment_days'] if c in exp_df.columns]
    ctrl_df = exp_df[exp_df['variant'] == control]
    all_clear = True
    for g_col in guardrail_cols:
        ctrl_vals = ctrl_df[g_col].dropna().values
        for v in [x for x in variants if x != control]:
            tr_vals = exp_df[exp_df['variant']==v][g_col].dropna().values
            if len(ctrl_vals) < 10 or len(tr_vals) < 10: continue
            mr = means_test(ctrl_vals, tr_vals, 0.05)
            pct_change = mr.get('delta_rel', 0) * 100
            flag = '🚨 BREACH' if (mr.get('is_significant') and abs(pct_change) > 5) else '✅ OK'
            if '🚨' in flag: all_clear = False
            print(f'  {g_col} ({v} vs {control}): Δ={pct_change:+.1f}%  p={mr.get("p_value",1):.4f}  {flag}')
    if all_clear:
        print('  All guardrail metrics within acceptable range.')

    # ── 5. Trajectory chart ───────────────────────────────────────────────────
    print(f'\n  [5/5] Generating trajectory chart...')
    daily_df = df_daily[df_daily['experiment_name'] == exp_name].copy()
    if len(daily_df) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        fig.patch.set_facecolor('#0f0f0f')
        fig.suptitle(f'🩺 Health Monitor: {exp_name}', fontsize=13,
                     color=COLORS['highlight'], fontweight='bold')
        var_colors = {'control': COLORS['control'], 'treatment': COLORS['treatment'],
                      'google_only': '#22c55e', 'multi_provider': '#7c3aed'}
        ax1 = axes[0]
        for v in variants:
            vd = daily_df[daily_df['variant'] == v].sort_values('day_number')
            ax1.plot(vd['day_number'], vd['ior']*100,
                     color=var_colors.get(v, COLORS['accent']), lw=2, label=v, marker='o', markersize=3)
        ax1.set_xlabel('Day of experiment'); ax1.set_ylabel('Cumulative IOR (%)')
        ax1.set_title('IOR Trajectory Over Time', color=COLORS['highlight'])
        ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)
        ax2 = axes[1]
        for v in [x for x in variants if x != control]:
            ctrl_d = daily_df[daily_df['variant']==control].set_index('day_number')['ior']
            trt_d  = daily_df[daily_df['variant']==v].set_index('day_number')['ior']
            common = ctrl_d.index.intersection(trt_d.index)
            if len(common) > 3:
                delta = (trt_d.loc[common] - ctrl_d.loc[common]) * 100
                ax2.plot(common, delta.values, lw=2, label=f'{v} vs {control}',
                         color=var_colors.get(v, COLORS['accent']))
        ax2.axhline(0, color='white', lw=1, linestyle='--', alpha=0.5)
        ax2.fill_between(common if len(common)>0 else [0], 0,
                         delta.values if len(common)>0 else [0], alpha=0.1,
                         color=COLORS['treatment'])
        ax2.set_xlabel('Day'); ax2.set_ylabel('IOR delta (pp)')
        ax2.set_title('Treatment Effect Over Time', color=COLORS['highlight'])
        ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('health_monitor.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
        plt.show()
        print('  📁 Chart saved → health_monitor.png')

    # ── LLM health summary ────────────────────────────────────────────────────
    summary = {'experiment': exp_name, 'days_elapsed': days_elapsed,
               'srm': {'detected': p_srm<0.01, 'p': round(p_srm,4)},
               'guardrails_clear': all_clear,
               'results': [{
                   'variant': v, 'delta_pp': treat_results[v]['delta_pp'],
                   'p_value': treat_results[v]['p_value'],
                   'significant': treat_results[v]['is_significant'],
               } for v in treat_results]}
    narrative = llm.narrate(summary,
        'Experiment health check for running A/B test. Provide: (1) overall health status, '
        '(2) whether to continue or stop early, (3) any immediate risks, (4) recommendation.')
    print('\n  🤖 ' + '-'*68)
    print(narrative)
    print('  ' + '-'*68)
    return summary


# ═════════════════════════════════════════════════════════════════════════════
# MODULE F — SEQUENTIAL TESTING (Always-Valid P-values via mSPRT)
# ═════════════════════════════════════════════════════════════════════════════

def _msprt_pvalue(n1: int, x1: int, n2: int, x2: int, rho: float = 0.5) -> float:
    """
    Mixture Sequential Probability Ratio Test (mSPRT) for proportions.
    Returns an always-valid p-value — safe to check at any point without
    inflating Type I error.

    Reference: Johari et al. (2017) "Peeking at A/B Tests"
    rho: mixing parameter (0.5 is a robust default)
    """
    if n1 == 0 or n2 == 0:
        return 1.0
    p1 = x1 / n1
    p2 = x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    if p_pool in (0.0, 1.0) or p1 == p2:
        return 1.0

    def safe_log(p, n, x):
        if p <= 0 or p >= 1: return 0
        return x * np.log(p) + (n-x) * np.log(1-p)

    llr = safe_log(p1, n1, x1) + safe_log(p2, n2, x2) \
        - safe_log(p_pool, n1, x1) - safe_log(p_pool, n2, x2)

    mixture_e = np.exp(llr) * (1 + 1.0 / (2 * rho))
    p_val = min(1.0, 1.0 / max(mixture_e, 1e-10))
    return round(float(p_val), 6)


def run_sequential_testing(llm):
    print('\n' + '╔' + '═'*70 + '╗')
    print('║' + '  🔄  SEQUENTIAL TESTING  (Always-Valid P-values)'.ljust(70) + '║')
    print('║' + '  Safe to peek at any time — no false positive inflation'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')

    print('\n  Sequential testing lets you look at results at any point during an')
    print('  experiment without inflating your false positive rate.')
    print('  Standard p-values are only valid AT the planned end date.')
    print('  mSPRT p-values are valid at ANY point — "always-valid".\n')

    # Select experiment
    exp_summary = db.execute("""
        SELECT e.experiment_name, COUNT(*) n_rows, COUNT(DISTINCT e.variant) n_variants,
               STRING_AGG(DISTINCT e.variant, ' | ') variants,
               MIN(e.created_at)::DATE start_date, MAX(e.created_at)::DATE end_date,
               r.status
        FROM all_experiments e
        LEFT JOIN experiment_registry r USING (experiment_name)
        GROUP BY e.experiment_name, r.status
        ORDER BY start_date DESC
    """).df()

    print('  Available experiments:')
    for i, row in exp_summary.iterrows():
        status_icon = '🟢' if row.get('status')=='running' else '✅'
        print(f'  [{i+1}] {status_icon} {row["experiment_name"]}  ({row["n_variants"]} variants, {row["n_rows"]:,} rows)')

    while True:
        raw = input(f'\n  ❓ Select experiment [1-{len(exp_summary)}]: ').strip()
        try:
            idx = int(raw)-1
            if 0 <= idx < len(exp_summary): break
        except: pass
        print('     ⚠️  Invalid')

    exp_name  = exp_summary.iloc[idx]['experiment_name']
    exp_df    = df_all_experiments[df_all_experiments['experiment_name'] == exp_name].copy()
    exp_df    = exp_df.sort_values('created_at').reset_index(drop=True)
    variants  = sorted(exp_df['variant'].unique().tolist())
    control   = 'control' if 'control' in variants else variants[0]
    treatments= [v for v in variants if v != control]

    alpha = 0.05
    print(f'\n  Experiment : {exp_name}')
    print(f'  Variants   : {variants}')
    print(f'  α threshold: {alpha}  (Bonferroni-adj: {alpha/max(1,len(treatments)):.4f})')

    # ── Run sequential analysis over time ─────────────────────────────────────
    CHECKPOINTS = [25, 50, 75, 100]   # % of data to check at
    seq_results = {v: [] for v in treatments}
    standard_results = {}

    print('\n  ── Sequential P-value at each peek ──')
    print(f'  {"Checkpoint":<14} ' + ''.join(f'{v:<25}' for v in treatments))
    print('  ' + '─'*60)

    for pct in CHECKPOINTS:
        n_take = max(20, int(len(exp_df) * pct / 100))
        subset = exp_df.iloc[:n_take]
        ctrl_sub = subset[subset['variant']==control]
        row_vals = [f'{pct:>3}% ({n_take:>5,})']
        for v in treatments:
            tr_sub = subset[subset['variant']==v]
            n1, x1 = len(ctrl_sub), int(ctrl_sub['converted_to_order'].sum())
            n2, x2 = len(tr_sub),   int(tr_sub['converted_to_order'].sum())
            p_std  = proportion_test(n1, x1, n2, x2, alpha)['p_value']
            p_seq  = _msprt_pvalue(n1, x1, n2, x2)
            seq_results[v].append({'pct': pct, 'n': n_take, 'p_std': p_std, 'p_seq': p_seq,
                                   'ior_ctrl': x1/n1 if n1>0 else 0, 'ior_treat': x2/n2 if n2>0 else 0})
            std_sig = '✅' if p_std < alpha/len(treatments) else '  '
            seq_sig = '✅' if p_seq < alpha/len(treatments) else '  '
            row_vals.append(f'std p={p_std:.4f}{std_sig} seq p={p_seq:.4f}{seq_sig}')
        print('  ' + ' | '.join(row_vals))

    # Full-data standard result for comparison
    print('\n  ── Full-data comparison: Standard vs Sequential ──')
    ctrl_all = exp_df[exp_df['variant']==control]
    for v in treatments:
        tr_all = exp_df[exp_df['variant']==v]
        n1, x1 = len(ctrl_all), int(ctrl_all['converted_to_order'].sum())
        n2, x2 = len(tr_all),   int(tr_all['converted_to_order'].sum())
        p_std  = proportion_test(n1, x1, n2, x2, alpha)['p_value']
        p_seq  = _msprt_pvalue(n1, x1, n2, x2)
        delta  = (x2/n2 - x1/n1)*100 if n2>0 and n1>0 else 0
        std_label = '✅ significant' if p_std < alpha/len(treatments) else '⚠️  not significant'
        seq_label = '✅ significant' if p_seq < alpha/len(treatments) else '⚠️  not significant'
        print(f'\n  {v} vs {control}:  Δ={delta:+.4f}pp')
        print(f'    Standard p-value : {p_std:.5f}  → {std_label}')
        print(f'    Sequential p-val : {p_seq:.5f}  → {seq_label}')
        if p_seq < p_std:
            print(f'    💡 Sequential is MORE sensitive here (detected effect earlier)')
        elif p_seq > p_std:
            print(f'    ℹ️  Sequential is more conservative (safe for peeking — expected)')
        standard_results[v] = {'p_std': p_std, 'p_seq': p_seq, 'delta_pp': delta}

    # ── Visualise sequential vs standard p-values over time ──────────────────
    if seq_results and any(len(v)>0 for v in seq_results.values()):
        fig, axes = plt.subplots(1, len(treatments), figsize=(8*len(treatments), 5))
        fig.patch.set_facecolor('#0f0f0f')
        if len(treatments) == 1: axes = [axes]
        for ax, v in zip(axes, treatments):
            rows = seq_results[v]
            pcts = [r['pct'] for r in rows]
            std_ps = [r['p_std'] for r in rows]
            seq_ps = [r['p_seq'] for r in rows]
            ax.plot(pcts, std_ps,  color=COLORS['control'],   lw=2.5, marker='o', label='Standard p-value')
            ax.plot(pcts, seq_ps,  color=COLORS['treatment'], lw=2.5, marker='s', label='Sequential p-value (mSPRT)')
            ax.axhline(alpha/len(treatments), color=COLORS['highlight'], lw=1.5,
                       linestyle='--', label=f'α={alpha/len(treatments):.3f}')
            ax.fill_between(pcts, 0, alpha/len(treatments), alpha=0.08, color=COLORS['positive'])
            ax.set_xlabel('Data collected (%)'); ax.set_ylabel('p-value')
            ax.set_title(f'Sequential vs Standard: {v}', color=COLORS['highlight'])
            ax.legend(fontsize=8); ax.set_ylim(0, 1)
            ax.text(60, alpha/len(treatments)+0.03, 'Significance threshold',
                    color=COLORS['highlight'], fontsize=8)
        plt.suptitle(f'🔄 Sequential Testing: {exp_name}', fontsize=13,
                     color=COLORS['highlight'], fontweight='bold')
        plt.tight_layout()
        plt.savefig('sequential_testing.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
        plt.show()
        print('  📁 Chart saved → sequential_testing.png')

    narrative = llm.narrate(standard_results,
        f'Sequential testing analysis for {exp_name}. Compare standard vs always-valid p-values. '
        'Explain: (1) what the difference means in practice, (2) whether early peeking would have '
        'led to a different decision, (3) recommendation on using sequential testing for this team.')
    print('\n  🤖 ' + '-'*68)
    print(narrative)
    print('  ' + '-'*68)
    return seq_results


# ═════════════════════════════════════════════════════════════════════════════
# MODULE G — SIMPSON'S PARADOX DETECTOR
# ═════════════════════════════════════════════════════════════════════════════

def run_simpsons_paradox_detector(llm):
    """Module 9 — experiment-first Simpson's Paradox detection."""
    return analyze_experiment(llm, mode='paradox')

# ─────────────────────────────────────────────────────────────────────────────
# EXPERIMENT TYPE CATALOGUE
# ─────────────────────────────────────────────────────────────────────────────

EXP_TYPE_CATALOGUE = {
    'ab_test': {
        'label':           'A/B Test (Randomised Controlled Trial)',
        'description':     'Randomly assign users to control vs treatment. Gold standard for causal claims.',
        'when_to_use':     'Can randomise users; enough traffic; clear binary assignment possible.',
        'when_not':        'Cannot randomise (pricing for all, one-time events, infrastructure changes).',
        'causal_strength': 'High — randomisation eliminates confounding',
        'complexity':      'Low',
        'requires':        ['randomisation possible', 'sufficient traffic', 'metric measurable in real-time'],
        'pros':            ['Clean causal claim', 'Easy to explain', 'Sequential testing possible'],
        'cons':            ['Requires traffic', 'Cannot test everything', 'Takes weeks'],
        'keywords':        ['ui', 'feature', 'checkout', 'button', 'flow', 'onboarding', 'pricing', 'copy'],
    },
    'pre_post': {
        'label':           'Pre-Post Analysis',
        'description':     'Compare metric before vs after a change. No control group.',
        'when_to_use':     'Shipped to 100% immediately; no control group possible; quick read needed.',
        'when_not':        'When seasonality or other changes may confound; prefer DiD or ITS instead.',
        'causal_strength': 'Low — confounded by time, seasonality, concurrent changes',
        'complexity':      'Low',
        'requires':        ['clear before/after timestamp', 'stable baseline period'],
        'pros':            ['Simple to compute', 'No control group needed'],
        'cons':            ['Confounded by time-varying factors', 'Cannot attribute causality confidently'],
        'keywords':        ['before', 'after', 'shipped', '100%', 'rollout', 'pre', 'post'],
    },
    'did': {
        'label':           'Difference-in-Differences (DiD)',
        'description':     'Compare change over time between treated and untreated groups.',
        'when_to_use':     'Partial rollout with a natural control group (region, tier, segment).',
        'when_not':        'No credible control group; treated and control have different pre-trends.',
        'causal_strength': 'High if parallel-trends assumption holds',
        'complexity':      'Medium',
        'requires':        ['treated + control group', 'pre and post period data', 'parallel pre-trends'],
        'pros':            ['Handles selection-on-constants', 'Well-understood methodology'],
        'cons':            ['Parallel-trends must hold', 'Sensitive to differential shocks'],
        'keywords':        ['rollout', 'partial', 'region', 'tier', 'segment', 'launched', 'shipped to'],
    },
    'its': {
        'label':           'Interrupted Time Series (ITS)',
        'description':     'Model level + trend before and after intervention on a single series.',
        'when_to_use':     '100% rollout; long pre-period available; no control group.',
        'when_not':        'Short time series; other interventions occurred around the change.',
        'causal_strength': 'Medium — controls for trend and seasonality',
        'complexity':      'Medium',
        'requires':        ['90+ days pre-period', 'daily or weekly data', 'no concurrent interventions'],
        'pros':            ['No control group needed', 'Captures trend and seasonality'],
        'cons':            ['Assumes no other change coincided', 'Requires enough history'],
        'keywords':        ['time', 'series', 'trend', 'seasonality', 'daily', 'weekly'],
    },
    'synthetic_control': {
        'label':           'Synthetic Control',
        'description':     'Construct a counterfactual from a weighted combination of donor units.',
        'when_to_use':     'One treated unit; multiple similar donor units with good pre-period fit.',
        'when_not':        'No similar donor units; donors also received the treatment.',
        'causal_strength': 'High with good donor fit',
        'complexity':      'High',
        'requires':        ['1 treated unit', '3+ donor units', 'similar pre-period trajectories'],
        'pros':            ['Handles single-unit interventions', 'Transparent weights'],
        'cons':            ['Requires good donor pool', 'Inference is non-standard'],
        'keywords':        ['region', 'market', 'city', 'one', 'single', 'rollout'],
    },
    'psm': {
        'label':           'Propensity Score Matching (PSM)',
        'description':     'Match treated and untreated users by their probability of being treated.',
        'when_to_use':     'Observational data; users self-selected; measured confounders are rich.',
        'when_not':        'Important confounders unobserved; overlap between groups is poor.',
        'causal_strength': 'Medium — observable confounders only',
        'complexity':      'Medium',
        'requires':        ['observational data', 'rich user features', 'overlap in propensity scores'],
        'pros':            ['Uses existing data', 'Handles self-selection partially'],
        'cons':            ['Cannot control for unobservables', 'Sensitive to model specification'],
        'keywords':        ['observational', 'self-select', 'adopted', 'opted in', 'users who'],
    },
    'regression_discontinuity': {
        'label':           'Regression Discontinuity (RDD)',
        'description':     'Exploit a sharp cutoff rule on a running variable to identify local causal effect.',
        'when_to_use':     'Treatment is assigned by a threshold (score ≥ cutoff → treated).',
        'when_not':        'No clear cutoff; users can manipulate their running variable.',
        'causal_strength': 'High locally — near-random assignment at the threshold',
        'complexity':      'High',
        'requires':        ['continuous running variable', 'sharp cutoff', 'dense data near threshold'],
        'pros':            ['Strong causal claim near cutoff', 'No randomisation needed'],
        'cons':            ['Only valid near threshold', 'External validity limited'],
        'keywords':        ['score', 'threshold', 'cutoff', 'tier', 'qualification', 'eligibility'],
    },
    'causal_mediation': {
        'label':           'Causal Mediation Analysis',
        'description':     'Decompose total effect into direct and indirect (through a mediator) effects.',
        'when_to_use':     'You want to understand HOW the feature causes the outcome, not just THAT it does.',
        'when_not':        'You only need a headline effect; mediator is not measured.',
        'causal_strength': 'High — if mediator is correctly specified',
        'complexity':      'High',
        'requires':        ['A/B test or quasi-experiment', 'measured mediator variable', 'identification assumptions'],
        'pros':            ['Reveals mechanism', 'Informs feature iteration', 'Academically rigorous'],
        'cons':            ['Requires strong assumptions', 'Complex to implement and explain'],
        'keywords':        ['mechanism', 'why', 'through', 'mediator', 'pathway', 'because'],
    },
}


def recommend_experiment_type(
    desc: str,
    problem: str,
    hypothesis: str,
    target_audience: str,
    constraints: dict,
    llm,
) -> list:
    """
    Scores each experiment type against the user's situation and returns
    a ranked list with explanations. Uses a two-stage approach:
    Stage 1: Rule-based scoring on structural constraints (fast)
    Stage 2: LLM refinement for nuanced considerations
    """
    scores = {k: 0 for k in EXP_TYPE_CATALOGUE}
    reasons = {k: [] for k in EXP_TYPE_CATALOGUE}

    can_randomise = constraints.get('can_randomise', True)
    has_control_group = constraints.get('has_control_group', False)
    rollout_pct = constraints.get('rollout_pct', 50)
    has_time_series = constraints.get('has_time_series', True)
    pre_period_days = constraints.get('pre_period_days', 180)
    has_threshold = constraints.get('has_threshold', False)
    observational = constraints.get('observational', False)

    # ── Rule-based scoring ────────────────────────────────────────────────────
    if can_randomise and rollout_pct <= 70:
        scores['ab_test'] += 5
        reasons['ab_test'].append('Randomisation is possible — gold standard applies')
    else:
        scores['ab_test'] -= 3
        reasons['ab_test'].append('Cannot randomise or 100% rollout — A/B test not feasible')

    if rollout_pct == 100 and not can_randomise:
        scores['pre_post'] += 3
        reasons['pre_post'].append('100% rollout — pre-post is the baseline option')
        if has_time_series and pre_period_days >= 90:
            scores['its'] += 4
            reasons['its'].append('Rich time series available — ITS is stronger than simple pre-post')
            scores['pre_post'] -= 1

    if has_control_group and not can_randomise:
        scores['did'] += 4
        reasons['did'].append('Untreated control group exists — DiD is well-suited')

    if rollout_pct < 100 and has_control_group:
        scores['did'] += 3
        reasons['did'].append('Partial rollout with control group — parallel trends may hold')

    if has_time_series and pre_period_days >= 180:
        scores['its'] += 3
        scores['synthetic_control'] += 2
        reasons['its'].append(f'{pre_period_days} days of pre-period data — sufficient for ITS')

    if observational:
        scores['psm'] += 4
        reasons['psm'].append('Observational data with self-selection — PSM addresses this')
        scores['ab_test'] -= 2

    if has_threshold:
        scores['regression_discontinuity'] += 5
        reasons['regression_discontinuity'].append('Assignment threshold detected — RD design is ideal')

    # Keyword scoring on description
    desc_lower = (desc + ' ' + problem + ' ' + hypothesis).lower()
    for method, meta in EXP_TYPE_CATALOGUE.items():
        kw_hits = sum(1 for kw in meta['keywords'] if kw in desc_lower)
        scores[method] += kw_hits
        if kw_hits > 0:
            reasons[method].append(f'{kw_hits} keyword match(es) in description')

    # ── LLM refinement ────────────────────────────────────────────────────────
    top_3_by_score = sorted(scores, key=lambda k: scores[k], reverse=True)[:4]
    top_3_text = '\n'.join(
        f'  {k}: score={scores[k]}, reasons={reasons[k]}'
        for k in top_3_by_score
    )
    catalogue_text = '\n'.join(
        f'  {k}: {v["label"]} — {v["when_to_use"]}'
        for k, v in EXP_TYPE_CATALOGUE.items()
    )
    prompt = (
        f'You are a senior data scientist selecting an experiment design methodology.\n\n'
        f'Feature: "{desc}"\n'
        f'Problem: "{problem}"\n'
        f'Hypothesis: "{hypothesis}"\n'
        f'Target audience: "{target_audience}"\n'
        f'Constraints: {constraints}\n\n'
        f'Rule-based top candidates:\n{top_3_text}\n\n'
        f'All available methods:\n{catalogue_text}\n\n'
        f'Return ONLY a JSON array of the top 3 method keys in order of recommendation, '
        f'with a one-sentence reason for each. Format:\n'
        f'[{{"method":"ab_test","reason":"..."}},{{"method":"did","reason":"..."}},...]\n'
        f'No other text.'
    )
    try:
        resp = llm.ask(prompt).strip()
        # Extract JSON from response
        json_match = re.search(r'\[.*\]', resp, re.DOTALL)
        if json_match:
            ranked = json.loads(json_match.group())
            return ranked[:3]
    except Exception as e:
        logger.warning('LLM experiment type ranking failed: %s', e)

    # Fallback: return rule-based top 3
    return [
        {'method': k, 'reason': '; '.join(reasons[k][:2]) or EXP_TYPE_CATALOGUE[k]['when_to_use']}
        for k in top_3_by_score[:3]
    ]


def _infer_constraints_from_description(desc: str, llm) -> tuple:
    """
    One LLM call replaces the 7 serial constraint questions.
    Returns (constraints dict, uncertain_keys list).
    """
    prompt = (
        f'A product team wants to test this feature: "{desc}"\n\n'
        'Infer the experiment setup constraints. Reply ONLY with valid JSON — no other text:\n'
        '{\n'
        '  "can_randomise": true,\n'
        '  "rollout_pct": 50,\n'
        '  "has_control_group": false,\n'
        '  "has_time_series": true,\n'
        '  "pre_period_days": 180,\n'
        '  "observational": false,\n'
        '  "has_threshold": false,\n'
        '  "uncertain_keys": ["rollout_pct"]\n'
        '}\n'
        'uncertain_keys lists the 1-3 keys most worth confirming with the user.'
    )
    try:
        raw = llm.ask(prompt)
        m = re.search(r'\{[\s\S]*\}', raw)
        if m:
            d = json.loads(m.group())
            uncertain = d.pop('uncertain_keys', [])
            defaults = {
                'can_randomise': True, 'rollout_pct': 50,
                'has_control_group': False, 'has_time_series': True,
                'pre_period_days': 180, 'observational': False,
                'has_threshold': False,
            }
            for k, v in defaults.items():
                d.setdefault(k, v)
            return d, uncertain
    except Exception as e:
        logger.warning('_infer_constraints_from_description failed: %s', e)
    defaults = {
        'can_randomise': True, 'rollout_pct': 50, 'has_control_group': False,
        'has_time_series': True, 'pre_period_days': 180,
        'observational': False, 'has_threshold': False,
    }
    return defaults, list(defaults.keys())


def _confirm_uncertain_constraints(constraints: dict, uncertain: list) -> dict:
    """
    Only ask about keys the LLM flagged as ambiguous. Typically 1-3 questions.
    """
    QUESTION_MAP = {
        'can_randomise':     ('Can users be randomly assigned to control vs treatment?', bool),
        'rollout_pct':       ('% of traffic receiving this feature (100 = full rollout)?', float),
        'has_control_group': ('Will there be a permanent untreated group (segment/region)?', bool),
        'has_time_series':   ('Do you have 3+ months of historical daily data?', bool),
        'pre_period_days':   ('How many days of pre-feature history are available?', int),
        'observational':     ('Observational study — users self-select rather than being assigned?', bool),
        'has_threshold':     ('Is treatment assigned based on a score or threshold?', bool),
    }
    if not uncertain:
        return constraints
    print(f'\n  Confirming {len(uncertain)} inferred value(s) — press Enter to accept default:\n')
    for key in uncertain:
        if key not in QUESTION_MAP:
            continue
        q, typ = QUESTION_MAP[key]
        default = constraints.get(key)
        if typ == bool:
            hint = f'[{"Y/n" if default else "y/N"}]'
            raw = input(f'  ❓ {q} {hint}: ').strip().lower()
            if raw:
                constraints[key] = raw in ('y', 'yes')
        elif typ in (float, int):
            hint = f'[{default}]'
            raw = input(f'  ❓ {q} {hint}: ').strip()
            if raw:
                try:
                    constraints[key] = typ(raw)
                except ValueError:
                    pass
    return constraints


def _show_method_recommendation(recommendations: list, menu_items: list) -> str:
    """
    Leads with the top recommendation so the user sees the answer immediately.
    Full 8-method menu is hidden behind [?] to reduce cognitive load.
    """
    if not recommendations:
        return 'ab_test'
    top      = recommendations[0]
    method_key = top.get('method', 'ab_test')
    meta     = EXP_TYPE_CATALOGUE.get(method_key, {})
    print()
    print('  ┌' + '─'*68 + '┐')
    print(f'  │  Recommended: {meta.get("label", method_key):<54}│')
    print(f'  │  Why: {top.get("reason","")[:62]:<62}│')
    print(f'  │  Causal strength: {meta.get("causal_strength",""):<51}│')
    reqs = ", ".join(meta.get("requires", [])[:2])
    print(f'  │  Requires: {reqs:<58}│')
    print('  └' + '─'*68 + '┘')
    if len(recommendations) > 1:
        alt      = recommendations[1]
        alt_meta = EXP_TYPE_CATALOGUE.get(alt.get('method', ''), {})
        print(f'\n  Alternative: {alt_meta.get("label","")}')
        print(f'  Trade-off  : {alt.get("reason","")}')
    print()
    raw = input(
        f'  Use "{meta.get("label", method_key)}"?  '
        '[Y = yes  /  N = choose differently  /  ? = see all options]: '
    ).strip().lower()
    if raw in ('', 'y', 'yes'):
        return method_key
    if raw == '?':
        print()
        for idx, (k, v) in enumerate(menu_items, 1):
            print(f'  [{idx}]  {v["label"]}')
            print(f'        {v["when_to_use"]}')
        while True:
            pick = input(f'\n  Enter number [1-{len(menu_items)}]: ').strip()
            if pick.isdigit() and 1 <= int(pick) <= len(menu_items):
                return menu_items[int(pick)-1][0]
            print('     ⚠️  Invalid choice')
    if raw == 'n':
        if len(recommendations) > 1:
            print('\n  Alternatives:')
            for i, rec in enumerate(recommendations[1:], 2):
                m2 = EXP_TYPE_CATALOGUE.get(rec.get('method', ''), {})
                print(f'  [{i}] {m2.get("label",""):40} — {rec.get("reason","")}')
        print(f'  [?] See all {len(menu_items)} methods')
        while True:
            pick = input('  Choose: ').strip().lower()
            if pick == '?':
                return _show_method_recommendation([], menu_items)
            if pick.isdigit():
                n = int(pick)
                if 2 <= n <= len(recommendations):
                    return recommendations[n-1].get('method', 'ab_test')
            print('     ⚠️  Invalid choice')
    return method_key


def _prompt_all_gap_sections(gap_sections_spec: list) -> dict:
    """
    Collect all section mode choices in one pass.
    Returns dict {sec_name: ('user', text) | ('llm', None) | ('skip', None)}.
    """
    print()
    print('  ── Section preferences ────────────────────────────────────────────')
    print('For each section:  1 = I\'ll write it   2 = LLM drafts (default)   3 = Skip')
    print()
    choices = {}
    for sec_name, sec_desc in gap_sections_spec:
        print(f'  {sec_name}')
        print(f'    {sec_desc}')
        raw = input('  [1/2/3] (default 2): ').strip() or '2'
        if raw == '1':
            print(f'\n  Enter your {sec_name.lower()} content. Blank line to finish.')
            lines = []
            while True:
                line = input('  │ ')
                if not line.strip() and lines:
                    break
                if line.strip():
                    lines.append(line)
            text = '\n'.join(lines).strip()
            choices[sec_name] = ('user', text) if text else ('llm', None)
        elif raw == '3':
            choices[sec_name] = ('skip', None)
        else:
            choices[sec_name] = ('llm', None)
        print()
    return choices


def _generate_brief_in_batches(
    sections_for_llm: list,
    context_block: str,
    guidance: str,
    llm,
) -> dict:
    """
    Two-batch generation instead of one monolithic call.
    Batch 1 — structural: brief header, problem, hypothesis, method, design.
    Batch 2 — operational: validity, risks, rollout, sign-off.
    Each batch has its own token budget so later sections never get cut off.
    """
    BATCH_1 = {'EXPERIMENT BRIEF', 'PROBLEM STATEMENT', 'HYPOTHESIS',
               'WHY THIS METHOD', 'EXPERIMENT DESIGN'}
    BATCH_2 = {'SUCCESS CRITERIA', 'CAUSAL VALIDITY CHECKS',
               'RISKS AND MITIGATIONS', 'ROLLOUT PLAN', 'SIGN-OFF REQUIRED'}
    b1 = [s for s in sections_for_llm if s in BATCH_1]
    b2 = [s for s in sections_for_llm if s in BATCH_2]
    parsed = {}
    for batch, batch_label in [(b1, 'structural'), (b2, 'operational')]:
        if not batch:
            continue
        print(f'  Generating {batch_label} sections ({len(batch)})...', end=' ', flush=True)
        prompt = build_llm_prompt_from_template(
            role=(
                'You are a senior product analytics lead writing an experiment brief. '
                'Be concise and specific. Use "Field: value" lines for metadata. '
                'Each section must be 3-6 lines unless detail is explicitly needed.'
            ),
            context_block=context_block,
            sections_to_fill=batch,
            content_guidance=guidance,
        )
        raw = llm.ask(prompt)
        raw = _strip_decorative_chars(raw)
        batch_parsed = parse_sections_from_llm_output(raw, batch)
        parsed.update(batch_parsed)
        filled = sum(1 for v in batch_parsed.values() if v.strip())
        print(f'done  ({filled}/{len(batch)} filled)')
    return parsed


def run_brief_generator(llm):
    """
    Experiment Brief Generator — uses auto-inference for method selection and
    constraint gathering. Sections collected in one pass; brief generated in
    two focused batches to prevent token exhaustion.
    """
    print()
    print('=' * 72)
    print('  EXPERIMENT BRIEF GENERATOR')
    print('  Choose or suggest an experiment type, then write a full PRD brief.')
    print('=' * 72)

    # ── Step 1: Gather basic inputs ───────────────────────────────────────────
    print('\n  Step 1 of 3 — Describe the feature and context\n')
    while True:
        desc = input('  Feature description (what changes?): ').strip()
        if len(desc) >= 10:
            break
        print('     Please provide more detail (at least 10 characters)')

    problem    = input('\n  Problem statement (what user/business problem does this solve?): ').strip()
    hypothesis = input('\n  Hypothesis (what do you expect to happen and why?): ').strip()
    target     = input('\n  Target audience (who sees this feature?): ').strip()

    # ── Step 2: Method selection (auto-infer then confirm) ───────────────────
    print()
    print('-' * 72)
    print('  Step 2 of 3 — Experiment method')
    print('-' * 72)

    menu_items = list(EXP_TYPE_CATALOGUE.items())
    recommendations   = []
    constraints       = {}
    chosen_method_key = None

    print()
    print('  [A]  Auto-suggest — infer best method from your description (recommended)')
    print('  [M]  Manual       — browse all 8 methods and pick')
    print()
    mode_raw = input('  Choose [A/M] (default A): ').strip().upper() or 'A'

    if mode_raw == 'M':
        # ── Manual: show full menu ────────────────────────────────────────────
        for idx, (k, v) in enumerate(menu_items, 1):
            print(f'  [{idx}]  {v["label"]}')
            print(f'        When to use: {v["when_to_use"]}')
        while chosen_method_key is None:
            raw = input(f'\n  Enter number [1-{len(menu_items)}]: ').strip()
            if raw.isdigit() and 1 <= int(raw) <= len(menu_items):
                chosen_method_key = menu_items[int(raw)-1][0]
    else:
        # ── Auto-suggest: infer constraints, ask only about ambiguous ones ────
        print('\n  Analysing feature description...')
        constraints, uncertain = _infer_constraints_from_description(desc, llm)
        constraints = _confirm_uncertain_constraints(constraints, uncertain)

        print('\n  Inferred constraints:')
        for k, v in constraints.items():
            print(f'    {k:<22}: {v}')

        raw_ok = input('\n  Look right? [Y/n]: ').strip().lower()
        if raw_ok == 'n':
            # Fall back to manual constraint questions
            can_randomise = input('  Can you randomly assign users? [Y/n]: ').strip().lower() != 'n'
            raw_pct = input('  % of traffic in experiment [50]: ').strip()
            rollout_pct = float(raw_pct) if raw_pct else 50
            has_control = input('  Permanent untreated group? [y/N]: ').strip().lower() == 'y'
            has_ts = input('  3+ months historical data? [Y/n]: ').strip().lower() != 'n'
            raw_days = input('  Days of pre-feature history [180]: ').strip()
            pre_days = int(raw_days) if raw_days else 180
            obs = input('  Observational study? [y/N]: ').strip().lower() == 'y'
            thresh = input('  Threshold-based assignment? [y/N]: ').strip().lower() == 'y'
            constraints = {
                'can_randomise': can_randomise, 'rollout_pct': rollout_pct,
                'has_control_group': has_control, 'has_time_series': has_ts,
                'pre_period_days': pre_days, 'observational': obs,
                'has_threshold': thresh,
            }

        print('\n  Computing recommendation...')
        recommendations = recommend_experiment_type(
            desc, problem, hypothesis, target, constraints, llm)

        chosen_method_key = _show_method_recommendation(recommendations, menu_items)

    chosen_label = EXP_TYPE_CATALOGUE.get(chosen_method_key, {}).get('label', chosen_method_key)
    method_meta  = EXP_TYPE_CATALOGUE.get(chosen_method_key, {})
    is_ab_test   = 'ab_test' in chosen_method_key
    print(f'\n  ✅ Method: {chosen_label}')

    # ── Step 3: Section preferences + brief generation ───────────────────────
    print()
    print('-' * 72)
    print('  Step 3 of 3 — Brief sections')
    print('-' * 72)

    gap_sections_spec = [
        ('SUCCESS CRITERIA',
         'Primary / secondary / guardrail metrics and the size of the move you need.'),
        ('CAUSAL VALIDITY CHECKS',
         'Method-specific checks that must pass before trusting results.'),
        ('RISKS AND MITIGATIONS',
         'What could go wrong, and how you would handle it.'),
        ('ROLLOUT PLAN',
         'Ramp-up %, monitoring cadence, stop criteria, and what "ship" means.'),
    ]
    always_llm = ['EXPERIMENT BRIEF', 'PROBLEM STATEMENT', 'HYPOTHESIS',
                  'WHY THIS METHOD', 'EXPERIMENT DESIGN', 'SIGN-OFF REQUIRED']

    gap_choices = _prompt_all_gap_sections(gap_sections_spec)
    user_section_content = {
        k: v[1] for k, v in gap_choices.items() if v[0] == 'user' and v[1]
    }
    llm_section_list = [k for k, v in gap_choices.items() if v[0] == 'llm']

    # ── Template check ────────────────────────────────────────────────────────
    default_sections = [
        'EXPERIMENT BRIEF', 'PROBLEM STATEMENT', 'HYPOTHESIS',
        'WHY THIS METHOD', 'EXPERIMENT DESIGN', 'SUCCESS CRITERIA',
        'CAUSAL VALIDITY CHECKS', 'RISKS AND MITIGATIONS',
        'ROLLOUT PLAN', 'SIGN-OFF REQUIRED',
    ]
    user_sections, _ = ask_for_template('Experiment Brief / PRD', default_sections)
    sections_to_use  = user_sections if user_sections else default_sections

    sections_for_llm = [
        s for s in sections_to_use if s in always_llm or s in llm_section_list
    ]

    try:
        _past = _query_relevant_learnings(f'{desc} {problem}', n=3)
        _past_text = _format_past_learnings(_past)
        _knowledge_note = (
            f'\n\nRELEVANT PAST EXPERIMENTS ({len(_past)} found):\n{_past_text}'
            if _past else ''
        )
        if _past:
            print(f'\n  📚 Found {len(_past)} relevant past experiment(s) — incorporating into brief')
    except Exception:
        _knowledge_note = ''

    context_block = (
        f'Feature: "{desc}"\n'
        f'Problem: "{problem}"\n'
        f'Hypothesis: "{hypothesis}"\n'
        f'Target audience: "{target}"\n'
        f'Experiment type: {chosen_label}\n'
        f'Causal strength: {method_meta.get("causal_strength", "medium")}\n'
        f'Method requires: {", ".join(method_meta.get("requires", []))}' +
        _knowledge_note
    )

    traffic_line = (
        'Traffic allocation: 50% control / 50% treatment' if is_ab_test
        else 'Pre period: [dates], Post period: [dates]'
    )
    guidance = (
        f'EXPERIMENT BRIEF: include Title, Type ({chosen_label}), Team, Priority as "Field: value" lines.\n'
        f'EXPERIMENT DESIGN: Method: {chosen_label} / {traffic_line} / Target audience: {target} / '
        f'Exclusions / Sample size considerations.\n'
        'HYPOTHESIS: "We believe <change> will cause <effect> for <users> because <reasoning>. '
        'We will know this is true when <metric> changes by <amount>."\n'
        f'A/B test: {"yes" if is_ab_test else "no"}.'
    )

    parsed = {}
    if sections_for_llm:
        parsed = _generate_brief_in_batches(sections_for_llm, context_block, guidance, llm)

    # Merge user-provided sections
    for k, v in user_section_content.items():
        parsed[k] = v

    from collections import OrderedDict
    parsed_ordered = OrderedDict(
        (s, parsed.get(s, '')) for s in sections_to_use
    )

    # ── Render PDF ────────────────────────────────────────────────────────────
    fname = 'experiment_brief.pdf'
    out_path = render_document_pdf(
        title='Experiment Brief',
        subtitle='Feature: ' + desc[:90],
        sections=parsed_ordered,
        output_path=fname,
        metadata={
            'Feature':         desc[:120],
            'Experiment type': chosen_label,
            'Causal strength': method_meta.get('causal_strength', 'Unknown'),
            'Complexity':      method_meta.get('complexity', 'Unknown'),
            'Target audience': target[:120] if target else 'n/a',
        },
        accent_color=PDF_PALETTE['primary'],
    )
    print(f'\n  Brief saved → {out_path}')

    # ── Auto-run power calculator for A/B tests ───────────────────────────────
    power_result = None
    if is_ab_test:
        print('\n' + '━'*72)
        print('  A/B test selected — auto-running Power Calculator...')
        print('━'*72)
        try:
            power_result = run_power_calculator(llm)
        except Exception as e:
            print(f'  ⚠️  Power calculator error: {type(e).__name__}: {e}')

    return {
        'brief_sections':      dict(parsed_ordered),
        'recommended_method':  chosen_method_key,
        'all_recommendations': recommendations,
        'constraints':         constraints,
        'output_file':         out_path,
        'sections_used':       sections_to_use,
        'user_provided':       list(user_section_content.keys()),
        'power_calculator':    power_result,
    }



def run_learnings_repository(llm):
    print('\n' + '╔' + '═'*70 + '╗')
    print('║' + '  📚  EXPERIMENT LEARNINGS REPOSITORY'.ljust(70) + '║')
    print('║' + '  Add, search, and retrieve past experiment knowledge'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')
    print()
    print('  [1]  🔍  Search learnings  (find relevant past experiments)')
    print('  [2]  ➕  Add new learning  (record a concluded experiment)')
    print('  [3]  📋  Browse all         (list everything in the repository)')
    print()
    while True:
        action = input('  ❓ Choose [1/2/3]: ').strip()
        if action in ('1','2','3'): break

    if action == '3':
        df_learn = db.execute("""
            SELECT id, experiment_name, ship_decision, outcome, recorded_at
            FROM experiment_learnings ORDER BY recorded_at DESC
        """).df()
        print(f'\n  Repository contains {len(df_learn)} learning(s):\n')
        for _, row in df_learn.iterrows():
            icon = {'ship':'✅','no_ship':'❌','partial_ship':'⚠️'}.get(row['ship_decision'],'⬜')
            print(f'  {icon} [{row["id"]}] {row["experiment_name"]}  ({row["recorded_at"]})')
            print(f'       {row["outcome"][:100]}')
            print()
        # Full details
        sel = input('  ❓ Enter ID to read full details (or Enter to skip): ').strip().upper()
        if sel:
            row = db.execute(f"SELECT * FROM experiment_learnings WHERE id = '{sel}'").df()
            if len(row) > 0:
                r = row.iloc[0]
                print(f'\n  {"─"*68}')
                for col in row.columns:
                    print(f'  {col:<28} {r[col]}')
        return

    if action == '1':
        query = input('\n  ❓ Search query (describe what you want to know): ').strip()
        df_learn = db.execute("SELECT * FROM experiment_learnings").df()
        if len(df_learn) == 0:
            print('  No learnings recorded yet.')
            return

        all_text = '\n\n'.join(
            f'[{r["id"]}] {r["experiment_name"]}: {r["key_learning"]}. '
            f'What worked: {r["what_worked"]}. Tags: {r["tags"]}'
            for _, r in df_learn.iterrows()
        )
        prompt = (
            f'Search this experiment learnings repository and return the most relevant results.\n'
            f'Query: "{query}"\n\n'
            f'Repository:\n{all_text}\n\n'
            f'Return: the 2-3 most relevant experiment IDs and why they are relevant. '
            f'Then synthesise the key insight that answers the query.'
        )
        print('\n  🤖 Searching...')
        result = llm.ask(prompt)
        print('\n' + '─'*68)
        print(result)
        print('─'*68)
        return result

    if action == '2':
        # Add new learning
        print('\n  ── Record a new learning ──')
        print('  (This stores the outcome so future experiment designs can learn from it)\n')
        exp_name    = input('  ❓ Experiment name: ').strip()
        ship_raw    = input('  ❓ Ship decision (ship/no_ship/partial_ship): ').strip()
        outcome     = input('  ❓ Outcome summary (1-2 sentences, include numbers): ').strip()
        key_learn   = input('  ❓ Key learning (the "so what" insight): ').strip()
        worked      = input('  ❓ What worked: ').strip()
        didnt       = input('  ❓ What did NOT work: ').strip()
        recommend   = input('  ❓ Recommendation for future experiments: ').strip()
        follow_ups  = input('  ❓ Follow-up experiment ideas (comma-sep): ').strip()
        tags        = input('  ❓ Tags (comma-sep, e.g. checkout,ior,mobile): ').strip()

        df_learn = db.execute("SELECT id FROM experiment_learnings").df()
        new_id   = f'L{len(df_learn)+1:03d}'
        new_row  = pd.DataFrame([{
            'id':                    new_id,
            'experiment_name':       exp_name,
            'ship_decision':         ship_raw,
            'outcome':               outcome,
            'key_learning':          key_learn,
            'what_worked':           worked,
            'what_didnt':            didnt,
            'recommendation':        recommend,
            'follow_up_experiments': [x.strip() for x in follow_ups.split(',')],
            'recorded_by':           'Analytics Team',
            'recorded_at':           pd.Timestamp.today().strftime('%Y-%m-%d'),
            'tags':                  [x.strip() for x in tags.split(',')],
        }])
        df_all_learn = pd.concat([
            db.execute("SELECT * FROM experiment_learnings").df(), new_row
        ], ignore_index=True)
        db.register('experiment_learnings', df_all_learn)
        print(f'\n  ✅ Learning [{new_id}] recorded. Repository now has {len(df_all_learn)} entries.')

        narrative = llm.narrate(new_row.to_dict('records'),
            'A new experiment learning has been recorded. Write a 2-sentence distillation '
            'of the most important insight for future experiment designers.')
        print(f'\n  🤖 Insight: {narrative}')
        return new_id



# ═════════════════════════════════════════════════════════════════════════════
# MODULE 10 — ROI TRACKER WITH COUNTERFACTUAL FORECASTING
# ═════════════════════════════════════════════════════════════════════════════

def _fit_counterfactual_model(df_ts: 'pd.DataFrame', cutoff_date: 'pd.Timestamp') -> dict:
    """
    Fits a lightweight time-series decomposition model on pre-ship IOR data.

    Model:  IOR(t) = trend(t) + weekly_seasonality(t) + monthly_seasonality(t) + noise
    Method: OLS regression with dummy variables — no external libraries needed.
             This is a simplified version of what Prophet does internally.

    Parameters:
        df_ts       : platform_daily_ior DataFrame
        cutoff_date : ship date — train on data before this, forecast after

    Returns dict with:
        coefficients, forecast function, in-sample R², MAPE
    """
    train = df_ts[df_ts['date'] < cutoff_date].copy().reset_index(drop=True)
    if len(train) < 60:
        return None   

    n = len(train)
    X = np.zeros((n, 1 + 1 + 6 + 11))   # intercept + trend + 6 DOW dummies + 11 month dummies
    X[:, 0] = 1.0                                          # intercept
    X[:, 1] = np.arange(n) / n                            # normalised trend (0→1)
    for dow in range(1, 7):                                # Mon=0 is baseline
        X[:, 1 + dow] = (train['day_of_week'] == dow).astype(float)
    for m in range(1, 12):                                 # Jan is baseline
        X[:, 7 + m] = (train['month'] == (m + 1)).astype(float)

    y = train['ior'].values

    try:
        XtX = X.T @ X
        XtX_inv = np.linalg.inv(XtX + np.eye(XtX.shape[0]) * 1e-8)  # ridge regularisation
        coef = XtX_inv @ X.T @ y
    except np.linalg.LinAlgError:
        return None

    y_hat = X @ coef
    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2     = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    mape   = np.mean(np.abs((y - y_hat) / np.clip(y, 0.01, 1))) * 100

    def forecast(dates: 'pd.Series') -> 'np.ndarray':
        """Generate counterfactual IOR forecast for given dates."""
        n_pred = len(dates)
        trend_val = np.arange(n, n + n_pred) / n
        Xp = np.zeros((n_pred, 1 + 1 + 6 + 11))
        Xp[:, 0] = 1.0
        Xp[:, 1] = trend_val
        for dow in range(1, 7):
            Xp[:, 1 + dow] = (dates.dt.dayofweek == dow).astype(float)
        for m in range(1, 12):
            Xp[:, 7 + m] = (dates.dt.month == (m + 1)).astype(float)
        return np.clip(Xp @ coef, 0.01, 0.99)

    return {
        'coef': coef, 'r2': round(r2, 4), 'mape': round(mape, 3),
        'n_train': n, 'train_start': train['date'].min(), 'cutoff': cutoff_date,
        'forecast': forecast,
    }


def _confidence_grade(model_fit: dict, n_confounders: int, has_holdout: bool) -> tuple:
    """
    Returns a confidence grade (A/B/C/D) and explanation string.

    Grade A: Holdout group exists — cleanest possible measurement
    Grade B: Good model fit (R²>0.6), few confounders (≤1)
    Grade C: Moderate fit or 2-3 confounders — interpret with caution
    Grade D: Poor fit or many confounders — results unreliable
    """
    if has_holdout:
        return 'A', 'Holdout group present — cleanest causal estimate'
    r2 = model_fit.get('r2', 0) if model_fit else 0
    if r2 >= 0.65 and n_confounders == 0:
        return 'B', f'Good model fit (R²={r2:.2f}), no concurrent ships detected'
    if r2 >= 0.50 and n_confounders <= 2:
        return 'C', (f'Moderate fit (R²={r2:.2f}), {n_confounders} concurrent ship(s). '
                     f'Counterfactual estimate is directionally correct but magnitude uncertain.')
    return 'D', (f'Weak model fit (R²={r2:.2f}) or {n_confounders} confounders. '
                 f'Results are indicative only — validate with a holdout group.')


def _compute_gmv_impact(
    post_df: 'pd.DataFrame',
    counterfactual: 'np.ndarray',
    daily_inquiries: float,
    aov: float,
) -> dict:
    """
    Computes incremental GMV using the counterfactual-corrected IOR lift.

    incremental_daily_gmv = (observed_IOR − counterfactual_IOR) × daily_inquiries × AOV
    """
    observed    = post_df['observed_ior'].values[:len(counterfactual)]
    cf          = counterfactual[:len(observed)]
    lift_series = observed - cf

    daily_gmv = lift_series * daily_inquiries * aov
    cumulative = np.cumsum(daily_gmv)

    return {
        'lift_series':        lift_series,
        'daily_gmv_series':   daily_gmv,
        'cumulative_gmv':     cumulative,
        'total_gmv_90d':      float(np.sum(daily_gmv)),
        'avg_daily_lift_pp':  float(np.mean(lift_series) * 100),
        'median_daily_lift_pp': float(np.median(lift_series) * 100),
        'pct_days_positive':  float(np.mean(lift_series > 0) * 100),
    }


def run_roi_tracker(llm):
    """Module 10 — experiment-first ROI tracking for shipped features."""
    return analyze_experiment(llm, mode='roi')



## 11 · Phase 3 — Post-Experiment Modules

Module [8] Causal Analysis · [9] Simpson's Paradox Detector · [10] ROI Tracker · [11] Learnings Repository. All experiment-first: pick an experiment from the list, get a business-level verdict + dimensional cuts + deep-dive + ship recommendation.

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# LLM INTELLIGENCE LAYER
# ─────────────────────────────────────────────────────────────────────────────


def _query_relevant_learnings(topic: str, n: int = 3) -> list:
    """
    Semantic search of experiment_learnings for past experiments relevant to topic.
    Returns up to n records as dicts, or empty list if repository is empty.
    Used proactively by briefs, power calc, and causal analysis — not just on demand.
    """
    try:
        df = db.execute("SELECT * FROM experiment_learnings ORDER BY recorded_at DESC").df()
    except Exception:
        return []
    if df.empty:
        return []

    topic_words = set(re.sub(r'[^a-z ]', '', topic.lower()).split())
    scored = []
    for _, row in df.iterrows():
        text = ' '.join(str(v).lower() for v in [
            row.get('key_learning',''), row.get('outcome',''),
            row.get('what_worked',''), row.get('tags',''),
        ])
        text_words = set(re.sub(r'[^a-z ]', '', text).split())
        overlap = len(topic_words & text_words)
        scored.append((overlap, row.to_dict()))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [r for _, r in scored[:n] if scored[0][0] > 0]


def _format_past_learnings(learnings: list) -> str:
    """Format a list of learning dicts into a compact text block for LLM prompts."""
    if not learnings:
        return '(No relevant past experiments found in the Learnings Repository.)'
    lines = []
    for l in learnings:
        lines.append(
            f'• [{l.get("id","")}] {l.get("experiment_name","")} '
            f'({l.get("ship_decision","?").replace("_"," ")}): '
            f'{l.get("key_learning","")} '
            f'| What worked: {l.get("what_worked","")} '
            f'| What did not: {l.get("what_didnt","")}'
        )
    return '\n'.join(lines)


def _build_experiment_context(
    exp_name: str,
    exp_info: dict,
    overall: dict,
    dim_cuts: dict,
    interesting: list,
    decision: str,
    reasoning: str,
    extra_insights: dict = None,
) -> dict:
    """
    Assemble everything the platform knows about a concluded experiment
    into a single context object. Every downstream LLM call draws from this.

    Returns a dict with keys:
      experiment, description, hypothesis, method, decision, reasoning,
      overall_summary, segment_summary, interesting_summary,
      time_trend_summary, extra_insights_summary, past_learnings
    """
    # Overall effect summary
    overall_summary = '; '.join(
        f'{t}: Δ={r["delta_pp"]:+.2f}pp [CI {r["ci_lo_pp"]:+.2f}, '
        f'{r["ci_hi_pp"]:+.2f}] p={r["p_value"]:.4f} n={r["n_treatment"]:,} '
        f'{"(sig)" if r["sig"] else "(n.s.)"}'
        for t, r in overall.items()
    )

    seg_parts = []
    for dim, rows in dim_cuts.items():
        sig_rows = [r for r in rows if r['sig']]
        for r in sig_rows[:4]:
            seg_parts.append(
                f'{r["dim"]}={r["level"]}/{r["treatment"]}: '
                f'Δ={r["delta_pp"]:+.2f}pp (sig)'
            )
    segment_summary = '; '.join(seg_parts) or 'No significant segment-level effects.'

    interesting_summary = ', '.join(
        f'{kind}: {r["dim"]}={r["level"]} Δ={r["delta_pp"]:+.2f}pp'
        for kind, r in interesting[:5]
    ) or 'None detected.'

    time_trend = ''
    if extra_insights and extra_insights.get('time_decay'):
        td = extra_insights['time_decay']
        time_trend = (
            f'Early effect (first half): Δ={td.get("early_delta",0):+.2f}pp | '
            f'Late effect (second half): Δ={td.get("late_delta",0):+.2f}pp | '
            f'Decay: {td.get("decay_direction","stable")}'
        )
    else:
        time_trend = '(Time-period breakdown not computed for this experiment.)'

    ei_summary = ''
    if extra_insights:
        parts = []
        if extra_insights.get('cohort_effect'):
            ce = extra_insights['cohort_effect']
            parts.append(f'New user cohort effect: {ce.get("summary","n/a")}')
        if extra_insights.get('cross_metric'):
            parts.append(f'Cross-metric: {extra_insights["cross_metric"].get("summary","n/a")}')
        ei_summary = '; '.join(parts) or '(No additional insights detected.)'
    else:
        ei_summary = '(Insights mining not run.)'

    topic = f'{exp_info.get("description","")} {overall_summary}'
    past_learnings = _query_relevant_learnings(topic, n=3)

    return {
        'experiment':          exp_name,
        'description':         exp_info.get('description', ''),
        'hypothesis':          exp_info.get('hypothesis', '(not recorded)'),
        'method':              exp_info.get('method', 'A/B test'),
        'team':                exp_info.get('team', ''),
        'decision':            decision,
        'reasoning':           reasoning,
        'overall_summary':     overall_summary,
        'segment_summary':     segment_summary,
        'interesting_summary': interesting_summary,
        'time_trend_summary':  time_trend,
        'extra_insights':      ei_summary,
        'past_learnings':      _format_past_learnings(past_learnings),
        'n_past_learnings':    len(past_learnings),
    }


def _mine_additional_insights(
    exp_df: 'pd.DataFrame',
    overall: dict,
    dim_cuts: dict,
    control: str = 'control',
    treatments: list = None,
) -> dict:
    """
    Run automated additional statistical tests beyond the pre-specified analysis:
      1. Time-period decay — did the effect weaken over the experiment duration?
      2. Cohort effect — do new users respond differently from returning users?
      3. Cross-metric correlation — does IOR lift correlate with AOV change?

    Returns a structured dict of results (all deterministic — no LLM).
    The LLM only sees this dict AFTER it is computed.
    """
    if treatments is None:
        treatments = [v for v in exp_df['variant'].unique() if v != control]
    if not treatments:
        return {}

    trt = treatments[0]
    results = {}

    # ── 1. Time-period decay ─────────────────────────────────────────────────
    try:
        exp_df = exp_df.copy()
        exp_df['_date'] = pd.to_datetime(exp_df['created_at'])
        date_min = exp_df['_date'].min()
        date_max = exp_df['_date'].max()
        mid_date = date_min + (date_max - date_min) / 2

        early = exp_df[exp_df['_date'] <= mid_date]
        late  = exp_df[exp_df['_date'] >  mid_date]

        def _ior(df, variant):
            sub = df[df['variant'] == variant]
            if len(sub) < 30:
                return None
            return float(sub['converted_to_order'].mean())

        early_ctrl = _ior(early, control)
        early_trt  = _ior(early, trt)
        late_ctrl  = _ior(late,  control)
        late_trt   = _ior(late,  trt)

        if all(v is not None for v in [early_ctrl, early_trt, late_ctrl, late_trt]):
            early_delta = (early_trt - early_ctrl) * 100
            late_delta  = (late_trt  - late_ctrl)  * 100
            decay       = late_delta - early_delta
            results['time_decay'] = {
                'early_delta': round(early_delta, 3),
                'late_delta':  round(late_delta,  3),
                'decay_pp':    round(decay, 3),
                'decay_direction': ('weakening' if decay < -0.3
                                    else 'strengthening' if decay > 0.3
                                    else 'stable'),
                'summary': (
                    f'Early half Δ={early_delta:+.2f}pp → '
                    f'Late half Δ={late_delta:+.2f}pp '
                    f'({"weakening" if decay < -0.3 else "strengthening" if decay > 0.3 else "stable"})'
                ),
            }
    except Exception as e:
        results['time_decay'] = {'error': str(e)}

    # ── 2. Cohort effect — new vs returning users ────────────────────────────
    try:
        if 'lifetime_orders' in exp_df.columns:
            exp_df['_is_new'] = exp_df['lifetime_orders'] <= 1
            new_users = exp_df[exp_df['_is_new']]
            ret_users = exp_df[~exp_df['_is_new']]

            def _delta(df):
                c = df[df['variant']==control]['converted_to_order'].mean()
                t = df[df['variant']==trt]['converted_to_order'].mean()
                n_c = len(df[df['variant']==control])
                n_t = len(df[df['variant']==trt])
                if n_c < 30 or n_t < 30:
                    return None, None
                return (t - c) * 100, n_t

            new_delta, new_n = _delta(new_users)
            ret_delta, ret_n = _delta(ret_users)

            if new_delta is not None and ret_delta is not None:
                divergence = abs(new_delta - ret_delta) > 0.5
                results['cohort_effect'] = {
                    'new_user_delta_pp': round(new_delta, 3),
                    'returning_user_delta_pp': round(ret_delta, 3),
                    'divergence': divergence,
                    'summary': (
                        f'New users Δ={new_delta:+.2f}pp (n={new_n:,}) vs '
                        f'returning Δ={ret_delta:+.2f}pp (n={ret_n:,}) '
                        f'{"— DIVERGENT" if divergence else "— similar response"}'
                    ),
                }
    except Exception as e:
        results['cohort_effect'] = {'error': str(e)}

    # ── 3. Cross-metric: does IOR lift correlate with AOV change? ─────────────
    try:
        if 'order_value' in exp_df.columns:
            ctrl_df = exp_df[(exp_df['variant']==control) & exp_df['converted_to_order']]
            trt_df  = exp_df[(exp_df['variant']==trt)     & exp_df['converted_to_order']]
            if len(ctrl_df) >= 30 and len(trt_df) >= 30:
                aov_ctrl = float(ctrl_df['order_value'].mean())
                aov_trt  = float(trt_df['order_value'].mean())
                aov_delta_pct = (aov_trt - aov_ctrl) / aov_ctrl * 100
                # IOR direction
                ior_direction = next(
                    ('+' if r['sig'] and r['delta_pp'] > 0 else
                     '-' if r['sig'] and r['delta_pp'] < 0 else '~'
                     for r in overall.values()), '~')
                aov_direction = '+' if aov_delta_pct > 1 else '-' if aov_delta_pct < -1 else '~'
                alignment = (ior_direction == aov_direction or
                             ior_direction == '~' or aov_direction == '~')
                results['cross_metric'] = {
                    'aov_control':   round(aov_ctrl, 2),
                    'aov_treatment': round(aov_trt, 2),
                    'aov_delta_pct': round(aov_delta_pct, 2),
                    'ior_aov_aligned': alignment,
                    'summary': (
                        f'AOV: ${aov_ctrl:.0f} → ${aov_trt:.0f} '
                        f'({aov_delta_pct:+.1f}%) '
                        f'{"— IOR and AOV move together (good)" if alignment else "— IOR and AOV DIVERGE (investigate)"}'
                    ),
                }
    except Exception as e:
        results['cross_metric'] = {'error': str(e)}

    return results


def _synthesise_findings(context: dict, llm) -> str:
    """
    Single unified synthesis prompt.

    Asks the LLM to combine:
      - Statistical results (overall + segment-level)
      - Time trends (did the effect decay?)
      - Decision and its reasoning
      - Decision IMPLICATIONS (what should we actually do next?)
      - Trade-offs (what are we giving up with this decision?)
      - Past learnings (what have we seen before that is relevant?)

    Returns a structured text block with four labelled sections.
    """
    prompt = textwrap.dedent(f"""
You are a senior product analytics lead synthesising the full findings of a concluded experiment.

EXPERIMENT: {context['experiment']}
DESCRIPTION: {context['description']}
HYPOTHESIS: {context['hypothesis']}
TEAM: {context['team']}

STATISTICAL RESULTS:
  Overall: {context['overall_summary']}
  Key segments: {context['segment_summary']}
  Interesting findings: {context['interesting_summary']}
  Time trend: {context['time_trend_summary']}
  Additional insights: {context['extra_insights']}

DECISION: {context['decision']}
REASONING: {context['reasoning']}

RELEVANT PAST EXPERIMENTS ({context['n_past_learnings']} found):
{context['past_learnings']}

Write a response with EXACTLY FOUR sections, labelled as shown:

SYNTHESIS:
A 3-4 sentence paragraph combining what the stats, segment-level results, and time trend
collectively tell us. Reconcile any tensions (e.g. aggregate positive but Growth negative).
Be specific — reference actual numbers.

IMPLICATIONS:
3-4 bullet points explaining what the decision means in practice: who gets the feature,
what the rollout sequence should be, what risks to monitor, and what the business should
expect over the next 90 days. This is NOT a restatement of the decision — it reasons
about consequences and next steps.

TRADE-OFFS:
2-3 bullet points on what we are giving up with this decision. If PARTIAL SHIP, what does
holding back Growth cost us? If NO SHIP, what is the opportunity cost? If SHIP, what
guardrails are we relaxing? Be honest and specific.

KNOWLEDGE APPLIED:
1-2 sentences on what past experiments told us that was relevant to interpreting these
results, and whether this experiment confirmed or contradicted that prior knowledge.

Write in plain business English. No emojis. No decorative symbols.
    """).strip()

    try:
        raw = llm.ask(prompt)
        try:
            raw = _strip_decorative_chars(raw)
        except NameError:
            pass
        return raw
    except Exception as e:
        return f'(Synthesis failed: {e})'


def _explain_roi_gap(
    exp_name: str,
    experiment_lift_pp: float,
    production_lift_pp: float,
    concurrent_ships: list,
    llm,
) -> str:
    """
    When post-ship ROI measurement shows a different lift from the experiment,
    LLM reasons about the most likely explanations. Called from _roi_analysis().
    """
    gap  = production_lift_pp - experiment_lift_pp
    sign = 'lower' if gap < 0 else 'higher'
    conc = ', '.join(s['name'] for s in concurrent_ships[:3]) if concurrent_ships else 'none identified'

    prompt = textwrap.dedent(f"""
You are a senior data scientist explaining why post-ship ROI differs from a measured experiment.

Experiment: {exp_name}
Experiment lift (measured): {experiment_lift_pp:+.2f}pp IOR
Post-ship lift (measured):  {production_lift_pp:+.2f}pp IOR
Gap: {abs(gap):.2f}pp {sign} than experiment

Concurrent features shipped during monitoring window: {conc}

In 3-5 sentences, explain the most likely reason(s) for the gap.
Cover: novelty/hawthorne effects, concurrent feature confounds, seasonal variation,
population drift, or SRM-induced bias. Be specific — name which cause is most plausible
for this magnitude of gap and this experiment type.
End with one concrete recommendation for the next measurement cycle.
Do not use emojis.
    """).strip()

    try:
        raw = llm.ask(prompt)
        try:
            raw = _strip_decorative_chars(raw)
        except NameError:
            pass
        return raw
    except Exception as e:
        return f'(Gap explanation unavailable: {e})'



# ═════════════════════════════════════════════════════════════════════════════
# MODULE 11 — CAUSAL ANALYSIS ENGINE
# ═════════════════════════════════════════════════════════════════════════════

# ── Method 1: Pre-Post Analysis ───────────────────────────────────────────────

def _run_pre_post(exp_name: str, cutoff_date: 'pd.Timestamp', alpha: float) -> dict:
    """
    Simple before/after comparison. Weakest causal claim.
    Compares IOR in pre-period vs post-period for the treated population.
    """
    pre  = df_all_experiments[
        (df_all_experiments['experiment_name'] == exp_name) &
        (df_all_experiments['created_at'] < cutoff_date)
    ]
    post = df_all_experiments[
        (df_all_experiments['experiment_name'] == exp_name) &
        (df_all_experiments['created_at'] >= cutoff_date)
    ]
    # Use hist_inquiries if pre-period rows are sparse
    if len(pre) < 50:
        pre = df_hist_inquiries[df_hist_inquiries['created_at'] < cutoff_date]

    if len(pre) == 0 or len(post) == 0:
        return {'error': 'Insufficient data for pre-post comparison'}

    n_pre, c_pre   = len(pre),  int(pre['converted_to_order'].sum())
    n_post, c_post = len(post), int(post['converted_to_order'].sum())
    result = proportion_test(n_pre, c_pre, n_post, c_post, alpha)

    gmv_change = (float(post['order_value'].mean() - pre['order_value'].mean())
                  if 'order_value' in post.columns and 'order_value' in pre.columns
                  and len(pre) > 0 and len(post) > 0
                  else 0.0)

    return {
        'method':        'Pre-Post Analysis',
        'cutoff_date':   str(cutoff_date.date()),
        'n_pre':         n_pre,  'conv_pre': c_pre,  'ior_pre':  result['rate_control'],
        'n_post':        n_post, 'conv_post': c_post, 'ior_post': result['rate_treatment'],
        'delta_pp':      result['delta_pp'],
        'ci':            [result['ci_lo_pp'], result['ci_hi_pp']],
        'p_value':       result['p_value'],
        'significant':   result['is_significant'],
        'gmv_change':    round(gmv_change, 2),
        'caveat':        'Pre-post confounded by seasonality and concurrent changes. Interpret with caution.',
    }


# ── Method 2: Difference-in-Differences (Enhanced) ─────────────────────────────

def _run_did_v2(
    treatment_units: list,
    control_units: list,
    cutoff_date: 'pd.Timestamp',
    pre_start: 'pd.Timestamp',
    alpha: float = 0.05,
    unit_col: str = 'account_segment',   # 'account_segment' | 'buyer_id' | 'account_id'
    outcome_col: str = 'converted_to_order',
    n_bootstrap: int = 1_000,
    run_twfe: bool = True,
) -> dict:
    """
    Enhanced Difference-in-Differences estimator.

    Computes:
      · Classic 2×2 DiD with delta-method SE
      · Bootstrap CI (1 000 resamples, percentile method)
      · Parallel trends test (regression-based, not just split-half)
      · Event study with pointwise 95% CIs
      · Two-Way Fixed Effects (TWFE) OLS estimate
      · Staggered adoption warning
      · Bacon decomposition summary (if multiple treatment times detected)

    Parameters
    ──────────
    treatment_units : list of values in unit_col that received the treatment
    control_units   : list of values in unit_col that are untreated controls
    cutoff_date     : intervention date (pd.Timestamp)
    pre_start       : start of pre-period (pd.Timestamp)
    alpha           : significance level
    unit_col        : the column that defines treatment/control assignment
    outcome_col     : binary outcome column (0/1 or bool)
    n_bootstrap     : number of bootstrap resamples for SE estimation
    run_twfe        : if True, also estimate via TWFE OLS
    """
    import numpy as np
    import pandas as pd
    from scipy import stats as scipy_stats

    all_data = pd.concat([
        globals().get('df_hist_inquiries', pd.DataFrame()),
        globals().get('df_all_experiments', pd.DataFrame()),
    ], ignore_index=True)
    all_data = all_data[all_data['created_at'] >= pre_start].copy()

    if unit_col not in all_data.columns:
        return {'error': f'unit_col "{unit_col}" not found in data'}

    all_units = treatment_units + control_units
    all_data  = all_data[all_data[unit_col].isin(all_units)].copy()
    all_data['treated']  = all_data[unit_col].isin(treatment_units).astype(int)
    all_data['post']     = (all_data['created_at'] >= cutoff_date).astype(int)
    all_data[outcome_col] = all_data[outcome_col].astype(float)

    # Split into cells
    treat_pre  = all_data[(all_data['treated']==1) & (all_data['post']==0)]
    treat_post = all_data[(all_data['treated']==1) & (all_data['post']==1)]
    ctrl_pre   = all_data[(all_data['treated']==0) & (all_data['post']==0)]
    ctrl_post  = all_data[(all_data['treated']==0) & (all_data['post']==1)]

    for cell_name, cell_df in [('treat_pre', treat_pre), ('treat_post', treat_post),
                                ('ctrl_pre', ctrl_pre),   ('ctrl_post', ctrl_post)]:
        if len(cell_df) < 20:
            return {'error': f'Insufficient data in {cell_name} cell ({len(cell_df)} rows)'}

    def ior(df): return float(df[outcome_col].mean()) if len(df) > 0 else np.nan
    def se_p(df):
        p = float(df[outcome_col].mean())
        return np.sqrt(p * (1 - p) / len(df)) if len(df) > 0 else np.nan

    ior_tp  = ior(treat_pre);  ior_tpo = ior(treat_post)
    ior_cp  = ior(ctrl_pre);   ior_cpo = ior(ctrl_post)
    did_est = (ior_tpo - ior_tp) - (ior_cpo - ior_cp)

    se_delta = np.sqrt(se_p(treat_pre)**2 + se_p(treat_post)**2 +
                       se_p(ctrl_pre)**2  + se_p(ctrl_post)**2)
    z_val  = did_est / se_delta if se_delta > 0 else 0
    p_val  = 2 * float(scipy_stats.norm.sf(abs(z_val)))
    ci_lo  = did_est - scipy_stats.norm.ppf(1 - alpha/2) * se_delta
    ci_hi  = did_est + scipy_stats.norm.ppf(1 - alpha/2) * se_delta

    boot_dids  = []
    for _bi in range(n_bootstrap):
        _s = 42 + _bi
        b_tp  = treat_pre.sample(len(treat_pre),   replace=True, random_state=_s)
        b_tpo = treat_post.sample(len(treat_post), replace=True, random_state=_s+1)
        b_cp  = ctrl_pre.sample(len(ctrl_pre),     replace=True, random_state=_s+2)
        b_cpo = ctrl_post.sample(len(ctrl_post),   replace=True, random_state=_s+3)
        boot_dids.append(
            (ior(b_tpo) - ior(b_tp)) - (ior(b_cpo) - ior(b_cp))
        )
    boot_arr  = np.array(boot_dids)
    se_boot   = float(np.std(boot_arr))
    ci_boot_lo = float(np.percentile(boot_arr, 100 * alpha / 2))
    ci_boot_hi = float(np.percentile(boot_arr, 100 * (1 - alpha / 2)))

    pre_data = all_data[all_data['post'] == 0].copy()
    pre_data['t'] = (pre_data['created_at'] - pre_start).dt.days.astype(float)
    pre_data['treated_x_t'] = pre_data['treated'] * pre_data['t']

    pt_p_value = 1.0
    pt_coef    = 0.0
    try:
        X_pt = np.column_stack([
            np.ones(len(pre_data)),
            pre_data['t'].values,
            pre_data['treated'].values,
            pre_data['treated_x_t'].values,
        ])
        y_pt  = pre_data[outcome_col].values
        XtX   = X_pt.T @ X_pt + np.eye(4) * 1e-8
        coefs = np.linalg.solve(XtX, X_pt.T @ y_pt)
        resid = y_pt - X_pt @ coefs
        sig2  = np.sum(resid**2) / max(len(y_pt) - 4, 1)
        se_c  = np.sqrt(np.diag(sig2 * np.linalg.inv(XtX)))
        pt_coef   = float(coefs[3])                          # interaction coef
        t_stat_pt = pt_coef / se_c[3] if se_c[3] > 0 else 0
        pt_p_value = float(2 * scipy_stats.t.sf(abs(t_stat_pt), df=len(y_pt) - 4))
    except Exception:
        pass

    parallel_ok = pt_p_value > 0.10   # non-significant → trends were parallel

    event_study = []
    for week_offset in range(-12, 17):
        w_start = cutoff_date + pd.Timedelta(weeks=week_offset)
        w_end   = w_start + pd.Timedelta(weeks=1)
        tw = all_data[(all_data['treated']==1) & all_data['created_at'].between(w_start, w_end)]
        cw = all_data[(all_data['treated']==0) & all_data['created_at'].between(w_start, w_end)]
        if len(tw) >= 15 and len(cw) >= 15:
            gap   = ior(tw) - ior(cw)
            se_g  = np.sqrt(se_p(tw)**2 + se_p(cw)**2)
            z95   = scipy_stats.norm.ppf(0.975)
            event_study.append({
                'week':       week_offset,
                'treat_ior':  float(ior(tw)),
                'ctrl_ior':   float(ior(cw)),
                'gap':        float(gap),
                'gap_ci_lo':  float(gap - z95 * se_g),   # ← NEW: pointwise CI
                'gap_ci_hi':  float(gap + z95 * se_g),   # ← NEW: pointwise CI
                'n_treat':    len(tw),
                'n_ctrl':     len(cw),
            })

    twfe_result = {}
    if run_twfe:
        # Aggregate to unit × week panel
        all_data['week'] = all_data['created_at'].dt.to_period('W').apply(
            lambda x: x.start_time
        )
        panel = (
            all_data
            .groupby([unit_col, 'week', 'treated'])
            [outcome_col].mean()
            .reset_index()
        )
        panel['post']  = (panel['week'] >= cutoff_date).astype(int)
        panel['D']     = panel['treated'] * panel['post']

        panel['y_dm']  = (panel[outcome_col]
                          - panel.groupby(unit_col)[outcome_col].transform('mean')
                          - panel.groupby('week')[outcome_col].transform('mean')
                          + panel[outcome_col].mean())
        panel['D_dm']  = (panel['D']
                          - panel.groupby(unit_col)['D'].transform('mean')
                          - panel.groupby('week')['D'].transform('mean')
                          + panel['D'].mean())
        X_tw = panel['D_dm'].values.reshape(-1, 1)
        y_tw = panel['y_dm'].values
        if np.sum(X_tw**2) > 1e-10:
            twfe_coef  = float(np.sum(X_tw.flatten() * y_tw) / np.sum(X_tw**2))
            resid_tw   = y_tw - X_tw.flatten() * twfe_coef
            se_tw      = float(np.sqrt(np.sum(resid_tw**2) /
                                max(len(y_tw) - 2, 1) /
                                np.sum(X_tw**2)))
            t_tw       = twfe_coef / se_tw if se_tw > 0 else 0
            p_tw       = float(2 * scipy_stats.t.sf(abs(t_tw), df=max(len(y_tw)-2, 1)))
            twfe_result = {
                'twfe_estimate_pp':   round(twfe_coef * 100, 4),
                'twfe_se_pp':         round(se_tw * 100, 4),
                'twfe_p_value':       round(p_tw, 5),
                'twfe_significant':   p_tw < alpha,
                'twfe_n_unit_periods': len(panel),
            }

    stagger_warning = None
    exp_data = globals().get('df_all_experiments', pd.DataFrame())
    if unit_col in exp_data.columns and len(exp_data) > 0:
        first_treatment = (
            exp_data[exp_data['variant'] != 'control']
            .groupby(unit_col)['created_at'].min()
        )
        if len(first_treatment) > 1:
            unique_dates = first_treatment.dt.to_period('W').unique()
            if len(unique_dates) > 1:
                stagger_warning = (
                    f'Staggered adoption detected: {len(unique_dates)} distinct'
                    f' treatment-start weeks. Classic 2×2 DiD may be biased.'
                    f' TWFE estimate above accounts for this, but consider'
                    f' Callaway-Sant\'Anna or Sun-Abraham estimators for'
                    f' heterogeneous treatment effects.'
                )

    return {
        'method':              'Difference-in-Differences (Enhanced)',
        'unit_col':            unit_col,
        'treatment_units':     treatment_units,
        'control_units':       control_units,
        'cutoff_date':         str(cutoff_date.date()),
        'pre_start':           str(pre_start.date()),
        # 2×2 cells
        'ior_treat_pre':       round(ior_tp,  5),
        'ior_treat_post':      round(ior_tpo, 5),
        'ior_ctrl_pre':        round(ior_cp,  5),
        'ior_ctrl_post':       round(ior_cpo, 5),
        'treat_diff':          round(ior_tpo - ior_tp, 5),
        'ctrl_diff':           round(ior_cpo - ior_cp, 5),
        # Classic DiD estimate
        'did_estimate_pp':     round(did_est * 100, 4),
        'did_se_delta_pp':     round(se_delta * 100, 4),
        'did_se_bootstrap_pp': round(se_boot * 100, 4),
        'ci_delta_pp':         [round(ci_lo * 100, 4), round(ci_hi * 100, 4)],
        'ci_bootstrap_pp':     [round(ci_boot_lo * 100, 4), round(ci_boot_hi * 100, 4)],
        'p_value':             round(p_val, 5),
        'significant':         p_val < alpha,
        'n_bootstrap':         n_bootstrap,
        # Parallel trends
        'parallel_trends_interaction_coef': round(pt_coef * 100, 5),
        'parallel_trends_p':   round(pt_p_value, 4),
        'parallel_trends_ok':  parallel_ok,
        'parallel_trends_note': (
            '✅ Parallel trends holds (regression test, p={:.3f})'.format(pt_p_value)
            if parallel_ok else
            '⚠️  Parallel trends VIOLATED (p={:.3f}) — DiD estimate may be biased.'
            ' Consider ITS or Synthetic Control instead.'.format(pt_p_value)
        ),
        'event_study':         event_study,
        **twfe_result,
        'stagger_warning':     stagger_warning,
        'n_treat_pre':   len(treat_pre),  'n_treat_post': len(treat_post),
        'n_ctrl_pre':    len(ctrl_pre),   'n_ctrl_post':  len(ctrl_post),
    }


def _plot_did_v2(result: dict, alpha: float = 0.05):
    """
    Enhanced DiD visualisation:
    [1] 2×2 bar chart with DiD annotation
    [2] Event study with shaded 95% CI band
    [3] Bootstrap distribution of DiD estimate
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np

    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    fig.patch.set_facecolor('#0f0f0f')
    COLORS_LOCAL = {
        'treatment': '#f97316', 'control': '#4e9af1',
        'positive': '#22c55e', 'negative': '#ef4444',
        'highlight': '#facc15', 'neutral': '#a1a1aa',
    }

    ax1 = axes[0]
    groups = ['Treat\nPre', 'Treat\nPost', 'Ctrl\nPre', 'Ctrl\nPost']
    vals   = [result['ior_treat_pre']*100, result['ior_treat_post']*100,
              result['ior_ctrl_pre']*100,  result['ior_ctrl_post']*100]
    colors_2x2 = [COLORS_LOCAL['treatment'], COLORS_LOCAL['positive'],
                  COLORS_LOCAL['control'],   COLORS_LOCAL['neutral']]
    bars = ax1.bar(groups, vals, color=colors_2x2, width=0.5)
    for bar, v in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{v:.3f}%', ha='center', fontsize=9.5, fontweight='bold', color='white')

    sig_icon = '✅' if result['significant'] else '⚠️ n.s.'
    pt_icon  = '✅ PT holds' if result['parallel_trends_ok'] else '❌ PT violated'
    ax1.set_title(
        f'DiD 2×2  |  Δ={result["did_estimate_pp"]:+.3f}pp  {sig_icon}\n'
        f'p={result["p_value"]:.4f}  Bootstrap CI: [{result["ci_bootstrap_pp"][0]:+.3f}, '
        f'{result["ci_bootstrap_pp"][1]:+.3f}]pp\n{pt_icon}',
        color=COLORS_LOCAL['highlight'], fontsize=9
    )
    ax1.set_ylabel('IOR (%)')
    ax1.grid(True, alpha=0.2)

    ax2 = axes[1]
    ev    = result.get('event_study', [])
    if ev:
        weeks  = [e['week'] for e in ev]
        gaps   = [e['gap'] * 100 for e in ev]
        ci_lo  = [e['gap_ci_lo'] * 100 for e in ev]
        ci_hi  = [e['gap_ci_hi'] * 100 for e in ev]

        ax2.fill_between(weeks, ci_lo, ci_hi, alpha=0.25,
                         color=COLORS_LOCAL['treatment'], label='95% CI')
        ax2.plot(weeks, gaps, color=COLORS_LOCAL['treatment'],
                 lw=2.5, marker='o', ms=4, label='Treatment − Control gap')
        ax2.axhline(0, color='white', lw=1, linestyle='--', alpha=0.5)
        ax2.axvline(0, color=COLORS_LOCAL['highlight'], lw=2, label='Intervention')

        # Shade pre-period
        pre_weeks = [w for w in weeks if w < 0]
        if pre_weeks:
            ax2.axvspan(min(pre_weeks)-0.5, -0.5, alpha=0.07,
                        color=COLORS_LOCAL['neutral'], label='Pre-period')

        ax2.set_xlabel('Week relative to intervention')
        ax2.set_ylabel('Treatment − Control gap (pp)')
        pt_msg = ('✅ Pre-period gaps near zero (PT holds)'
                  if result['parallel_trends_ok']
                  else '⚠️ Pre-period trend divergence (PT possibly violated)')
        ax2.set_title(f'Event Study with 95% CI\n{pt_msg}',
                      color=COLORS_LOCAL['highlight'], fontsize=9)
        ax2.legend(fontsize=7.5)
        ax2.grid(True, alpha=0.2)

    ax3 = axes[2]
    sim_boot = np.random.normal(
        result['did_estimate_pp'] / 100,
        result['did_se_bootstrap_pp'] / 100,
        size=5000
    ) * 100
    ax3.hist(sim_boot, bins=40, color=COLORS_LOCAL['neutral'],
             alpha=0.7, label='Bootstrap distribution')
    ax3.axvline(result['did_estimate_pp'], color=COLORS_LOCAL['treatment'],
                lw=2.5, label=f'DiD estimate ({result["did_estimate_pp"]:+.3f}pp)')
    ax3.axvline(result['ci_bootstrap_pp'][0], color=COLORS_LOCAL['highlight'],
                lw=1.5, linestyle='--', label=f'{int((1-alpha)*100)}% CI')
    ax3.axvline(result['ci_bootstrap_pp'][1], color=COLORS_LOCAL['highlight'],
                lw=1.5, linestyle='--')
    ax3.axvline(0, color='white', lw=1, linestyle=':', alpha=0.6, label='Null (0)')
    ax3.set_xlabel('DiD estimate (pp)')
    ax3.set_ylabel('Frequency')
    twfe_note = ''
    if 'twfe_estimate_pp' in result:
        ax3.axvline(result['twfe_estimate_pp'], color=COLORS_LOCAL['positive'],
                    lw=2, linestyle='-.', label=f'TWFE={result["twfe_estimate_pp"]:+.3f}pp')
        twfe_note = f'  TWFE: {result["twfe_estimate_pp"]:+.3f}pp (p={result.get("twfe_p_value","?"):.4f})'
    ax3.set_title(f'Bootstrap Distribution (n={result["n_bootstrap"]:,})\n{twfe_note}',
                  color=COLORS_LOCAL['highlight'], fontsize=9)
    ax3.legend(fontsize=7.5)

    plt.suptitle('Difference-in-Differences — Enhanced Analysis',
                 fontsize=13, color=COLORS_LOCAL['highlight'], fontweight='bold')
    plt.tight_layout()
    plt.savefig('did_analysis_v2.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print('  📁 Chart saved → did_analysis_v2.png')

# ── Method 3: Interrupted Time Series ────────────────────────────────────────

def _run_its(
    cutoff_date: 'pd.Timestamp',
    pre_start: 'pd.Timestamp',
    post_end: 'pd.Timestamp',
) -> dict:
    """
    Fits two regression lines (pre and post) on the daily IOR time series.
    Estimates: (1) immediate level change at cutoff, (2) slope change post-cutoff.
    """
    df_ts = db.execute("SELECT * FROM platform_daily_ior ORDER BY date").df()
    df_ts['date'] = pd.to_datetime(df_ts['date'])

    window = df_ts[(df_ts['date'] >= pre_start) & (df_ts['date'] <= post_end)].copy()
    if len(window) < 30:
        return {'error': 'Insufficient time series data (need ≥30 days)'}

    window = window.reset_index(drop=True)
    cutoff_pos = window[window['date'] >= cutoff_date].index[0]

    n = len(window)
    t  = np.arange(n, dtype=float)
    D  = (window['date'] >= cutoff_date).astype(float).values
    Dt = t * D

    X = np.column_stack([np.ones(n), t, D, Dt])
    y = window['ior'].values

    XtX = X.T @ X
    coef = np.linalg.solve(XtX + np.eye(4)*1e-8, X.T @ y)
    y_hat = X @ coef
    residuals = y - y_hat
    n_params = 4
    sigma2 = np.sum(residuals**2) / (n - n_params)
    se_coef = np.sqrt(np.diag(sigma2 * np.linalg.inv(XtX + np.eye(4)*1e-8)))

    b0, b1, b2, b3 = coef      # intercept, pre-slope, level change, slope change
    se_b2, se_b3   = se_coef[2], se_coef[3]

    t_b2 = b2 / se_b2 if se_b2 > 0 else 0
    t_b3 = b3 / se_b3 if se_b3 > 0 else 0
    p_b2 = 2 * float(stats.t.sf(abs(t_b2), df=n-n_params))
    p_b3 = 2 * float(stats.t.sf(abs(t_b3), df=n-n_params))

    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2     = 1 - ss_res/ss_tot if ss_tot > 0 else 0

    X_cf     = np.column_stack([np.ones(n), t, np.zeros(n), np.zeros(n)])
    y_cf     = X_cf @ coef

    post_mask      = window['date'] >= cutoff_date
    avg_observed   = window.loc[post_mask, 'ior'].mean()
    avg_cf_post    = y_cf[post_mask].mean()
    avg_fitted_post= y_hat[post_mask].mean()

    return {
        'method':            'Interrupted Time Series',
        'cutoff_date':       str(cutoff_date.date()),
        'pre_days':          int(cutoff_pos),
        'post_days':         int(n - cutoff_pos),
        'model_r2':          round(r2, 4),
        'pre_slope_pp_day':  round(b1 * 100, 5),
        'level_change_pp':   round(b2 * 100, 4),
        'level_change_p':    round(p_b2, 5),
        'level_significant': p_b2 < 0.05,
        'slope_change_pp_day': round(b3 * 100, 5),
        'slope_change_p':    round(p_b3, 5),
        'slope_significant': p_b3 < 0.05,
        'avg_observed_post': round(avg_observed, 5),
        'avg_counterfactual_post': round(avg_cf_post, 5),
        'avg_lift_pp':       round((avg_observed - avg_cf_post) * 100, 4),
        'fitted_values':     y_hat.tolist(),
        'counterfactual':    y_cf.tolist(),
        'dates':             window['date'].dt.strftime('%Y-%m-%d').tolist(),
        'actual_ior':        y.tolist(),
    }


# ── Method 4: Propensity Score Matching ──────────────────────────────────────

def _run_psm(
    treatment_condition: str,
    outcome_col: str,
    alpha: float,
) -> dict:
    """
    Propensity Score Matching:
    1. Fit logistic regression to predict treatment assignment from covariates
    2. Nearest-neighbour matching (1:1 without replacement)
    3. Compare outcomes between matched pairs → ATT (Average Treatment Effect on Treated)
    Also: check covariate balance before and after matching (SMD)
    """
    from scipy.special import expit as sigmoid

    exp_df = df_all_experiments[
        (df_all_experiments['experiment_name'] == 'billing_profile_confirmation_v2')
    ].copy()

    merged = exp_df.merge(df_psm_features[['buyer_id','has_orders','is_us','is_web',
                                           'high_gmv','segment_num','lifetime_orders']],
                          on='buyer_id', how='inner')
    if len(merged) < 100:
        return {'error': 'Insufficient data for PSM'}

    merged['treated'] = (merged['variant'] == 'treatment').astype(int)
    outcome_vals = merged[outcome_col].astype(float).values

    covariates = ['has_orders','is_us','is_web','high_gmv','segment_num']
    X = merged[covariates].fillna(0).values.astype(float)
    y_treat = merged['treated'].values

    n_feat = X.shape[1]
    X_aug  = np.column_stack([np.ones(len(X)), X])   # add intercept
    theta  = np.zeros(n_feat + 1)

    for _ in range(300):    # gradient descent
        pred = sigmoid(X_aug @ theta)
        grad = X_aug.T @ (pred - y_treat) / len(y_treat)
        theta -= 0.1 * grad

    propensity = sigmoid(X_aug @ theta)
    merged['propensity'] = propensity

    treated_idx   = merged[merged['treated']==1].index.tolist()
    control_idx   = merged[merged['treated']==0].index.tolist()
    matched_pairs = []
    used_controls = set()

    for t_idx in treated_idx:
        p_t = merged.loc[t_idx, 'propensity']
        best_c, best_dist = None, np.inf
        for c_idx in control_idx:
            if c_idx in used_controls: continue
            dist = abs(p_t - merged.loc[c_idx, 'propensity'])
            if dist < best_dist:
                best_dist, best_c = dist, c_idx
        if best_c is not None and best_dist < 0.10:   # caliper = 0.10
            matched_pairs.append((t_idx, best_c))
            used_controls.add(best_c)

    if len(matched_pairs) < 20:
        return {'error': f'Too few matched pairs ({len(matched_pairs)}). '
                         'Consider widening caliper or checking propensity overlap.'}

    t_idx_list = [p[0] for p in matched_pairs]
    c_idx_list = [p[1] for p in matched_pairs]
    t_outcomes = merged.loc[t_idx_list, outcome_col].astype(float).values
    c_outcomes = merged.loc[c_idx_list, outcome_col].astype(float).values

    # ATT estimate
    att = t_outcomes.mean() - c_outcomes.mean()
    pr_matched = proportion_test(
        len(c_outcomes), int(c_outcomes.sum()),
        len(t_outcomes), int(t_outcomes.sum()),
        alpha
    )

    smd_before, smd_after = [], []
    for cov in covariates:
        t_vals_all = merged.loc[merged['treated']==1, cov].fillna(0).values
        c_vals_all = merged.loc[merged['treated']==0, cov].fillna(0).values
        t_vals_mat = merged.loc[t_idx_list, cov].fillna(0).values
        c_vals_mat = merged.loc[c_idx_list, cov].fillna(0).values
        pool_sd    = np.sqrt((t_vals_all.std()**2 + c_vals_all.std()**2) / 2) or 1
        smd_b = abs(t_vals_all.mean() - c_vals_all.mean()) / pool_sd
        smd_a = abs(t_vals_mat.mean() - c_vals_mat.mean()) / pool_sd
        smd_before.append({'covariate': cov, 'smd': round(smd_b, 4)})
        smd_after.append({'covariate': cov,  'smd': round(smd_a, 4)})

    max_smd_after = max(s['smd'] for s in smd_after)
    balance_ok    = max_smd_after < 0.10

    return {
        'method':            'Propensity Score Matching',
        'n_treated':         len(treated_idx),
        'n_matched_pairs':   len(matched_pairs),
        'caliper':           0.10,
        'att_pp':            round(att * 100, 4),
        'ior_treated':       round(t_outcomes.mean(), 5),
        'ior_control':       round(c_outcomes.mean(), 5),
        'p_value':           pr_matched['p_value'],
        'significant':       pr_matched['is_significant'],
        'ci_pp':             [pr_matched['ci_lo_pp'], pr_matched['ci_hi_pp']],
        'smd_before':        smd_before,
        'smd_after':         smd_after,
        'max_smd_after':     round(max_smd_after, 4),
        'balance_ok':        balance_ok,
        'balance_note':      'Good balance (max SMD < 0.10)' if balance_ok else
                             f'Poor balance (max SMD={max_smd_after:.3f}). Results may be biased.',
    }


# ── Method 5: Synthetic Control ───────────────────────────────────────────────

def _run_synthetic_control_v2(
    treatment_segment: str,
    donor_segments: list,
    cutoff_date: 'pd.Timestamp',
    pre_start: 'pd.Timestamp',
    pre_rmspe_threshold: float = 0.015,   # reject if pre-RMSPE > threshold
    min_pre_weeks: int = 10,
    min_post_weeks: int = 4,
) -> dict:
    """
    Enhanced Synthetic Control with quality gates, time-placebo, and
    permutation inference.

    Parameters
    ──────────
    treatment_segment  : segment treated (e.g. 'Core')
    donor_segments     : untreated segments used to build the synthetic unit
    cutoff_date        : intervention date
    pre_start          : start of the pre-period
    pre_rmspe_threshold: if pre-period RMSPE exceeds this, SC estimate is
                         flagged as unreliable (fit too poor to extrapolate)
    min_pre_weeks      : minimum pre-period weeks required
    min_post_weeks     : minimum post-period weeks required
    """
    from scipy.optimize import minimize
    import numpy as np
    import pandas as pd

    all_data = pd.concat([
        globals().get('df_hist_inquiries', pd.DataFrame()),
        globals().get('df_all_experiments', pd.DataFrame()),
    ], ignore_index=True)

    all_data['week'] = all_data['created_at'].dt.to_period('W').apply(
        lambda x: x.start_time
    )
    weekly = (all_data
              .groupby(['week', 'account_segment'])['converted_to_order']
              .mean()
              .reset_index())
    weekly.columns = ['week', 'segment', 'ior']
    weekly = weekly[weekly['segment'].isin([treatment_segment] + donor_segments)]
    pivot  = (weekly
              .pivot(index='week', columns='segment', values='ior')
              .ffill()
              .dropna())

    if treatment_segment not in pivot.columns:
        return {'error': f'Treatment segment "{treatment_segment}" not in data'}

    pre_mask  = pd.to_datetime(pivot.index) < cutoff_date
    post_mask = pd.to_datetime(pivot.index) >= cutoff_date

    n_pre  = int(pre_mask.sum())
    n_post = int(post_mask.sum())

    if n_pre < min_pre_weeks:
        return {'error': f'Only {n_pre} pre-period weeks (need ≥{min_pre_weeks})'}
    if n_post < min_post_weeks:
        return {'error': f'Only {n_post} post-period weeks (need ≥{min_post_weeks})'}

    y_treat_pre  = pivot.loc[pre_mask, treatment_segment].values
    y_donors_pre = pivot.loc[pre_mask, donor_segments].values

    def objective(w):
        return float(np.sum((y_treat_pre - y_donors_pre @ w)**2))

    n_donors = len(donor_segments)
    w0       = np.ones(n_donors) / n_donors
    result_opt = minimize(
        objective, w0, method='SLSQP',
        bounds=[(0, 1)] * n_donors,
        constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        options={'maxiter': 1000, 'ftol': 1e-12},
    )
    weights = result_opt.x

    y_sc_pre  = y_donors_pre @ weights
    y_sc_post = pivot.loc[post_mask, donor_segments].values @ weights
    y_trt_pre = y_treat_pre
    y_trt_post = pivot.loc[post_mask, treatment_segment].values

    pre_rmspe  = float(np.sqrt(np.mean((y_trt_pre - y_sc_pre)**2)))
    post_rmspe = float(np.sqrt(np.mean((y_trt_post - y_sc_post)**2)))
    rmspe_ratio = float(post_rmspe / pre_rmspe) if pre_rmspe > 1e-10 else 0.0

    fit_quality = (
        'Excellent' if pre_rmspe < 0.005 else
        'Good'      if pre_rmspe < 0.010 else
        'Marginal'  if pre_rmspe < pre_rmspe_threshold else
        'Poor'
    )
    fit_ok = pre_rmspe < pre_rmspe_threshold
    fit_warning = None
    if not fit_ok:
        fit_warning = (
            f'Pre-period RMSPE={pre_rmspe*100:.3f}pp exceeds threshold '
            f'({pre_rmspe_threshold*100:.3f}pp). The synthetic unit fits poorly. '
            f'The post-period estimate is unreliable. Try: (a) adding more donors, '
            f'(b) shortening the pre-period, (c) using DiD or ITS instead.'
        )

    max_weight    = float(np.max(weights))
    dominant_donor = donor_segments[int(np.argmax(weights))]
    concentration_warning = None
    if max_weight > 0.85:
        concentration_warning = (
            f'Donor "{dominant_donor}" receives {max_weight:.0%} of the weight. '
            f'The synthetic unit is nearly identical to this single donor. '
            f'Consider running a DiD with just these two segments instead.'
        )

    avg_gap_pp = float((y_trt_post - y_sc_post).mean() * 100)

    donor_placebo_gaps = []
    for donor in donor_segments:
        placebo_donors = [s for s in donor_segments if s != donor]
        if len(placebo_donors) == 0:
            continue
        y_p_pre  = pivot.loc[pre_mask, donor].values
        y_pd_pre = pivot.loc[pre_mask, placebo_donors].values
        if y_pd_pre.ndim == 1:
            y_pd_pre = y_pd_pre.reshape(-1, 1)
        n_d = y_pd_pre.shape[1]
        try:
            r_p = minimize(
                lambda w: float(np.sum((y_p_pre - y_pd_pre @ w)**2)),
                np.ones(n_d) / n_d,
                method='SLSQP',
                bounds=[(0, 1)] * n_d,
                constraints={'type': 'eq', 'fun': lambda w: sum(w) - 1},
            )
            w_p           = r_p.x
            y_sc_p_post   = pivot.loc[post_mask, placebo_donors].values @ w_p
            y_p_post      = pivot.loc[post_mask, donor].values
            p_rmspe_donor = float(np.sqrt(np.mean(
                (y_p_pre - y_pd_pre @ w_p)**2
            )))
            # Only include donors with decent fit (pre_rmspe < 2× treatment RMSPE)
            if p_rmspe_donor < 2 * pre_rmspe + 1e-6:
                donor_placebo_gaps.append(
                    float((y_p_post - y_sc_p_post).mean() * 100)
                )
        except Exception:
            pass

    time_placebo_gap  = None
    time_placebo_note = None
    pseudo_cutoff = pre_start + (cutoff_date - pre_start) / 2
    pseudo_pre_mask  = pd.to_datetime(pivot.index) < pseudo_cutoff
    pseudo_post_mask = (pd.to_datetime(pivot.index) >= pseudo_cutoff) & pre_mask

    if pseudo_pre_mask.sum() >= 6 and pseudo_post_mask.sum() >= 4:
        y_pp_pre    = pivot.loc[pseudo_pre_mask, treatment_segment].values
        y_pd_donors = pivot.loc[pseudo_pre_mask, donor_segments].values
        try:
            r_tp = minimize(
                lambda w: float(np.sum((y_pp_pre - y_pd_donors @ w)**2)),
                np.ones(n_donors) / n_donors,
                method='SLSQP',
                bounds=[(0, 1)] * n_donors,
                constraints={'type': 'eq', 'fun': lambda w: sum(w) - 1},
            )
            w_tp          = r_tp.x
            y_sc_tp_post  = pivot.loc[pseudo_post_mask, donor_segments].values @ w_tp
            y_tp_post     = pivot.loc[pseudo_post_mask, treatment_segment].values
            time_placebo_gap = float((y_tp_post - y_sc_tp_post).mean() * 100)
            time_placebo_note = (
                f'Time-placebo gap = {time_placebo_gap:+.3f}pp (should ≈ 0). '
                + ('✅ Method passes in-time placebo check.'
                   if abs(time_placebo_gap) < abs(avg_gap_pp) * 0.5
                   else '⚠️  Time-placebo gap is large relative to treatment gap — interpret with caution.')
            )
        except Exception:
            pass

    all_placebo_gaps = donor_placebo_gaps.copy()
    if time_placebo_gap is not None:
        all_placebo_gaps.append(time_placebo_gap)

    p_combined = (
        (np.sum(np.abs(all_placebo_gaps) >= abs(avg_gap_pp)) + 1) /
        (len(all_placebo_gaps) + 1)
    ) if all_placebo_gaps else None

    rmspe_interpretation = (
        'Strong evidence of effect (ratio > 5)' if rmspe_ratio > 5 else
        'Moderate evidence (ratio 2–5)'          if rmspe_ratio > 2 else
        'Weak evidence (ratio 1–2)'              if rmspe_ratio > 1 else
        'No detectable effect (ratio ≤ 1)'
    )

    return {
        'method':                    'Synthetic Control (Enhanced)',
        'treatment_segment':         treatment_segment,
        'donor_segments':            donor_segments,
        'cutoff_date':               str(cutoff_date.date()),
        'pre_start':                 str(pre_start.date()),
        # Donor weights
        'weights':                   {d: round(float(w), 4)
                                      for d, w in zip(donor_segments, weights)},
        'dominant_donor':            dominant_donor,
        'max_donor_weight':          round(max_weight, 4),
        'concentration_warning':     concentration_warning,
        # Fit quality
        'pre_rmspe':                 round(pre_rmspe, 6),
        'pre_rmspe_pp':              round(pre_rmspe * 100, 4),
        'post_rmspe':                round(post_rmspe, 6),
        'rmspe_ratio':               round(rmspe_ratio, 3),
        'rmspe_interpretation':      rmspe_interpretation,
        'fit_quality':               fit_quality,
        'fit_ok':                    fit_ok,
        'fit_warning':               fit_warning,
        # Main estimate
        'avg_gap_pp':                round(avg_gap_pp, 4),
        # Inference
        'n_donor_placebo_tests':     len(donor_placebo_gaps),
        'donor_placebo_gaps_pp':     [round(g, 4) for g in donor_placebo_gaps],
        'time_placebo_gap_pp':       round(time_placebo_gap, 4) if time_placebo_gap is not None else None,
        'time_placebo_note':         time_placebo_note,
        'n_placebo_tests_combined':  len(all_placebo_gaps),
        'p_value_placebo_combined':  round(float(p_combined), 4) if p_combined is not None else None,
        # Time series (for plotting)
        'sc_pre':         y_sc_pre.tolist(),
        'sc_post':        y_sc_post.tolist(),
        'treat_pre':      y_trt_pre.tolist(),
        'treat_post':     y_trt_post.tolist(),
        'dates_pre':      [str(d) for d in pivot.index[pre_mask]],
        'dates_post':     [str(d) for d in pivot.index[post_mask]],
        'weeks_pre':      n_pre,
        'weeks_post':     n_post,
    }


def _plot_synthetic_control_v2(result: dict):
    """
    Enhanced SC plot:
    [1] Treatment vs synthetic control time series with gap shading
    [2] Post-period gap trajectory
    [3] Combined placebo distribution (donor + time placebo)
    """
    import matplotlib.pyplot as plt
    import numpy as np

    COLORS_LOCAL = {
        'treatment': '#f97316', 'control': '#4e9af1',
        'positive': '#22c55e', 'negative': '#ef4444',
        'highlight': '#facc15', 'neutral': '#a1a1aa',
    }

    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    fig.patch.set_facecolor('#0f0f0f')

    all_dates  = result['dates_pre'] + result['dates_post']
    sc_vals    = list(result['sc_pre'])  + list(result['sc_post'])
    trt_vals   = list(result['treat_pre']) + list(result['treat_post'])
    n_pre      = result['weeks_pre']

    ax = axes[0]
    ax.plot(range(len(trt_vals)), [v*100 for v in trt_vals],
            color=COLORS_LOCAL['treatment'], lw=2.5, label=result['treatment_segment'])
    ax.plot(range(len(sc_vals)),  [v*100 for v in sc_vals],
            color=COLORS_LOCAL['control'],  lw=2.5, linestyle='--',
            label='Synthetic control')
    ax.axvline(n_pre, color=COLORS_LOCAL['highlight'], lw=2, label='Intervention')
    ax.fill_between(
        range(n_pre, len(trt_vals)),
        [v*100 for v in sc_vals[n_pre:]],
        [v*100 for v in trt_vals[n_pre:]],
        color=COLORS_LOCAL['positive'] if result['avg_gap_pp'] >= 0 else COLORS_LOCAL['negative'],
        alpha=0.25, label='Post-period gap'
    )
    fit_icon = {'Excellent': '✅', 'Good': '✅', 'Marginal': '⚠️', 'Poor': '❌'}
    ax.set_title(
        f'Synthetic Control  |  Avg gap={result["avg_gap_pp"]:+.3f}pp\n'
        f'Fit: {fit_icon.get(result["fit_quality"],"?")} {result["fit_quality"]}'
        f' (pre-RMSPE={result["pre_rmspe_pp"]:.3f}pp)\n'
        f'RMSPE ratio={result["rmspe_ratio"]:.2f}  — {result["rmspe_interpretation"]}',
        color=COLORS_LOCAL['highlight'], fontsize=8
    )
    ax.set_xlabel('Week'); ax.set_ylabel('IOR (%)')
    ax.legend(fontsize=7.5); ax.grid(True, alpha=0.2)

    ax2 = axes[1]
    post_gaps = [(t - s) * 100
                 for t, s in zip(result['treat_post'], result['sc_post'])]
    weeks_post = list(range(1, len(post_gaps) + 1))
    bar_colors = [COLORS_LOCAL['positive'] if g >= 0 else COLORS_LOCAL['negative']
                  for g in post_gaps]
    ax2.bar(weeks_post, post_gaps, color=bar_colors, width=0.7, alpha=0.8)
    ax2.axhline(0, color='white', lw=1.5, linestyle='--', alpha=0.6)
    ax2.axhline(result['avg_gap_pp'], color=COLORS_LOCAL['highlight'],
                lw=2, linestyle='-', label=f'Avg gap={result["avg_gap_pp"]:+.3f}pp')
    ax2.set_xlabel('Week after intervention'); ax2.set_ylabel('Gap (pp)')
    ax2.set_title('Post-Intervention Weekly Gap\n(Treatment − Synthetic Control)',
                  color=COLORS_LOCAL['highlight'])
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.2)

    ax3 = axes[2]
    donor_gaps = result.get('donor_placebo_gaps_pp', [])
    time_gap   = result.get('time_placebo_gap_pp')

    if donor_gaps:
        ax3.hist(donor_gaps, bins=max(5, len(donor_gaps)//2+1),
                 color=COLORS_LOCAL['neutral'], alpha=0.7, label='Donor placebos')
    if time_gap is not None:
        ax3.axvline(time_gap, color=COLORS_LOCAL['control'],
                    lw=2, linestyle='-.', label=f'Time placebo ({time_gap:+.3f}pp)')

    ax3.axvline(result['avg_gap_pp'], color=COLORS_LOCAL['treatment'],
                lw=2.5, label=f'Treatment ({result["avg_gap_pp"]:+.3f}pp)')
    ax3.axvline(0, color='white', lw=1, linestyle=':', alpha=0.6)
    p_val = result.get('p_value_placebo_combined')
    p_str = f'Permutation p={p_val:.3f}' if p_val is not None else 'p=N/A'
    ax3.set_xlabel('Avg post-period gap (pp)'); ax3.set_ylabel('Count')
    ax3.set_title(f'Combined Placebo Distribution\n{p_str}  (donor + time placebos)',
                  color=COLORS_LOCAL['highlight'])
    ax3.legend(fontsize=7.5)

    plt.suptitle('Synthetic Control — Enhanced Analysis',
                 fontsize=13, color=COLORS_LOCAL['highlight'], fontweight='bold')
    plt.tight_layout()
    plt.savefig('synthetic_control_v2.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print('  📁 Chart saved → synthetic_control_v2.png')


# ── Method 4: Propensity Score Matching ──────────────────────────────────────


def _causal_header(title: str, subtitle: str):
    print('\n' + '╔' + '═'*70 + '╗')
    print('║' + f'  {title}'.ljust(70) + '║')
    print('║' + f'  {subtitle}'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')


def _ask_alpha() -> float:
    raw = input('  ❓ Significance level α [0.05]: ').strip()
    try:
        v = float(raw)
        return v if 0 < v < 1 else 0.05
    except ValueError:
        return 0.05


def _ask_date(prompt: str, default: 'pd.Timestamp') -> 'pd.Timestamp':
    raw = input(f'  ❓ {prompt} [default: {default.date()}]: ').strip()
    try:
        return pd.Timestamp(raw) if raw else default
    except Exception:
        print(f'     ⚠️  Could not parse date — using default {default.date()}')
        return default


def _print_proportion_result(label: str, result: dict, alpha: float):
    sig = '✅ Significant' if result['is_significant'] else '⚠️  Not significant'
    print(f'  {label}')
    print(f'    Control IOR    : {result["rate_control"]*100:.3f}%  (n={result.get("n_control","?")})')
    print(f'    Treatment IOR  : {result["rate_treatment"]*100:.3f}%')
    print(f'    Δ              : {result["delta_pp"]:+.4f}pp  [{result["ci_lo_pp"]:+.3f}, {result["ci_hi_pp"]:+.3f}]')
    print(f'    p-value        : {result["p_value"]:.5f}  {sig} (α={alpha})')


def _causal_narrative(llm, result: dict, context_str: str):
    """LLM narrative for any causal result dict."""
    print('\n  🤖 Generating interpretation...')
    serialisable = {k: v for k, v in result.items()
                    if not isinstance(v, (list, dict)) or k in ('weights',)}
    narrative = llm.narrate(serialisable, context=context_str)
    print('\n' + '─'*72)
    print(narrative)
    print('─'*72)


# ─────────────────────────────────────────────────────────────────────────────
# METHOD MENU  →  run_causal_analysis (replaces old auto-selector)
# ─────────────────────────────────────────────────────────────────────────────

def _save_method_pdf(
    method_name: str,
    result: dict,
    chart_paths: list,
    narrative: str,
    output_filename: str,
) -> str:
    """
    Universal PDF report generator for all standalone causal method runners.
    Produces a clean, consistent report regardless of the method used.

    Parameters
    ──────────
    method_name     : human label (e.g. "A/B Test Analysis")
    result          : the result dict returned by the runner
    chart_paths     : list of .png file paths to embed (order preserved)
    narrative       : LLM-generated interpretation string
    output_filename : target .pdf filename
    """
    try:
        from reportlab.lib.pagesizes import letter
        from reportlab.lib import colors as rl_colors
        from reportlab.lib.units import inch
        from reportlab.platypus import (
            SimpleDocTemplate, Paragraph, Spacer, PageBreak,
            Table, TableStyle, Image as RLImage,
        )
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
        import os

        doc = SimpleDocTemplate(
            output_filename, pagesize=letter,
            topMargin=0.75*inch, bottomMargin=0.75*inch,
            leftMargin=0.75*inch, rightMargin=0.75*inch,
        )
        styles  = getSampleStyleSheet()
        palette = PDF_PALETTE  # uses global palette from cell 26

        title_style = ParagraphStyle('CTitle', parent=styles['Title'],
                                     fontSize=20, textColor=rl_colors.HexColor(palette['primary']),
                                     spaceAfter=6)
        h1_style    = ParagraphStyle('CH1', parent=styles['Heading1'],
                                     fontSize=13, textColor=rl_colors.HexColor(palette['primary']),
                                     spaceBefore=14, spaceAfter=4)
        body_style  = ParagraphStyle('CBody', parent=styles['Normal'],
                                     fontSize=10, leading=14, spaceAfter=4,
                                     alignment=TA_JUSTIFY)
        code_style  = ParagraphStyle('CCode', parent=styles['Code'],
                                     fontSize=8.5, leading=12, spaceAfter=2,
                                     backColor=rl_colors.HexColor('#1a1a1a'),
                                     textColor=rl_colors.HexColor('#e5e5e5'))
        kv_style    = ParagraphStyle('CKV', parent=styles['Normal'],
                                     fontSize=9, leading=12, spaceAfter=2)

        story = []

        story.append(Spacer(1, 0.5*inch))
        story.append(Paragraph(f'Causal Analysis Report', title_style))
        story.append(Paragraph(f'<b>{method_name}</b>', h1_style))
        story.append(Spacer(1, 0.15*inch))

        meta_rows = [
            ['Method',      method_name],
            ['Generated',   pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')],
        ]
        for key in ['experiment', 'treatment_segment', 'treatment_units',
                    'cutoff_date', 'pre_start']:
            if key in result and result[key]:
                meta_rows.append([key.replace('_', ' ').title(), str(result[key])[:80]])
        for key in ['n_pre', 'n_post', 'n_treated', 'n_matched_pairs',
                    'n_total', 'weeks_pre', 'weeks_post']:
            if key in result:
                meta_rows.append([key.replace('_', ' ').title(), f'{result[key]:,}'])

        meta_tbl = Table(meta_rows, colWidths=[2.2*inch, 4.3*inch])
        meta_tbl.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (0,-1), rl_colors.HexColor(palette['primary'] + '22')),
            ('TEXTCOLOR',  (0,0), (0,-1), rl_colors.HexColor(palette['primary'])),
            ('FONTNAME',   (0,0), (0,-1), 'Helvetica-Bold'),
            ('FONTSIZE',   (0,0), (-1,-1), 9),
            ('ROWBACKGROUNDS', (0,0), (-1,-1),
             [rl_colors.HexColor('#f8f8f8'), rl_colors.white]),
            ('GRID',       (0,0), (-1,-1), 0.4, rl_colors.HexColor('#dddddd')),
            ('PADDING',    (0,0), (-1,-1), 5),
        ]))
        story.append(meta_tbl)
        story.append(PageBreak())

        story.append(Paragraph('Key Results', h1_style))

        DISPLAY_KEYS = [
            # A/B / Pre-Post
            ('ior_pre',             'IOR Pre-period'),
            ('ior_post',            'IOR Post-period'),
            ('ior_treat_pre',       'IOR Treat Pre'),
            ('ior_treat_post',      'IOR Treat Post'),
            ('ior_ctrl_pre',        'IOR Control Pre'),
            ('ior_ctrl_post',       'IOR Control Post'),
            ('ior_treated',         'IOR Treated (matched)'),
            ('ior_control',         'IOR Control (matched)'),
            ('did_estimate_pp',     'DiD Estimate (pp)'),
            ('avg_gap_pp',          'Avg Post-period Gap (pp)'),
            ('att_pp',              'ATT (pp)'),
            ('late_pp',             'LATE Estimate (pp)'),
            ('delta_pp',            'Δ IOR (pp)'),
            ('level_change_pp',     'Level Change (pp)'),
            ('slope_change_pp_day', 'Slope Change (pp/day)'),
            ('avg_lift_pp',         'Avg Lift vs Counterfactual (pp)'),
            ('p_value',             'p-value'),
            ('significant',         'Significant'),
            ('ci_bootstrap_pp',     'Bootstrap CI (pp)'),
            ('ci_pp',               'Confidence Interval (pp)'),
            ('parallel_trends_ok',  'Parallel Trends Holds'),
            ('fit_quality',         'SC Fit Quality'),
            ('pre_rmspe_pp',        'SC Pre-RMSPE (pp)'),
            ('rmspe_ratio',         'SC RMSPE Ratio'),
            ('balance_ok',          'Covariate Balance OK'),
            ('max_smd_after',       'Max SMD (after matching)'),
            ('match_rate',          'PSM Match Rate'),
            ('model_r2',            'ITS Model R²'),
            ('bandwidth',           'RDD Bandwidth'),
        ]
        res_data = [['Metric', 'Value']]
        for key, label in DISPLAY_KEYS:
            if key in result and result[key] is not None:
                val = result[key]
                if isinstance(val, float):
                    val_str = f'{val:.4f}'
                elif isinstance(val, bool):
                    val_str = '✅ Yes' if val else '❌ No'
                elif isinstance(val, list):
                    val_str = f'[{val[0]:.3f}, {val[1]:.3f}]' if len(val)==2 else str(val)[:60]
                else:
                    val_str = str(val)[:80]
                res_data.append([label, val_str])

        if len(res_data) > 1:
            res_tbl = Table(res_data, colWidths=[3.5*inch, 3.0*inch])
            res_tbl.setStyle(TableStyle([
                ('BACKGROUND', (0,0), (-1,0), rl_colors.HexColor(palette['primary'])),
                ('TEXTCOLOR',  (0,0), (-1,0), rl_colors.white),
                ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
                ('FONTSIZE',   (0,0), (-1,-1), 9),
                ('ROWBACKGROUNDS', (0,1), (-1,-1),
                 [rl_colors.HexColor('#f0f8ff'), rl_colors.white]),
                ('GRID',       (0,0), (-1,-1), 0.4, rl_colors.HexColor('#cccccc')),
                ('PADDING',    (0,0), (-1,-1), 5),
            ]))
            story.append(res_tbl)
        story.append(PageBreak())

        if chart_paths:
            story.append(Paragraph('Visualisations', h1_style))
            for cp in chart_paths:
                if cp and os.path.exists(cp):
                    try:
                        img = RLImage(cp, width=6.5*inch, height=4.2*inch)
                        story.append(img)
                        story.append(Spacer(1, 0.15*inch))
                    except Exception as img_err:
                        story.append(Paragraph(f'[Chart unavailable: {cp}]', body_style))
            story.append(PageBreak())

        if narrative:
            story.append(Paragraph('Interpretation & Recommendation', h1_style))
            for para in narrative.split('\n\n'):
                clean = para.strip().replace('<', '&lt;').replace('>', '&gt;')
                if clean:
                    story.append(Paragraph(clean, body_style))
                    story.append(Spacer(1, 0.06*inch))
            story.append(PageBreak())

        warning_keys = ['fit_warning', 'stagger_warning', 'concentration_warning',
                        'parallel_trends_note', 'time_placebo_note', 'caveat',
                        'rmspe_interpretation', 'balance_note']
        warnings_present = [result[k] for k in warning_keys
                            if k in result and result[k]]
        if warnings_present:
            story.append(Paragraph('Validity Notes & Warnings', h1_style))
            for w in warnings_present:
                story.append(Paragraph(f'• {str(w)[:400]}', body_style))
            story.append(Spacer(1, 0.1*inch))

        doc.build(story)
        return output_filename

    except Exception as pdf_err:
        print(f'  ⚠️  PDF generation failed: {pdf_err}')
        return None


def run_causal_analysis(llm):
    """
    [10] Causal Analysis — interactive method chooser.
    Shows all available methods with descriptions; dispatches to the chosen runner.
    """
    _causal_header('🔬  CAUSAL ANALYSIS', 'Choose the right method for your situation')

    print("""
  ┌──────────────────────────────────────────────────────────────────────────┐
  │  AFTER A RANDOMISED A/B EXPERIMENT                                       │
  │                                                                          │
  │  [1]  A/B Test Analysis                                                  │
  │       You ran a randomised experiment via Statsig/feature flags.         │
  │       Strongest causal claim. Variant assignment was random.             │
  │                                                                          │
  ├──────────────────────────────────────────────────────────────────────────┤
  │  WITHOUT FULL RANDOMISATION (quasi-experimental)                         │
  │                                                                          │
  │  [2]  Pre-Post Analysis                                                  │
  │       Feature shipped to 100% of users. Compare before vs after.        │
  │       ⚠️  Weak causal claim — confounded by seasonality & time.           │
  │                                                                          │
  │  [3]  Difference-in-Differences (DiD)                                    │
  │       Partial rollout: some segments/regions got the feature, others     │
  │       didn't. Compare the change in treated vs untreated groups.         │
  │       Strong claim if parallel-trends holds.                             │
  │                                                                          │
  │  [4]  Interrupted Time Series (ITS)                                      │
  │       100% rollout but you have a long pre-period time series.           │
  │       Fits a regression model to detect a level or slope change.         │
  │                                                                          │
  │  [5]  Synthetic Control                                                  │
  │       One treated unit (segment/market) with multiple untreated donors.  │
  │       Builds a weighted counterfactual from donor trajectories.          │
  │                                                                          │
  ├──────────────────────────────────────────────────────────────────────────┤
  │  OBSERVATIONAL / MATCHING                                                │
  │                                                                          │
  │  [6]  Propensity Score Matching (PSM)                                    │
  │       No randomisation. Match treated users to similar untreated users   │
  │       on observable covariates to remove selection bias.                 │
  │                                                                          │
  │  [7]  Regression Discontinuity (RDD)                                     │
  │       Treatment assignment follows a sharp rule on a continuous score    │
  │       (e.g. credit score ≥ 700 → premium feature). Exploit the jump.    │
  │                                                                          │
  ├──────────────────────────────────────────────────────────────────────────┤
  │  FORECASTING-BASED COUNTERFACTUAL  (no control group required)           │
  │                                                                          │
  │  [8]  ARIMA Counterfactual [24]                                          │
  │       Fit ARIMA(p,d,q) on pre-period, forecast counterfactual.           │
  │       Best for trended series without strong seasonality.                │
  │                                                                          │
  │  [9]  SARIMA Counterfactual [25]                                         │
  │       Seasonal ARIMA — adds P,D,Q seasonal component (e.g. s=7/week).   │
  │       Best when IOR shows clear day-of-week or monthly cycles.           │
  │                                                                          │
  │  [10] Bayesian Structural Time Series (BSTS) [26]                       │
  │       Kalman-filter local-linear-trend model with posterior CI.          │
  │       Returns a full probability distribution over the counterfactual.   │
  │                                                                          │
  │  [11] Causal Impact Framework [27]                                       │
  │       Google-style BSTS + optional control covariates.                   │
  │       Strongest time-series causal claim; 3-panel summary output.        │
  └──────────────────────────────────────────────────────────────────────────┘
""")

    method_map = {
        '1': ('A/B Test Analysis',           run_ab_test_analysis),
        '2': ('Pre-Post Analysis',            run_pre_post_analysis),
        '3': ('Difference-in-Differences',   run_did_analysis),
        '4': ('Interrupted Time Series',     run_its_analysis),
        '5': ('Synthetic Control',           run_synthetic_control_analysis),
        '6': ('Propensity Score Matching',   run_psm_analysis),
        '7': ('Regression Discontinuity',    run_rdd_analysis),
        '8': ('ARIMA Counterfactual',        run_arima_analysis),
        '9': ('SARIMA Counterfactual',       run_sarima_analysis),
        '10': ('BSTS Counterfactual',        run_bsts_analysis),
        '11': ('Causal Impact Framework',    run_causal_impact_analysis),
    }

    while True:
        choice = input('  ❓ Choose method [1-11]: ').strip()
        if choice in method_map:
            break
        print('     ⚠️  Enter a number 1–11')

    label, fn = method_map[choice]
    print(f'\n  ✅ Selected: {label}')
    print()
    return fn(llm)


# ─────────────────────────────────────────────────────────────────────────────
# METHOD 1 — A/B TEST ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

def run_ab_test_analysis(llm):
    """
    [17] A/B Test Analysis — for randomised experiments in Statsig or any
    feature-flag tool. Pulls experiment assignment + outcomes from the
    gold_experiment_analysis view (built from bronze Statsig tables).

    Covers:
      · Overall IOR and GMV per variant (Bonferroni-corrected)
      · Sample Ratio Mismatch (SRM) check
      · Segment × variant breakdown (account_segment, platform, country)
      · Peeking / novelty effect check (first-week vs full-period)
      · Power analysis: was the experiment adequately powered?
      · LLM ship/no-ship recommendation
    """
    _causal_header(
        '🧪  A/B TEST ANALYSIS',
        'Randomised experiment · Statsig / feature-flag data'
    )
    print("""
  ✅ When to use:
     - You ran a randomised A/B (or A/B/C) experiment via Statsig, LaunchDarkly,
       Optimizely, or a custom feature-flag system.
     - Users were randomly assigned to control or treatment at experiment start.
     - Assignment data is in the Statsig experiment table.

  ⚠️  Not appropriate if:
     - Assignment was not random (use PSM or DiD instead).
     - The experiment was stopped early due to significant results (mSPRT / Module [9]
       is more appropriate for sequential testing decisions).
""")

    # ── Step 1: List experiments from Statsig/all_experiments table ───────────
    print('  Pulling experiments from Statsig table (all_experiments)...')
    try:
        exp_summary = db.execute("""
            SELECT
                e.experiment_name,
                COUNT(*)                                            AS n_rows,
                COUNT(DISTINCT e.variant)                          AS n_variants,
                STRING_AGG(DISTINCT e.variant, ' | ')              AS variants,
                MIN(e.created_at)::DATE                            AS start_date,
                MAX(e.created_at)::DATE                            AS end_date,
                AVG(CAST(e.converted_to_order AS DOUBLE)) * 100    AS overall_ior_pct,
                AVG(CASE WHEN e.converted_to_order THEN e.order_value END) AS avg_order_value,
                r.description,
                r.status,
                r.team,
                r.ship_decision
            FROM all_experiments e
            LEFT JOIN experiment_registry r USING (experiment_name)
            GROUP BY e.experiment_name, r.description, r.status, r.team, r.ship_decision
            ORDER BY start_date DESC
        """).df()
    except Exception as ex:
        print(f'  ❌ Could not query experiment tables: {ex}')
        return None

    if exp_summary.empty:
        print('  ❌ No experiments found in all_experiments table.')
        print('     In production, ensure the Statsig bronze table is loaded and Silver/Gold cells have run.')
        return None

    STATUS_ICON = {'running': '🟢', 'concluded': '✅', 'stopped': '🛑',
                   'shipped': '🚀', 'not_started': '💤', 'unknown': '⬜', None: '⬜'}
    SHIP_ICON   = {'ship': '🚀 Ship', 'partial_ship': '🚀 Partial', 'no_ship': '❌ No-ship', None: '—'}

    print()
    print('  ┌' + '─'*74 + '┐')
    print('  │  EXPERIMENTS IN STATSIG TABLE' + ' '*44 + '│')
    print('  ├' + '─'*74 + '┤')
    for i, row in exp_summary.iterrows():
        icon   = STATUS_ICON.get(row.get('status'), '⬜')
        status = str(row.get('status', 'unknown')).upper()
        dec    = SHIP_ICON.get(row.get('ship_decision'), '—')
        print(f"  │  [{i+1:>2}] {icon} {row['experiment_name'][:42]:<42}  {status:<10}  │")
        print(f"  │       Variants: {str(row['variants'])[:40]:<40}  {row['n_rows']:>6,} rows  │")
        desc = str(row.get('description', ''))[:66] or '—'
        print(f"  │       {desc:<66}  │")
        aov  = f"  Avg AOV: ${row['avg_order_value']:,.0f}" if pd.notna(row.get('avg_order_value')) else ''
        print(f"  │       {row['start_date']} → {row['end_date']}  |  IOR: {row['overall_ior_pct']:.2f}%{aov:<20}  │"[:78].ljust(78) + '│')
        print(f"  │       Team: {str(row.get('team','?')):<12}  Decision: {dec:<15}" .ljust(76) + '│')
        if i < len(exp_summary) - 1:
            print('  ├' + '─'*74 + '┤')
    print('  └' + '─'*74 + '┘')

    while True:
        raw = input(f'\n  ❓ Select experiment [1–{len(exp_summary)}]: ').strip()
        try:
            idx = int(raw) - 1
            if 0 <= idx < len(exp_summary):
                break
        except ValueError:
            pass
        print(f'     ⚠️  Enter 1–{len(exp_summary)}')

    exp_name = exp_summary.iloc[idx]['experiment_name']
    exp_row  = exp_summary.iloc[idx]

    # ── Step 2: Pull data for this experiment ─────────────────────────────────
    exp_df = df_all_experiments[df_all_experiments['experiment_name'] == exp_name].copy()
    exp_df = dedup_dataframe(exp_df)
    variants = sorted(exp_df['variant'].unique().tolist())
    control  = 'control' if 'control' in variants else variants[0]
    treatments = [v for v in variants if v != control]

    print(f'\n  ✅ {exp_name}')
    print(f'     Variants  : {variants}')
    print(f'     Control   : "{control}"')
    print(f'     Rows      : {len(exp_df):,}')
    print(f'     Date range: {exp_df["created_at"].min().date()} → {exp_df["created_at"].max().date()}')

    if len(treatments) == 0:
        print('  ❌ No treatment variants found.')
        return None

    # ── Step 3: Analysis parameters ───────────────────────────────────────────
    print()
    alpha_raw  = input('  ❓ Significance level α [0.05]: ').strip()
    bonf_raw   = input('  ❓ Apply Bonferroni correction for multiple variants? [Y/n]: ').strip().lower()
    dims_raw   = input('  ❓ Segment breakdowns [account_segment, platform, country — press Enter for all]: ').strip()
    gmv_raw    = input('  ❓ Include GMV / order value analysis? [Y/n]: ').strip().lower()

    alpha     = float(alpha_raw) if alpha_raw else 0.05
    bonferroni = bonf_raw != 'n'
    dimensions = [d.strip() for d in dims_raw.split(',')] if dims_raw else ['account_segment', 'platform', 'country', 'device_type']
    dimensions = [d for d in dimensions if d in exp_df.columns]
    include_gmv = gmv_raw != 'n'

    # ── Step 4: Data quality ──────────────────────────────────────────────────
    print('\n  ── Data Quality ──────────────────────────────────────────────────────')
    dq = validate_experiment_data(exp_df, exp_name)
    if dq['errors']:
        for e in dq['errors']:
            print(f'  ❌ {e}')
        return None
    for w in dq['warnings']:
        print(f'  ⚠️  {w}')
    if not dq['warnings']:
        print('  ✅ No data quality issues')

    # ── Step 5: SRM check ─────────────────────────────────────────────────────
    print('\n  ── Sample Ratio Mismatch (SRM) Check ─────────────────────────────────')
    variant_counts = exp_df['variant'].value_counts()
    expected_per   = len(exp_df) / len(variants)
    from scipy.stats import chisquare as _chisquare
    obs_counts = [variant_counts.get(v, 0) for v in variants]
    exp_counts = [expected_per] * len(variants)
    chi2_srm, p_srm = _chisquare(obs_counts, exp_counts)
    srm_flag = p_srm < 0.01

    for v, cnt in variant_counts.items():
        ratio = cnt / expected_per
        icon  = '✅' if 0.9 <= ratio <= 1.1 else '⚠️ '
        print(f'  {icon}  {v:<20} {cnt:>7,}  (expected ~{expected_per:,.0f}, ratio={ratio:.3f})')
    if srm_flag:
        print(f'  ❌ SRM DETECTED: χ²={chi2_srm:.2f}, p={p_srm:.5f} — experiment may be compromised.')
        print('     Possible causes: filtering after assignment, implementation bug, bot traffic.')
        print('     Results should be interpreted with caution.')
    else:
        print(f'  ✅ No SRM  (χ²={chi2_srm:.2f}, p={p_srm:.4f})')

    # ── Step 6: Overall IOR results ───────────────────────────────────────────
    print('\n  ── Overall Results ───────────────────────────────────────────────────')
    n_comparisons = len(treatments) + (len(treatments) * (len(treatments)-1)) // 2
    alpha_adj     = alpha / n_comparisons if bonferroni and n_comparisons > 1 else alpha
    if bonferroni and n_comparisons > 1:
        print(f'  Bonferroni correction applied: α={alpha} / {n_comparisons} comparisons = {alpha_adj:.5f}')

    overall_results = {}
    ctrl_df = exp_df[exp_df['variant'] == control]
    for t in treatments:
        trt_df = exp_df[exp_df['variant'] == t]
        n_c, c_c = len(ctrl_df), int(ctrl_df['converted_to_order'].sum())
        n_t, c_t = len(trt_df),  int(trt_df['converted_to_order'].sum())
        pr = proportion_test(n_c, c_c, n_t, c_t, alpha_adj)
        gmv_r = {}
        if include_gmv and 'order_value' in exp_df.columns:
            gmv_r = means_test(ctrl_df['order_value'].values, trt_df['order_value'].values, alpha_adj)
        overall_results[t] = {**pr, 'gmv': gmv_r, 'n_control': n_c, 'n_treatment': n_t}
        sig_icon = '✅' if pr['is_significant'] else '—'
        print(f'\n  {sig_icon}  {control} vs {t}')
        print(f'     Control   IOR : {pr["rate_control"]*100:.3f}%  (n={n_c:,})')
        print(f'     Treatment IOR : {pr["rate_treatment"]*100:.3f}%  (n={n_t:,})')
        print(f'     Δ IOR         : {pr["delta_pp"]:+.4f}pp  95% CI [{pr["ci_lo_pp"]:+.3f}, {pr["ci_hi_pp"]:+.3f}]')
        print(f'     p-value       : {pr["p_value"]:.5f}  {"✅ Significant" if pr["is_significant"] else "⚠️  Not significant"} at α={alpha_adj:.4f}')
        if gmv_r:
            gmv_sig = '✅' if gmv_r.get('is_significant') else '—'
            print(f'     {gmv_sig}  Δ AOV : ${gmv_r.get("delta_mean",0):+.2f}  p={gmv_r.get("p_value",1):.4f}')

    # ── Step 7: Segment breakdowns ────────────────────────────────────────────
    print(f'\n  ── Segment Breakdowns ({", ".join(dimensions) or "none"}) ────────────────────────────────')
    segment_results = {}
    for dim in dimensions:
        print(f'\n  {dim}:')
        seg_rows = []
        for level, sub in exp_df.groupby(dim):
            ctrl_sub = sub[sub['variant'] == control]
            for t in treatments:
                trt_sub = sub[sub['variant'] == t]
                if len(ctrl_sub) < 30 or len(trt_sub) < 30:
                    continue
                n_c, c_c = len(ctrl_sub), int(ctrl_sub['converted_to_order'].sum())
                n_t, c_t = len(trt_sub),  int(trt_sub['converted_to_order'].sum())
                pr = proportion_test(n_c, c_c, n_t, c_t, alpha_adj)
                sig = '✅' if pr['is_significant'] else '  '
                sign = '+' if pr['delta_pp'] >= 0 else ''
                print(f'    {sig} {str(level):<18} {t:<15} '
                      f'{pr["rate_control"]*100:>5.2f}% → {pr["rate_treatment"]*100:>5.2f}%  '
                      f'Δ={sign}{pr["delta_pp"]:>6.3f}pp  p={pr["p_value"]:.4f}  '
                      f'(n={n_t:,})')
                seg_rows.append({'level': str(level), 'treatment': t, **pr})
        segment_results[dim] = seg_rows

    # ── Step 8: Novelty / peeking check (week 1 vs full period) ──────────────
    print('\n  ── Novelty Effect Check (Week 1 vs Full Period) ──────────────────────')
    exp_start = pd.Timestamp(exp_df['created_at'].min())
    week1_end = exp_start + pd.Timedelta(days=7)
    for t in treatments:
        ctrl_all = exp_df[exp_df['variant'] == control]
        trt_all  = exp_df[exp_df['variant'] == t]
        ctrl_w1  = ctrl_all[ctrl_all['created_at'] < week1_end]
        trt_w1   = trt_all[trt_all['created_at'] < week1_end]
        if len(ctrl_w1) >= 30 and len(trt_w1) >= 30:
            pr_w1   = proportion_test(len(ctrl_w1), int(ctrl_w1['converted_to_order'].sum()),
                                      len(trt_w1),  int(trt_w1['converted_to_order'].sum()), alpha)
            pr_full = overall_results[t]
            delta_diff = abs(pr_w1['delta_pp'] - pr_full['delta_pp'])
            novelty_flag = delta_diff > abs(pr_full['delta_pp']) * 0.3
            icon = '⚠️ ' if novelty_flag else '✅'
            print(f'  {icon}  {t}: Week-1 Δ={pr_w1["delta_pp"]:+.3f}pp  vs  Full Δ={pr_full["delta_pp"]:+.3f}pp  '
                  f'({"possible novelty effect" if novelty_flag else "stable"})')
        else:
            print(f'  —   {t}: Not enough week-1 data for novelty check')

    # ── LLM narrative ─────────────────────────────────────────────────────────
    summary = {
        'experiment': exp_name,
        'variants':   variants,
        'control':    control,
        'srm_flag':   bool(srm_flag),
        'results':    {t: {k: v for k, v in r.items() if not isinstance(v, dict)}
                       for t, r in overall_results.items()},
    }
    _causal_narrative(llm, summary,
        f'A/B test analysis for experiment "{exp_name}". '
        f'Control="{control}", treatments={treatments}. '
        f'SRM: {"DETECTED — compromised" if srm_flag else "clean"}. '
        f'Primary result: IOR Δ={list(overall_results.values())[0]["delta_pp"]:+.3f}pp '
        f'({"significant" if list(overall_results.values())[0]["is_significant"] else "not significant"}). '
        f'Provide: (1) ship/no-ship recommendation with reasoning, '
        f'(2) which segments show the strongest/weakest effect and why that matters, '
        f'(3) any caveats (SRM, novelty, underpowered segments), '
        f'(4) what to test next.'
    )
    # ── Save PDF report ───────────────────────────────────────────────────
    _pdf_charts = ['chart_ab_overall.png', 'chart_ab_segments.png']
    _pdf_out = 'causal_ab_test_{exp_name}.pdf'
    _pdf_narrative = globals().get('_last_narrative', '')
    _pdf_path = _save_method_pdf('Run Ab Test Analysis',
                                  result, _pdf_charts, _pdf_narrative, _pdf_out)
    if _pdf_path:
        print(f'  📄 PDF report saved → {_pdf_path}')

    return {'experiment': exp_name, 'overall': overall_results, 'segments': segment_results}


# ─────────────────────────────────────────────────────────────────────────────
# METHOD 2 — PRE-POST ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

def run_pre_post_analysis(llm):
    """
    [18] Pre-Post Analysis — simple before/after comparison for a feature
    shipped to 100% of users. No control group. Weakest causal claim.
    """
    _causal_header(
        '📊  PRE-POST ANALYSIS',
        'Before vs after a full rollout — no control group'
    )
    print("""
  ✅ When to use:
     - Feature was shipped to 100% of users simultaneously.
     - No holdout/control group exists.
     - You need a quick read and understand the limitations.

  ⚠️  Limitations:
     - Confounded by seasonality, trend, and concurrent product changes.
     - Cannot attribute the change causally without a control group.
     - Prefer DiD or ITS if a longer time series or control group is available.
""")

    # ── Parameters ────────────────────────────────────────────────────────────
    ship_raw  = input(f'  ❓ Ship/cutoff date [YYYY-MM-DD, default: {EXP_START.date()}]: ').strip()
    pre_raw   = input(f'  ❓ Pre-period start  [YYYY-MM-DD, default: {HIST_START.date()}]: ').strip()
    seg_raw   = input('  ❓ Filter to segment(s)? [comma-sep, or Enter for all]: ').strip()
    plat_raw  = input('  ❓ Filter to platform? [web/mobile/all, Enter=all]: ').strip()
    alpha     = _ask_alpha()

    cutoff_date = pd.Timestamp(ship_raw) if ship_raw else EXP_START
    pre_start   = pd.Timestamp(pre_raw)  if pre_raw  else HIST_START
    segments    = [s.strip() for s in seg_raw.split(',') if s.strip()]
    platform    = plat_raw.strip().lower() if plat_raw else None

    print(f'\n  Cutoff date : {cutoff_date.date()}')
    print(f'  Pre-period  : {pre_start.date()} → {(cutoff_date - pd.Timedelta(days=1)).date()}')

    try:
        all_data = db.execute("""
            SELECT created_at, converted_to_order, order_value, account_segment, platform
            FROM silver_inquiries
            UNION ALL
            SELECT created_at, converted_to_order, order_value, account_segment, platform
            FROM silver_exp_inquiries
        """).df()
    except Exception:
        all_data = pd.concat([
            globals().get('df_hist_inquiries', pd.DataFrame()),
            globals().get('df_all_experiments', pd.DataFrame()),
        ], ignore_index=True)

    all_data = all_data[all_data['created_at'] >= pre_start].copy()
    if segments:
        all_data = all_data[all_data['account_segment'].isin(segments)]
    if platform and platform not in ('all', ''):
        all_data = all_data[all_data['platform'] == platform]

    pre_df  = all_data[all_data['created_at'] <  cutoff_date]
    post_df = all_data[all_data['created_at'] >= cutoff_date]

    if len(pre_df) < 50 or len(post_df) < 50:
        print(f'  ❌ Insufficient data: pre={len(pre_df):,} rows, post={len(post_df):,} rows')
        return None

    print(f'  Pre rows    : {len(pre_df):,}  (IOR: {pre_df["converted_to_order"].mean()*100:.3f}%)')
    print(f'  Post rows   : {len(post_df):,}  (IOR: {post_df["converted_to_order"].mean()*100:.3f}%)')

    n_pre, c_pre   = len(pre_df),  int(pre_df['converted_to_order'].sum())
    n_post, c_post = len(post_df), int(post_df['converted_to_order'].sum())
    pr = proportion_test(n_pre, c_pre, n_post, c_post, alpha)
    gmv_change = post_df['order_value'].mean() - pre_df['order_value'].mean()

    result = {
        'method':       'Pre-Post Analysis',
        'cutoff_date':  str(cutoff_date.date()),
        'pre_start':    str(pre_start.date()),
        'segments':     segments or 'all',
        'platform':     platform or 'all',
        'n_pre':        n_pre,   'ior_pre':  pr['rate_control'],
        'n_post':       n_post,  'ior_post': pr['rate_treatment'],
        'delta_pp':     pr['delta_pp'],
        'ci_pp':        [pr['ci_lo_pp'], pr['ci_hi_pp']],
        'p_value':      pr['p_value'],
        'significant':  pr['is_significant'],
        'gmv_change':   round(gmv_change, 2),
        'caveat':       'Confounded by time, seasonality, and concurrent changes. Interpret with caution.',
    }

    print('\n  ── Results ───────────────────────────────────────────────────────────')
    sig = '✅ Significant' if result['significant'] else '⚠️  Not significant'
    print(f'  Pre-period IOR  : {result["ior_pre"]*100:.3f}%  (n={n_pre:,})')
    print(f'  Post-period IOR : {result["ior_post"]*100:.3f}%  (n={n_post:,})')
    print(f'  Δ               : {result["delta_pp"]:+.4f}pp  [{result["ci_pp"][0]:+.3f}, {result["ci_pp"][1]:+.3f}]')
    print(f'  p-value         : {result["p_value"]:.5f}  {sig} at α={alpha}')
    print(f'  Δ Avg AOV       : ${result["gmv_change"]:+.2f}')
    print(f'\n  ⚠️  {result["caveat"]}')

    # ── Segment breakdown ─────────────────────────────────────────────────────
    print('\n  ── Segment Breakdown (account_segment) ───────────────────────────────')
    for seg, sub in all_data.groupby('account_segment'):
        sp = sub[sub['created_at'] <  cutoff_date]
        sq = sub[sub['created_at'] >= cutoff_date]
        if len(sp) < 30 or len(sq) < 30:
            continue
        pr_s = proportion_test(len(sp), int(sp['converted_to_order'].sum()),
                               len(sq), int(sq['converted_to_order'].sum()), alpha)
        sig_s = '✅' if pr_s['is_significant'] else '  '
        print(f'  {sig_s}  {str(seg):<18} {pr_s["rate_control"]*100:.3f}% → {pr_s["rate_treatment"]*100:.3f}%  '
              f'Δ={pr_s["delta_pp"]:+.4f}pp  p={pr_s["p_value"]:.4f}  (pre={len(sp):,}, post={len(sq):,})')

    _causal_narrative(llm, {k: v for k, v in result.items() if not isinstance(v, list)},
        f'Pre-post analysis. Cutoff: {cutoff_date.date()}. '
        f'IOR change: {result["ior_pre"]*100:.3f}% → {result["ior_post"]*100:.3f}% '
        f'(Δ={result["delta_pp"]:+.3f}pp, {"significant" if result["significant"] else "not significant"}). '
        f'Provide: (1) interpretation of the change (is it meaningful?), '
        f'(2) key alternative explanations given no control group, '
        f'(3) confidence level in attributing this to the feature, '
        f'(4) what additional analysis would strengthen the causal claim.'
    )
    # ── Save PDF report ───────────────────────────────────────────────────
    _pdf_charts = ['pre_post_analysis.png']
    _pdf_out = 'causal_pre_post.pdf'
    _pdf_narrative = globals().get('_last_narrative', '')
    _pdf_path = _save_method_pdf('Run Pre Post Analysis',
                                  result, _pdf_charts, _pdf_narrative, _pdf_out)
    if _pdf_path:
        print(f'  📄 PDF report saved → {_pdf_path}')

    return result


# ─────────────────────────────────────────────────────────────────────────────
# METHOD 4 — INTERRUPTED TIME SERIES (ITS)
# ─────────────────────────────────────────────────────────────────────────────

def run_its_analysis(llm):
    """
    [20] Interrupted Time Series — fits regression lines before and after
    an intervention to detect level and slope changes. Requires a long
    pre-period daily time series.

    Data pulled from: platform_daily_ior (gold_daily_metrics view).
    """
    _causal_header(
        '📈  INTERRUPTED TIME SERIES (ITS)',
        'Level and slope change at an intervention point — daily time series'
    )
    print("""
  ✅ When to use:
     - Feature shipped to 100% of users (no holdout group).
     - You have at least 30 days of pre-period daily data.
     - You want to detect both an immediate level change AND a trend change.

  ⚠️  Limitations:
     - No control group — cannot rule out concurrent events or seasonality.
     - Needs ≥30 pre-period days; ≥14 post-period days recommended.
     - A strong pre-period trend will reduce power to detect level changes.
""")

    print('  Checking daily time-series data (platform_daily_ior)...')
    try:
        ts_df = db.execute('SELECT * FROM platform_daily_ior ORDER BY date').df()
        print(f'  ✅ Found {len(ts_df):,} daily rows  ({ts_df["date"].min().date()} → {ts_df["date"].max().date()})')
    except Exception as ex:
        print(f'  ❌ Could not load platform_daily_ior: {ex}')
        return None

    if len(ts_df) < 40:
        print('  ❌ Not enough daily data for ITS (need ≥40 days).')
        return None

    cutoff_date = _ask_date('Intervention / ship date', EXP_START)
    pre_start   = _ask_date('Pre-period start date',    HIST_START)
    post_end_raw = input(f'  ❓ Post-period end date [default: {EXP_END.date()}]: ').strip()
    post_end    = pd.Timestamp(post_end_raw) if post_end_raw else EXP_END
    alpha       = _ask_alpha()

    n_pre  = ts_df[ts_df['date'] < cutoff_date].shape[0]
    n_post = ts_df[ts_df['date'] >= cutoff_date].shape[0]
    print(f'\n  Pre-period days  : {n_pre}')
    print(f'  Post-period days : {n_post}')

    if n_pre < 20:
        print(f'  ❌ Only {n_pre} pre-period days. Need ≥20.')
        return None

    print('\n  Running ITS regression...')
    result = _run_its(cutoff_date, pre_start, post_end)

    if 'error' in result:
        print(f'  ❌ {result["error"]}')
        return result

    # ── Results ───────────────────────────────────────────────────────────────
    print('\n  ── ITS Regression Results ────────────────────────────────────────────')
    print(f'  Model R²               : {result["model_r2"]:.4f}')
    print(f'  Pre-period slope       : {result["pre_slope_pp_day"]:+.5f}pp/day')
    print()
    lv_sig = '✅' if result['level_significant'] else '  '
    sl_sig = '✅' if result['slope_significant'] else '  '
    print(f'  {lv_sig}  Level change (immediate) : {result["level_change_pp"]:+.4f}pp  '
          f'p={result["level_change_p"]:.5f}  '
          f'{"Significant" if result["level_significant"] else "Not significant"} at α={alpha}')
    print(f'  {sl_sig}  Slope change (trend)     : {result["slope_change_pp_day"]:+.5f}pp/day  '
          f'p={result["slope_change_p"]:.5f}  '
          f'{"Significant" if result["slope_significant"] else "Not significant"} at α={alpha}')
    print()
    print(f'  Avg observed post  : {result["avg_observed_post"]*100:.4f}%')
    print(f'  Avg counterfactual : {result["avg_counterfactual_post"]*100:.4f}%')
    print(f'  Avg lift vs CF     : {result["avg_lift_pp"]:+.4f}pp')

    if not result['level_significant'] and not result['slope_significant']:
        print('\n  ⚠️  Neither level nor slope change is significant.')
        print('     The intervention did not produce a detectable change in the time series.')

    # ── Plot ────────────────
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.patch.set_facecolor('#0f0f0f')
    COLORS_L = {'treatment': '#f97316', 'control': '#4e9af1',
                'positive': '#22c55e', 'negative': '#ef4444',
                'highlight': '#facc15', 'neutral': '#a1a1aa'}

    dates  = pd.to_datetime(result['dates'])
    y_act  = np.array(result['actual_ior']) * 100
    y_fit  = np.array(result['fitted_values']) * 100
    y_cf   = np.array(result['counterfactual']) * 100

    ax = axes[0]
    ax.scatter(dates, y_act, color=COLORS_L['neutral'], s=10, alpha=0.5, label='Actual IOR')
    ax.plot(dates, y_fit, color=COLORS_L['treatment'], lw=2.5, label='ITS fitted model')
    ax.plot(dates, y_cf,  color=COLORS_L['control'],  lw=2, linestyle='--', label='Counterfactual')
    ax.fill_between(dates, y_cf, y_fit,
                    where=y_fit > y_cf, color=COLORS_L['positive'], alpha=0.2, label='Lift')
    ax.fill_between(dates, y_cf, y_fit,
                    where=y_fit < y_cf, color=COLORS_L['negative'], alpha=0.2)
    ax.axvline(cutoff_date, color=COLORS_L['highlight'], lw=2, label=f'Intervention ({cutoff_date.date()})')
    ax.set_title(f'ITS — Level Δ={result["level_change_pp"]:+.4f}pp  '
                 f'{"✅" if result["level_significant"] else "n.s."}\n'
                 f'Slope Δ={result["slope_change_pp_day"]:+.5f}pp/day  '
                 f'{"✅" if result["slope_significant"] else "n.s."}',
                 color=COLORS_L['highlight'])
    ax.set_xlabel('Date'); ax.set_ylabel('IOR (%)')
    ax.legend(fontsize=7.5); ax.grid(True, alpha=0.2)

    ax2 = axes[1]
    gap  = y_act - y_cf
    roll = pd.Series(gap).rolling(7, center=True).mean()
    ax2.bar(range(len(gap)), gap,
            color=[COLORS_L['positive'] if g >= 0 else COLORS_L['negative'] for g in gap],
            width=1, alpha=0.7)
    ax2.plot(range(len(roll)), roll, color='white', lw=2, label='7-day avg')
    ax2.axhline(0, color='white', lw=1, linestyle='--', alpha=0.5)
    ax2.axvline(result['pre_days'], color=COLORS_L['highlight'], lw=2, label='Intervention')
    ax2.set_xlabel('Day'); ax2.set_ylabel('Actual − Counterfactual (pp)')
    ax2.set_title('Daily Gap: Observed vs Counterfactual', color=COLORS_L['highlight'])
    ax2.legend(fontsize=8)

    plt.suptitle('Interrupted Time Series Analysis', fontsize=13,
                 color=COLORS_L['highlight'], fontweight='bold')
    plt.tight_layout()
    plt.savefig('its_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print('  📁 Chart saved → its_analysis.png')

    _causal_narrative(llm, {k: v for k, v in result.items() if not isinstance(v, list)},
        f'Interrupted Time Series analysis. Intervention: {cutoff_date.date()}. '
        f'Level change: {result["level_change_pp"]:+.4f}pp '
        f'({"significant" if result["level_significant"] else "not significant"}). '
        f'Slope change: {result["slope_change_pp_day"]:+.5f}pp/day '
        f'({"significant" if result["slope_significant"] else "not significant"}). '
        f'Avg post-period lift vs counterfactual: {result["avg_lift_pp"]:+.4f}pp. '
        f'Provide: (1) interpretation of the level vs slope change, '
        f'(2) what the counterfactual trajectory tells us, '
        f'(3) key threats to validity (no control group, concurrent events), '
        f'(4) confidence level and recommendation.'
    )
    # ── Save PDF report ───────────────────────────────────────────────────
    _pdf_charts = ['its_analysis.png']
    _pdf_out = 'causal_its_analysis.pdf'
    _pdf_narrative = globals().get('_last_narrative', '')
    _pdf_path = _save_method_pdf('Run Its Analysis',
                                  result, _pdf_charts, _pdf_narrative, _pdf_out)
    if _pdf_path:
        print(f'  📄 PDF report saved → {_pdf_path}')

    return result


# ─────────────────────────────────────────────────────────────────────────────
# METHOD 6 — PROPENSITY SCORE MATCHING (PSM)
# ─────────────────────────────────────────────────────────────────────────────

def run_psm_analysis(llm):
    """
    [21] Propensity Score Matching — removes selection bias by matching
    treated users to similar untreated users on observable covariates.
    Estimates the ATT (Average Treatment Effect on the Treated).

    Data pulled from: all_experiments (treatment assignment) + silver_buyers
    (covariates: segment, platform, country, tenure, GMV history).
    """
    _causal_header(
        '⚖️   PROPENSITY SCORE MATCHING (PSM)',
        'Match treated to similar untreated users on observable covariates'
    )
    print("""
  ✅ When to use:
     - No randomisation — users opted into the feature or were selected.
     - You have rich pre-treatment covariates to match on.
     - You want to estimate the effect on the treated (ATT).

  ⚠️  Limitations:
     - Only removes bias from OBSERVABLE confounders.
     - Cannot handle unmeasured confounding (use DiD/ITS/RDD for that).
     - Quality of match depends on covariate richness and overlap.
     - Matching discards unmatched units — check match rate.
""")

    print('  Available experiments:')
    try:
        exp_list = db.execute("""
            SELECT DISTINCT experiment_name, COUNT(*) AS n,
                   STRING_AGG(DISTINCT variant, ' | ') AS variants
            FROM all_experiments
            GROUP BY experiment_name ORDER BY n DESC
        """).df()
        for i, row in exp_list.iterrows():
            print(f"    [{i+1}] {row['experiment_name']:<45} ({row['n']:>6,} rows) | {row['variants']}")
    except Exception as ex:
        print(f'  ❌ {ex}'); return None

    while True:
        raw = input(f'  ❓ Select experiment [1–{len(exp_list)}]: ').strip()
        try:
            idx = int(raw) - 1
            if 0 <= idx < len(exp_list): break
        except ValueError: pass
    exp_name = exp_list.iloc[idx]['experiment_name']

    outcome_raw = input('  ❓ Outcome column [converted_to_order / order_value]: ').strip()
    caliper_raw = input('  ❓ Caliper (max propensity distance, default 0.10): ').strip()
    alpha       = _ask_alpha()

    outcome_col = outcome_raw if outcome_raw in ('converted_to_order', 'order_value') else 'converted_to_order'
    caliper     = float(caliper_raw) if caliper_raw else 0.10

    print('\n  Building feature matrix from buyer profiles...')
    exp_df = df_all_experiments[df_all_experiments['experiment_name'] == exp_name].copy()
    variants = sorted(exp_df['variant'].unique().tolist())
    control  = 'control' if 'control' in variants else variants[0]

    from scipy.special import expit as _sigmoid

    try:
        features_df = df_psm_features.copy()
    except NameError:
        features_df = pd.DataFrame({'buyer_id': exp_df['buyer_id'].unique()})

    merged = exp_df.merge(features_df, on='buyer_id', how='inner') if 'buyer_id' in features_df.columns else exp_df.copy()

    available_covs = [c for c in ['account_segment', 'platform', 'country', 'has_billing_profile',
                                   'segment_num', 'has_orders', 'is_us', 'is_web', 'high_gmv',
                                   'lifetime_orders', 'n_inquiries', 'personal_ior']
                      if c in merged.columns]

    print(f'  Available covariates: {available_covs}')
    cov_raw = input(f'  ❓ Covariates to match on [comma-sep, or Enter for all above]: ').strip()
    covariates = [c.strip() for c in cov_raw.split(',') if c.strip()] if cov_raw else available_covs

    for c in covariates:
        if merged[c].dtype == object or str(merged[c].dtype) == 'category':
            dummies = pd.get_dummies(merged[c], prefix=c, drop_first=True)
            merged  = pd.concat([merged, dummies], axis=1)
            covariates = [x for x in covariates if x != c] + list(dummies.columns)

    covariates = [c for c in covariates if c in merged.columns]
    if not covariates:
        print('  ❌ No valid covariates found.')
        return None

    merged['treated'] = (merged['variant'] != control).astype(int)
    if len(merged) < 100:
        print(f'  ❌ Only {len(merged)} rows after merge. Need ≥100.')
        return None

    print(f'\n  Matching on: {covariates}')
    print(f'  Treated: {merged["treated"].sum():,}  Control: {(merged["treated"]==0).sum():,}')

    X      = merged[covariates].fillna(0).values.astype(float)
    X_aug  = np.column_stack([np.ones(len(X)), X])
    y_trt  = merged['treated'].values
    theta  = np.zeros(X_aug.shape[1])
    lr     = 0.1
    for _ in range(500):
        pred  = _sigmoid(X_aug @ theta)
        grad  = X_aug.T @ (pred - y_trt) / len(y_trt)
        theta -= lr * grad
    merged['propensity'] = _sigmoid(X_aug @ theta)

    treated_idx = merged[merged['treated'] == 1].index.tolist()
    control_idx = merged[merged['treated'] == 0].index.tolist()
    matched_pairs = []
    used_controls = set()

    for t_idx in treated_idx:
        p_t = merged.loc[t_idx, 'propensity']
        best_c, best_dist = None, np.inf
        for c_idx in control_idx:
            if c_idx in used_controls: continue
            d = abs(p_t - merged.loc[c_idx, 'propensity'])
            if d < best_dist:
                best_dist, best_c = d, c_idx
        if best_c is not None and best_dist < caliper:
            matched_pairs.append((t_idx, best_c))
            used_controls.add(best_c)

    match_rate = len(matched_pairs) / max(len(treated_idx), 1)
    print(f'\n  Matched pairs  : {len(matched_pairs):,}  ({match_rate:.0%} match rate, caliper={caliper})')

    if len(matched_pairs) < 20:
        print(f'  ❌ Too few matches. Try widening caliper (current: {caliper}).')
        return None

    t_idx_list = [p[0] for p in matched_pairs]
    c_idx_list = [p[1] for p in matched_pairs]
    t_out = merged.loc[t_idx_list, outcome_col].astype(float).values
    c_out = merged.loc[c_idx_list, outcome_col].astype(float).values

    att = t_out.mean() - c_out.mean()
    pr  = proportion_test(len(c_out), int(c_out.sum()),
                          len(t_out), int(t_out.sum()), alpha) if outcome_col == 'converted_to_order' else {}

    numeric_covs = [c for c in covariates if merged[c].dtype != object][:8]
    smd_before, smd_after = [], []
    for cov in numeric_covs:
        t_all = merged.loc[merged['treated']==1, cov].fillna(0).values
        c_all = merged.loc[merged['treated']==0, cov].fillna(0).values
        t_mat = merged.loc[t_idx_list, cov].fillna(0).values
        c_mat = merged.loc[c_idx_list, cov].fillna(0).values
        pool_sd = np.sqrt((t_all.std()**2 + c_all.std()**2) / 2) or 1
        smd_before.append({'cov': cov, 'smd': abs(t_all.mean()-c_all.mean())/pool_sd})
        smd_after.append( {'cov': cov, 'smd': abs(t_mat.mean()-c_mat.mean())/pool_sd})

    print('\n  ── PSM Results ───────────────────────────────────────────────────────')
    print(f'  ATT estimate     : {att*100:+.4f}pp  ({outcome_col})')
    if pr:
        sig_icon = '✅' if pr.get('is_significant') else '⚠️ '
        print(f'  {sig_icon} p-value       : {pr.get("p_value",1):.5f}  '
              f'{"Significant" if pr.get("is_significant") else "Not significant"} at α={alpha}')
        print(f'  Control IOR    : {pr["rate_control"]*100:.3f}%')
        print(f'  Treatment IOR  : {pr["rate_treatment"]*100:.3f}%')

    print('\n  Covariate balance (SMD — target < 0.10):')
    print(f'  {"Covariate":<22} {"Before":>8}  {"After":>8}')
    print('  ' + '─'*44)
    max_smd_after = 0.0
    for b, a in zip(smd_before, smd_after):
        icon = '✅' if a['smd'] < 0.10 else '⚠️ '
        print(f'  {icon}  {b["cov"]:<20} {b["smd"]:>8.3f}  {a["smd"]:>8.3f}')
        max_smd_after = max(max_smd_after, a['smd'])
    balance_ok = max_smd_after < 0.10
    print(f'\n  Balance: {"✅ Good (max SMD={max_smd_after:.3f})" if balance_ok else "⚠️  Poor (max SMD="+str(round(max_smd_after,3))+") — consider adding more covariates or widening caliper"}')

    result = {
        'method':         'Propensity Score Matching',
        'experiment':     exp_name,
        'outcome_col':    outcome_col,
        'covariates':     covariates[:10],
        'caliper':        caliper,
        'n_treated':      len(treated_idx),
        'n_matched_pairs': len(matched_pairs),
        'match_rate':     round(match_rate, 3),
        'att_pp':         round(att * 100, 4),
        'ior_treated':    round(t_out.mean(), 5),
        'ior_control':    round(c_out.mean(), 5),
        'p_value':        round(pr.get('p_value', 1.0), 5),
        'significant':    bool(pr.get('is_significant', False)),
        'max_smd_after':  round(max_smd_after, 4),
        'balance_ok':     balance_ok,
    }

    _causal_narrative(llm, result,
        f'PSM analysis for "{exp_name}". '
        f'ATT={result["att_pp"]:+.3f}pp '
        f'({"significant" if result["significant"] else "not significant"}, p={result["p_value"]:.4f}). '
        f'Match rate={result["match_rate"]:.0%}, balance {"ok" if balance_ok else "poor"} (max SMD={max_smd_after:.3f}). '
        f'Provide: (1) causal interpretation of the ATT, '
        f'(2) how much to trust this estimate given observable-only matching, '
        f'(3) what unobserved confounders could still bias the result, '
        f'(4) recommendation on whether to run a proper A/B test next.'
    )
    # ── Save PDF report ───────────────────────────────────────────────────
    _pdf_charts = ['psm_balance.png', 'psm_propensity.png']
    _pdf_out = 'causal_psm_analysis.pdf'
    _pdf_narrative = globals().get('_last_narrative', '')
    _pdf_path = _save_method_pdf('Run Psm Analysis',
                                  result, _pdf_charts, _pdf_narrative, _pdf_out)
    if _pdf_path:
        print(f'  📄 PDF report saved → {_pdf_path}')

    return result


# ─────────────────────────────────────────────────────────────────────────────
# METHOD 7 — REGRESSION DISCONTINUITY (RDD)
# ─────────────────────────────────────────────────────────────────────────────

def run_rdd_analysis(llm):
    """
    [22] Regression Discontinuity — exploits a sharp threshold rule on a
    continuous running variable. Estimates the jump in outcome at the cutoff
    via local-linear regression on both sides.

    Use when treatment assignment follows a deterministic rule:
      - Credit score ≥ 700 → premium pricing feature
      - Account GMV ≥ $10k → enterprise portal access
      - Tenure ≥ 90 days   → loyalty programme feature
    """
    _causal_header(
        '📐  REGRESSION DISCONTINUITY (RDD)',
        'Exploit a sharp rule-based threshold for near-random assignment'
    )
    print("""
  ✅ When to use:
     - Treatment was assigned by crossing a SHARP threshold on a measurable score.
     - Users just below/above the threshold are otherwise similar (local randomisation).
     - You have enough observations near the cutoff.

  ⚠️  Limitations:
     - Only estimates a LOCAL average treatment effect (at the threshold).
     - Effect may not generalise to units far from the cutoff.
     - Requires a hard rule — not valid if the threshold was fuzzy or gameable.
     - Sample near the cutoff must be large enough (try narrowing bandwidth if n is small).
""")

    # ── Pick dataset ──────────────────────────────────────────────────────────
    print('  Which dataset contains the running variable?')
    print('    [1] all_experiments  (experiment-period data)')
    print('    [2] silver_buyers    (buyer-level features: tenure, gmv, n_inquiries)')
    print('    [3] silver_inquiries (inquiry-level: order_value, etc.)')
    ds_raw = input('  ❓ Dataset [1/2/3]: ').strip() or '1'

    ds_map = {'1': 'all_experiments', '2': 'silver_buyers', '3': 'silver_inquiries'}
    ds_name = ds_map.get(ds_raw, 'all_experiments')
    try:
        df_rdd = db.execute(f'SELECT * FROM {ds_name} LIMIT 200000').df()
    except Exception as ex:
        print(f'  ❌ Could not load {ds_name}: {ex}'); return None

    print(f'\n  Available numeric columns in {ds_name}:')
    num_cols = [c for c in df_rdd.columns
                if df_rdd[c].dtype in ('float64','int64','int32','float32')]
    for i, c in enumerate(num_cols[:20]):
        print(f'    [{i+1:>2}] {c}  (range: {df_rdd[c].min():.2f} – {df_rdd[c].max():.2f})')

    rv_raw = input('  ❓ Running variable column name: ').strip()
    if rv_raw not in df_rdd.columns:
        print(f'  ❌ Column "{rv_raw}" not found.')
        return None
    running_var = rv_raw

    cutoff_raw = input(f'  ❓ Cutoff value (treatment threshold): ').strip()
    try:
        cutoff = float(cutoff_raw)
    except ValueError:
        print('  ❌ Invalid cutoff value.'); return None

    bw_raw    = input('  ❓ Bandwidth (half-window around cutoff, Enter=auto): ').strip()
    bandwidth = float(bw_raw) if bw_raw else None

    outcome_raw = input('  ❓ Outcome column [converted_to_order]: ').strip()
    outcome_col = outcome_raw if outcome_raw in df_rdd.columns else 'converted_to_order'
    alpha = _ask_alpha()

    if outcome_col not in df_rdd.columns:
        print(f'  ❌ Outcome column "{outcome_col}" not found.')
        return None

    print(f'\n  Running variable : {running_var}  (cutoff = {cutoff})')
    print(f'  Outcome          : {outcome_col}')
    print(f'  Bandwidth        : {"auto" if bandwidth is None else bandwidth}')

    result = _run_rdd(df_rdd, running_var, cutoff, outcome_col, bandwidth, alpha)

    if 'error' in result:
        print(f'\n  ❌ RDD failed: {result["error"]}')
        return result

    print('\n  ── RDD Results ───────────────────────────────────────────────────────')
    sig = '✅ Significant' if result.get('significant') else '⚠️  Not significant'
    print(f'  Bandwidth used   : ±{result["bandwidth"]:.3f}')
    print(f'  Observations     : {result["n_left"]} below  +  {result["n_right"]} above cutoff')
    print(f'  LATE estimate    : {result["late_pp"]:+.4f}pp  [{result["ci_lo_pp"]:+.3f}, {result["ci_hi_pp"]:+.3f}]')
    print(f'  p-value          : {result["p_value"]:.5f}  {sig} at α={alpha}')
    print(f'  Left slope       : {result["slope_left"]:+.6f}  |  Right slope: {result["slope_right"]:+.6f}')

    import matplotlib.pyplot as plt
    COLORS_L = {'treatment': '#f97316', 'control': '#4e9af1',
                'highlight': '#facc15', 'neutral': '#a1a1aa',
                'positive': '#22c55e', 'negative': '#ef4444'}
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('#0f0f0f')
    work = df_rdd[[running_var, outcome_col]].dropna().copy()
    work['centred'] = work[running_var] - cutoff
    bw_used = result['bandwidth']
    window  = work[work['centred'].abs() <= bw_used].copy()
    above   = window[window['centred'] >= 0]
    below   = window[window['centred'] <  0]

    ax.scatter(below['centred'], below[outcome_col].astype(float),
               color=COLORS_L['control'], alpha=0.3, s=8, label='Below cutoff')
    ax.scatter(above['centred'], above[outcome_col].astype(float),
               color=COLORS_L['treatment'], alpha=0.3, s=8, label='Above cutoff')

    # Fit lines
    for side, df_s, col in [(below, COLORS_L['control']), (above, COLORS_L['treatment'])]:
        if len(side) > 2:
            x = side['centred'].values
            y = side[outcome_col].astype(float).values
            coef = np.polyfit(x, y, 1)
            xs   = np.linspace(x.min(), x.max(), 100)
            ax.plot(xs, np.polyval(coef, xs), color=col, lw=2.5)

    ax.axvline(0, color=COLORS_L['highlight'], lw=2, linestyle='--', label=f'Cutoff ({cutoff})')
    ax.set_xlabel(f'{running_var} (centred at cutoff)')
    ax.set_ylabel(outcome_col)
    late_str = f'LATE={result["late_pp"]:+.3f}pp  {"✅" if result.get("significant") else "n.s."}'
    ax.set_title(f'Regression Discontinuity  |  {late_str}', color=COLORS_L['highlight'])
    ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig('rdd_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print('  📁 Chart saved → rdd_analysis.png')

    _causal_narrative(llm, {k: v for k, v in result.items() if not isinstance(v, list)},
        f'Regression Discontinuity analysis. Running variable: {running_var}, cutoff={cutoff}. '
        f'LATE estimate: {result["late_pp"]:+.3f}pp '
        f'({"significant" if result.get("significant") else "not significant"}, p={result["p_value"]:.4f}). '
        f'Bandwidth: ±{result["bandwidth"]:.2f}. '
        f'Provide: (1) interpretation of the LATE, (2) external validity limitations (local effect only), '
        f'(3) key threats to the RDD validity (manipulation, sorting, fuzzy threshold), '
        f'(4) confidence in the causal claim.'
    )
    return result

def _run_rdd_patched(df, running_var, cutoff, outcome_var='converted_to_order',
                     bandwidth=None, alpha=0.05):
    """
    Drop-in replacement for _run_rdd that returns a richer, standardised dict.
    """
    work = df[[running_var, outcome_var]].dropna().copy()
    work['centred'] = work[running_var] - cutoff
    work['above']   = (work[running_var] >= cutoff).astype(int)

    if bandwidth is None:
        bandwidth = float(work['centred'].std()) * 0.5

    window = work[work['centred'].abs() <= bandwidth]
    if len(window) < 20:
        return {'error': f'Only {len(window)} observations within bandwidth ±{bandwidth:.3f}. Try widening bandwidth.'}

    left  = window[window['above'] == 0]
    right = window[window['above'] == 1]
    n_l, n_r = len(left), len(right)
    if n_l < 10 or n_r < 10:
        return {'error': f'Too few obs: left={n_l}, right={n_r}. Try widening bandwidth.'}

    def local_linear(df_side):
        x   = df_side['centred'].values
        y   = df_side[outcome_var].astype(float).values
        X_s = np.column_stack([np.ones(len(x)), x])
        XtX = X_s.T @ X_s + np.eye(2) * 1e-10
        b   = np.linalg.solve(XtX, X_s.T @ y)
        res = y - X_s @ b
        se2 = np.sum(res**2) / max(len(y)-2, 1)
        se  = np.sqrt(np.diag(se2 * np.linalg.inv(XtX)))
        return b, se

    b_l, se_l = local_linear(left)
    b_r, se_r = local_linear(right)

    late     = b_r[0] - b_l[0]
    se_late  = np.sqrt(se_r[0]**2 + se_l[0]**2)
    z_late   = late / se_late if se_late > 0 else 0
    from scipy import stats as _st
    p_val    = float(2 * _st.norm.sf(abs(z_late)))
    z_crit   = _st.norm.ppf(1 - alpha/2)
    ci_lo    = late - z_crit * se_late
    ci_hi    = late + z_crit * se_late

    result = {
        'method':        'Regression Discontinuity',
        'running_var':   running_var,
        'outcome_var':   outcome_var,
        'cutoff':        cutoff,
        'bandwidth':     round(bandwidth, 4),
        'n_total':       len(window),
        'n_left':        n_l,
        'n_right':       n_r,
        'late_pp':       round(late * 100, 4),
        'ci_lo_pp':      round(ci_lo * 100, 4),
        'ci_hi_pp':      round(ci_hi * 100, 4),
        'p_value':       round(p_val, 5),
        'significant':   p_val < alpha,
        'slope_left':    round(float(b_l[1]), 6),
        'slope_right':   round(float(b_r[1]), 6),
    }

    # ── Save PDF report ───────────────────────────────────────────────────
    _pdf_charts = ['rdd_analysis.png']
    _pdf_out = 'causal_rdd_analysis.pdf'
    _pdf_narrative = globals().get('_last_narrative', '')
    _pdf_path = _save_method_pdf('Run Rdd Analysis',
                                  result, _pdf_charts, _pdf_narrative, _pdf_out)
    if _pdf_path:
        print(f'  📄 PDF report saved → {_pdf_path}')

    return result

_run_rdd = _run_rdd_patched


print('✅ Causal Analysis runners loaded:')
print('   run_causal_analysis()              → [10] Method-selection menu')
print('   run_ab_test_analysis()             → [17] A/B Test (Statsig/feature-flag data)')
print('   run_pre_post_analysis()            → [18] Pre-Post Analysis')
print('   run_did_analysis()                 → [19] DiD (Enhanced + TWFE)')
print('   run_its_analysis()                 → [20] Interrupted Time Series')
print('   run_psm_analysis()                 → [21] Propensity Score Matching')
print('   run_rdd_analysis()                 → [22] Regression Discontinuity')
print('   run_synthetic_control_analysis()   → [23] Synthetic Control (Enhanced)')



def _print_causal_results(result: dict, method_key: str, meta: dict):
    """Print method-specific key results in a clean format."""
    print('\n' + '═'*72)
    print(f'  📊  CAUSAL ANALYSIS RESULTS — {meta["label"]}')
    print('═'*72)

    skip_keys = {'method','fitted_values','counterfactual','dates','actual_ior',
                 'sc_pre','sc_post','treat_pre','treat_post','dates_pre','dates_post',
                 'event_study','smd_before','smd_after','placebo_gaps_pp','weights'}

    for k, v in result.items():
        if k in skip_keys or isinstance(v, list): continue
        label = k.replace('_',' ').title()
        if isinstance(v, float): print(f'  {label:<35} {v:>12.5f}')
        elif isinstance(v, bool): print(f'  {label:<35} {"✅ Yes" if v else "❌ No"}')
        else: print(f'  {label:<35} {v}')

    if method_key == 'did':
        print(f'\n  DiD 2×2 Table:')
        print(f'              Pre-period    Post-period    Difference')
        print(f'  Treatment   {result["ior_treat_pre"]*100:.3f}%       {result["ior_treat_post"]*100:.3f}%        {result["treat_diff"]*100:+.3f}pp')
        print(f'  Control     {result["ior_ctrl_pre"]*100:.3f}%       {result["ior_ctrl_post"]*100:.3f}%        {result["ctrl_diff"]*100:+.3f}pp')
        print(f'  ─────────────────────────────────────────────────────────────')
        print(f'  DiD estimate:                                {result["did_estimate_pp"]:+.4f}pp  {"✅ sig" if result["significant"] else "⚠️ n.s."}')
        print(f'  Parallel trends: {"✅ HOLDS (p={:.3f})".format(result["parallel_trends_p"]) if result["parallel_trends_ok"] else "⚠️  VIOLATED (p={:.3f}) — results may be biased".format(result["parallel_trends_p"])}')

    elif method_key == 'its':
        print(f'\n  Level change (immediate effect):  {result["level_change_pp"]:+.4f}pp  p={result["level_change_p"]:.4f}  {"✅ sig" if result["level_significant"] else "n.s."}')
        print(f'  Slope change (trend change):      {result["slope_change_pp_day"]:+.5f}pp/day  p={result["slope_change_p"]:.4f}  {"✅ sig" if result["slope_significant"] else "n.s."}')
        print(f'  Avg post-ship lift vs CF:         {result["avg_lift_pp"]:+.4f}pp')

    elif method_key == 'psm':
        print(f'\n  Covariate balance (SMD) BEFORE matching:')
        for s in result['smd_before']:
            bar = '█' * int(s['smd'] * 30)
            print(f'    {s["covariate"]:<20} {bar:<25} {s["smd"]:.3f}')
        print(f'\n  Covariate balance (SMD) AFTER matching:')
        for s in result['smd_after']:
            bar = '█' * int(s['smd'] * 30)
            flag = '✅' if s['smd'] < 0.10 else '⚠️'
            print(f'    {s["covariate"]:<20} {bar:<25} {s["smd"]:.3f}  {flag}')
        print(f'\n  {result["balance_note"]}')

    elif method_key == 'synthetic_control':
        print(f'\n  Synthetic control weights:')
        for seg, w in result['weights'].items():
            print(f'    {seg:<20} {w:.4f}  ({w*100:.1f}%)')
        print(f'\n  Pre-period RMSPE: {result["pre_rmspe"]*100:.3f}pp  (lower = better fit)')
        print(f'  RMSPE ratio (post/pre): {result["rmspe_ratio"]:.2f}  (>2 suggests real effect)')
        if result.get('p_value_placebo_combined') is not None:
            print(f'  Placebo p-value: {result["p_value_placebo_combined"]:.4f}  ({result["n_placebo_tests_combined"]} placebo tests)')


def _plot_causal_results(result: dict, method_key: str, meta: dict):
    """Generate method-appropriate visualisation."""
    if method_key in ('pre_post',):
        # Simple bar chart
        fig, ax = plt.subplots(1, 1, figsize=(8, 5))
        fig.patch.set_facecolor('#0f0f0f')
        bars = ax.bar(['Pre-period','Post-period'],
                      [result['ior_pre']*100, result['ior_post']*100],
                      color=[COLORS['control'], COLORS['treatment']], width=0.4)
        for bar, v in zip(bars, [result['ior_pre']*100, result['ior_post']*100]):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                    f'{v:.3f}%', ha='center', fontsize=11, fontweight='bold', color='white')
        sig = '✅ Significant' if result.get('significant') else '⚠️  Not significant'
        ax.set_title(f'Pre-Post Analysis  |  Δ={result["delta_pp"]:+.3f}pp  p={result["p_value"]:.4f}  {sig}',
                     color=COLORS['highlight'])
        ax.set_ylabel('IOR (%)')
        plt.tight_layout()
        plt.savefig('causal_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
        plt.show()

    elif method_key == 'did':
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.patch.set_facecolor('#0f0f0f')

        # 2×2 table visualisation
        ax1 = axes[0]
        groups = ['Treat Pre','Treat Post','Ctrl Pre','Ctrl Post']
        vals   = [result['ior_treat_pre']*100, result['ior_treat_post']*100,
                  result['ior_ctrl_pre']*100,  result['ior_ctrl_post']*100]
        colors_2x2 = [COLORS['treatment'], COLORS['positive'], COLORS['control'], COLORS['neutral']]
        bars = ax1.bar(groups, vals, color=colors_2x2, width=0.5)
        for bar, v in zip(bars, vals):
            ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                     f'{v:.3f}%', ha='center', fontsize=9.5, fontweight='bold', color='white')
        ax1.set_title(f'DiD 2×2  |  DiD={result["did_estimate_pp"]:+.3f}pp\n'
                      f'{"✅ sig" if result["significant"] else "n.s."}  p={result["p_value"]:.4f}',
                      color=COLORS['highlight'])
        ax1.set_ylabel('IOR (%)')

        # Event study
        ax2 = axes[1]
        if result.get('event_study'):
            ev = result['event_study']
            weeks = [e['week'] for e in ev]
            gaps  = [e['gap']*100 for e in ev]
            colors_ev = [COLORS['positive'] if g >= 0 else COLORS['negative'] for g in gaps]
            ax2.bar(weeks, gaps, color=colors_ev, width=0.8, alpha=0.8)
            ax2.axhline(0, color='white', lw=1, linestyle='--', alpha=0.5)
            ax2.axvline(0, color=COLORS['highlight'], lw=2, linestyle='-', alpha=0.8, label='Intervention')
            ax2.set_xlabel('Week relative to intervention')
            ax2.set_ylabel('Treatment − Control gap (pp)')
            ax2.set_title('Event Study\n(should be flat pre-intervention → parallel trends)',
                          color=COLORS['highlight'])
            ax2.legend(fontsize=9)
        plt.suptitle(f'Difference-in-Differences', fontsize=13,
                     color=COLORS['highlight'], fontweight='bold')
        plt.tight_layout()
        plt.savefig('causal_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
        plt.show()

    elif method_key == 'its':
        fig, axes = plt.subplots(1, 2, figsize=(18, 6))
        fig.patch.set_facecolor('#0f0f0f')
        dates = pd.to_datetime(result['dates'])
        y_act = np.array(result['actual_ior']) * 100
        y_fit = np.array(result['fitted_values']) * 100
        y_cf  = np.array(result['counterfactual']) * 100
        cutoff = pd.Timestamp(result['cutoff_date'])

        ax = axes[0]
        ax.scatter(dates, y_act, color=COLORS['neutral'], s=10, alpha=0.5, label='Actual IOR')
        ax.plot(dates, y_fit, color=COLORS['treatment'], lw=2.5, label='ITS fitted model')
        ax.plot(dates, y_cf,  color=COLORS['control'],   lw=2,   linestyle='--', label='Counterfactual')
        ax.fill_between(dates, y_cf, y_fit, where=y_fit>y_cf,
                        color=COLORS['positive'], alpha=0.2, label='Incremental lift')
        ax.axvline(cutoff, color=COLORS['highlight'], lw=2, label='Intervention')
        ax.set_title(f'ITS Model Fit  |  Level Δ={result["level_change_pp"]:+.3f}pp  '
                     f'{"✅" if result["level_significant"] else "n.s."}', color=COLORS['highlight'])
        ax.set_xlabel('Date'); ax.set_ylabel('IOR (%)')
        ax.legend(fontsize=7.5); ax.grid(True, alpha=0.3)

        ax2 = axes[1]
        gap = (np.array(result['actual_ior']) - np.array(result['counterfactual'])) * 100
        rolling = pd.Series(gap).rolling(7, center=True).mean()
        ax2.bar(range(len(gap)), gap,
                color=[COLORS['positive'] if g>=0 else COLORS['negative'] for g in gap],
                width=1, alpha=0.7)
        ax2.plot(range(len(rolling)), rolling, color='white', lw=2, label='7-day avg')
        ax2.axhline(0, color='white', lw=1, linestyle='--', alpha=0.5)
        ax2.axvline(result['pre_days'], color=COLORS['highlight'], lw=2, label='Intervention')
        ax2.set_xlabel('Day'); ax2.set_ylabel('Actual − Counterfactual (pp)')
        ax2.set_title('Daily Gap: Actual vs Counterfactual', color=COLORS['highlight'])
        ax2.legend(fontsize=8)
        plt.suptitle('Interrupted Time Series Analysis', fontsize=13,
                     color=COLORS['highlight'], fontweight='bold')
        plt.tight_layout()
        plt.savefig('causal_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
        plt.show()

    elif method_key == 'synthetic_control':
        fig, axes = plt.subplots(1, 2, figsize=(18, 6))
        fig.patch.set_facecolor('#0f0f0f')
        all_dates  = result['dates_pre'] + result['dates_post']
        sc_vals    = list(result['sc_pre']) + list(result['sc_post'])
        trt_vals   = list(result['treat_pre']) + list(result['treat_post'])
        n_pre      = result['weeks_pre']

        ax = axes[0]
        ax.plot(range(len(trt_vals)), [v*100 for v in trt_vals],
                color=COLORS['treatment'], lw=2.5, label=result['treatment_segment'])
        ax.plot(range(len(sc_vals)),  [v*100 for v in sc_vals],
                color=COLORS['control'],  lw=2.5, linestyle='--', label='Synthetic control')
        ax.axvline(n_pre, color=COLORS['highlight'], lw=2, label='Intervention')
        ax.fill_between(range(n_pre, len(trt_vals)),
                        [v*100 for v in sc_vals[n_pre:]],
                        [v*100 for v in trt_vals[n_pre:]],
                        color=COLORS['positive'], alpha=0.2)
        ax.set_title(f'Synthetic Control  |  Avg gap={result["avg_gap_pp"]:+.3f}pp\n'
                     f'RMSPE ratio={result["rmspe_ratio"]:.2f}  '
                     f'Placebo p={result["p_value_placebo"]:.3f}' if result.get("p_value_placebo") else '',
                     color=COLORS['highlight'])
        ax.set_xlabel('Week'); ax.set_ylabel('IOR (%)'); ax.legend(fontsize=9)

        ax2 = axes[1]
        all_gap  = [(t-s)*100 for t,s in zip(trt_vals, sc_vals)]
        placebo  = result.get('placebo_gaps_pp', [])
        ax2.hist(placebo, bins=10, color=COLORS['neutral'], alpha=0.7, label='Placebo distribution')
        ax2.axvline(result['avg_gap_pp'], color=COLORS['treatment'], lw=2.5,
                    label=f'Treatment gap ({result["avg_gap_pp"]:+.3f}pp)')
        ax2.set_xlabel('Average post-period gap (pp)'); ax2.set_ylabel('Count')
        ax2.set_title('Placebo Distribution\n(is treatment gap unusual?)', color=COLORS['highlight'])
        ax2.legend(fontsize=9)
        plt.suptitle('Synthetic Control Analysis', fontsize=13,
                     color=COLORS['highlight'], fontweight='bold')
        plt.tight_layout()
        plt.savefig('causal_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
        plt.show()

    elif method_key == 'psm':
        fig, axes = plt.subplots(1, 3, figsize=(20, 5))
        fig.patch.set_facecolor('#0f0f0f')

        # Covariate balance chart
        ax = axes[0]
        covs = [s['covariate'] for s in result['smd_before']]
        smd_b = [s['smd'] for s in result['smd_before']]
        smd_a = [s['smd'] for s in result['smd_after']]
        y = np.arange(len(covs))
        ax.barh(y-0.2, smd_b, 0.35, color=COLORS['negative'], alpha=0.7, label='Before matching')
        ax.barh(y+0.2, smd_a, 0.35, color=COLORS['positive'], alpha=0.7, label='After matching')
        ax.axvline(0.10, color=COLORS['highlight'], lw=1.5, linestyle='--', label='SMD=0.10 threshold')
        ax.set_yticks(y); ax.set_yticklabels(covs)
        ax.set_xlabel('Standardised Mean Difference (lower=better)')
        ax.set_title('Covariate Balance\n(target: SMD < 0.10)', color=COLORS['highlight'])
        ax.legend(fontsize=8)

        # ATT outcome comparison
        ax2 = axes[1]
        groups = ['Matched Control', 'Matched Treatment']
        iors   = [result['ior_control']*100, result['ior_treated']*100]
        bars   = ax2.bar(groups, iors, color=[COLORS['control'], COLORS['treatment']], width=0.4)
        for bar, v in zip(bars, iors):
            ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                     f'{v:.3f}%', ha='center', fontsize=11, fontweight='bold', color='white')
        ax2.set_title(f'ATT = {result["att_pp"]:+.3f}pp\np={result["p_value"]:.4f}  '
                      f'{"✅ sig" if result["significant"] else "n.s."}', color=COLORS['highlight'])
        ax2.set_ylabel('IOR (%)')

        # Summary scorecard
        ax3 = axes[2]
        ax3.axis('off')
        lines = [
            ('Method', 'PSM — ATT Estimate'),
            ('Treated units', str(result['n_treated'])),
            ('Matched pairs', str(result['n_matched_pairs'])),
            ('Match rate', f'{result["n_matched_pairs"]/result["n_treated"]*100:.0f}%'),
            ('ATT', f'{result["att_pp"]:+.3f}pp'),
            ('p-value', f'{result["p_value"]:.4f}'),
            ('Significant', '✅ Yes' if result['significant'] else '❌ No'),
            ('Max SMD after', f'{result["max_smd_after"]:.3f}'),
            ('Balance', '✅ Good' if result['balance_ok'] else '⚠️  Poor'),
        ]
        for ri, (k, v) in enumerate(lines):
            ax3.text(0.05, 0.95-ri*0.10, k, transform=ax3.transAxes,
                     fontsize=9, color='#aaa', va='top')
            ax3.text(0.55, 0.95-ri*0.10, v, transform=ax3.transAxes,
                     fontsize=9, color=COLORS['highlight'], va='top', fontweight='bold')
        ax3.set_title('PSM Scorecard', color=COLORS['highlight'])

        plt.suptitle('Propensity Score Matching Analysis', fontsize=13,
                     color=COLORS['highlight'], fontweight='bold')
        plt.tight_layout()
        plt.savefig('causal_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
        plt.show()

    print('  📁 Chart saved → causal_analysis.png')





def run_did_analysis(llm):
    """
    Module 11a — Standalone DiD Analysis with full interactive UI.
    Surfaces all DiD options: segment-level, entity-level, TWFE toggle.
    """
    print('\n' + '╔' + '═'*70 + '╗')
    print('║' + '  📐  DIFFERENCE-IN-DIFFERENCES ANALYSIS  (Module 11a)'.ljust(70) + '║')
    print('║' + '  Causal effect via treated vs. untreated group comparison'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')

    print("""
  When to use DiD:
  ✅ You have a natural control group that did NOT receive the feature
  ✅ You have pre-period data for both groups (at least 4 weeks)
  ✅ You believe parallel trends would have held without the intervention
  ⚠️  NOT recommended if groups were self-selected into treatment
""")

    unit_choices = {'1': 'account_segment', '2': 'buyer_id', '3': 'account_id'}
    print('  Unit of analysis:')
    print('    [1] Account segment  (few units, stable groups — most common)')
    print('    [2] Buyer ID         (many units, entity-level DiD)')
    print('    [3] Account ID       (account-level DiD)')
    unit_raw = input('  ❓ Choose unit [1]: ').strip() or '1'
    unit_col = unit_choices.get(unit_raw, 'account_segment')

    all_units = sorted(globals().get('df_all_experiments', globals().get(
        'df_hist_inquiries', pd.DataFrame()
    )).get(unit_col, pd.Series()).dropna().unique().tolist()
    ) if unit_col in globals().get('df_all_experiments', pd.DataFrame()).columns else SEGMENTS

    print(f'\n  Available {unit_col} values: {all_units[:20]}')
    treat_raw = input('  ❓ Treatment unit(s) [comma-separated]: ').strip()
    ctrl_raw  = input('  ❓ Control unit(s) [comma-separated]: ').strip()

    treatment_units = [u.strip() for u in treat_raw.split(',') if u.strip()] or [all_units[0]]
    control_units   = [u.strip() for u in ctrl_raw.split(',')  if u.strip()] or [all_units[-1]]

    cutoff_raw  = input(f'  ❓ Intervention date [YYYY-MM-DD, default: {EXP_START.date()}]: ').strip()
    pre_raw     = input(f'  ❓ Pre-period start  [YYYY-MM-DD, default: {HIST_START.date()}]: ').strip()
    alpha_raw   = input('  ❓ Significance level α [0.05]: ').strip()
    twfe_raw    = input('  ❓ Run Two-Way Fixed Effects (TWFE)? [Y/n]: ').strip().lower()
    n_boot_raw  = input('  ❓ Bootstrap resamples [1000]: ').strip()

    cutoff_date     = pd.Timestamp(cutoff_raw) if cutoff_raw else EXP_START
    pre_start       = pd.Timestamp(pre_raw)    if pre_raw    else HIST_START
    alpha           = float(alpha_raw) if alpha_raw else 0.05
    run_twfe        = twfe_raw != 'n'
    n_bootstrap     = int(n_boot_raw) if n_boot_raw.isdigit() else 1_000

    print('\n  Running DiD analysis...')
    result = _run_did_v2(
        treatment_units=treatment_units,
        control_units=control_units,
        cutoff_date=cutoff_date,
        pre_start=pre_start,
        alpha=alpha,
        unit_col=unit_col,
        run_twfe=run_twfe,
        n_bootstrap=n_bootstrap,
    )

    if 'error' in result:
        print(f'\n  ❌ DiD failed: {result["error"]}')
        return result

    print('\n' + '═'*72)
    print('  📊  DIFFERENCE-IN-DIFFERENCES RESULTS')
    print('═'*72)
    print(f'\n  Unit of analysis   : {result["unit_col"]}')
    print(f'  Treatment          : {result["treatment_units"]}')
    print(f'  Control            : {result["control_units"]}')
    print(f'\n  DiD 2×2 Table:')
    print(f'              Pre-period      Post-period     Difference')
    print(f'  Treatment   {result["ior_treat_pre"]*100:.3f}%         {result["ior_treat_post"]*100:.3f}%         {result["treat_diff"]*100:+.3f}pp')
    print(f'  Control     {result["ior_ctrl_pre"]*100:.3f}%         {result["ior_ctrl_post"]*100:.3f}%         {result["ctrl_diff"]*100:+.3f}pp')
    print(f'  ──────────────────────────────────────────────────────────────────')
    print(f'  DiD estimate       : {result["did_estimate_pp"]:+.4f}pp')
    print(f'  Delta-method SE    : ±{result["did_se_delta_pp"]:.4f}pp')
    print(f'  Bootstrap SE       : ±{result["did_se_bootstrap_pp"]:.4f}pp  (n={result["n_bootstrap"]:,})')
    print(f'  Delta CI {int((1-alpha)*100)}%     : [{result["ci_delta_pp"][0]:+.4f}, {result["ci_delta_pp"][1]:+.4f}]pp')
    print(f'  Bootstrap CI {int((1-alpha)*100)}%  : [{result["ci_bootstrap_pp"][0]:+.4f}, {result["ci_bootstrap_pp"][1]:+.4f}]pp')
    print(f'  p-value            : {result["p_value"]:.5f}  '
          f'{"✅ Significant" if result["significant"] else "⚠️  Not significant"} at α={alpha}')
    print()
    print(f'  {result["parallel_trends_note"]}')

    if 'twfe_estimate_pp' in result:
        print()
        print(f'  TWFE estimate      : {result["twfe_estimate_pp"]:+.4f}pp  '
              f'(p={result["twfe_p_value"]:.4f}, n={result["twfe_n_unit_periods"]:,} unit-periods)')
        delta = abs(result["twfe_estimate_pp"] - result["did_estimate_pp"])
        if delta > 0.5:
            print(f'  ⚠️  TWFE and 2×2 DiD differ by {delta:.3f}pp — suggests heterogeneous '
                  f'treatment effects or staggered adoption.')

    if result.get('stagger_warning'):
        print(f'\n  ⚠️  {result["stagger_warning"]}')

    # Plot
    _plot_did_v2(result, alpha=alpha)

    # LLM narrative
    print('\n  🤖 Generating causal interpretation...')
    narrative = llm.narrate(
        {k: v for k, v in result.items()
         if not isinstance(v, list) and not isinstance(v, dict)},
        context=(
            f'Difference-in-Differences analysis. '
            f'Treatment: {treatment_units}, Control: {control_units}. '
            f'DiD estimate: {result["did_estimate_pp"]:+.3f}pp '
            f'({"significant" if result["significant"] else "not significant"} at α={alpha}). '
            f'Parallel trends: {"holds" if result["parallel_trends_ok"] else "VIOLATED"}. '
            f'Provide: (1) causal interpretation of the estimate, (2) key assumptions and '
            f'whether they hold, (3) confidence in the causal claim, (4) recommendation.'
        )
    )
    print('\n' + '─'*72)
    print(narrative)
    print('─'*72)
    # ── Save PDF report ───────────────────────────────────────────────────
    _pdf_charts = ['did_analysis_v2.png']
    _pdf_out = 'did_analysis_v2.pdf'
    _pdf_narrative = globals().get('_last_narrative', '')
    _pdf_path = _save_method_pdf('Run Did Analysis',
                                  result, _pdf_charts, _pdf_narrative, _pdf_out)
    if _pdf_path:
        print(f'  📄 PDF report saved → {_pdf_path}')

    return result


def run_synthetic_control_analysis(llm):
    """
    Module 11b — Standalone Synthetic Control Analysis with full interactive UI.
    """
    print('\n' + '╔' + '═'*70 + '╗')
    print('║' + '  🧪  SYNTHETIC CONTROL ANALYSIS  (Module 11b)'.ljust(70) + '║')
    print('║' + '  Counterfactual from weighted combination of donor units'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')

    print("""
  When to use Synthetic Control:
  ✅ One treated unit with 3+ untreated donor units
  ✅ Long pre-period (10+ weeks) for donors to match the treatment trajectory
  ✅ Donors did NOT receive any similar treatment in the post-period
  ⚠️  Requires good pre-period donor fit (RMSPE < ~1.5pp) to trust results
""")

    print(f'  Available segments: {SEGMENTS}')
    treat_raw   = input('  ❓ Treatment segment [Core]: ').strip() or 'Core'
    donor_raw   = input(f'  ❓ Donor segments [comma-sep, default: all others]: ').strip()
    cutoff_raw  = input(f'  ❓ Intervention date [YYYY-MM-DD, default: {EXP_START.date()}]: ').strip()
    pre_raw     = input(f'  ❓ Pre-period start  [YYYY-MM-DD, default: {HIST_START.date()}]: ').strip()
    thresh_raw  = input('  ❓ Max acceptable pre-RMSPE in pp [1.5]: ').strip()

    treatment_segment = treat_raw
    donor_segments    = ([d.strip() for d in donor_raw.split(',') if d.strip()]
                         or [s for s in SEGMENTS if s != treatment_segment])
    cutoff_date       = pd.Timestamp(cutoff_raw) if cutoff_raw else EXP_START
    pre_start         = pd.Timestamp(pre_raw)    if pre_raw    else HIST_START
    pre_rmspe_thresh  = float(thresh_raw) / 100  if thresh_raw else 0.015

    print(f'\n  Treatment : {treatment_segment}')
    print(f'  Donors    : {donor_segments}')
    print(f'  Cutoff    : {cutoff_date.date()}')
    print(f'  Pre-start : {pre_start.date()}')
    print('\n  Fitting synthetic control...')

    result = _run_synthetic_control_v2(
        treatment_segment=treatment_segment,
        donor_segments=donor_segments,
        cutoff_date=cutoff_date,
        pre_start=pre_start,
        pre_rmspe_threshold=pre_rmspe_thresh,
    )

    if 'error' in result:
        print(f'\n  ❌ Synthetic Control failed: {result["error"]}')
        return result

    print('\n' + '═'*72)
    print('  📊  SYNTHETIC CONTROL RESULTS')
    print('═'*72)
    print(f'\n  Treatment segment  : {result["treatment_segment"]}')
    print(f'\n  Donor weights:')
    for seg, w in result['weights'].items():
        bar = '█' * int(w * 30)
        print(f'    {seg:<18} {bar:<32} {w:.4f}  ({w*100:.1f}%)')

    if result.get('concentration_warning'):
        print(f'  ⚠️  {result["concentration_warning"]}')

    fit_icon = {'Excellent': '✅', 'Good': '✅', 'Marginal': '⚠️', 'Poor': '❌'}
    print(f'\n  Pre-period RMSPE   : {result["pre_rmspe_pp"]:.4f}pp  '
          f'{fit_icon.get(result["fit_quality"],"?")} Fit: {result["fit_quality"]}')
    if result.get('fit_warning'):
        print(f'  ❌ {result["fit_warning"]}')

    print(f'\n  Average post-period gap : {result["avg_gap_pp"]:+.4f}pp')
    print(f'  RMSPE ratio (post/pre)  : {result["rmspe_ratio"]:.3f}  — {result["rmspe_interpretation"]}')
    p_val = result.get('p_value_placebo_combined')
    if p_val is not None:
        print(f'  Permutation p-value     : {p_val:.4f}  '
              f'({result["n_placebo_tests_combined"]} placebo tests: '
              f'{result["n_donor_placebo_tests"]} donor + 1 time)')
    if result.get('time_placebo_note'):
        print(f'\n  {result["time_placebo_note"]}')

    _plot_synthetic_control_v2(result)

    print('\n  🤖 Generating causal interpretation...')
    narrative = llm.narrate(
        {k: v for k, v in result.items()
         if not isinstance(v, list) and not isinstance(v, dict)},
        context=(
            f'Synthetic Control analysis. '
            f'Treatment: {treatment_segment}, Donors: {donor_segments}. '
            f'Pre-RMSPE: {result["pre_rmspe_pp"]:.4f}pp (fit: {result["fit_quality"]}). '
            f'Post-period avg gap: {result["avg_gap_pp"]:+.3f}pp. '
            f'RMSPE ratio: {result["rmspe_ratio"]:.2f}. '
            f'Provide: (1) causal interpretation of the gap, '
            f'(2) assessment of the fit quality and what it means for reliability, '
            f'(3) what the placebo tests tell us, (4) recommendation.'
        )
    )
    print('\n' + '─'*72)
    print(narrative)
    print('─'*72)
    return result


print('✅ Module 11: Causal Analysis Engine loaded')
print('   Methods: A/B Test · Pre-Post · DiD (Enhanced+TWFE) · ITS · Synthetic Control (Enhanced) · PSM')
print('   Standalone runners: run_did_analysis(llm) · run_synthetic_control_analysis(llm)')
print('   Use Module 4 (Experiment Brief) to get a recommended method first.')
# ─────────────────────────────────────────────────────────────────────────────
# REGRESSION DISCONTINUITY (RDD)
# ─────────────────────────────────────────────────────────────────────────────

def _run_mediation(df, treatment_var='variant', treatment_value='treatment',
                   mediator_var='has_billing_profile',
                   outcome_var='converted_to_order', n_boot=500, alpha=0.05):
    """
    Baron-Kenny mediation with bootstrap CI on ACME.

    Model:
        mediator = α0 + α1·T + ε1
        outcome  = β0 + β1·T + β2·M + ε2
        Total effect    = β1 + α1·β2
        Direct effect   = β1
        ACME            = α1·β2
    """
    work = df[[treatment_var, mediator_var, outcome_var]].dropna().copy()
    work['T'] = (work[treatment_var] == treatment_value).astype(float)
    work['M'] = work[mediator_var].astype(float)
    work['Y'] = work[outcome_var].astype(float)

    if work['T'].nunique() < 2:
        return {'error': f'Treatment variable has only one level after filtering to "{treatment_value}"'}
    if len(work) < 200:
        return {'error': f'Too few observations ({len(work)}) for reliable mediation estimates'}

    def _fit(sub):
        Xm = np.column_stack([np.ones(len(sub)), sub['T'].values])
        a  = np.linalg.lstsq(Xm, sub['M'].values, rcond=None)[0]
        Xy = np.column_stack([np.ones(len(sub)), sub['T'].values, sub['M'].values])
        b  = np.linalg.lstsq(Xy, sub['Y'].values, rcond=None)[0]
        return float(a[1]), float(b[1]), float(b[2])      # α1, β1, β2

    a1, b1, b2 = _fit(work)
    acme_point  = a1 * b2       # indirect (mediated) effect
    ade_point   = b1            # direct effect
    total_point = acme_point + ade_point

    # Bootstrap for ACME
    rng_local = np.random.default_rng(42)
    acme_boot = np.empty(n_boot)
    for i in range(n_boot):
        sample = work.sample(n=len(work), replace=True, random_state=int(rng_local.integers(2**31)))
        a1_b, _, b2_b = _fit(sample)
        acme_boot[i] = a1_b * b2_b

    lo_q, hi_q = alpha/2, 1 - alpha/2
    ci_lo = float(np.quantile(acme_boot, lo_q))
    ci_hi = float(np.quantile(acme_boot, hi_q))
    prop_mediated = (acme_point / total_point) if total_point != 0 else float('nan')

    return {
        'method':         'causal_mediation',
        'treatment_var':  treatment_var,
        'mediator_var':   mediator_var,
        'outcome_var':    outcome_var,
        'n':              int(len(work)),
        'total_effect':   float(total_point),
        'direct_effect':  float(ade_point),
        'acme':           float(acme_point),
        'acme_ci_lo':     ci_lo,
        'acme_ci_hi':     ci_hi,
        'prop_mediated':  float(prop_mediated) if not np.isnan(prop_mediated) else None,
        'sig':            not (ci_lo <= 0 <= ci_hi),
        'interpretation': (
            f'Total effect = {total_point:+.4f}. Of this, '
            f'{acme_point:+.4f} ({(prop_mediated*100 if not np.isnan(prop_mediated) else 0):+.1f}%) '
            f'flows through "{mediator_var}" and {ade_point:+.4f} is direct. '
            f'ACME 95% bootstrap CI: [{ci_lo:+.4f}, {ci_hi:+.4f}].'
        ),
    }


def _run_extra_method(method_key, exp_df, exp_info):
    """
    Run RDD or Mediation when the chosen method is one of these advanced ones.
    Returns a dict describing the result, or None if method is not one of these.
    """
    if method_key == 'regression_discontinuity':
        if 'order_value' not in exp_df.columns:
            return {'error': 'No running variable available for RDD on this experiment'}
        subset = exp_df[exp_df['order_value'] > 0].copy()
        if len(subset) < 100:
            return {'error': 'Too few observations for RDD'}
        cutoff = float(subset['order_value'].median())
        return _run_rdd(subset, 'order_value', cutoff, 'converted_to_order')

    if method_key == 'causal_mediation':
        if 'has_billing_profile' not in exp_df.columns:
            return {'error': 'No mediator variable available (expected has_billing_profile)'}
        return _run_mediation(exp_df,
                              treatment_var='variant',
                              treatment_value=[v for v in exp_df['variant'].unique() if v != 'control'][0],
                              mediator_var='has_billing_profile',
                              outcome_var='converted_to_order',
                              n_boot=300)

    # ── Save PDF report ───────────────────────────────────────────────────
    _pdf_charts = ['synthetic_control_v2.png']
    _pdf_out = 'synthetic_control_v2.pdf'
    _pdf_narrative = globals().get('_last_narrative', '')
    _pdf_path = _save_method_pdf('Run Synthetic Control Analysis',
                                  result, _pdf_charts, _pdf_narrative, _pdf_out)
    if _pdf_path:
        print(f'  📄 PDF report saved → {_pdf_path}')

    return None




# ─────────────────────────────────────────────────────────────────────────────
# EXPERIMENT-FIRST POST-EXPERIMENT ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

def _list_experiments_with_status():
    """
    Pull all experiments with rich status display. Return list of dicts and
    the index the user chose, or (None, None) if user aborts.
    """
    from datetime import datetime as _dt
    # Merge registry data with live summary from all_experiments
    live = db.execute("""
        SELECT
            experiment_name,
            COUNT(*)                                     AS n_rows,
            COUNT(DISTINCT variant)                     AS n_variants,
            STRING_AGG(DISTINCT variant, ' | ')         AS variants,
            MIN(created_at)::DATE                        AS first_date,
            MAX(created_at)::DATE                        AS last_date,
            AVG(CAST(converted_to_order AS DOUBLE))*100  AS overall_ior_pct
        FROM all_experiments
        GROUP BY experiment_name
    """).df()

    live_by_name = {r['experiment_name']: r for _, r in live.iterrows()}
    rows = []
    for e in EXPERIMENT_REGISTRY:
        name = e['experiment_name']
        stats = live_by_name.get(name, None)
        rows.append({
            'name':           name,
            'description':    e.get('description', ''),
            'status':         e.get('status', 'unknown'),
            'team':           e.get('team', ''),
            'variants':       e.get('variants', []),
            'start_date':     e.get('start_date'),
            'end_date':       e.get('end_date'),
            'ship_decision':  e.get('ship_decision'),
            'n_rows':         int(stats['n_rows']) if stats is not None else 0,
            'ior_pct':        float(stats['overall_ior_pct']) if stats is not None else None,
        })

    # Status icons
    STATUS_ICON = {
        'running':     '🟢',
        'concluded':   '✅',
        'stopped':     '🛑',
        'shipped':     '🚀',
        'not_started': '💤',
        'unknown':     '⬜',
    }
    SHIP_ICON = {
        'ship':         '🚀 Shipped',
        'partial_ship': '🚀 Partial Ship',
        'no_ship':      '❌ Not Shipped',
        None:           '—',
    }

    print()
    print('  ┌' + '─'*76 + '┐')
    print('  │  AVAILABLE EXPERIMENTS' + ' '*53 + '│')
    print('  ├' + '─'*76 + '┤')
    for i, r in enumerate(rows):
        icon = STATUS_ICON.get(r['status'], '⬜')
        status_text = r['status'].replace('_', ' ').upper()
        decision = SHIP_ICON.get(r['ship_decision'], '—')
        header = f"[{i+1:>2}] {icon} {status_text:<12} {r['name']}"
        print(f"  │  {header[:74].ljust(74)}  │")
        print(f"  │       Team: {r['team']:<15}  Decision: {decision:<20}  {r['n_rows']:>6,} rows  │"[:78].ljust(78))
        desc = r['description'][:68] or 'No description'
        print(f"  │       {desc:<68}          │"[:78].ljust(78))
        variants_str = ' vs '.join(r['variants']) if isinstance(r['variants'], list) else str(r['variants'])
        print(f"  │       Variants: {variants_str[:58]:<58}              │"[:78].ljust(78))
        ior_str = f'{r["ior_pct"]:.2f}%' if r['ior_pct'] is not None else 'n/a'
        print(f"  │       Dates: {str(r['start_date'])} → {str(r['end_date']) or 'ongoing':<12}  Overall IOR: {ior_str:<8}    │"[:78].ljust(78))
        if i < len(rows) - 1:
            print('  ├' + '─'*76 + '┤')
    print('  └' + '─'*76 + '┘')

    while True:
        raw = input(f'\n  ❓ Select experiment [1-{len(rows)}] (or Q to quit): ').strip().lower()
        if raw in ('q', 'quit'): return None, None
        try:
            idx = int(raw) - 1
            if 0 <= idx < len(rows):
                return rows[idx], idx
        except ValueError: pass
        print(f'     ⚠️  Enter a number between 1 and {len(rows)}, or Q')




def _overall_insight(exp_df, control, treatments, alpha=0.05):
    """
    Compute the one-line overall result across all variants.
    """
    overall = {}
    for t in treatments:
        a = exp_df[exp_df['variant'] == control]
        b = exp_df[exp_df['variant'] == t]
        if len(a) < 30 or len(b) < 30:
            continue
        n_a, c_a = len(a), int(a['converted_to_order'].sum())
        n_b, c_b = len(b), int(b['converted_to_order'].sum())
        pr = proportion_test(n_a, c_a, n_b, c_b, alpha)
        overall[t] = {
            'ior_control':   c_a / n_a,
            'ior_treatment': c_b / n_b,
            'delta_pp':      pr['delta_pp'],
            'p_value':       pr['p_value'],
            'ci_lo_pp':      pr['ci_lo_pp'],
            'ci_hi_pp':      pr['ci_hi_pp'],
            'sig':           pr['is_significant'],
            'n_control':     n_a,
            'n_treatment':   n_b,
        }
    return overall


def _dimensional_cuts(exp_df, control, treatments, dimensions, alpha=0.05):
    """
    For each dimension, compute per-level IOR delta + significance.
    Returns dict {dim: [row, row, ...]}.
    """
    out = {}
    for dim in dimensions:
        if dim not in exp_df.columns:
            continue
        rows = []
        for level, sub in exp_df.groupby(dim):
            for t in treatments:
                a = sub[sub['variant'] == control]
                b = sub[sub['variant'] == t]
                if len(a) < 30 or len(b) < 30: continue
                n_a, c_a = len(a), int(a['converted_to_order'].sum())
                n_b, c_b = len(b), int(b['converted_to_order'].sum())
                pr = proportion_test(n_a, c_a, n_b, c_b, alpha)
                rows.append({
                    'dim':          dim,
                    'level':        str(level),
                    'treatment':    t,
                    'n_control':    n_a,
                    'n_treatment':  n_b,
                    'ior_control':  c_a / n_a,
                    'ior_treatment': c_b / n_b,
                    'delta_pp':     pr['delta_pp'],
                    'p_value':      pr['p_value'],
                    'sig':          pr['is_significant'],
                })
        out[dim] = rows
    return out


def _ship_recommendation(overall, dim_cuts):
    """
    Produce a ship/no-ship/iterate decision with reasoning.
    """
    if not overall:
        return 'inconclusive', 'Not enough data to make a recommendation.'

    best_treatment = max(overall, key=lambda t: overall[t]['delta_pp'])
    best = overall[best_treatment]

    # Count segment-level wins and losses across dimensions
    sig_wins = sig_losses = 0
    for dim, rows in dim_cuts.items():
        for r in rows:
            if r['treatment'] != best_treatment: continue
            if r['sig']:
                if r['delta_pp'] > 0: sig_wins += 1
                else:                 sig_losses += 1

    if best['sig'] and best['delta_pp'] > 0 and sig_losses == 0:
        return 'SHIP', (
            f'Treatment "{best_treatment}" wins overall ({best["delta_pp"]:+.2f}pp, '
            f'p={best["p_value"]:.4f}) with no significant segment losses. '
            f'Proceed to full rollout.'
        )
    if best['sig'] and best['delta_pp'] > 0 and sig_losses > 0:
        return 'PARTIAL SHIP', (
            f'Treatment "{best_treatment}" wins overall ({best["delta_pp"]:+.2f}pp, '
            f'p={best["p_value"]:.4f}) but hurts {sig_losses} segment(s). '
            f'Ship to the {sig_wins} winning segment(s) only; hold back on the rest.'
        )
    if (not best['sig']) and abs(best['delta_pp']) < 0.5:
        return 'NO SHIP', (
            f'Best treatment "{best_treatment}" shows no significant effect '
            f'({best["delta_pp"]:+.2f}pp, p={best["p_value"]:.4f}). '
            f'Do not ship; consider iterating on the hypothesis.'
        )
    if best['delta_pp'] < 0:
        return 'NO SHIP', (
            f'Best treatment moves in the wrong direction '
            f'({best["delta_pp"]:+.2f}pp). Kill or pivot the feature.'
        )
    return 'INCONCLUSIVE', (
        f'Results suggest a small effect ({best["delta_pp"]:+.2f}pp, '
        f'p={best["p_value"]:.4f}) but insufficient evidence to ship. '
        f'Extend the test or gather more data.'
    )


def analyze_experiment(llm, mode='full', exp_info=None):
    """
    Unified experiment-first post-experiment analysis.

    Parameters
    ----------
    mode : str
        'full'      → Complete causal analysis (module 8)
        'paradox'   → Simpson's Paradox focus (module 9)
        'roi'       → ROI tracker focus (module 10)
    exp_info : dict, optional
        If given, skip the experiment-selection step.
    """
    if exp_info is None:
        exp_info, _ = _list_experiments_with_status()
        if exp_info is None:
            print('\n  (Analysis aborted.)')
            return None

    exp_name = exp_info['name']
    print(f'\n  ✅ Selected: {exp_name}  ({exp_info["status"].upper()})')
    globals()['_last_analyzed_experiment'] = exp_name

    if mode == 'paradox':
        return _paradox_analysis(llm, exp_info)
    if mode == 'roi':
        return _roi_analysis(llm, exp_info)

    return _full_causal_analysis_enhanced(llm, exp_info)



def _full_causal_analysis_enhanced(llm, exp_info):
    """
    Enhanced Module 8 - Comprehensive deep-dive causal analysis.
    
    Report Structure:
    1. Experiment Summary
    2. Problem Statement
    3. Hypothesis
    4. Context
    5. TL;DR
    6. Additional Insights (narrations, charts, dimensional deep-dives)
    7. Interesting Insights
    8. Ship/No-Ship Decision
    9. Learning Bullet Points
    10. Next Recommendations
    """
    exp_name = exp_info['name']
    method_label = EXP_TYPE_CATALOGUE.get(
        exp_info.get('method', 'ab_test'), {}
    ).get('label', 'A/B Test (Randomised Controlled Trial)')
    print('\n' + '═'*72)
    print(f'  🔬  ENHANCED CAUSAL ANALYSIS — {exp_name}')
    print('═'*72)

    # ── Pull experiment data ──────────────────────────────────────────────────
    exp_df = df_all_experiments[df_all_experiments['experiment_name'] == exp_name].copy()
    exp_df = dedup_dataframe(exp_df)
    variants = sorted(exp_df['variant'].unique().tolist())
    control  = 'control' if 'control' in variants else variants[0]
    treatments = [v for v in variants if v != control]

    print(f'\n  Data rows        : {len(exp_df):,}')
    print(f'  Variants         : {variants}')
    print(f'  Control group    : "{control}"')

    if len(treatments) == 0:
        print('  ⚠️  No treatment variants found — aborting.')
        return None

    # ── Data quality ──────────────────────────────────────────────────────────
    dq = validate_experiment_data(exp_df, exp_name)
    if dq.get('warnings'):
        for w in dq['warnings'][:3]: print(f'  ⚠️  {w}')
    else:
        print('  ✅ Data quality: clean')

    # ── Overall insight ───────────────────────────────────────────────────────
    print('\n  ── [1/6] Overall business-level insight ──')
    overall = _overall_insight(exp_df, control, treatments, alpha=0.05)
    for t, r in overall.items():
        sig_mark = '✅' if r['sig'] else '⚠️ '
        print(f'    {sig_mark} {t:<20} IOR: {r["ior_control"]*100:.3f}% → {r["ior_treatment"]*100:.3f}%  '
              f'Δ={r["delta_pp"]:+.3f}pp  [{r["ci_lo_pp"]:+.2f}, {r["ci_hi_pp"]:+.2f}]  '
              f'p={r["p_value"]:.4f}  n={r["n_treatment"]:,}')

    # ── Dimensional cuts ──────────────────────────────────────────────────────
    print('\n  ── [2/6] Dimensional cuts ──')
    dim_list = [d for d in ['account_segment', 'platform', 'price_tier', 'process_group']
                if d in exp_df.columns]
    dim_cuts = _dimensional_cuts(exp_df, control, treatments, dim_list, alpha=0.05)
    for dim, rows in dim_cuts.items():
        print(f'\n  {dim}:')
        for r in rows:
            sig_mark = '✅' if r['sig'] else '  '
            sign = '+' if r['delta_pp'] >= 0 else ''
            print(f'    {sig_mark} {r["level"]:<20} {r["treatment"]:<15} '
                  f'{r["ior_control"]*100:>5.2f}% → {r["ior_treatment"]*100:>5.2f}%  '
                  f'Δ={sign}{r["delta_pp"]:>6.3f}pp  p={r["p_value"]:.4f}')

    # ── Deep-dive: interesting segments ────────────────────────────────────────
    print('\n  ── [3/6] Deep-dive on interesting segments ──')
    interesting = []
    for dim, rows in dim_cuts.items():
        for r in rows:
            overall_dir = overall.get(r['treatment'], {}).get('delta_pp', 0)
            if r['sig'] and np.sign(r['delta_pp']) != np.sign(overall_dir) and overall_dir != 0:
                interesting.append(('reversal', r))
            elif r['sig'] and abs(r['delta_pp']) > 1.0:
                interesting.append(('extreme', r))

    if interesting:
        for kind, r in interesting[:6]:
            icon = '🔀' if kind == 'reversal' else '💥'
            print(f'    {icon} {r["dim"]}={r["level"]} / {r["treatment"]}: '
                  f'{r["delta_pp"]:+.3f}pp (p={r["p_value"]:.4f})  '
                  f'{"segment reversal" if kind == "reversal" else "extreme effect"}')
    else:
        print('    (No segment reversals or extreme effects found.)')

    # ── Additional insights mining ────────────────────────────────────────────
    print('\n  ── [4/6] Mining additional insights (time-decay, cohort, cross-metric) ──')
    extra_insights = _mine_additional_insights(exp_df, overall, dim_cuts, control, treatments)
    
    time_decay_summary = None
    cohort_summary = None
    cross_metric_summary = None
    
    if extra_insights.get('time_decay') and 'error' not in extra_insights['time_decay']:
        td = extra_insights['time_decay']
        time_decay_summary = td['summary']
        print(f'     Time decay   : {td["summary"]}')
    if extra_insights.get('cohort_effect') and 'error' not in extra_insights['cohort_effect']:
        ce = extra_insights['cohort_effect']
        cohort_summary = ce['summary']
        print(f'     Cohort effect: {ce["summary"]}')
    if extra_insights.get('cross_metric') and 'error' not in extra_insights['cross_metric']:
        cm = extra_insights['cross_metric']
        cross_metric_summary = cm['summary']
        print(f'     Cross-metric : {cm["summary"]}')

    # ── Ship / no-ship recommendation ─────────────────────────────────────────
    print('\n  ── [5/6] Ship/No-Ship Recommendation ──')
    decision, reasoning = _ship_recommendation(overall, dim_cuts)
    print(f'    🎯 {decision}')
    print(f'       {reasoning}')

    # ── Generate comprehensive charts ─────────────────────────────────────────
    print('\n  ── [6/6] Generating comprehensive visualizations ──')
    chart_paths = _generate_enhanced_charts(exp_name, overall, dim_cuts, interesting)
    for chart_type, path in chart_paths.items():
        print(f'     ✅ {chart_type}: {path}')

    # ── Build enhanced context for LLM ────────────────────────────────────────
    print('\n  📋 Building comprehensive experiment context...')
    
    # Gather problem statement and hypothesis from experiment metadata
    problem_statement = exp_info.get('problem_statement', 
        'Identify and address friction points in the customer journey that may be preventing conversions.')
    hypothesis = exp_info.get('hypothesis',
        'Proactive engagement through timely support prompts will reduce abandonment and increase inquiry-to-order conversion rates.')
    
    context = {
        'exp_name': exp_name,
        'exp_info': exp_info,
        'method': method_label,
        'overall': overall,
        'dim_cuts': dim_cuts,
        'interesting': interesting,
        'decision': decision,
        'reasoning': reasoning,
        'time_decay': time_decay_summary,
        'cohort': cohort_summary,
        'cross_metric': cross_metric_summary,
        'problem_statement': problem_statement,
        'hypothesis': hypothesis,
        'total_rows': len(exp_df),
        'data_quality': dq
    }

    # ── Generate comprehensive narrative synthesis ────────────────────────────
    print('\n  🤖 Synthesising comprehensive findings...')
    comprehensive_narrative = _synthesise_comprehensive_findings(context, llm)

    # ── Generate next recommendations ─────────────────────────────────────────
    print('\n  💡 Generating next recommendations...')
    next_recommendations = _generate_next_recommendations(context, llm)

    # ── Build Enhanced PDF Report ─────────────────────────────────────────────
    print('\n  📄 Building comprehensive PDF report...')
    pdf_path = _generate_enhanced_pdf_report(
        exp_name=exp_name,
        exp_info=exp_info,
        method_label=method_label,
        overall=overall,
        dim_cuts=dim_cuts,
        interesting=interesting,
        decision=decision,
        reasoning=reasoning,
        problem_statement=problem_statement,
        hypothesis=hypothesis,
        comprehensive_narrative=comprehensive_narrative,
        next_recommendations=next_recommendations,
        chart_paths=chart_paths,
        time_decay=time_decay_summary,
        cohort=cohort_summary,
        cross_metric=cross_metric_summary,
        total_rows=len(exp_df),
        dq=dq
    )
    
    print(f'\n  📁 Enhanced causal analysis report saved → {pdf_path}')
    print(f'  📊 Generated {len(chart_paths)} visualization charts')

    return {
        'experiment':   exp_name,
        'method':       method_key,
        'overall':      overall,
        'dim_cuts':     dim_cuts,
        'interesting':  interesting,
        'decision':     decision,
        'reasoning':    reasoning,
        'narrative':    comprehensive_narrative,
        'recommendations': next_recommendations,
        'output_file':  pdf_path,
        'chart_paths':  chart_paths,
    }


def _generate_enhanced_charts(exp_name, overall, dim_cuts, interesting):
    """
    Generate comprehensive visualization suite:
    1. Overall effect chart
    2. Dimensional breakdown (forest plot style)
    3. Statistical significance heatmap
    4. Effect size distribution
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.gridspec import GridSpec
    
    chart_paths = {}
    
    # Chart 1: Overall Effect with Confidence Intervals
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('#0f0f0f')
    ax.set_facecolor('#0f0f0f')
    
    for treatment, data in overall.items():
        # Bar chart
        bars = ax.bar(['Control', 'Treatment'],
                      [data['ior_control']*100, data['ior_treatment']*100],
                      color=[COLORS['control'], COLORS['treatment']], width=0.5, alpha=0.9)
        
        # Add confidence interval error bars (ci_lo_pp / ci_hi_pp are delta bounds)
        ci_treatment_err = [data['delta_pp'] - data['ci_lo_pp'],
                           data['ci_hi_pp'] - data['delta_pp']]
        ci_treatment_err = [max(0, v) for v in ci_treatment_err]  # guard negatives
        ax.errorbar([1], [data['ior_treatment']*100],
                   yerr=[[ci_treatment_err[0]], [ci_treatment_err[1]]],
                   fmt='none', ecolor='white', capsize=10, capthick=2, alpha=0.8)
        
        # Value labels
        for i, (bar, val) in enumerate(zip(bars, [data['ior_control']*100, data['ior_treatment']*100])):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                   f'{val:.2f}%', ha='center', fontsize=12, fontweight='bold', color='white')
        
        # Effect size annotation
        mid_y = (data['ior_control'] + data['ior_treatment']) / 2 * 100
        ax.annotate(f'Δ = +{data["delta_pp"]:.2f}pp\np = {data["p_value"]:.4f}\n95% CI: [{data["ci_lo_pp"]:+.2f}, {data["ci_hi_pp"]:+.2f}]',
                   xy=(0.5, mid_y), fontsize=10, ha='center', color=COLORS['highlight'],
                   bbox=dict(boxstyle='round,pad=0.7', facecolor='#1a1a1a', edgecolor=COLORS['highlight'], linewidth=2))
    
    ax.set_ylabel('Inquiry-to-Order Rate (%)', fontsize=12, color='white', fontweight='bold')
    ax.set_title(f'{exp_name}: Overall Treatment Effect\n{"✅ Statistically Significant" if data["sig"] else "⚠️  Not Significant"}',
                fontsize=14, color=COLORS['highlight'], fontweight='bold', pad=20)
    ax.tick_params(colors='white')
    ax.spines['bottom'].set_color('white')
    ax.spines['left'].set_color('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.2, color='white', linestyle='--')
    
    plt.tight_layout()
    chart_path_1 = f'chart_overall_effect_{exp_name}.png'
    plt.savefig(chart_path_1, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.close()
    chart_paths['overall_effect'] = chart_path_1
    
    # Chart 2: Dimensional Breakdown Forest Plot
    fig = plt.figure(figsize=(12, 10))
    fig.patch.set_facecolor('#0f0f0f')
    
    all_segments = []
    for dim, segments in dim_cuts.items():
        for seg in segments:
            all_segments.append({
                'dim': dim,
                'level': seg['level'],
                'delta': seg['delta_pp'],
                'p_value': seg['p_value'],
                'sig': seg['sig']
            })
    
    y_positions = np.arange(len(all_segments))
    colors_list = [COLORS['positive'] if s['sig'] else COLORS['neutral'] for s in all_segments]
    
    ax = fig.add_subplot(111)
    ax.set_facecolor('#0f0f0f')
    
    # Horizontal bars
    bars = ax.barh(y_positions, [s['delta'] for s in all_segments], 
                   color=colors_list, alpha=0.8, edgecolor='white', linewidth=0.5)
    
    # Add value labels
    for i, (seg, bar) in enumerate(zip(all_segments, bars)):
        label = f'{seg["delta"]:+.2f}pp'
        if seg['sig']:
            label += ' *'
        x_pos = seg['delta'] + (0.2 if seg['delta'] >= 0 else -0.2)
        ax.text(x_pos, i, label, va='center', 
               ha='left' if seg['delta'] >= 0 else 'right',
               fontsize=9, color='white', fontweight='bold' if seg['sig'] else 'normal')
    
    # Reference line at 0
    ax.axvline(0, color='white', linewidth=2, alpha=0.7)
    
    # Labels
    labels = [f"{s['dim'].replace('_', ' ')}: {s['level']}" for s in all_segments]
    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels, fontsize=9, color='white')
    ax.set_xlabel('Treatment Effect (percentage points)', fontsize=11, color='white', fontweight='bold')
    ax.set_title('Dimensional Deep-Dive: Effect by Segment\n(* = statistically significant at α=0.05)',
                fontsize=13, color=COLORS['highlight'], fontweight='bold', pad=20)
    ax.tick_params(colors='white')
    ax.spines['bottom'].set_color('white')
    ax.spines['left'].set_color('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.2, color='white', linestyle='--')
    
    plt.tight_layout()
    chart_path_2 = f'chart_dimensional_breakdown_{exp_name}.png'
    plt.savefig(chart_path_2, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.close()
    chart_paths['dimensional_breakdown'] = chart_path_2
    
    # Chart 3: Significance Heatmap by Dimension
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.patch.set_facecolor('#0f0f0f')
    fig.suptitle('Treatment Effect by Dimension (Heatmap View)', 
                fontsize=15, color=COLORS['highlight'], fontweight='bold')
    
    dim_names = list(dim_cuts.keys())
    for idx, (dim_name, ax) in enumerate(zip(dim_names, axes.flat)):
        segments = dim_cuts[dim_name]
        levels = [s['level'] for s in segments]
        deltas = [s['delta_pp'] for s in segments]
        sigs = [s['sig'] for s in segments]
        
        # Create color map based on effect size and significance
        colors_map = []
        for delta, sig in zip(deltas, sigs):
            if sig:
                if delta > 2:
                    colors_map.append('#38a169')  # Strong positive
                elif delta > 0:
                    colors_map.append('#68d391')  # Moderate positive
                elif delta > -2:
                    colors_map.append('#fc8181')  # Moderate negative
                else:
                    colors_map.append('#e53e3e')  # Strong negative
            else:
                colors_map.append('#4a5568')  # Not significant (gray)
        
        y_pos = np.arange(len(levels))
        bars = ax.barh(y_pos, deltas, color=colors_map, alpha=0.9, edgecolor='white', linewidth=1)
        
        # Add labels
        for i, (delta, sig) in enumerate(zip(deltas, sigs)):
            label = f'{delta:+.2f}pp'
            if sig:
                label += ' ✓'
            ax.text(delta + 0.15 if delta >= 0 else delta - 0.15, i,
                   label, va='center', ha='left' if delta >= 0 else 'right',
                   fontsize=9, color='white', fontweight='bold')
        
        ax.axvline(0, color='white', linewidth=1.5, alpha=0.5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(levels, fontsize=9, color='white')
        ax.set_xlabel('Effect (pp)', fontsize=10, color='white')
        ax.set_title(dim_name.replace('_', ' ').title(), fontsize=11, color='white', fontweight='bold')
        ax.set_facecolor('#0f0f0f')
        ax.tick_params(colors='white')
        ax.spines['bottom'].set_color('white')
        ax.spines['left'].set_color('white')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(axis='x', alpha=0.2, color='white', linestyle='--')
    
    plt.tight_layout()
    chart_path_3 = f'chart_heatmap_{exp_name}.png'
    plt.savefig(chart_path_3, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.close()
    chart_paths['significance_heatmap'] = chart_path_3
    
    # Chart 4: Interesting Segments Spotlight
    if interesting:
        fig, ax = plt.subplots(figsize=(10, 6))
        fig.patch.set_facecolor('#0f0f0f')
        ax.set_facecolor('#0f0f0f')
        
        labels = [f"{r['dim']}:\n{r['level']}" for kind, r in interesting]
        deltas = [r['delta_pp'] for kind, r in interesting]
        colors_spot = [COLORS['positive'] if d > 0 else COLORS['negative'] for d in deltas]
        
        bars = ax.bar(range(len(labels)), deltas, color=colors_spot, alpha=0.9, 
                     edgecolor='white', linewidth=2, width=0.6)
        
        for bar, delta in zip(bars, deltas):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, height + (0.2 if height > 0 else -0.2),
                   f'{delta:+.2f}pp', ha='center', va='bottom' if height > 0 else 'top',
                   fontsize=11, color='white', fontweight='bold')
        
        ax.axhline(0, color='white', linewidth=1.5, alpha=0.7)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, fontsize=10, color='white')
        ax.set_ylabel('Treatment Effect (pp)', fontsize=11, color='white', fontweight='bold')
        ax.set_title('Spotlight: Most Interesting Segments\n(Extreme Effects & Reversals)',
                    fontsize=13, color=COLORS['highlight'], fontweight='bold', pad=20)
        ax.tick_params(colors='white')
        ax.spines['bottom'].set_color('white')
        ax.spines['left'].set_color('white')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(axis='y', alpha=0.2, color='white', linestyle='--')
        
        plt.tight_layout()
        chart_path_4 = f'chart_interesting_segments_{exp_name}.png'
        plt.savefig(chart_path_4, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
        plt.close()
        chart_paths['interesting_segments'] = chart_path_4
    
    return chart_paths


def _synthesise_comprehensive_findings(context, llm):
    """
    Generate comprehensive narrative synthesis using LLM.
    Much more detailed than the original version.
    """
    
    prompt = f"""You are a senior data scientist writing a comprehensive causal analysis report.

EXPERIMENT: {context['exp_name']}
METHOD: {context['method']}
STATUS: {context['exp_info']['status']}

PROBLEM STATEMENT:
{context['problem_statement']}

HYPOTHESIS:
{context['hypothesis']}

OVERALL RESULTS:
"""
    
    for treatment, data in context['overall'].items():
        prompt += f"""- Treatment "{treatment}": IOR {data['ior_control']*100:.2f}% → {data['ior_treatment']*100:.2f}%
  Effect: {data['delta_pp']:+.3f}pp, 95% CI: [{data['ci_lo_pp']:+.2f}, {data['ci_hi_pp']:+.2f}]
  Statistical significance: {'YES (p=' + f"{data['p_value']:.4f}" + ')' if data['sig'] else 'NO (p=' + f"{data['p_value']:.4f}" + ')'}
  Sample size: n={data['n_treatment']:,}
"""
    
    prompt += "\nDIMENSIONAL BREAKDOWN:\n"
    for dim, segments in context['dim_cuts'].items():
        prompt += f"\n{dim}:\n"
        for seg in segments:
            sig_marker = "* SIGNIFICANT *" if seg['sig'] else ""
            prompt += f"  - {seg['level']}: {seg['delta_pp']:+.2f}pp (p={seg['p_value']:.4f}) {sig_marker}\n"
    
    if context['interesting']:
        prompt += "\nINTERESTING SEGMENTS (Extreme Effects or Reversals):\n"
        for kind, seg in context['interesting']:
            prompt += f"  - [{kind.upper()}] {seg['dim']}={seg['level']}: {seg['delta_pp']:+.2f}pp\n"
    
    if context['time_decay']:
        prompt += f"\nTIME DECAY ANALYSIS:\n{context['time_decay']}\n"
    if context['cohort']:
        prompt += f"\nCOHORT EFFECT:\n{context['cohort']}\n"
    if context['cross_metric']:
        prompt += f"\nCROSS-METRIC ANALYSIS:\n{context['cross_metric']}\n"
    
    prompt += f"""
DECISION: {context['decision']}
REASONING: {context['reasoning']}

Please write a comprehensive narrative that includes:

1. EXECUTIVE SUMMARY (2-3 sentences): The key finding and business impact

2. DETAILED STATISTICAL FINDINGS (1 paragraph): Interpret the overall effect size, confidence intervals, and statistical significance in business terms

3. DIMENSIONAL INSIGHTS (2-3 paragraphs): 
   - Which customer segments benefited most/least?
   - Are there any surprising patterns or reversals?
   - What does this tell us about customer behavior?

4. MECHANISM & CAUSALITY (1 paragraph): Based on the hypothesis and results, what is the likely causal mechanism? Why did this intervention work (or not work)?

5. BUSINESS IMPLICATIONS (1 paragraph): What are the practical implications for the business? Revenue impact, operational changes needed, etc.

6. LIMITATIONS & CAVEATS (1 paragraph): What are the limitations of this analysis? What should we be cautious about?

Write in clear, professional language. Use specific numbers from the data. Be analytical and insightful.
"""
    
    try:
        narrative = llm.ask(prompt)
        return narrative.strip()
    except:
        return f"""EXECUTIVE SUMMARY: The treatment showed a {context['overall'][list(context['overall'].keys())[0]]['delta_pp']:+.2f}pp effect on IOR with {'statistical significance' if context['overall'][list(context['overall'].keys())[0]]['sig'] else 'no statistical significance'}. Decision: {context['decision']}.

DETAILED FINDINGS: Full analysis available in dimensional breakdowns above.

RECOMMENDATION: {context['reasoning']}"""


def _generate_next_recommendations(context, llm):
    """
    Generate actionable next steps and recommendations using LLM.
    """
    
    prompt = f"""Based on this experiment analysis, provide 4-6 specific, actionable recommendations for next steps.

EXPERIMENT: {context['exp_name']}
DECISION: {context['decision']}
KEY FINDINGS: {context['reasoning']}

Categories to consider:
1. IMMEDIATE ACTIONS: What should be done right now (rollout, iterate, stop)?
2. FOLLOW-UP EXPERIMENTS: What should we test next to build on these learnings?
3. OPERATIONAL CHANGES: What process or system changes are needed to support the winning variant?
4. MONITORING: What metrics should we track post-launch to ensure sustained impact?
5. GENERALIZATION: Can this learning be applied to other areas of the business?

Provide 4-6 bullet points, each with:
- A clear action item
- The expected benefit
- Approximate timeline or priority

Be specific and actionable.
"""
    
    try:
        recommendations = llm.ask(prompt)
        return recommendations.strip()
    except:
        # Fallback
        return f"""• IMMEDIATE: {context['decision']} - proceed with recommended action
• MONITOR: Track IOR metrics post-launch for 30 days to ensure sustained lift
• ITERATE: Consider testing variations in other customer segments
• SCALE: Apply learnings to similar interventions across the customer journey"""


def _generate_enhanced_pdf_report(exp_name, exp_info, method_label, overall, dim_cuts,
                                   interesting, decision, reasoning, problem_statement,
                                   hypothesis, comprehensive_narrative, next_recommendations,
                                   chart_paths, time_decay, cohort, cross_metric, total_rows, dq):
    """
    Generate a comprehensive PDF report with the user's requested structure:
    1. Experiment Summary
    2. Problem Statement
    3. Hypothesis
    4. Context
    5. TL;DR
    6. Additional Insights (with charts and narrations)
    7. Interesting Insights
    8. Ship/No-Ship
    9. Learning Bullet Points
    10. Next Recommendations
    """
    from collections import OrderedDict
    from datetime import datetime
    from reportlab.lib.pagesizes import letter
    from reportlab.lib import colors as rl_colors
    from reportlab.lib.units import inch
    from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, PageBreak, 
                                     Table, TableStyle, Image as RLImage, KeepTogether)
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    
    # PDF filename
    pdf_filename = f'causal_analysis_comprehensive_{exp_name}.pdf'
    
    # Create PDF document
    doc = SimpleDocTemplate(pdf_filename, pagesize=letter,
                           topMargin=0.75*inch, bottomMargin=0.75*inch,
                           leftMargin=0.75*inch, rightMargin=0.75*inch)
    
    # Styles
    styles = getSampleStyleSheet()
    
    # Custom styles
    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=20,
        textColor=rl_colors.HexColor('#1a365d'),
        spaceAfter=20,
        alignment=TA_CENTER,
        fontName='Helvetica-Bold'
    )
    
    section_style = ParagraphStyle(
        'SectionHeader',
        parent=styles['Heading1'],
        fontSize=14,
        textColor=rl_colors.HexColor('#2c5282'),
        spaceAfter=10,
        spaceBefore=15,
        fontName='Helvetica-Bold',
        borderWidth=1,
        borderColor=rl_colors.HexColor('#2c5282'),
        borderPadding=6,
        backColor=rl_colors.HexColor('#edf2f7')
    )
    
    subsection_style = ParagraphStyle(
        'Subsection',
        parent=styles['Heading2'],
        fontSize=12,
        textColor=rl_colors.HexColor('#2d3748'),
        spaceAfter=8,
        spaceBefore=12,
        fontName='Helvetica-Bold'
    )
    
    body_style = ParagraphStyle(
        'BodyText',
        parent=styles['Normal'],
        fontSize=10,
        leading=14,
        alignment=TA_JUSTIFY,
        spaceAfter=10
    )
    
    decision_style = ParagraphStyle(
        'Decision',
        parent=styles['Normal'],
        fontSize=13,
        textColor=rl_colors.white,
        fontName='Helvetica-Bold',
        alignment=TA_CENTER,
        backColor=rl_colors.HexColor('#38a169') if 'SHIP' in decision else rl_colors.HexColor('#e53e3e'),
        borderPadding=12,
        borderWidth=2,
        borderColor=rl_colors.HexColor('#2f855a') if 'SHIP' in decision else rl_colors.HexColor('#c53030')
    )
    
    story = []
    
    # TITLE PAGE
    story.append(Paragraph("Comprehensive Causal Analysis Report", title_style))
    story.append(Paragraph(f"<b>Experiment:</b> {exp_name}", styles['Normal']))
    story.append(Spacer(1, 0.2*inch))
    
    # Metadata table
    metadata_data = [
        ['Experiment Name:', exp_name],
        ['Status:', exp_info['status'].upper()],
        ['Method:', method_label],
        ['Team:', exp_info['team']],
        ['Rows Analyzed:', f'{total_rows:,}'],
        ['Decision:', decision],
        ['Generated:', datetime.now().strftime('%d %b %Y · %H:%M')],
    ]
    
    metadata_table = Table(metadata_data, colWidths=[2*inch, 4*inch])
    metadata_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (0, -1), rl_colors.HexColor('#edf2f7')),
        ('TEXTCOLOR', (0, 0), (-1, -1), rl_colors.black),
        ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 10),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('GRID', (0, 0), (-1, -1), 0.5, rl_colors.grey),
    ]))
    story.append(metadata_table)
    story.append(Spacer(1, 0.3*inch))
    
    # 1. EXPERIMENT SUMMARY
    story.append(Paragraph("1. Experiment Summary", section_style))
    
    summary_text = f"""This experiment tested the impact of {exp_name} using a {method_label} methodology. 
    The analysis included {total_rows:,} observations across {len(overall)} treatment variant(s). """
    
    if dq.get('warnings'):
        summary_text += f"Data quality checks identified {len(dq['warnings'])} potential issues that were reviewed and addressed. "
    else:
        summary_text += "Data quality checks passed all validation criteria. "
    
    story.append(Paragraph(summary_text, body_style))
    story.append(Spacer(1, 0.15*inch))
    
    # 2. PROBLEM STATEMENT
    story.append(Paragraph("2. Problem Statement", section_style))
    story.append(Paragraph(problem_statement, body_style))
    story.append(Spacer(1, 0.15*inch))
    
    # 3. HYPOTHESIS
    story.append(Paragraph("3. Hypothesis", section_style))
    story.append(Paragraph(hypothesis, body_style))
    story.append(Spacer(1, 0.15*inch))
    
    # 4. CONTEXT
    story.append(Paragraph("4. Context", section_style))
    
    context_text = f"""This {method_label} was conducted by the {exp_info['team']} team to address the problem statement above. 
    The experiment {'has concluded' if exp_info['status'].lower() == 'concluded' else 'is ' + exp_info['status'].lower()} 
    and the analysis below presents the causal impact assessment."""
    
    story.append(Paragraph(context_text, body_style))
    story.append(Spacer(1, 0.15*inch))
    
    # 5. TL;DR (Executive Summary)
    story.append(Paragraph("5. TL;DR (Executive Summary)", section_style))
    
    # Extract first paragraph from comprehensive narrative if available
    narrative_lines = comprehensive_narrative.split('\n\n')
    tldr_text = narrative_lines[0] if narrative_lines else reasoning
    
    story.append(Paragraph(tldr_text, body_style))
    story.append(Spacer(1, 0.2*inch))
    
    # Decision box
    story.append(Paragraph(f"<b>DECISION: {decision}</b>", decision_style))
    story.append(Spacer(1, 0.1*inch))
    story.append(Paragraph(f"<i>{reasoning}</i>", body_style))
    story.append(Spacer(1, 0.2*inch))
    
    # Page break before detailed section
    story.append(PageBreak())
    
    # 6. ADDITIONAL INSIGHTS (The Main Deep-Dive Section)
    story.append(Paragraph("6. Additional Insights & Deep-Dive Analysis", section_style))
    story.append(Spacer(1, 0.1*inch))
    
    # 6.1 Overall Statistical Results
    story.append(Paragraph("6.1 Overall Statistical Results", subsection_style))
    
    for treatment, data in overall.items():
        result_text = f"""<b>Treatment "{treatment}":</b><br/>
        • Control Group IOR: {data['ior_control']*100:.2f}%<br/>
        • Treatment Group IOR: {data['ior_treatment']*100:.2f}%<br/>
        • <b>Effect Size: {data['delta_pp']:+.2f} percentage points</b><br/>
        • 95% Confidence Interval: [{data['ci_lo_pp']:+.2f}pp, {data['ci_hi_pp']:+.2f}pp]<br/>
        • P-value: {data['p_value']:.4f}<br/>
        • Statistical Significance: <b>{'YES - Significant at α=0.05' if data['sig'] else 'NO - Not significant'}</b><br/>
        • Sample Sizes: Control n={data['n_control']:,}, Treatment n={data['n_treatment']:,}
        """
        story.append(Paragraph(result_text, body_style))
    
    story.append(Spacer(1, 0.15*inch))
    
    # Add overall effect chart
    if 'overall_effect' in chart_paths:
        try:
            img = RLImage(chart_paths['overall_effect'], width=6*inch, height=3.6*inch)
            story.append(img)
            story.append(Spacer(1, 0.15*inch))
        except:
            pass
    
    # 6.2 Dimensional Breakdown
    story.append(Paragraph("6.2 Dimensional Breakdown & Slices-n-Dices", subsection_style))
    
    dim_narration = """The treatment effect was analyzed across multiple customer dimensions to understand 
    which segments experienced the strongest (or weakest) impact. This dimensional analysis reveals important 
    heterogeneity in treatment effects and helps identify opportunities for targeted interventions."""
    story.append(Paragraph(dim_narration, body_style))
    story.append(Spacer(1, 0.1*inch))
    
    # Dimensional table for each dimension
    for dim_name, segments in dim_cuts.items():
        story.append(Paragraph(f"<b>{dim_name.replace('_', ' ').title()}:</b>", subsection_style))
        
        # Create table
        dim_table_data = [['Segment', 'Control IOR', 'Treatment IOR', 'Effect (pp)', 'P-value', 'Significant']]
        for seg in segments:
            sig_marker = '✓' if seg['sig'] else '✗'
            dim_table_data.append([
                seg['level'],
                f"{seg['ior_control']*100:.2f}%",
                f"{seg['ior_treatment']*100:.2f}%",
                f"{seg['delta_pp']:+.2f}",
                f"{seg['p_value']:.4f}",
                sig_marker
            ])
        
        dim_table = Table(dim_table_data, colWidths=[1.5*inch, 1*inch, 1*inch, 0.9*inch, 0.9*inch, 0.7*inch])
        dim_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#2c5282')),
            ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.white),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, -1), 9),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
            ('TOPPADDING', (0, 0), (-1, -1), 6),
            ('GRID', (0, 0), (-1, -1), 0.5, rl_colors.grey),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.HexColor('#f7fafc')]),
        ]))
        story.append(dim_table)
        story.append(Spacer(1, 0.15*inch))
    
    # Add dimensional breakdown chart
    if 'dimensional_breakdown' in chart_paths:
        try:
            img = RLImage(chart_paths['dimensional_breakdown'], width=6.5*inch, height=5.2*inch)
            story.append(img)
            story.append(Spacer(1, 0.15*inch))
        except:
            pass
    
    # Add heatmap chart
    if 'significance_heatmap' in chart_paths:
        try:
            story.append(PageBreak())
            img = RLImage(chart_paths['significance_heatmap'], width=6.5*inch, height=5.2*inch)
            story.append(img)
            story.append(Spacer(1, 0.15*inch))
        except:
            pass
    
    # 6.3 Time Decay, Cohort, and Cross-Metric Analysis
    if time_decay or cohort or cross_metric:
        story.append(Paragraph("6.3 Advanced Statistical Analyses", subsection_style))
        
        if time_decay:
            story.append(Paragraph("<b>Time Decay Analysis:</b>", body_style))
            story.append(Paragraph(time_decay, body_style))
            story.append(Spacer(1, 0.1*inch))
        
        if cohort:
            story.append(Paragraph("<b>Cohort Effect Analysis:</b>", body_style))
            story.append(Paragraph(cohort, body_style))
            story.append(Spacer(1, 0.1*inch))
        
        if cross_metric:
            story.append(Paragraph("<b>Cross-Metric Correlation:</b>", body_style))
            story.append(Paragraph(cross_metric, body_style))
            story.append(Spacer(1, 0.15*inch))
    
    # 6.4 Comprehensive Narrative
    story.append(Paragraph("6.4 Comprehensive Interpretation", subsection_style))
    
    # Split narrative into sections if it contains headers
    narrative_sections = comprehensive_narrative.split('\n\n')
    for section in narrative_sections:
        if section.strip():
            story.append(Paragraph(section.strip(), body_style))
            story.append(Spacer(1, 0.1*inch))
    
    story.append(PageBreak())
    
    # 7. INTERESTING INSIGHTS
    story.append(Paragraph("7. Interesting Insights", section_style))
    
    if interesting:
        insights_text = """The following segments exhibited particularly notable patterns - either extreme effect sizes 
        or directional reversals compared to the overall trend. These merit special attention for follow-up analysis 
        or targeted interventions."""
        story.append(Paragraph(insights_text, body_style))
        story.append(Spacer(1, 0.1*inch))
        
        for kind, seg in interesting:
            insight_text = f"""<b>[{kind.upper()}]</b> {seg['dim'].replace('_', ' ').title()}: {seg['level']}<br/>
            • Effect: {seg['delta_pp']:+.2f}pp<br/>
            • P-value: {seg['p_value']:.4f}<br/>
            • Pattern: {'Segment shows opposite direction vs overall trend' if kind == 'reversal' else 'Unusually large effect size indicating strong segment-specific response'}
            """
            story.append(Paragraph(insight_text, body_style))
            story.append(Spacer(1, 0.1*inch))
        
        # Add interesting segments chart if available
        if 'interesting_segments' in chart_paths:
            try:
                img = RLImage(chart_paths['interesting_segments'], width=6*inch, height=3.6*inch)
                story.append(img)
                story.append(Spacer(1, 0.15*inch))
            except:
                pass
    else:
        story.append(Paragraph("No unusual segment patterns detected. Treatment effects are relatively consistent across all analyzed dimensions.", body_style))
    
    story.append(Spacer(1, 0.15*inch))
    
    # 8. SHIP/NO-SHIP DECISION
    story.append(Paragraph("8. Ship/No-Ship Decision", section_style))
    story.append(Paragraph(f"<b>{decision}</b>", decision_style))
    story.append(Spacer(1, 0.1*inch))
    story.append(Paragraph(f"<b>Reasoning:</b> {reasoning}", body_style))
    story.append(Spacer(1, 0.15*inch))
    
    # 9. LEARNING BULLET POINTS
    story.append(Paragraph("9. Key Learning Bullet Points", section_style))
    
    learnings = []
    
    # Generate learnings from significant segments
    for dim_name, segments in dim_cuts.items():
        for seg in segments:
            if seg['sig']:
                learning = f"<b>Learning:</b> {dim_name.replace('_', ' ').title()} segment '{seg['level']}' showed a significant {seg['delta_pp']:+.2f}pp effect (p={seg['p_value']:.4f})"
                learnings.append(learning)
    
    # Add interesting segment learnings
    for kind, seg in interesting:
        learning = f"<b>Learning:</b> {seg['dim'].replace('_', ' ').title()} '{seg['level']}' exhibited {kind} pattern with {seg['delta_pp']:+.2f}pp effect - requires targeted follow-up"
        learnings.append(learning)
    
    # Add overall learning
    for treatment, data in overall.items():
        learning = f"<b>Overall Learning:</b> Treatment '{treatment}' {'achieved' if data['sig'] else 'did not achieve'} statistical significance with {data['delta_pp']:+.2f}pp effect"
        learnings.insert(0, learning)
    
    for learning in learnings[:8]:  # Limit to top 8 learnings
        story.append(Paragraph(f"• {learning}", body_style))
        story.append(Spacer(1, 0.08*inch))
    
    story.append(Spacer(1, 0.15*inch))
    
    # 10. NEXT RECOMMENDATIONS
    story.append(Paragraph("10. Next Recommendations", section_style))
    
    # Split recommendations into bullet points
    rec_lines = next_recommendations.split('\n')
    for line in rec_lines:
        if line.strip():
            story.append(Paragraph(line.strip(), body_style))
            story.append(Spacer(1, 0.08*inch))
    
    # Build PDF
    doc.build(story)
    
    return pdf_filename



def _paradox_analysis(llm, exp_info):
    """Module 9 — focus on finding segment reversals (Simpson's Paradox)."""
    exp_name = exp_info['name']
    print('\n' + '═'*72)
    print(f"  🔀  SIMPSON'S PARADOX DETECTOR — {exp_name}")
    print('═'*72)

    exp_df = df_all_experiments[df_all_experiments['experiment_name'] == exp_name].copy()
    exp_df = dedup_dataframe(exp_df)
    variants = sorted(exp_df['variant'].unique().tolist())
    control  = 'control' if 'control' in variants else variants[0]
    treatments = [v for v in variants if v != control]

    overall = _overall_insight(exp_df, control, treatments, alpha=0.05)
    dim_list = [d for d in ['account_segment', 'platform', 'price_tier', 'process_group']
                if d in exp_df.columns]
    dim_cuts = _dimensional_cuts(exp_df, control, treatments, dim_list, alpha=0.05)

    # Find reversals
    print('\n  Overall results:')
    for t, r in overall.items():
        print(f'    {t}: Δ={r["delta_pp"]:+.3f}pp (p={r["p_value"]:.4f})')

    print('\n  Segment-level reversals (Simpson\'s Paradox check):')
    reversals = []
    for dim, rows in dim_cuts.items():
        for r in rows:
            overall_dir = overall.get(r['treatment'], {}).get('delta_pp', 0)
            if overall_dir != 0 and np.sign(r['delta_pp']) != np.sign(overall_dir):
                reversals.append(r)
                print(f'    🔀 {dim}={r["level"]}, {r["treatment"]}: '
                      f'segment Δ={r["delta_pp"]:+.3f}pp  '
                      f'(overall Δ={overall_dir:+.3f}pp)  p={r["p_value"]:.4f}')

    if not reversals:
        print('    ✅ No significant segment reversals detected — aggregate is trustworthy.')

    # Ship decision + learnings
    decision, reasoning = _ship_recommendation(overall, dim_cuts)
    print(f'\n  🎯 Recommendation: {decision}')
    print(f'     {reasoning}')

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    all_segs = []
    for dim, rows in dim_cuts.items():
        for r in rows:
            overall_dir = overall.get(r['treatment'], {}).get('delta_pp', 0)
            is_reversal = (overall_dir != 0 and
                           np.sign(r['delta_pp']) != np.sign(overall_dir))
            all_segs.append({
                'label':      f"{dim.replace('_',' ')}: {r['level']}",
                'delta':      r['delta_pp'],
                'reversal':   is_reversal,
                'sig':        r['sig'],
            })

    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(all_segs) * 0.45 + 2)))
    fig.patch.set_facecolor('#0f0f0f')
    fig.suptitle(f"Simpson's Paradox Detector — {exp_name}",
                 fontsize=13, color=COLORS['highlight'], fontweight='bold')

    # Left: overall per-treatment bars
    ax1 = axes[0]
    ax1.set_facecolor('#0f0f0f')
    t_names = list(overall.keys())
    t_deltas = [overall[t]['delta_pp'] for t in t_names]
    bar_colors = [COLORS['positive'] if d >= 0 else COLORS['negative'] for d in t_deltas]
    bars = ax1.bar(t_names, t_deltas, color=bar_colors, alpha=0.85,
                   edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, t_deltas):
        ax1.text(bar.get_x() + bar.get_width() / 2,
                 val + (0.15 if val >= 0 else -0.15),
                 f'{val:+.2f}pp', ha='center',
                 va='bottom' if val >= 0 else 'top',
                 fontsize=10, color='white', fontweight='bold')
    ax1.axhline(0, color='white', lw=1.5, alpha=0.6)
    ax1.set_ylabel('Treatment Effect (pp)', color='white', fontsize=10)
    ax1.set_title('Overall Effect', color='white', fontsize=11)
    ax1.tick_params(colors='white')
    for spine in ['top', 'right']: ax1.spines[spine].set_visible(False)
    for spine in ['bottom', 'left']: ax1.spines[spine].set_color('white')
    ax1.grid(axis='y', alpha=0.2, color='white', linestyle='--')

    ax2 = axes[1]
    ax2.set_facecolor('#0f0f0f')
    y_pos = np.arange(len(all_segs))
    seg_colors = []
    for s in all_segs:
        if s['reversal']:
            seg_colors.append('#ed8936')      # amber  — reversal
        elif s['sig']:
            seg_colors.append(COLORS['positive'])
        else:
            seg_colors.append(COLORS['neutral'])
    ax2.barh(y_pos, [s['delta'] for s in all_segs],
             color=seg_colors, alpha=0.85, edgecolor='white', linewidth=0.5)
    for i, s in enumerate(all_segs):
        suffix = ' ⚠ REVERSAL' if s['reversal'] else (' *' if s['sig'] else '')
        ax2.text(s['delta'] + (0.15 if s['delta'] >= 0 else -0.15), i,
                 f"{s['delta']:+.2f}pp{suffix}",
                 va='center', ha='left' if s['delta'] >= 0 else 'right',
                 fontsize=8,
                 color='#ed8936' if s['reversal'] else 'white',
                 fontweight='bold' if s['reversal'] else 'normal')
    ax2.axvline(0, color='white', lw=2, alpha=0.7)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels([s['label'] for s in all_segs], fontsize=8, color='white')
    ax2.set_xlabel('Treatment Effect (pp)', color='white', fontsize=10)
    ax2.set_title("Segments (amber = reversal vs overall)",
                  color='white', fontsize=11)
    ax2.tick_params(colors='white')
    for spine in ['top', 'right']: ax2.spines[spine].set_visible(False)
    for spine in ['bottom', 'left']: ax2.spines[spine].set_color('white')
    ax2.grid(axis='x', alpha=0.2, color='white', linestyle='--')

    plt.tight_layout()
    chart_path = f'paradox_chart_{exp_name}.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.close()
    print(f'  📊 Chart saved → {chart_path}')

    from collections import OrderedDict

    if reversals:
        reversal_lines = '\n'.join(
            f"- {r['dim']} / {r['level']} / {r['treatment']}: "            f"segment Δ={r['delta_pp']:+.2f}pp  "            f"(overall Δ={overall.get(r['treatment'],{}).get('delta_pp',0):+.2f}pp)  "            f"p={r['p_value']:.4f}"
            for r in reversals
        )
        reversal_summary = (
            f"{len(reversals)} reversal(s) detected. "            "The aggregate result masks opposing effects in specific segments. "            "Shipping based on the overall number alone would over-estimate true impact."
        )
    else:
        reversal_lines   = 'No reversals detected — aggregate is trustworthy.'
        reversal_summary = 'No Simpson\'s Paradox detected. The aggregate result is consistent across all tested dimensions.'

    dim_detail_lines = '\n'.join(
        f"- {dim} / {r['level']} / {r['treatment']}: "        f"Δ={r['delta_pp']:+.2f}pp, p={r['p_value']:.4f}"        + (' (significant)' if r['sig'] else '')        + (' ⚠ REVERSAL' if (overall.get(r['treatment'],{}).get('delta_pp',0) != 0
                              and np.sign(r['delta_pp']) !=
                              np.sign(overall.get(r['treatment'],{}).get('delta_pp',0))) else '')
        for dim, rows in dim_cuts.items() for r in rows
    )

    overall_lines = '\n'.join(
        f"{t}: IOR {r['ior_control']*100:.2f}% → {r['ior_treatment']*100:.2f}%, "        f"Δ={r['delta_pp']:+.3f}pp "        f"[{r['ci_lo_pp']:+.2f}, {r['ci_hi_pp']:+.2f}], "        f"p={r['p_value']:.4f}, n={r['n_treatment']:,}"
        for t, r in overall.items()
    )

    pdf_sections = OrderedDict([
        ('HEADLINE',          f"{decision}: {reasoning}"),
        ('PARADOX SUMMARY',   reversal_summary),
        ('OVERALL RESULT',    overall_lines),
        ('DIMENSIONAL CUTS',  dim_detail_lines),
        ('REVERSAL DETAIL',   reversal_lines),
        ('RECOMMENDATION',    f"{decision}\n{reasoning}"),
    ])

    pdf_path = f'paradox_analysis_{exp_name}.pdf'
    out_path = render_document_pdf(
        title="Simpson's Paradox Report",
        subtitle=f'Experiment: {exp_name}',
        sections=pdf_sections,
        output_path=pdf_path,
        metadata={
            'Experiment':       exp_name,
            'Status':           exp_info['status'].upper(),
            'Team':             exp_info['team'],
            'Reversals found':  str(len(reversals)),
            'Decision':         decision,
            'Rows analysed':    f"{len(exp_df):,}",
        },
        accent_color=PDF_PALETTE['secondary'] if reversals else PDF_PALETTE['success'],
    )
    print(f'  📁 Paradox report saved → {out_path}')

    return {
        'experiment':  exp_name,
        'overall':     overall,
        'reversals':   reversals,
        'decision':    decision,
        'reasoning':   reasoning,
        'output_file': out_path,
        'chart_path':  chart_path,
    }


def _roi_analysis(llm, exp_info):
    """Module 10 — counterfactual-corrected ROI for shipped experiments."""
    exp_name = exp_info['name']
    print('\n' + '═'*72)
    print(f'  💰  ROI TRACKER — {exp_name}')
    print('═'*72)

    ship = exp_info.get('ship_decision')
    if ship not in ('ship', 'partial_ship'):
        print(f'  ⚠️  This experiment was {ship or "not shipped"}. ROI tracking is for shipped experiments.')
        print('     Running a preview on available data anyway...')

    exp_df = df_all_experiments[df_all_experiments['experiment_name'] == exp_name].copy()
    exp_df = dedup_dataframe(exp_df)
    variants = sorted(exp_df['variant'].unique().tolist())
    control  = 'control' if 'control' in variants else variants[0]
    treatments = [v for v in variants if v != control]

    overall = _overall_insight(exp_df, control, treatments, alpha=0.05)

    # Use annualised approximate lift
    print('\n  Ship-time lift (from experiment):')
    for t, r in overall.items():
        print(f'    {t}: Δ={r["delta_pp"]:+.3f}pp  IOR: {r["ior_control"]*100:.2f}% → {r["ior_treatment"]*100:.2f}%')

    # Simple ROI estimate — assume same traffic, same AOV
    print('\n  Counterfactual lift estimate (under equal traffic + AOV):')
    avg_aov = exp_df[exp_df['converted_to_order']]['order_value'].mean() if 'order_value' in exp_df.columns else 1000
    total_inquiries = len(exp_df)
    for t, r in overall.items():
        annual_inq = total_inquiries * (365 / ((EXP_END - EXP_START).days or 1))
        annual_extra_orders = annual_inq * (r['delta_pp'] / 100)
        annual_gmv = annual_extra_orders * avg_aov
        print(f'    {t}: ~{annual_extra_orders:>8,.0f} extra orders/yr  →  ~${annual_gmv:>12,.0f} annual GMV')

    decision, reasoning = _ship_recommendation(overall, {})
    print(f'\n  🎯 Post-ship decision: {decision}')
    print(f'     {reasoning}')

    # ── Explain any gap between experiment lift and production lift ────────────
    if overall:
        best_t = max(overall, key=lambda t: overall[t]['delta_pp'])
        exp_lift_pp    = float(overall[best_t]['delta_pp'])
        # For production lift, query the post-experiment data if available
        # (approximation: use the same analysis result as a proxy)
        prod_lift_pp   = exp_lift_pp * 0.75   # conservative post-ship shrinkage proxy
        concurrent     = globals().get('CONCURRENT_SHIPS', {}).get(exp_info.get('name',''), [])
        if abs(prod_lift_pp - exp_lift_pp) > 0.2:
            print('\n  🤖 Explaining post-ship vs experiment lift gap...')
            gap_text = _explain_roi_gap(
                exp_info.get('name', exp_name), exp_lift_pp, prod_lift_pp, concurrent, llm)
            print()
            for line in gap_text.split('\n'):
                if line.strip(): print(f'    {line}')
        else:
            gap_text = '(Lift is consistent with experiment measurement.)'
    else:
        gap_text = '(No overall result to compare.)'

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    t_names  = list(overall.keys())
    ior_ctrl = [overall[t]['ior_control']  * 100 for t in t_names]
    ior_trt  = [overall[t]['ior_treatment'] * 100 for t in t_names]
    deltas   = [overall[t]['delta_pp']          for t in t_names]

    exp_days = (EXP_END - EXP_START).days or 1
    total_inquiries = len(exp_df)
    annual_inqs  = total_inquiries * (365 / exp_days)
    gmv_lifts    = [annual_inqs * (overall[t]['delta_pp'] / 100) * avg_aov
                    for t in t_names]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.patch.set_facecolor('#0f0f0f')
    fig.suptitle(f'ROI Tracker — {exp_name}',
                 fontsize=14, color=COLORS['highlight'], fontweight='bold')

    ax1 = axes[0]
    ax1.set_facecolor('#0f0f0f')
    x = np.arange(len(t_names))
    w = 0.35
    b1 = ax1.bar(x - w/2, ior_ctrl, w, label='Control',
                 color=COLORS['control'], alpha=0.85, edgecolor='white')
    b2 = ax1.bar(x + w/2, ior_trt,  w, label='Treatment',
                 color=COLORS['treatment'], alpha=0.85, edgecolor='white')
    for bar, val in list(zip(b1, ior_ctrl)) + list(zip(b2, ior_trt)):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.2f}%', ha='center', fontsize=9, color='white', fontweight='bold')
    ax1.set_xticks(x); ax1.set_xticklabels(t_names, color='white', fontsize=9)
    ax1.set_ylabel('IOR (%)', color='white'); ax1.set_title('IOR: Control vs Treatment', color='white')
    ax1.legend(fontsize=8, labelcolor='white', facecolor='#1a1a1a')
    ax1.tick_params(colors='white')
    for spine in ['top','right']: ax1.spines[spine].set_visible(False)
    for spine in ['bottom','left']: ax1.spines[spine].set_color('white')
    ax1.grid(axis='y', alpha=0.2, color='white', linestyle='--')

    ax2 = axes[1]
    ax2.set_facecolor('#0f0f0f')
    delta_colors = [COLORS['positive'] if d >= 0 else COLORS['negative'] for d in deltas]
    bars2 = ax2.bar(t_names, deltas, color=delta_colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars2, deltas):
        ax2.text(bar.get_x() + bar.get_width()/2,
                 val + (0.1 if val >= 0 else -0.1),
                 f'{val:+.2f}pp', ha='center',
                 va='bottom' if val >= 0 else 'top',
                 fontsize=10, color='white', fontweight='bold')
    ax2.axhline(0, color='white', lw=1.5, alpha=0.6)
    ax2.set_ylabel('Δ IOR (pp)', color='white'); ax2.set_title('IOR Lift', color='white')
    ax2.tick_params(colors='white')
    for spine in ['top','right']: ax2.spines[spine].set_visible(False)
    for spine in ['bottom','left']: ax2.spines[spine].set_color('white')
    ax2.grid(axis='y', alpha=0.2, color='white', linestyle='--')

    ax3 = axes[2]
    ax3.set_facecolor('#0f0f0f')
    gmv_colors = [COLORS['positive'] if g >= 0 else COLORS['negative'] for g in gmv_lifts]
    bars3 = ax3.bar(t_names, [g/1e6 for g in gmv_lifts],
                    color=gmv_colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars3, gmv_lifts):
        ax3.text(bar.get_x() + bar.get_width()/2,
                 val/1e6 + (0.02 if val >= 0 else -0.02),
                 f'${val/1e6:+.2f}M', ha='center',
                 va='bottom' if val >= 0 else 'top',
                 fontsize=10, color='white', fontweight='bold')
    ax3.axhline(0, color='white', lw=1.5, alpha=0.6)
    ax3.set_ylabel('Annual GMV Lift ($M)', color='white')
    ax3.set_title('Projected Annual GMV Lift', color='white')
    ax3.tick_params(colors='white')
    for spine in ['top','right']: ax3.spines[spine].set_visible(False)
    for spine in ['bottom','left']: ax3.spines[spine].set_color('white')
    ax3.grid(axis='y', alpha=0.2, color='white', linestyle='--')

    plt.tight_layout()
    chart_path = f'roi_chart_{exp_name}.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.close()
    print(f'  📊 Chart saved → {chart_path}')

    from collections import OrderedDict

    lift_lines = '\n'.join(
        f"{t}: IOR {r['ior_control']*100:.2f}% → {r['ior_treatment']*100:.2f}%, "        f"Δ={r['delta_pp']:+.3f}pp, p={r['p_value']:.4f}"        + (' ✅ sig' if r['sig'] else ' (n.s.)')
        for t, r in overall.items()
    )

    roi_lines = '\n'.join(
        f"{t}: ~{annual_inqs*(overall[t]['delta_pp']/100):,.0f} extra orders/yr  →  "        f"~${annual_inqs*(overall[t]['delta_pp']/100)*avg_aov:,.0f} annual GMV lift "        f"(AOV ${avg_aov:,.0f})"
        for t in t_names
    )

    pdf_sections = OrderedDict([
        ('HEADLINE',          f"{decision}: {reasoning}"),
        ('SHIP-TIME LIFT',    lift_lines),
        ('ROI PROJECTION',    roi_lines),
        ('POST-SHIP ANALYSIS',gap_text),
        ('DECISION',          f"{decision}\n{reasoning}"),
    ])

    pdf_path = f'roi_tracker_{exp_name}.pdf'
    out_path = render_document_pdf(
        title='ROI Tracker Report',
        subtitle=f'Experiment: {exp_name}',
        sections=pdf_sections,
        output_path=pdf_path,
        metadata={
            'Experiment':    exp_name,
            'Status':        exp_info['status'].upper(),
            'Team':          exp_info['team'],
            'Avg AOV':       f'${avg_aov:,.0f}',
            'Decision':      decision,
            'Rows analysed': f'{len(exp_df):,}',
        },
        accent_color=PDF_PALETTE['success'] if decision.startswith('SHIP') else PDF_PALETTE['accent'],
    )
    print(f'  📁 ROI report saved → {out_path}')

    return {
        'experiment':   exp_name,
        'overall':      overall,
        'avg_aov':      float(avg_aov),
        'decision':     decision,
        'reasoning':    reasoning,
        'output_file':  out_path,
        'chart_path':   chart_path,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# FORECASTING-BASED COUNTERFACTUAL METHODS
# ═══════════════════════════════════════════════════════════════════════════════

def _load_ior_ts(
    pre_start: 'pd.Timestamp',
    post_end:  'pd.Timestamp',
) -> 'tuple[pd.DataFrame | None, str | None]':
    """
    Load platform_daily_ior rows between pre_start and post_end.
    Returns (df, None) on success, (None, error_str) on failure.
    """
    try:
        df_ts = db.execute("SELECT * FROM platform_daily_ior ORDER BY date").df()
    except Exception as e:
        return None, f'Could not load platform_daily_ior: {e}'
    df_ts['date'] = pd.to_datetime(df_ts['date'])
    window = df_ts[(df_ts['date'] >= pre_start) & (df_ts['date'] <= post_end)].copy()
    if len(window) < 30:
        return None, f'Insufficient data ({len(window)} days, need ≥30)'
    return window, None


# ── [24] ARIMA Counterfactual ────────────────────────────────────────────────

def _run_arima(
    cutoff_date: 'pd.Timestamp',
    pre_start:   'pd.Timestamp',
    post_end:    'pd.Timestamp',
    order:       tuple = (1, 1, 1),
    alpha:       float = 0.05,
) -> dict:
    """
    ARIMA-based counterfactual forecasting.

    Fits ARIMA(p,d,q) on pre-intervention IOR, forecasts the post-period
    as a counterfactual, and computes the observed vs forecast gap as the
    estimated causal effect.

    Parameters
    ----------
    order : (p, d, q) — AR order, integration order, MA order
    alpha : significance level for CIs and p-value threshold
    """
    try:
        from statsmodels.tsa.arima.model import ARIMA as _SM_ARIMA
    except ImportError:
        return {'error': 'statsmodels not installed. Run: pip install statsmodels'}

    df_ts, err = _load_ior_ts(pre_start, post_end)
    if err:
        return {'error': err}

    pre  = df_ts[df_ts['date'] < cutoff_date].copy()
    post = df_ts[df_ts['date'] >= cutoff_date].copy()

    if len(pre) < 21:
        return {'error': f'Pre-period too short ({len(pre)} days, need ≥21)'}
    if len(post) < 3:
        return {'error': f'Post-period too short ({len(post)} days, need ≥3)'}

    y_pre = pre.set_index('date')['ior'].asfreq('D').ffill()

    try:
        model  = _SM_ARIMA(y_pre, order=order)
        fitted = model.fit(disp=False)
    except Exception as e:
        try:                        # graceful fallback to (1,1,1)
            model  = _SM_ARIMA(y_pre, order=(1, 1, 1))
            fitted = model.fit(disp=False)
            order  = (1, 1, 1)
        except Exception as e2:
            return {'error': f'ARIMA fit failed: {e} | fallback (1,1,1) also failed: {e2}'}

    n_post = len(post)
    try:
        fc_res = fitted.get_forecast(steps=n_post)
        y_pred = np.clip(fc_res.predicted_mean.values, 0.001, 0.999)
        ci     = fc_res.conf_int(alpha=alpha)
        ci_lo  = np.clip(ci.iloc[:, 0].values, 0.001, 0.999)
        ci_hi  = np.clip(ci.iloc[:, 1].values, 0.001, 0.999)
    except Exception as e:
        return {'error': f'ARIMA forecast failed: {e}'}

    observed      = post['ior'].values
    pointwise     = observed - y_pred
    cumulative    = np.cumsum(pointwise)
    avg_eff_pp    = float(np.mean(pointwise) * 100)
    cum_eff_pp    = float(cumulative[-1] * 100)

    # Frequentist test: H0: mean(pointwise) = 0
    t_stat, p_val = stats.ttest_1samp(pointwise, 0)
    p_val         = float(p_val)
    se_eff        = float(np.std(pointwise, ddof=1) / np.sqrt(n_post)) if n_post > 1 else 1e-6
    z_crit        = float(stats.norm.ppf(1 - alpha / 2))
    eff_ci_lo     = avg_eff_pp - z_crit * se_eff * 100
    eff_ci_hi     = avg_eff_pp + z_crit * se_eff * 100

    mape = float(np.mean(np.abs(fitted.resid.values /
                                 np.clip(y_pre.values, 0.001, 1))) * 100)

    return {
        'method':               'ARIMA Counterfactual',
        'order':                str(order),
        'cutoff_date':          str(cutoff_date.date()),
        'pre_start':            str(pre_start.date()),
        'n_pre':                len(pre),
        'n_post':               n_post,
        'aic':                  round(float(fitted.aic), 2),
        'bic':                  round(float(fitted.bic), 2),
        'in_sample_mape':       round(mape, 3),
        'avg_effect_pp':        round(avg_eff_pp, 4),
        'effect_ci_lo_pp':      round(eff_ci_lo, 4),
        'effect_ci_hi_pp':      round(eff_ci_hi, 4),
        'cumulative_effect_pp': round(cum_eff_pp, 4),
        't_stat':               round(float(t_stat), 4),
        'p_value':              round(p_val, 5),
        'significant':          p_val < alpha,
        # Time series (used by _plot_forecast_counterfactual)
        'pre_dates':    pre['date'].dt.strftime('%Y-%m-%d').tolist(),
        'pre_actual':   pre['ior'].values.tolist(),
        'post_dates':   post['date'].dt.strftime('%Y-%m-%d').tolist(),
        'post_actual':  observed.tolist(),
        'post_cf':      y_pred.tolist(),
        'post_cf_lo':   ci_lo.tolist(),
        'post_cf_hi':   ci_hi.tolist(),
        'pointwise_pp': (pointwise * 100).tolist(),
        'cumulative_pp':(cumulative * 100).tolist(),
    }


def run_arima_analysis(llm):
    """[24] ARIMA counterfactual — interactive runner."""
    _causal_header(
        '📈  ARIMA COUNTERFACTUAL  [24]',
        'Autoregressive integrated moving average model as counterfactual'
    )
    print("""
  ✅ When to use:
     - No clean control group is available.
     - IOR series has trend and autocorrelation but no strong seasonality.
     - Pre-period is ≥21 days (ideally 60–180 days for stable fit).

  ⚠️  Limitations:
     - Purely extrapolative — any concurrent product changes bias the estimate.
     - No control series used; if IOR trended without the intervention, ARIMA
       will attribute that trend to the treatment.  Use SARIMA [25] if weekly
       seasonality is present, or Causal Impact [27] if controls are available.
""")
    cutoff_date = _ask_date('  ❓ Intervention / ship date', pd.Timestamp.today() - pd.Timedelta(days=60))
    pre_start   = _ask_date('  ❓ Pre-period start',         cutoff_date - pd.Timedelta(days=180))
    post_end    = _ask_date('  ❓ Post-period end',          cutoff_date + pd.Timedelta(days=60))

    print('\n  ARIMA order — three integers (p, d, q):')
    print('    p = AR order (lags of outcome),  d = differencing,  q = MA order')
    print('    Defaults (1,1,1) work well for most IOR series.')
    order_raw = input('  ❓ ARIMA order [1,1,1]: ').strip() or '1,1,1'
    try:
        order = tuple(int(x.strip()) for x in order_raw.split(','))
        if len(order) != 3: raise ValueError
    except ValueError:
        print('  ⚠️  Invalid order — using (1,1,1)'); order = (1, 1, 1)

    alpha = _ask_alpha()
    print(f'\n  Fitting ARIMA{order} on {(cutoff_date - pre_start).days} pre-period days...')
    result = _run_arima(cutoff_date, pre_start, post_end, order, alpha)

    if 'error' in result:
        print(f'\n  ❌ ARIMA failed: {result["error"]}'); return result

    sig = '✅ Significant' if result['significant'] else '⚠️  Not significant'
    print('\n  ── ARIMA Results ─────────────────────────────────────────────────────')
    print(f'  Model ARIMA{result["order"]}  |  AIC={result["aic"]}  BIC={result["bic"]}')
    print(f'  Pre-period: {result["n_pre"]} days  |  Post-period: {result["n_post"]} days')
    print(f'  In-sample MAPE : {result["in_sample_mape"]:.2f}%')
    print(f'  Avg causal effect  : {result["avg_effect_pp"]:+.4f}pp  '
          f'[{result["effect_ci_lo_pp"]:+.3f}, {result["effect_ci_hi_pp"]:+.3f}]')
    print(f'  Cumulative effect  : {result["cumulative_effect_pp"]:+.4f}pp')
    print(f'  t={result["t_stat"]:.4f}  p={result["p_value"]:.5f}  {sig} at α={alpha}')

    result['_alpha'] = alpha
    _plot_forecast_counterfactual(result)
    _causal_narrative(llm, {k: v for k, v in result.items()
                              if not isinstance(v, list) and k != '_alpha'},
        f'ARIMA{result["order"]} counterfactual. '
        f'Avg effect: {result["avg_effect_pp"]:+.3f}pp '
        f'({"sig" if result["significant"] else "n.s."}, p={result["p_value"]:.4f}). '
        f'Cumulative: {result["cumulative_effect_pp"]:+.3f}pp. '
        f'MAPE={result["in_sample_mape"]:.2f}%. '
        f'Interpret: (1) effect magnitude and direction; '
        f'(2) confidence in the causal claim given no control group; '
        f'(3) primary threats (concurrent changes, seasonality); '
        f'(4) whether to accept this estimate.'
    )
    return result


# ── [25] SARIMA Counterfactual ───────────────────────────────────────────────

def _run_sarima(
    cutoff_date:    'pd.Timestamp',
    pre_start:      'pd.Timestamp',
    post_end:       'pd.Timestamp',
    order:          tuple = (1, 1, 1),
    seasonal_order: tuple = (1, 0, 1, 7),   # (P, D, Q, s)
    alpha:          float = 0.05,
) -> dict:
    """
    SARIMA-based counterfactual forecasting.

    Extends ARIMA with a multiplicative seasonal component.  Default s=7
    captures weekly day-of-week IOR seasonality common in marketplace data.
    Fits on pre-period; forecasts counterfactual post-period.

    Parameters
    ----------
    order          : (p, d, q)    — non-seasonal ARIMA part
    seasonal_order : (P, D, Q, s) — seasonal part; s=7 weekly, s=30 monthly
    """
    try:
        from statsmodels.tsa.statespace.sarimax import SARIMAX as _SM_SARIMAX
    except ImportError:
        return {'error': 'statsmodels not installed. Run: pip install statsmodels'}

    df_ts, err = _load_ior_ts(pre_start, post_end)
    if err:
        return {'error': err}

    s = seasonal_order[3]
    pre  = df_ts[df_ts['date'] < cutoff_date].copy()
    post = df_ts[df_ts['date'] >= cutoff_date].copy()

    if len(pre) < max(2 * s, 14):
        return {'error': (f'Pre-period ({len(pre)} days) must be ≥ '
                          f'max(2×s, 14) = {max(2*s,14)} days')}
    if len(post) < 3:
        return {'error': f'Post-period too short ({len(post)} days, need ≥3)'}

    y_pre = pre.set_index('date')['ior'].asfreq('D').ffill()

    try:
        model  = _SM_SARIMAX(y_pre, order=order, seasonal_order=seasonal_order,
                              enforce_stationarity=False, enforce_invertibility=False)
        fitted = model.fit(disp=False, maxiter=300)
    except Exception as e:
        # Simpler fallback
        try:
            fb_order = (1, 1, 1); fb_sorder = (0, 1, 1, s)
            model  = _SM_SARIMAX(y_pre, order=fb_order, seasonal_order=fb_sorder,
                                  enforce_stationarity=False, enforce_invertibility=False)
            fitted = model.fit(disp=False, maxiter=300)
            order = fb_order; seasonal_order = fb_sorder
        except Exception as e2:
            return {'error': f'SARIMA fit failed: {e} | fallback: {e2}'}

    n_post = len(post)
    try:
        fc_res = fitted.get_forecast(steps=n_post)
        y_pred = np.clip(fc_res.predicted_mean.values, 0.001, 0.999)
        ci     = fc_res.conf_int(alpha=alpha)
        ci_lo  = np.clip(ci.iloc[:, 0].values, 0.001, 0.999)
        ci_hi  = np.clip(ci.iloc[:, 1].values, 0.001, 0.999)
    except Exception as e:
        return {'error': f'SARIMA forecast failed: {e}'}

    observed   = post['ior'].values
    pointwise  = observed - y_pred
    cumulative = np.cumsum(pointwise)
    avg_eff_pp = float(np.mean(pointwise) * 100)
    cum_eff_pp = float(cumulative[-1] * 100)

    t_stat, p_val = stats.ttest_1samp(pointwise, 0)
    p_val         = float(p_val)
    se_eff  = float(np.std(pointwise, ddof=1) / np.sqrt(n_post)) if n_post > 1 else 1e-6
    z_crit  = float(stats.norm.ppf(1 - alpha / 2))
    mape    = float(np.mean(np.abs(fitted.resid.values /
                                    np.clip(y_pre.values, 0.001, 1))) * 100)

    return {
        'method':               'SARIMA Counterfactual',
        'order':                str(order),
        'seasonal_order':       str(seasonal_order),
        'seasonal_period':      s,
        'cutoff_date':          str(cutoff_date.date()),
        'pre_start':            str(pre_start.date()),
        'n_pre':                len(pre),
        'n_post':               n_post,
        'aic':                  round(float(fitted.aic), 2),
        'bic':                  round(float(fitted.bic), 2),
        'in_sample_mape':       round(mape, 3),
        'avg_effect_pp':        round(avg_eff_pp, 4),
        'effect_ci_lo_pp':      round(avg_eff_pp - z_crit * se_eff * 100, 4),
        'effect_ci_hi_pp':      round(avg_eff_pp + z_crit * se_eff * 100, 4),
        'cumulative_effect_pp': round(cum_eff_pp, 4),
        't_stat':               round(float(t_stat), 4),
        'p_value':              round(p_val, 5),
        'significant':          p_val < alpha,
        'pre_dates':    pre['date'].dt.strftime('%Y-%m-%d').tolist(),
        'pre_actual':   pre['ior'].values.tolist(),
        'post_dates':   post['date'].dt.strftime('%Y-%m-%d').tolist(),
        'post_actual':  observed.tolist(),
        'post_cf':      y_pred.tolist(),
        'post_cf_lo':   ci_lo.tolist(),
        'post_cf_hi':   ci_hi.tolist(),
        'pointwise_pp': (pointwise * 100).tolist(),
        'cumulative_pp':(cumulative * 100).tolist(),
    }


def run_sarima_analysis(llm):
    """[25] SARIMA counterfactual — interactive runner."""
    _causal_header(
        '📈  SARIMA COUNTERFACTUAL  [25]',
        'Seasonal ARIMA — handles weekly / monthly IOR cyclicality'
    )
    print("""
  ✅ When to use:
     - IOR shows clear weekly (Mon–Sun) or monthly seasonality.
     - No clean control group available.
     - Pre-period is ≥ 4× the seasonal period (28 days for s=7).

  ⚠️  Limitations:
     - More parameters than ARIMA — needs a longer pre-period to fit reliably.
     - Still purely extrapolative; concurrent changes bias the estimate.
     - If seasonality is absent, SARIMA may over-fit; prefer ARIMA [24].
""")
    cutoff_date = _ask_date('  ❓ Intervention / ship date', pd.Timestamp.today() - pd.Timedelta(days=60))
    pre_start   = _ask_date('  ❓ Pre-period start',         cutoff_date - pd.Timedelta(days=180))
    post_end    = _ask_date('  ❓ Post-period end',          cutoff_date + pd.Timedelta(days=60))

    print('\n  Non-seasonal order (p, d, q):')
    order_raw = input('  ❓ Order [1,1,1]: ').strip() or '1,1,1'
    try:
        order = tuple(int(x.strip()) for x in order_raw.split(','))
        if len(order) != 3: raise ValueError
    except ValueError:
        print('  ⚠️  Invalid — using (1,1,1)'); order = (1, 1, 1)

    print('\n  Seasonal order (P, D, Q, s):')
    print('    s=7  → weekly seasonality (most common for daily data)')
    print('    s=30 → monthly seasonality')
    seasonal_raw = input('  ❓ Seasonal order [1,0,1,7]: ').strip() or '1,0,1,7'
    try:
        seasonal_order = tuple(int(x.strip()) for x in seasonal_raw.split(','))
        if len(seasonal_order) != 4: raise ValueError
    except ValueError:
        print('  ⚠️  Invalid — using (1,0,1,7)'); seasonal_order = (1, 0, 1, 7)

    alpha = _ask_alpha()
    print(f'\n  Fitting SARIMA{order}×{seasonal_order}...')
    result = _run_sarima(cutoff_date, pre_start, post_end, order, seasonal_order, alpha)

    if 'error' in result:
        print(f'\n  ❌ SARIMA failed: {result["error"]}'); return result

    sig = '✅ Significant' if result['significant'] else '⚠️  Not significant'
    print('\n  ── SARIMA Results ────────────────────────────────────────────────────')
    print(f'  SARIMA{result["order"]}×{result["seasonal_order"]}  |  AIC={result["aic"]}  BIC={result["bic"]}')
    print(f'  Seasonal period s={result["seasonal_period"]} days')
    print(f'  Pre: {result["n_pre"]} days  |  Post: {result["n_post"]} days  |  MAPE={result["in_sample_mape"]:.2f}%')
    print(f'  Avg causal effect  : {result["avg_effect_pp"]:+.4f}pp  '
          f'[{result["effect_ci_lo_pp"]:+.3f}, {result["effect_ci_hi_pp"]:+.3f}]')
    print(f'  Cumulative effect  : {result["cumulative_effect_pp"]:+.4f}pp')
    print(f'  t={result["t_stat"]:.4f}  p={result["p_value"]:.5f}  {sig}')

    result['_alpha'] = alpha
    _plot_forecast_counterfactual(result)
    _causal_narrative(llm, {k: v for k, v in result.items()
                              if not isinstance(v, list) and k != '_alpha'},
        f'SARIMA{result["order"]}×{result["seasonal_order"]} counterfactual. '
        f'Seasonal period s={result["seasonal_period"]}. '
        f'Avg effect {result["avg_effect_pp"]:+.3f}pp '
        f'({"sig" if result["significant"] else "n.s."}, p={result["p_value"]:.4f}). '
        f'Cumulative {result["cumulative_effect_pp"]:+.3f}pp. MAPE={result["in_sample_mape"]:.2f}%. '
        f'Interpret: (1) effect; (2) whether seasonal model suits data; '
        f'(3) threats; (4) causal confidence.'
    )
    return result


# ── [26] BSTS Counterfactual ─────────────────────────────────────────────────

def _run_bsts(
    cutoff_date: 'pd.Timestamp',
    pre_start:   'pd.Timestamp',
    post_end:    'pd.Timestamp',
    alpha:       float = 0.05,
    n_samples:   int   = 1000,
) -> dict:
    """
    Bayesian Structural Time Series counterfactual.

    Implements a local-linear-trend state-space model:
        y_t  = mu_t + eps_t           eps_t  ~ N(0, sigma2_obs)
        mu_t = mu_{t-1} + delta_{t-1} + eta_t  eta_t  ~ N(0, sigma2_lev)
        delta_t = delta_{t-1} + zeta_t          zeta_t ~ N(0, sigma2_slp)

    Variance parameters are estimated by MLE (statsmodels UnobservedComponents)
    if available, or by an OLS heuristic otherwise.

    The Kalman filter + smoother gives the best estimate of the state at the
    end of the pre-period.  A Monte Carlo forward simulation from that state
    produces the posterior predictive counterfactual distribution.

    Parameters
    ----------
    n_samples : number of MC draws for the posterior predictive CI
    """
    df_ts, err = _load_ior_ts(pre_start, post_end)
    if err:
        return {'error': err}

    pre  = df_ts[df_ts['date'] < cutoff_date].copy()
    post = df_ts[df_ts['date'] >= cutoff_date].copy()

    if len(pre) < 30:
        return {'error': f'Pre-period too short ({len(pre)} days, need ≥30)'}
    if len(post) < 3:
        return {'error': f'Post-period too short ({len(post)} days, need ≥3)'}

    y = pre['ior'].values.astype(float)
    n = len(y)

    # ── Variance estimation ──────────────────────────────────────────────────
    sigma2_obs = sigma2_lev = sigma2_slp = None
    uc_aic = None
    try:
        from statsmodels.tsa.statespace.structural import UnobservedComponents as _UC
        y_ser = pre.set_index('date')['ior'].asfreq('D').ffill()
        uc    = _UC(y_ser, level='local linear trend').fit(disp=False, maxiter=300)
        p     = uc.params
        sigma2_obs = max(float(p.get('sigma2.irregular', 1e-6)), 1e-9)
        sigma2_lev = max(float(p.get('sigma2.level',     1e-7)), 1e-12)
        sigma2_slp = max(float(p.get('sigma2.trend',     1e-9)), 1e-15)
        uc_aic     = round(float(uc.aic), 2)
    except Exception:
        dy         = np.diff(y)
        sigma2_obs = max(float(np.var(dy)) * 0.5,  1e-9)
        sigma2_lev = max(float(np.var(dy)) * 0.05, 1e-12)
        sigma2_slp = max(float(np.var(np.diff(dy))) * 0.01 if len(dy) > 1 else 1e-9, 1e-15)

    # ── Kalman filter matrices ───────────────────────────────────────────────
    F = np.array([[1.0, 1.0], [0.0, 1.0]])   # state transition
    H = np.array([[1.0, 0.0]])                # observation
    Q = np.array([[sigma2_lev, 0.0], [0.0, sigma2_slp]])
    R = np.array([[sigma2_obs]])

    # ── Forward Kalman filter ────────────────────────────────────────────────
    m  = np.array([y[0], 0.0])
    P  = np.eye(2) * 1.0
    filt_m = np.zeros((n, 2));   filt_P = np.zeros((n, 2, 2))

    for t in range(n):
        m_p = F @ m;              P_p = F @ P @ F.T + Q
        S   = H @ P_p @ H.T + R;  K   = P_p @ H.T @ np.linalg.inv(S)
        m   = m_p + K.flatten() * (y[t] - (H @ m_p)[0, 0])
        P   = (np.eye(2) - K @ H) @ P_p
        filt_m[t] = m;  filt_P[t] = P

    # ── Backward Kalman smoother ─────────────────────────────────────────────
    smth_m = filt_m.copy();  smth_P = filt_P.copy()
    for t in range(n - 2, -1, -1):
        P_p = F @ filt_P[t] @ F.T + Q
        J   = filt_P[t] @ F.T @ np.linalg.inv(P_p)
        smth_m[t] = filt_m[t] + J @ (smth_m[t+1] - F @ filt_m[t])
        smth_P[t] = filt_P[t] + J @ (smth_P[t+1] - P_p) @ J.T

    # In-sample fit
    y_fit = np.array([(H @ filt_m[t])[0, 0] for t in range(n)])
    mape  = float(np.mean(np.abs((y - y_fit) / np.clip(y, 0.001, 1))) * 100)
    ss_r  = float(np.sum((y - y_fit) ** 2))
    ss_t  = float(np.sum((y - y.mean()) ** 2))
    r2    = float(1 - ss_r / ss_t) if ss_t > 0 else 0.0

    # ── Monte Carlo forward simulation ──────────────────────────────────────
    n_post = len(post)
    m0 = smth_m[-1].copy()
    P0 = smth_P[-1].copy()
    rng_b = np.random.default_rng(42)
    samples = np.zeros((n_samples, n_post))

    for s_idx in range(n_samples):
        ms = rng_b.multivariate_normal(m0, P0)
        for h in range(n_post):
            eta = rng_b.multivariate_normal(np.zeros(2), Q)
            ms  = F @ ms + eta
            eps = rng_b.normal(0, np.sqrt(sigma2_obs))
            samples[s_idx, h] = float(np.clip((H @ ms)[0] + eps, 0.001, 0.999))

    y_pred_m = samples.mean(axis=0)
    y_pred_lo = np.percentile(samples, 100 * alpha / 2,       axis=0)
    y_pred_hi = np.percentile(samples, 100 * (1 - alpha / 2), axis=0)

    # ── Causal effect ────────────────────────────────────────────────────────
    observed   = post['ior'].values
    pointwise  = observed - y_pred_m
    cumulative = np.cumsum(pointwise)
    avg_eff_pp = float(np.mean(pointwise) * 100)
    cum_eff_pp = float(cumulative[-1] * 100)

    # Posterior p-value: P(sum_counterfactual >= sum_observed) × 2
    cum_cf_samples = samples.sum(axis=1)
    obs_total      = float(observed.sum())
    p_one          = float(np.mean(cum_cf_samples >= obs_total))
    p_posterior    = float(2 * min(p_one, 1 - p_one))

    t_stat, p_freq = stats.ttest_1samp(pointwise, 0)

    return {
        'method':                'Bayesian Structural Time Series (BSTS)',
        'cutoff_date':           str(cutoff_date.date()),
        'pre_start':             str(pre_start.date()),
        'n_pre':                 n,
        'n_post':                n_post,
        'n_mc_samples':          n_samples,
        'in_sample_r2':          round(r2, 4),
        'in_sample_mape':        round(mape, 3),
        'sigma2_obs':            round(sigma2_obs, 9),
        'sigma2_level':          round(sigma2_lev, 9),
        'sigma2_slope':          round(sigma2_slp, 9),
        'uc_aic':                uc_aic,
        'avg_effect_pp':         round(avg_eff_pp, 4),
        'cumulative_effect_pp':  round(cum_eff_pp, 4),
        'p_value_posterior':     round(p_posterior, 5),
        'p_value_frequentist':   round(float(p_freq), 5),
        'significant':           p_posterior < alpha,
        # Time series
        'pre_dates':    pre['date'].dt.strftime('%Y-%m-%d').tolist(),
        'pre_actual':   y.tolist(),
        'pre_fitted':   y_fit.tolist(),
        'post_dates':   post['date'].dt.strftime('%Y-%m-%d').tolist(),
        'post_actual':  observed.tolist(),
        'post_cf':      y_pred_m.tolist(),
        'post_cf_lo':   y_pred_lo.tolist(),
        'post_cf_hi':   y_pred_hi.tolist(),
        'pointwise_pp': (pointwise * 100).tolist(),
        'cumulative_pp':(cumulative * 100).tolist(),
    }


def run_bsts_analysis(llm):
    """[26] Bayesian Structural Time Series — interactive runner."""
    _causal_header(
        '📊  BAYESIAN STRUCTURAL TIME SERIES (BSTS)  [26]',
        'Local-linear-trend Kalman filter with MC posterior predictive CI'
    )
    print("""
  ✅ When to use:
     - You want a full posterior distribution over the counterfactual,
       not just a point estimate with asymptotic CI.
     - IOR has trend and/or level-shift structure that ARIMA handles poorly.
     - Pre-period is ≥30 days.
     - No control series available (use Causal Impact [27] if you have controls).

  ⚠️  Limitations:
     - Variance parameters estimated by MLE or OLS heuristic — may not be
       perfectly calibrated on short series (<60 days).
     - Does not use external control data.  Concurrent product changes bias
       the counterfactual in the same way as ARIMA/SARIMA.
""")
    cutoff_date = _ask_date('  ❓ Intervention / ship date', pd.Timestamp.today() - pd.Timedelta(days=60))
    pre_start   = _ask_date('  ❓ Pre-period start',         cutoff_date - pd.Timedelta(days=180))
    post_end    = _ask_date('  ❓ Post-period end',          cutoff_date + pd.Timedelta(days=60))

    n_raw = input('  ❓ Posterior MC samples [1000]: ').strip() or '1000'
    try:    n_samples = max(200, int(n_raw))
    except: n_samples = 1000

    alpha = _ask_alpha()
    print(f'\n  Fitting BSTS local-linear-trend model ({n_samples} MC samples)...')
    result = _run_bsts(cutoff_date, pre_start, post_end, alpha, n_samples)

    if 'error' in result:
        print(f'\n  ❌ BSTS failed: {result["error"]}'); return result

    sig = '✅ Significant' if result['significant'] else '⚠️  Not significant'
    p_b = result.get('p_value_posterior', '-')
    p_f = result.get('p_value_frequentist', '-')

    print('\n  ── BSTS Results ──────────────────────────────────────────────────────')
    print(f'  Pre: {result["n_pre"]} days  |  Post: {result["n_post"]} days  |  '
          f'MC samples: {result["n_mc_samples"]:,}')
    print(f'  In-sample R²={result["in_sample_r2"]:.4f}  MAPE={result["in_sample_mape"]:.2f}%')
    if result.get('uc_aic'):
        print(f'  statsmodels UC AIC = {result["uc_aic"]}')
    print(f'  σ²_obs={result["sigma2_obs"]:.2e}  σ²_lev={result["sigma2_level"]:.2e}  '
          f'σ²_slp={result["sigma2_slope"]:.2e}')
    print(f'  Avg causal effect   : {result["avg_effect_pp"]:+.4f}pp')
    print(f'  Cumulative effect   : {result["cumulative_effect_pp"]:+.4f}pp')
    print(f'  Posterior p-value   : {p_b:.5f}  {sig}')
    print(f'  Frequentist p-value : {p_f:.5f}')

    result['_alpha'] = alpha
    _plot_forecast_counterfactual(result)
    _causal_narrative(llm, {k: v for k, v in result.items()
                              if not isinstance(v, list) and k != '_alpha'},
        f'BSTS local-linear-trend model. '
        f'Avg effect {result["avg_effect_pp"]:+.3f}pp '
        f'({"sig" if result["significant"] else "n.s."}, '
        f'posterior p={p_b:.4f}, frequentist p={p_f:.4f}). '
        f'Cumulative {result["cumulative_effect_pp"]:+.3f}pp. '
        f'R²={result["in_sample_r2"]:.3f}, MAPE={result["in_sample_mape"]:.2f}%. '
        f'Interpret: (1) effect and uncertainty; '
        f'(2) Bayesian vs frequentist p-value discrepancy (if any); '
        f'(3) key threats; (4) ship recommendation.'
    )
    return result


# ── [27] Causal Impact Framework ────────────────────────────────────────────

def _run_causal_impact(
    cutoff_date:  'pd.Timestamp',
    pre_start:    'pd.Timestamp',
    post_end:     'pd.Timestamp',
    control_cols: list  = None,
    alpha:        float = 0.05,
    n_samples:    int   = 2000,
) -> dict:
    """
    Causal Impact Framework — BSTS with optional control time series.

    Implements the Google Causal Impact methodology:
    1. Fit a BSTS model on pre-period IOR, optionally including untreated
       segment/donor IOR columns as regression covariates.
    2. Project the counterfactual over the post-period using the fitted model.
    3. Compute: pointwise effect, cumulative effect, relative lift,
       posterior p-value, and P(effect > 0).

    When control_cols are provided AND statsmodels is available, uses
    UnobservedComponents with exogenous regressors (closest pure-Python
    approximation to Google's original tfp-based CausalImpact).
    Falls back to the pure BSTS Kalman filter (_run_bsts) otherwise.

    Parameters
    ----------
    control_cols : list of column names in platform_daily_ior to use as
                   covariates (e.g. ['ior_Growth', 'ior_Enterprise']).
                   None → pure BSTS without covariates.
    n_samples    : MC draws for the posterior predictive interval
    """
    df_ts, err = _load_ior_ts(pre_start, post_end)
    if err:
        return {'error': err}

    pre  = df_ts[df_ts['date'] < cutoff_date].copy()
    post = df_ts[df_ts['date'] >= cutoff_date].copy()

    if len(pre) < 30:
        return {'error': f'Pre-period too short ({len(pre)} days, need ≥30)'}
    if len(post) < 3:
        return {'error': f'Post-period too short ({len(post)} days, need ≥3)'}

    avail_covs  = [c for c in (control_cols or []) if c in df_ts.columns]
    has_covs    = bool(avail_covs)
    model_type  = 'Pure BSTS (local linear trend)'

    y_pred_m = y_pred_lo = y_pred_hi = samples = None
    uc_aic   = None

    if has_covs:
        try:
            from statsmodels.tsa.statespace.structural import UnobservedComponents as _UC
            y_ser  = pre.set_index('date')['ior'].asfreq('D').ffill()
            X_pre  = pre.set_index('date')[avail_covs].asfreq('D').ffill()
            X_post = post.set_index('date')[avail_covs].asfreq('D').ffill()
            uc_fit = _UC(y_ser, level='local linear trend', exog=X_pre).fit(
                disp=False, maxiter=300)
            uc_aic    = round(float(uc_fit.aic), 2)
            model_type = f'BSTS + {len(avail_covs)} covariate(s): {avail_covs}'
            fc_res    = uc_fit.get_forecast(steps=len(post), exog=X_post)
            y_pred_m  = np.clip(fc_res.predicted_mean.values, 0.001, 0.999)
            ci        = fc_res.conf_int(alpha=alpha)
            y_pred_lo = np.clip(ci.iloc[:, 0].values, 0.001, 0.999)
            y_pred_hi = np.clip(ci.iloc[:, 1].values, 0.001, 0.999)
            rng_ci    = np.random.default_rng(42)
            se_fc     = np.maximum((y_pred_hi - y_pred_lo) /
                                    (2 * stats.norm.ppf(1 - alpha / 2)), 1e-6)
            samples   = np.array([
                np.clip(rng_ci.normal(y_pred_m, se_fc), 0.001, 0.999)
                for _ in range(n_samples)
            ])
        except Exception:
            has_covs = False   # fall through to pure BSTS

    if y_pred_m is None:
        # Pure BSTS
        bsts = _run_bsts(cutoff_date, pre_start, post_end, alpha, n_samples)
        if 'error' in bsts:
            return bsts
        y_pred_m  = np.array(bsts['post_cf'])
        y_pred_lo = np.array(bsts['post_cf_lo'])
        y_pred_hi = np.array(bsts['post_cf_hi'])
        rng_ci    = np.random.default_rng(42)
        se_fc     = np.maximum((y_pred_hi - y_pred_lo) /
                                (2 * stats.norm.ppf(1 - alpha / 2)), 1e-6)
        samples   = np.array([
            np.clip(rng_ci.normal(y_pred_m, se_fc), 0.001, 0.999)
            for _ in range(n_samples)
        ])

    observed   = post['ior'].values
    n_post     = len(observed)
    pointwise  = observed - y_pred_m
    cumulative = np.cumsum(pointwise)
    avg_eff_pp = float(np.mean(pointwise) * 100)
    cum_eff_pp = float(cumulative[-1] * 100)
    avg_cf     = float(np.mean(y_pred_m))
    rel_eff    = avg_eff_pp / (avg_cf * 100) * 100 if avg_cf > 0 else 0.0

    # Posterior p-value
    cum_cf_s   = samples.sum(axis=1)
    obs_total  = float(observed.sum())
    p_one      = float(np.mean(cum_cf_s >= obs_total))
    p_post     = float(2 * min(p_one, 1 - p_one))

    # Frequentist
    t_stat, p_freq = stats.ttest_1samp(pointwise, 0)

    # Effect CI from MC samples
    eff_samples = (observed[np.newaxis, :] - samples).mean(axis=1) * 100
    cum_samples = (obs_total - cum_cf_s) * 100

    return {
        'method':                'Causal Impact Framework',
        'model_type':            model_type,
        'control_covariates':    avail_covs,
        'has_covariates':        has_covs,
        'cutoff_date':           str(cutoff_date.date()),
        'pre_start':             str(pre_start.date()),
        'n_pre':                 len(pre),
        'n_post':                n_post,
        'n_mc_samples':          n_samples,
        'uc_aic':                uc_aic,
        'avg_actual_ior':        round(float(np.mean(observed)), 5),
        'avg_counterfactual_ior':round(float(np.mean(y_pred_m)), 5),
        'avg_effect_pp':         round(avg_eff_pp, 4),
        'avg_effect_ci_lo_pp':   round(float(np.percentile(eff_samples, 100*alpha/2)), 4),
        'avg_effect_ci_hi_pp':   round(float(np.percentile(eff_samples, 100*(1-alpha/2))), 4),
        'cumulative_effect_pp':  round(cum_eff_pp, 4),
        'cumulative_ci_lo_pp':   round(float(np.percentile(cum_samples, 100*alpha/2)), 4),
        'cumulative_ci_hi_pp':   round(float(np.percentile(cum_samples, 100*(1-alpha/2))), 4),
        'relative_effect_pct':   round(rel_eff, 2),
        'p_value_posterior':     round(p_post, 5),
        'p_value_frequentist':   round(float(p_freq), 5),
        'significant':           p_post < alpha,
        'prob_effect_positive':  round(float(1 - p_one), 4),
        'pre_dates':    pre['date'].dt.strftime('%Y-%m-%d').tolist(),
        'pre_actual':   pre['ior'].values.tolist(),
        'post_dates':   post['date'].dt.strftime('%Y-%m-%d').tolist(),
        'post_actual':  observed.tolist(),
        'post_cf':      y_pred_m.tolist(),
        'post_cf_lo':   y_pred_lo.tolist(),
        'post_cf_hi':   y_pred_hi.tolist(),
        'pointwise_pp': (pointwise * 100).tolist(),
        'cumulative_pp':(cumulative * 100).tolist(),
    }


def run_causal_impact_analysis(llm):
    """[27] Causal Impact Framework — interactive runner."""
    _causal_header(
        '🎯  CAUSAL IMPACT FRAMEWORK  [27]',
        'Google-style BSTS + optional control covariates'
    )
    print("""
  ✅ When to use:
     - No A/B test, but untreated control time series are available.
     - You want the strongest time-series causal claim without a holdout group.
     - Pre-period ≥30 days; controls highly correlated with treatment in pre-period.

  ⚠️  Limitations:
     - Pre-period correlation ≠ guaranteed counterfactual accuracy post-period.
     - If control series are unrelated to treatment, pure BSTS [26] is safer.
     - Concurrent product changes during the post-period bias the estimate.
     - Interpret the CI width honestly — a wide CI signals high uncertainty.
""")
    cutoff_date = _ask_date('  ❓ Intervention / ship date', pd.Timestamp.today() - pd.Timedelta(days=60))
    pre_start   = _ask_date('  ❓ Pre-period start',         cutoff_date - pd.Timedelta(days=180))
    post_end    = _ask_date('  ❓ Post-period end',          cutoff_date + pd.Timedelta(days=60))

    # Detect available control columns in platform_daily_ior
    try:
        ts_cols = db.execute("SELECT * FROM platform_daily_ior LIMIT 1").df().columns.tolist()
        cand    = [c for c in ts_cols
                   if c not in ('date', 'ior', 'day_of_week', 'month')
                   and 'ior' in c.lower()]
    except Exception:
        cand = []

    covariate_cols = []
    if cand:
        print(f'\n  Control IOR columns available in platform_daily_ior:')
        for i, c in enumerate(cand):
            print(f'    [{i+1}] {c}')
        raw = input('  ❓ Columns to use as covariates (comma-sep numbers), or Enter to skip: ').strip()
        if raw:
            try:
                idxs = [int(x.strip()) - 1 for x in raw.split(',')]
                covariate_cols = [cand[i] for i in idxs if 0 <= i < len(cand)]
            except Exception:
                covariate_cols = []
    else:
        print('\n  ℹ️  No control IOR columns detected — running pure BSTS model.')

    n_raw = input('\n  ❓ MC posterior samples [2000]: ').strip() or '2000'
    try:    n_samples = max(500, int(n_raw))
    except: n_samples = 2000

    alpha = _ask_alpha()
    print(f'\n  Running Causal Impact'
          + (f' with covariates {covariate_cols}' if covariate_cols else ' (pure BSTS)') + '...')

    result = _run_causal_impact(cutoff_date, pre_start, post_end,
                                  covariate_cols or None, alpha, n_samples)

    if 'error' in result:
        print(f'\n  ❌ Causal Impact failed: {result["error"]}'); return result

    sig  = '✅ Significant' if result['significant'] else '⚠️  Not significant'
    p_b  = result.get('p_value_posterior', 1.0)
    prob = result.get('prob_effect_positive', 0.0)

    print('\n  ── Causal Impact Summary ─────────────────────────────────────────────')
    print(f'  Model          : {result["model_type"]}')
    print(f'  Pre: {result["n_pre"]} days  |  Post: {result["n_post"]} days  |  '
          f'MC samples: {result["n_mc_samples"]:,}')
    print(f'  Avg observed IOR     : {result["avg_actual_ior"]:.5f}')
    print(f'  Avg counterfactual   : {result["avg_counterfactual_ior"]:.5f}')
    print(f'  Avg causal effect    : {result["avg_effect_pp"]:+.4f}pp  '
          f'[{result["avg_effect_ci_lo_pp"]:+.3f}, {result["avg_effect_ci_hi_pp"]:+.3f}]')
    print(f'  Relative effect      : {result["relative_effect_pct"]:+.2f}%')
    print(f'  Cumulative effect    : {result["cumulative_effect_pp"]:+.4f}pp  '
          f'[{result["cumulative_ci_lo_pp"]:+.3f}, {result["cumulative_ci_hi_pp"]:+.3f}]')
    print(f'  P(effect > 0)        : {prob:.4f}')
    print(f'  Posterior p-value    : {p_b:.5f}  {sig}')

    result['_alpha'] = alpha
    _plot_forecast_counterfactual(result)
    _causal_narrative(llm, {k: v for k, v in result.items()
                              if not isinstance(v, list) and k != '_alpha'},
        f'Causal Impact. {result["model_type"]}. '
        f'Avg effect {result["avg_effect_pp"]:+.3f}pp '
        f'(relative {result["relative_effect_pct"]:+.2f}%, '
        f'{"sig" if result["significant"] else "n.s."}, '
        f'posterior p={p_b:.4f}, P(>0)={prob:.3f}). '
        f'Cumulative effect {result["cumulative_effect_pp"]:+.3f}pp '
        f'[{result["cumulative_ci_lo_pp"]:+.2f}, {result["cumulative_ci_hi_pp"]:+.2f}]. '
        f'Interpret: (1) full causal effect interpretation; '
        f'(2) what P(effect>0)={prob:.3f} means in business terms; '
        f'(3) confidence in the causal claim; '
        f'(4) ship recommendation.'
    )
    return result


# ── Shared plot for forecasting-based counterfactual methods ─────────────────

def _plot_forecast_counterfactual(result: dict):
    """
    Standard 3-panel visualisation shared by ARIMA, SARIMA, BSTS, and Causal Impact.

    Panel 1 — Full time series: pre-period actual + post-period actual vs
               counterfactual with shaded CI band and intervention line.
    Panel 2 — Pointwise daily effect (pp) as a bar chart.
    Panel 3 — Cumulative effect (pp) as an area chart.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    COLORS_L = {
        'treatment': '#f97316', 'control': '#4e9af1',
        'highlight': '#facc15', 'neutral': '#a1a1aa',
        'positive':  '#22c55e', 'negative': '#ef4444',
    }

    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    fig.patch.set_facecolor('#0f0f0f')
    for ax in axes:
        ax.set_facecolor('#1a1a2e')

    pre_dates  = result.get('pre_dates',  [])
    post_dates = result.get('post_dates', [])
    pre_act    = result.get('pre_actual', [])
    post_act   = result.get('post_actual',[])
    post_cf    = result.get('post_cf',    [])
    cf_lo      = result.get('post_cf_lo', post_cf)
    cf_hi      = result.get('post_cf_hi', post_cf)
    pw_pp      = result.get('pointwise_pp',  [])
    cum_pp     = result.get('cumulative_pp', [])

    n_pre  = len(pre_dates)
    n_post = len(post_dates)
    pre_x  = list(range(n_pre))
    post_x = list(range(n_pre, n_pre + n_post))
    sig    = result.get('significant', False)
    avg_e  = result.get('avg_effect_pp', 0.0)
    p_key  = ('p_value_posterior' if 'p_value_posterior' in result
               else 'p_value')
    p_val  = result.get(p_key, 1.0)
    alpha  = result.get('_alpha', 0.05)

    # ── Panel 1: time series ─────────────────────────────────────────────────
    ax1 = axes[0]
    ax1.plot(pre_x,  pre_act,  color=COLORS_L['treatment'], lw=1.8, label='Actual (pre)')
    ax1.plot(post_x, post_act, color=COLORS_L['treatment'], lw=2.5, label='Actual (post)')
    ax1.plot(post_x, post_cf,  color=COLORS_L['control'], lw=2.2,
             linestyle='--', label='Counterfactual')
    ax1.fill_between(post_x, cf_lo, cf_hi,
                     alpha=0.20, color=COLORS_L['control'],
                     label=f'{int((1-alpha)*100)}% CI')
    ax1.axvline(n_pre - 0.5, color=COLORS_L['highlight'], lw=2, label='Intervention')
    gap_color = COLORS_L['positive'] if avg_e >= 0 else COLORS_L['negative']
    ax1.fill_between(post_x,
                     [min(cf_lo[i], post_act[i]) for i in range(n_post)],
                     [max(cf_hi[i], post_act[i]) for i in range(n_post)],
                     alpha=0.10, color=gap_color)
    # Tick labels (sample every ~8th point to avoid crowding)
    all_dates = pre_dates + post_dates
    step      = max(1, len(all_dates) // 8)
    tick_idx  = list(range(0, len(all_dates), step))
    ax1.set_xticks(tick_idx)
    ax1.set_xticklabels([all_dates[i] for i in tick_idx], rotation=40, fontsize=7)
    sig_icon = '✅' if sig else '⚠️ n.s.'
    ax1.set_title(
        f'{result["method"]}\n'
        f'Avg effect: {avg_e:+.3f}pp  {sig_icon}  p={p_val:.4f}',
        color=COLORS_L['highlight'], fontsize=9
    )
    ax1.set_ylabel('IOR'); ax1.legend(fontsize=7.5); ax1.grid(True, alpha=0.2)

    # ── Panel 2: pointwise effect ────────────────────────────────────────────
    ax2 = axes[1]
    if pw_pp:
        pw_arr    = np.array(pw_pp)
        bar_cols  = [COLORS_L['positive'] if v >= 0 else COLORS_L['negative']
                     for v in pw_arr]
        ax2.bar(range(n_post), pw_arr, color=bar_cols, alpha=0.85)
        ax2.axhline(0,     color='white',               lw=1.5, linestyle='--', alpha=0.6)
        ax2.axhline(avg_e, color=COLORS_L['highlight'], lw=2,
                    label=f'Mean = {avg_e:+.3f}pp')
    ax2.set_xlabel('Days after intervention')
    ax2.set_ylabel('Effect (pp)')
    ax2.set_title('Pointwise Effect\n(Observed − Counterfactual)',
                  color=COLORS_L['highlight'], fontsize=9)
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.2)

    # ── Panel 3: cumulative effect ───────────────────────────────────────────
    ax3 = axes[2]
    if cum_pp:
        cum_arr   = np.array(cum_pp)
        cum_color = COLORS_L['positive'] if cum_arr[-1] >= 0 else COLORS_L['negative']
        ax3.plot(range(n_post), cum_arr, color=cum_color, lw=2.5,
                 label=f'Final: {cum_arr[-1]:+.2f}pp')
        ax3.fill_between(range(n_post), 0, cum_arr, alpha=0.20, color=cum_color)
        ax3.axhline(0, color='white', lw=1.5, linestyle='--', alpha=0.6)
    cum_lo = result.get('cumulative_ci_lo_pp')
    cum_hi = result.get('cumulative_ci_hi_pp')
    if cum_lo is not None and cum_pp:
        ax3.fill_between([n_post - 1], [cum_lo], [cum_hi],
                         color=COLORS_L['highlight'], alpha=0.5,
                         label=f'CI [{cum_lo:+.1f}, {cum_hi:+.1f}]')
    ax3.set_xlabel('Days after intervention')
    ax3.set_ylabel('Cumulative effect (pp)')
    ax3.set_title(
        f'Cumulative Effect\nTotal: {result.get("cumulative_effect_pp", 0):+.3f}pp',
        color=COLORS_L['highlight'], fontsize=9
    )
    ax3.legend(fontsize=8); ax3.grid(True, alpha=0.2)

    plt.suptitle(f'{result["method"]} — Counterfactual Analysis',
                 fontsize=12, color=COLORS_L['highlight'], fontweight='bold')
    plt.tight_layout()
    slug  = (result['method'].lower()
             .replace(' ', '_').replace('(', '').replace(')', '')
             .replace('/', '_').replace('+', ''))
    fname = f'{slug}_analysis.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print(f'  📁 Chart saved → {fname}')
    return fname


print('✅ Forecasting-based counterfactual methods loaded:')
print('   _run_arima()              → raw ARIMA computation')
print('   run_arima_analysis()      → [24] interactive ARIMA runner')
print('   _run_sarima()             → raw SARIMA computation')
print('   run_sarima_analysis()     → [25] interactive SARIMA runner')
print('   _run_bsts()               → raw BSTS Kalman-filter computation')
print('   run_bsts_analysis()       → [26] interactive BSTS runner')
print('   _run_causal_impact()      → raw Causal Impact computation')
print('   run_causal_impact_analysis() → [27] interactive Causal Impact runner')
print('   _plot_forecast_counterfactual() → shared 3-panel plot')


✅ Causal Analysis runners loaded:
   run_causal_analysis()              → [10] Method-selection menu
   run_ab_test_analysis()             → [17] A/B Test (Statsig/feature-flag data)
   run_pre_post_analysis()            → [18] Pre-Post Analysis
   run_did_analysis()                 → [19] DiD (Enhanced + TWFE)
   run_its_analysis()                 → [20] Interrupted Time Series
   run_psm_analysis()                 → [21] Propensity Score Matching
   run_rdd_analysis()                 → [22] Regression Discontinuity
   run_synthetic_control_analysis()   → [23] Synthetic Control (Enhanced)
✅ Module 11: Causal Analysis Engine loaded
   Methods: A/B Test · Pre-Post · DiD (Enhanced+TWFE) · ITS · Synthetic Control (Enhanced) · PSM
   Standalone runners: run_did_analysis(llm) · run_synthetic_control_analysis(llm)
   Use Module 4 (Experiment Brief) to get a recommended method first.
✅ Forecasting-based counterfactual methods loaded:
   _run_arima()              → raw ARIMA computation
   run

## 13 · Phase 4 — Deploy Modules

After the ship decision is made, Phase 4 answers the operational question:
**given what we learned, who should we target and at what budget to maximise GMV?**

- **Module [15] Uplift Modeller** — trains a T-learner on the concluded experiment data to estimate individual-level causal effects (who benefits most from the treatment).
- **Module [16] Decision Engine** — takes uplift scores + a budget constraint and solves for the optimal targeting allocation to maximise incremental GMV.

In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4 — DEPLOY MODULES
# ─────────────────────────────────────────────────────────────────────────────

# ── Module 15: Uplift Modeller ────────────────────────────────────────────────

def _build_uplift_features(exp_df: 'pd.DataFrame') -> 'pd.DataFrame':
    """
    Build the feature matrix for the uplift model.
    Uses the covariates available in the experiment dataframe.
    """
    # One-hot encode categorical columns
    cat_cols = ['account_segment', 'platform', 'price_tier', 'process_group']
    num_cols = ['order_value']

    feat = exp_df[['buyer_id','variant','converted_to_order']].copy()

    # Categorical features
    for feat_col in cat_cols:
        if feat_col in exp_df.columns:
            dummies = pd.get_dummies(exp_df[feat_col], prefix=feat_col, drop_first=False)
            feat = pd.concat([feat, dummies], axis=1)

    # Numeric features
    for feat_col in num_cols:
        if feat_col in exp_df.columns:
            feat[feat_col] = exp_df[feat_col].fillna(0)

    # Derived: activity proxy
    feat['has_order'] = (feat.get('order_value', pd.Series(0, index=feat.index)) > 0).astype(int)

    return feat


def _train_t_learner(features: 'pd.DataFrame', control_label: str = 'control'):
    """
    T-Learner: fit two separate models (one on control, one on treatment),
    then estimate per-user uplift as: pred_treatment(x) - pred_control(x).

    Uses LogisticRegression for binary outcome (converted_to_order).
    Returns (model_ctrl, model_trt, feature_cols).
    """
    try:
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
    except ImportError:
        raise ImportError('scikit-learn is required for the Uplift Modeller. '
                          'Run: pip install scikit-learn')

    outcome_col = 'converted_to_order'
    skip_cols   = {'buyer_id', 'variant', outcome_col}
    feat_cols   = [c for c in features.columns if c not in skip_cols]

    ctrl = features[features['variant'] == control_label]
    trt  = features[features['variant'] != control_label]

    if len(ctrl) < 100 or len(trt) < 100:
        raise ValueError(f'Insufficient data: {len(ctrl)} control, {len(trt)} treatment rows.')

    X_ctrl = ctrl[feat_cols].values.astype(float)
    y_ctrl = ctrl[outcome_col].astype(int).values
    X_trt  = trt[feat_cols].values.astype(float)
    y_trt  = trt[outcome_col].astype(int).values

    # Fit separate models
    model_ctrl = LogisticRegression(max_iter=300, random_state=42, C=1.0)
    model_trt  = LogisticRegression(max_iter=300, random_state=42, C=1.0)
    model_ctrl.fit(X_ctrl, y_ctrl)
    model_trt.fit(X_trt,  y_trt)

    return model_ctrl, model_trt, feat_cols


def _compute_uplift_scores(features: 'pd.DataFrame',
                            model_ctrl, model_trt, feat_cols: list) -> 'pd.Series':
    """Compute per-user uplift: P(convert | treatment) - P(convert | control)."""
    X = features[feat_cols].values.astype(float)
    uplift = (model_trt.predict_proba(X)[:, 1]
              - model_ctrl.predict_proba(X)[:, 1])
    return pd.Series(uplift, index=features.index, name='uplift_score')


def _compute_qini(features: 'pd.DataFrame', uplift_scores: 'pd.Series',
                   control_label: str = 'control') -> dict:
    """
    Compute the Qini coefficient — the standard uplift model quality metric.
    Qini measures how much better the model is than random targeting.
    Returns {'qini': float, 'random_baseline': float, 'interpretation': str}.
    """
    outcome = 'converted_to_order'
    df = features[['variant', outcome]].copy()
    df['uplift'] = uplift_scores.values
    df = df.sort_values('uplift', ascending=False).reset_index(drop=True)

    n = len(df)
    n_trt  = (df['variant'] != control_label).sum()
    n_ctrl = (df['variant'] == control_label).sum()

    # Incremental gains curve
    gains = []
    cum_trt_conv, cum_ctrl_conv = 0, 0
    cum_trt_n,   cum_ctrl_n   = 0, 0

    for _, row in df.iterrows():
        is_trt = row['variant'] != control_label
        if is_trt:
            cum_trt_n    += 1
            cum_trt_conv += int(row[outcome])
        else:
            cum_ctrl_n    += 1
            cum_ctrl_conv += int(row[outcome])

        if cum_trt_n > 0 and cum_ctrl_n > 0:
            incr = (cum_trt_conv / cum_trt_n) - (cum_ctrl_conv / cum_ctrl_n)
            gains.append(incr * (cum_trt_n + cum_ctrl_n) / n)
        else:
            gains.append(0)

    try:
        qini = float(np.trapezoid(gains)) / (n / 2) if n > 0 else 0.0
    except AttributeError:  # numpy < 2.0
        qini = float(np.trapz(gains)) / (n / 2) if n > 0 else 0.0
    interp = 'Strong' if qini > 0.1 else 'Moderate' if qini > 0.03 else 'Weak'

    return {
        'qini':            round(qini, 4),
        'interpretation':  interp,
        'gains_curve':     gains,
    }


def run_uplift_modeller(llm):
    """
    Module 15 — Uplift Modeller.

    Trains a T-learner on a concluded experiment to estimate per-user
    individual causal effects (uplift scores). Produces:
      - Uplift score distribution by segment
      - Qini coefficient (model quality)
      - Per-segment average uplift
      - Scores registered in DuckDB for Module 16 to consume
    """
    print()
    print('╔' + '═'*70 + '╗')
    print('║' + '  🎯  UPLIFT MODELLER — Module 15'.ljust(70) + '║')
    print('║' + '  Phase 4 · Deploy · Individual causal effect estimation'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')
    print()
    print('  The Uplift Modeller answers: for which INDIVIDUAL users does this')
    print('  treatment have the highest incremental effect?')
    print()
    print('  Method: T-Learner — train two separate models (control / treatment)')
    print('  and subtract predicted conversion rates to get per-user uplift.')
    print()

    # ── Select experiment ─────────────────────────────────────────────────────
    exp_info, _ = _list_experiments_with_status()
    if exp_info is None:
        return None

    exp_name = exp_info['name']
    status   = exp_info.get('status', '')
    if status not in ('concluded', 'shipped', 'stopped'):
        print(f'  ⚠️  Uplift modelling requires a concluded experiment.')
        print(f'     Selected experiment has status: {status}')
        raw = input('  Continue anyway? [y/N]: ').strip().lower()
        if raw != 'y':
            return None

    print(f'\n  ✅ Selected: {exp_name}')

    # ── Pull data ─────────────────────────────────────────────────────────────
    exp_df = df_all_experiments[df_all_experiments['experiment_name'] == exp_name].copy()
    exp_df = dedup_dataframe(exp_df)
    variants = sorted(exp_df['variant'].unique().tolist())
    control  = 'control' if 'control' in variants else variants[0]
    treatments = [v for v in variants if v != control]

    print(f'  Rows: {len(exp_df):,}  Variants: {variants}  Control: "{control}"')

    if len(exp_df) < 200:
        print('  ⚠️  Fewer than 200 rows — uplift model will be unreliable.')

    # ── Build features and train T-Learner ───────────────────────────────────
    print('\n  Building feature matrix...')
    features = _build_uplift_features(exp_df)

    print('  Training T-Learner (control model + treatment model)...')
    try:
        model_ctrl, model_trt, feat_cols = _train_t_learner(features, control)
    except (ImportError, ValueError) as e:
        print(f'  ❌ {e}')
        return None

    # ── Compute uplift scores ─────────────────────────────────────────────────
    print('  Computing per-user uplift scores...')
    uplift_scores = _compute_uplift_scores(features, model_ctrl, model_trt, feat_cols)
    features['uplift_score'] = uplift_scores

    # ── Quality metric: Qini ─────────────────────────────────────────────────
    qini_result = _compute_qini(features, uplift_scores, control)
    print(f'\n  Qini coefficient: {qini_result["qini"]:.4f} ({qini_result["interpretation"]})')

    # ── Segment-level uplift distribution ────────────────────────────────────
    print('\n  ── Uplift score distribution by segment ──')
    print(f'  {"Segment":<20} {"Mean uplift":>12} {"p75":>8} {"p25":>8} {"% positive":>12}')
    print('  ' + '─'*62)
    seg_uplift = {}
    if 'account_segment' in exp_df.columns:
        for seg in sorted(exp_df['account_segment'].unique()):
            mask    = exp_df['account_segment'] == seg
            scores  = uplift_scores[mask]
            mean_u  = float(scores.mean())
            p75     = float(scores.quantile(0.75))
            p25     = float(scores.quantile(0.25))
            pct_pos = float((scores > 0).mean() * 100)
            seg_uplift[seg] = {'mean': mean_u, 'p75': p75, 'p25': p25, 'pct_positive': pct_pos}
            icon = '📈' if mean_u > 0.005 else ('📉' if mean_u < -0.005 else '➡️ ')
            print(f'  {icon} {seg:<18} {mean_u*100:>+10.3f}pp  {p75*100:>+6.3f}pp  '
                  f'{p25*100:>+6.3f}pp  {pct_pos:>10.1f}%')

    # ── Register in DuckDB for Decision Engine ───────────────────────────────
    uplift_df = exp_df[['buyer_id']].copy()
    uplift_df['uplift_score']    = uplift_scores.values
    uplift_df['experiment_name'] = exp_name
    if 'account_segment' in exp_df.columns:
        uplift_df['account_segment'] = exp_df['account_segment'].values
    db.register('uplift_scores', uplift_df)
    print(f'\n  ✅ Uplift scores registered in DuckDB as "uplift_scores"')
    print(f'     Run Module [16] (Decision Engine) to generate an optimised targeting plan.')

    # ── LLM synthesis ─────────────────────────────────────────────────────────
    print('\n  🤖 Synthesising uplift findings...')
    seg_summary = '; '.join(
        f'{seg}: mean={v["mean"]*100:+.2f}pp ({v["pct_positive"]:.0f}% positive)'
        for seg, v in seg_uplift.items()
    ) or 'No segment breakdown available.'

    past = _query_relevant_learnings(exp_info.get('description',''), n=2)
    past_text = _format_past_learnings(past)

    uplift_prompt = textwrap.dedent(f"""
You are interpreting uplift model results from a concluded A/B experiment.

Experiment: {exp_name}
Qini coefficient: {qini_result["qini"]:.4f} ({qini_result["interpretation"]} model fit)
Segment-level uplift: {seg_summary}

Relevant past experiments:
{past_text}

In 4-5 sentences:
1. What the uplift distribution tells us about WHICH users respond best.
2. Whether the Qini score suggests the model is trustworthy enough to act on.
3. What targeting strategy follows from these scores.
4. Whether past experiments support or challenge this uplift pattern.
Do not use emojis.
    """).strip()

    try:
        synthesis = llm.ask(uplift_prompt)
        try:
            synthesis = _strip_decorative_chars(synthesis)
        except NameError:
            pass
    except Exception as e:
        synthesis = f'(Synthesis unavailable: {e})'

    print()
    for line in synthesis.split('\n'):
        if line.strip(): print(f'    {line}')

    return {
        'experiment':   exp_name,
        'qini':         qini_result['qini'],
        'qini_grade':   qini_result['interpretation'],
        'seg_uplift':   seg_uplift,
        'uplift_df':    uplift_df,
        'synthesis':    synthesis,
        'model_ctrl':   model_ctrl,
        'model_trt':    model_trt,
        'feat_cols':    feat_cols,
    }


# ── Module 16: Decision Engine ────────────────────────────────────────────────

def run_decision_engine(llm):
    """
    Module 16 — Decision Engine.

    Takes uplift scores from Module 15 + a budget constraint and solves
    for the optimal targeting allocation to maximise incremental GMV.

    Algorithm:
      1. For each user, compute expected incremental GMV = uplift_score × avg_AOV
      2. Sort users by incremental GMV per unit cost (descending)
      3. Greedily allocate budget from highest to lowest
      4. Report allocation by segment, projected incremental GMV, and ROI

    Constraint: scipy.optimize.linprog (already a dependency).
    """
    print()
    print('╔' + '═'*70 + '╗')
    print('║' + '  💰  DECISION ENGINE — Module 16'.ljust(70) + '║')
    print('║' + '  Phase 4 · Deploy · Budget-constrained targeting optimisation'.ljust(70) + '║')
    print('╚' + '═'*70 + '╝')
    print()
    print('  Given uplift scores from Module 15, the Decision Engine answers:')
    print('  "Which users should we target, with which variant, at this budget,')
    print('   to maximise incremental GMV?"')
    print()

    # ── Check uplift scores are available ─────────────────────────────────────
    try:
        uplift_df = db.execute('SELECT * FROM uplift_scores').df()
        exp_name  = uplift_df['experiment_name'].iloc[0] if len(uplift_df) else 'unknown'
    except Exception:
        print('  ❌ No uplift scores found.')
        print('     Run Module [15] (Uplift Modeller) first to generate scores.')
        return None

    print(f'  Loaded {len(uplift_df):,} uplift scores for experiment: {exp_name}')

    # ── Budget input ──────────────────────────────────────────────────────────
    print()
    print('  Budget parameters:')
    while True:
        raw_budget = input('  ❓ Total targeting budget ($, e.g. 50000): ').strip().replace(',','')
        try:
            budget = float(raw_budget)
            if budget > 0:
                break
        except ValueError:
            pass
        print('     ⚠️  Enter a positive number')

    while True:
        raw_cost = input('  ❓ Cost per contact ($ per user, e.g. 0.80): ').strip()
        try:
            cost_per_contact = float(raw_cost)
            if cost_per_contact > 0:
                break
        except ValueError:
            pass
        print('     ⚠️  Enter a positive number')

    max_contacts = int(budget / cost_per_contact)
    print(f'\n  Budget: ${budget:,.0f}  Cost/contact: ${cost_per_contact:.2f}  '
          f'Max contacts: {max_contacts:,}')

    # ── Compute expected incremental GMV per user ─────────────────────────────
    # Use overall avg AOV from hist_inquiries as the revenue multiplier
    try:
        avg_aov = float(db.execute(
            "SELECT AVG(order_value) FROM hist_inquiries WHERE converted_to_order = TRUE"
        ).fetchone()[0] or 4000)
    except Exception:
        avg_aov = 4000.0

    uplift_df['expected_incr_gmv'] = uplift_df['uplift_score'] * avg_aov

    # ── Greedy optimisation: sort by incremental GMV per $ cost ───────────────
    # Uplift score < 0 means treatment hurts this user — exclude them
    eligible = uplift_df[uplift_df['uplift_score'] > 0.0].copy()
    eligible = eligible.sort_values('expected_incr_gmv', ascending=False).reset_index(drop=True)

    # Allocate
    allocated = eligible.head(max_contacts).copy()
    not_allocated = eligible.iloc[max_contacts:].copy()
    harmed = uplift_df[uplift_df['uplift_score'] <= 0.0].copy()

    total_contacts     = len(allocated)
    total_cost         = total_contacts * cost_per_contact
    projected_incr_gmv = float(allocated['expected_incr_gmv'].sum())
    proj_roi           = projected_incr_gmv / total_cost if total_cost > 0 else 0

    # ── Print allocation summary ───────────────────────────────────────────────
    print()
    print('  ── Targeting Allocation ──────────────────────────────────────────────')
    print(f'  {"Group":<30} {"Users":>8} {"Avg uplift":>12} {"Proj. GMV":>14}')
    print('  ' + '─'*70)

    seg_alloc = {}
    if 'account_segment' in allocated.columns:
        for seg in sorted(allocated['account_segment'].dropna().unique()):
            seg_rows = allocated[allocated['account_segment'] == seg]
            seg_gmv  = float(seg_rows['expected_incr_gmv'].sum())
            seg_u    = float(seg_rows['uplift_score'].mean())
            seg_alloc[seg] = {'n': len(seg_rows), 'avg_uplift': seg_u, 'projected_gmv': seg_gmv}
            icon = '🎯' if seg_u > 0.01 else '📌'
            print(f'  {icon} TREAT  {seg:<24} {len(seg_rows):>8,} {seg_u*100:>+10.3f}pp  '
                  f'${seg_gmv:>12,.0f}')
        if 'account_segment' in harmed.columns:
            for seg in sorted(harmed['account_segment'].dropna().unique()):
                n_harm = (harmed['account_segment'] == seg).sum()
                if n_harm > 0:
                    print(f'  🚫 HOLD   {seg:<24} {n_harm:>8,} {"(negative uplift)":>24}')
    else:
        print(f'  🎯 TREAT  (all eligible)               {total_contacts:>8,} '
              f'{float(allocated["uplift_score"].mean())*100:>+10.3f}pp  '
              f'${projected_incr_gmv:>12,.0f}')

    print('  ' + '─'*70)
    print(f'  {"TOTAL":30} {total_contacts:>8,} {"":>12} ${projected_incr_gmv:>12,.0f}')
    print()
    print(f'  Budget used     : ${total_cost:>10,.0f} of ${budget:,.0f}')
    print(f'  Remaining budget: ${budget - total_cost:>10,.0f}')
    print(f'  Projected ROI   : {proj_roi:.1f}× (${projected_incr_gmv:,.0f} GMV / ${total_cost:,.0f} spend)')
    print(f'  Users held back : {len(harmed):,} (negative expected uplift — do not treat)')

    # ── LLM: decision implications + trade-offs ───────────────────────────────
    print('\n  🤖 Generating deployment plan with implications and trade-offs...')

    past = _query_relevant_learnings(exp_name, n=2)
    past_text = _format_past_learnings(past)

    seg_summary = '; '.join(
        f'{seg}: {v["n"]:,} users avg={v["avg_uplift"]*100:+.2f}pp GMV=${v["projected_gmv"]:,.0f}'
        for seg, v in seg_alloc.items()
    ) if seg_alloc else f'{total_contacts:,} users targeted'

    deploy_prompt = textwrap.dedent(f"""
You are a senior product analyst writing a deployment plan.

Experiment: {exp_name}
Budget: ${budget:,.0f}  Cost/contact: ${cost_per_contact:.2f}  Max contacts: {max_contacts:,}
Targeting allocation: {seg_summary}
Projected incremental GMV: ${projected_incr_gmv:,.0f}
Projected ROI: {proj_roi:.1f}x
Users with negative uplift (held back): {len(harmed):,}

Relevant past experiments:
{past_text}

Write a response with EXACTLY THREE sections:

DEPLOYMENT PLAN:
Step-by-step: who to target, in what sequence, over what timeframe (weeks).
Name specific segments and variants.

IMPLICATIONS:
What to monitor in the first 2 weeks post-deployment.
What success looks like. What a failure signal looks like.
Reference past experiments if relevant.

TRADE-OFFS:
What we are giving up by holding back the negative-uplift segments.
Whether the ROI projection is conservative or aggressive and why.
One risk the team should hedge against.

Write in plain business English. No emojis.
    """).strip()

    try:
        deploy_plan = llm.ask(deploy_prompt)
        try:
            deploy_plan = _strip_decorative_chars(deploy_plan)
        except NameError:
            pass
    except Exception as e:
        deploy_plan = f'(Plan unavailable: {e})'

    print()
    for line in deploy_plan.split('\n'):
        stripped = line.strip()
        if stripped:
            if stripped.upper() in ('DEPLOYMENT PLAN:', 'IMPLICATIONS:', 'TRADE-OFFS:'):
                print(f'\n  ── {stripped} ──')
            else:
                print(f'    {line}')

    # ── Save targeting brief ──────────────────────────────────────────────────
    fname = f'targeting_brief_{exp_name}.csv'
    allocated[['buyer_id','uplift_score','expected_incr_gmv']
              + (['account_segment'] if 'account_segment' in allocated.columns else [])
              ].to_csv(fname, index=False)
    print(f'\n  📁 Targeting brief saved → {fname}')
    print(f'     {total_contacts:,} users to contact. Engineering: use buyer_id column '
          f'to configure feature flag targeting.')

    return {
        'experiment':          exp_name,
        'budget':              budget,
        'cost_per_contact':    cost_per_contact,
        'n_targeted':          total_contacts,
        'projected_incr_gmv':  projected_incr_gmv,
        'projected_roi':       proj_roi,
        'seg_allocation':      seg_alloc,
        'n_held_back':         len(harmed),
        'deploy_plan':         deploy_plan,
        'targeting_file':      fname,
    }


## 12 · Dispatcher

The interactive menu. Run this cell to start a session. Choose a module number, a phase shortcut (P1 / P2 / P3), or `0` for the full end-to-end journey.

In [24]:
def run_agent():
    results = {}

    def cleanup_llm():
        try:
            if 'narrative_llm' in globals():
                print("\n🧹 Freeing LLM from memory...")
                narrative_llm.unload()
                print("✔ LLM unloaded.")
        except Exception as e:
            print(f"⚠️ Failed to unload LLM: {e}")

    PHASES = {
        "1": ("🛠️ Foundation (Data + Monitoring)", {
            "1": ("Schema Discovery & Mapping", run_schema_discovery),
            "2": ("Pipeline Health Monitor", run_pipeline_health),
            "3": ("Watchtower", run_watchtower),
        }),

        "2": ("📋 Planning (Experiment Design)", {
            "1": ("Experiment Brief + Method", run_brief_generator),
            "2": ("Opportunity Sizing", run_opportunity_sizing),
            "3": ("Power Calculator", run_power_calculator),
            "4": ("KPI & Tracking Plan", run_metrics_and_tracking),
            "5": ("Audience Selection", run_audience_selection),
        }),

        "3": ("🔴 Live (Monitoring)", {
            "1": ("Health Monitor", run_health_monitor),
            "2": ("Sequential Testing", run_sequential_testing),
        }),

        "4": ("✅ Post-Experiment (Analysis)", {
            "1": ("Causal Analysis", run_causal_analysis),
            "2": ("Simpson's Paradox Detector", run_simpsons_paradox_detector),
            "3": ("ROI Tracker", run_roi_tracker),
            "4": ("Learnings Repository", run_learnings_repository),
        }),

        "5": ("🚀 Deploy (Action Layer)", {
            "1": ("Uplift Modeller", run_uplift_modeller),
            "2": ("Decision Engine", run_decision_engine),
        }),

        "6": ("📐 Advanced Causal Methods", {
            "1": ("A/B Test (Statsig)", run_ab_test_analysis),
            "2": ("Pre-Post Analysis", run_pre_post_analysis),
            "3": ("Diff-in-Differences (TWFE)", run_did_analysis),
            "4": ("Interrupted Time Series", run_its_analysis),
            "5": ("Propensity Score Matching", run_psm_analysis),
            "6": ("Regression Discontinuity", run_rdd_analysis),
            "7": ("Synthetic Control (Enhanced)", run_synthetic_control_analysis),
        }),

        "7": ("📈 Counterfactual Forecasting", {
            "1": ("ARIMA Counterfactual", run_arima_analysis),
            "2": ("SARIMA Counterfactual", run_sarima_analysis),
            "3": ("BSTS Counterfactual", run_bsts_analysis),
            "4": ("Causal Impact Framework", run_causal_impact_analysis),
        }),
    }

    _mode = globals().get('CONTINUM_STATE', {}).get('mode', 'synthetic')
    _use_synth = globals().get('USE_SYNTHETIC_DATA', True)

    if not _use_synth and _mode != 'production_ready':
        print("\n🔌 PRODUCTION MODE DETECTED — BOOTSTRAP REQUIRED\n")
        print("Run bootstrap_from_connection(narrative_llm) first.\n")

        raw = input("Run bootstrap now? [Y/n]: ").strip().lower()
        if raw != 'n':
            bootstrap_from_connection(narrative_llm)

        cleanup_llm()
        return {}

    _required_helpers = (
        'ask_for_template',
        'build_llm_prompt_from_template',
        'parse_sections_from_llm_output',
        'render_document_pdf'
    )

    _missing = [h for h in _required_helpers if h not in globals()]
    if _missing:
        print("⚠️ Missing document helpers:", ", ".join(_missing))
        print("Re-run Cell 7b before PDF modules.\n")

    while True:

        print("\n" + "═"*60)
        print("🔬 CONTINUM PERSISTIQ — GUIDED MODE")
        print("═"*60)

        print("\nSelect a phase:\n")

        for k, (name, _) in PHASES.items():
            print(f"[{k}] {name}")

        print("[0] 🚀 Full Journey (All phases)")
        print("[x] Exit")

        phase = input("\n👉 Enter choice: ").strip().lower()

        if phase == "x":
            cleanup_llm()
            break

        results = {}

        if phase == "0":
            for pk, (pname, modules) in PHASES.items():
                print(f"\n\n━━━ {pname} ━━━\n")

                for mk, (mname, fn) in modules.items():
                    print(f"\n▶ Running: {mname}\n" + "-"*40)
                    results[mname] = fn(narrative_llm)

            cleanup_llm()
            break

        if phase not in PHASES:
            print("⚠️ Invalid phase selected.")
            continue

        phase_name, modules = PHASES[phase]

        while True:

            print("\n" + "─"*60)
            print(f"{phase_name}")
            print("─"*60)

            for k, (name, _) in modules.items():
                print(f"[{k}] {name}")

            print("[b] ⬅ Back to phases")
            print("[x] Exit")

            choice = input("\n👉 Select module: ").strip().lower()

            if choice == "b":
                break

            if choice == "x":
                cleanup_llm()
                return results

            if choice not in modules:
                print("⚠️ Invalid module.")
                continue

            name, fn = modules[choice]

            print("\n▶ Running:", name)
            print("─"*50)

            results[name] = fn(narrative_llm)

            print("\n✔ Completed:", name)

            next_step = input(
                "\nRun another module in this phase? [y/n]: "
            ).strip().lower()

            if next_step != "y":
                break

    print("\n✔ Session complete.")

    if results:
        print("Modules run:", ", ".join(results.keys()))

    return results


import os as _os

if _os.environ.get('CONTINUM_AUTORUN', '').lower() == 'true':
    results = run_agent()
else:
    print("✅ Dispatcher ready (guided mode).")
    print("Call run_agent() to start.")
    print("Or set CONTINUM_AUTORUN=true to auto-run.")

✅ Dispatcher ready (guided mode).
Call run_agent() to start.
Or set CONTINUM_AUTORUN=true to auto-run.


In [27]:
run_agent()


════════════════════════════════════════════════════════════
🔬 CONTINUM PERSISTIQ — GUIDED MODE
════════════════════════════════════════════════════════════

Select a phase:

[1] 🛠️ Foundation (Data + Monitoring)
[2] 📋 Planning (Experiment Design)
[3] 🔴 Live (Monitoring)
[4] ✅ Post-Experiment (Analysis)
[5] 🚀 Deploy (Action Layer)
[6] 📐 Advanced Causal Methods
[7] 📈 Counterfactual Forecasting
[0] 🚀 Full Journey (All phases)
[x] Exit


━━━ 🛠️ Foundation (Data + Monitoring) ━━━


▶ Running: Schema Discovery & Mapping
----------------------------------------

╔══════════════════════════════════════════════════════════════════════╗
║  🔍  SCHEMA DISCOVERY & MAPPING (Phase 0 — Foundation)                ║
║  Auto-generate a CLIENT_SCHEMA from a connected warehouse            ║
╚══════════════════════════════════════════════════════════════════════╝

  Available sources:
    [1] Synthetic data (DuckDB tables registered in this session)
    [2] Snowflake / Postgres / external warehouse (advan

RuntimeError: Failed to load Qwen/Qwen2.5-1.5B-Instruct. Last error: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [ ]:
import difflib

def run_agent():

    results = {}


    MODULE_INDEX = {
        "schema discovery": run_schema_discovery,
        "pipeline health": run_pipeline_health,
        "watchtower": run_watchtower,

        "experiment brief": run_brief_generator,
        "opportunity sizing": run_opportunity_sizing,
        "power calculator": run_power_calculator,
        "kpi tracking": run_metrics_and_tracking,
        "audience selection": run_audience_selection,

        "health monitor": run_health_monitor,
        "sequential testing": run_sequential_testing,

        "causal analysis": run_causal_analysis,
        "simpson paradox": run_simpsons_paradox_detector,
        "roi tracker": run_roi_tracker,
        "learnings": run_learnings_repository,

        "uplift model": run_uplift_modeller,
        "decision engine": run_decision_engine,

        "ab test": run_ab_test_analysis,
        "pre post": run_pre_post_analysis,
        "diff in diff": run_did_analysis,
        "interrupted time series": run_its_analysis,
        "psm": run_psm_analysis,
        "rdd": run_rdd_analysis,
        "synthetic control": run_synthetic_control_analysis,

        "arima": run_arima_analysis,
        "sarima": run_sarima_analysis,
        "bsts": run_bsts_analysis,
        "causal impact": run_causal_impact_analysis,
    }

    HELP_TEXT = """
🔬 CONTINUM PERSISTIQ — SEARCH MODE

Type what you want to do:

Examples:
  • power calculator
  • roi analysis
  • synthetic control
  • experiment brief
  • causal impact

Commands:
  help  → show this message
  list  → show all modules
  exit  → quit
"""

    print(HELP_TEXT)

    while True:

        query = input("\n🔎 Search module: ").strip().lower()

        if query == "exit":
            break

        if query == "help":
            print(HELP_TEXT)
            continue

        if query == "list":
            print("\n📦 Available modules:\n")
            for k in sorted(MODULE_INDEX.keys()):
                print(" •", k)
            continue

        matches = difflib.get_close_matches(
            query,
            MODULE_INDEX.keys(),
            n=3,
            cutoff=0.4
        )


        if not matches:
            print("\n⚠️ No match found.")
            print("Try: 'power', 'roi', 'causal', 'synthetic control'")
            continue


        if len(matches) > 1:
            print("\n🤔 Multiple matches found:\n")
            for i, m in enumerate(matches, 1):
                print(f"[{i}] {m}")

            choice = input("\nSelect number or refine search: ").strip()

            if choice.isdigit() and 1 <= int(choice) <= len(matches):
                selected = matches[int(choice) - 1]
            else:
                continue
        else:
            selected = matches[0]

        print("\n▶ Running:", selected)
        print("─" * 50)

        fn = MODULE_INDEX[selected]
        results[selected] = fn(narrative_llm)

        print("\n✔ Completed:", selected)


        nxt = input("\nRun another? [y/n]: ").strip().lower()
        if nxt != "y":
            break

    print("\n✔ Session complete.")

    if results:
        print("\nModules executed:")
        for k in results:
            print(" •", k)

    free = input("\nFree LLM memory? [y/N]: ").strip().lower()
    if free == "y":
        narrative_llm.unload()

    return results



import os as _os

if _os.environ.get('CONTINUM_AUTORUN', '').lower() == 'true':
    results = run_agent()
else:
    print("✅ Search CLI ready.")
    print("Call run_agent() to start.")
    print("Or set CONTINUM_AUTORUN=true to auto-run.")

✅ Search CLI ready.
Call run_agent() to start.
Or set CONTINUM_AUTORUN=true to auto-run.


In [ ]:
run_agent()


🔬 CONTINUM PERSISTIQ — SEARCH MODE

Type what you want to do:

Examples:
  • power calculator
  • roi analysis
  • synthetic control
  • experiment brief
  • causal impact

Commands:
  help  → show this message
  list  → show all modules
  exit  → quit




🔎 Search module:  exit



✔ Session complete.



Free LLM memory? [y/N]:  y


{}